# PyTorch from First Principles to Production
### A self-study notebook for undergraduates who know the theory and are ready to write the code

*Companion to "Neural Networks & Deep Learning: A Self-Study Tutorial for Engineering Undergraduates" (sections 1–12). Written for **PyTorch 2.14 / torchvision 0.29** (current stable releases as of October 2026) and tested against them. It runs top to bottom on a free Google Colab T4 GPU.*

---

You have already done the hard intellectual work. You derived backpropagation by hand, you know why He initialization keeps the signal alive, you know what AdamW fixes about Adam, and you know why BatchNorm behaves differently in `train` and `eval` mode. What you have **not** done yet is turn that knowledge into working code that trains fast, fails loudly, and can be handed to a colleague or shipped to production.

That is what this notebook is for. By the end you will have:

- written every piece of a training pipeline yourself, from raw tensors to a reusable `fit()` function;
- trained an MLP and a CNN on Fashion-MNIST, a ResNet on CIFAR-10, and fine-tuned an ImageNet model on a 1,020-image flowers dataset;
- used mixed precision, `torch.compile`, and the profiler, and measured what each one actually buys you;
- debugged deliberately broken training code, the way you will in your first job;
- exported a model with `torch.export` and ONNX and served it from a standalone inference script;
- turned notebook code into a tested, configurable training script.

## Contents

| Part | Chapter | What you will be able to do |
| --- | --- | --- |
| **0. Orientation** | 0. How to use this notebook | Set up Colab and the notebook conventions |
| **I. The foundations** | 1. Tensors | Create, reshape, broadcast and move tensors without shape bugs |
| | 2. Autograd | Let PyTorch do the backprop of Tutorial §6, and know exactly what it is doing |
| | 3. Linear regression four ways | See exactly what each PyTorch abstraction replaces |
| **II. Building and training models** | 4. `nn.Module` in depth | Write, inspect, initialise and instrument models |
| | 5. Loss functions and the logits rule | Pick the right loss and target format every time |
| | 6. Data: `Dataset`, `DataLoader`, transforms | Build correct, fast input pipelines |
| | 7. The training loop and your first project | Train and evaluate an MLP on Fashion-MNIST |
| | 8. Optimizers, schedules and regularisation in code | Turn Tutorial §7, §10 and §12 into working code |
| **III. Engineering discipline** | 9. Checkpointing and reproducibility | Save, resume and reproduce runs exactly |
| | 10. Debugging training | Find silent bugs systematically |
| | 11. Performance | Use AMP, `torch.compile`, profiling and memory tricks |
| **IV. Convolutional networks** | 12. Convolution in PyTorch | Understand and implement convolution, pooling and receptive fields |
| | 13. Residual networks and the CIFAR-10 project | Build and train a ResNet to the low 90s |
| | 14. Transfer learning | Fine-tune a pretrained model on a small dataset |
| **V. Production** | 15. Inference, export and deployment | Ship a model with `torch.export`, ONNX and a CLI |
| | 16. From notebook to codebase | Write tested, configurable training code |
| **VI. Review** | 17. Interview questions, error decoder, cheat sheet | Consolidate and self-assess |

**Time budget.** About 30–40 hours of study. GPU time is roughly 1 hour for the main path, and up to about 1.5–2 hours if you also run every exercise solution. These are estimates: the notebook logs your actual epoch times and results (§17.6).

## 0. How to use this notebook

### 0.1 Setting up Colab
1. Open the notebook in Google Colab.
2. **Runtime → Change runtime type → T4 GPU → Save.**
3. Run the setup cell below. It should print `device: cuda` and the GPU name. If it says `cpu`, step 2 did not take effect.

Colab gives you a fresh machine each session, so downloaded datasets and saved files disappear when the runtime is recycled. That is fine for this notebook, because everything re-downloads automatically. If you want to keep checkpoints, mount Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) and point `ARTIFACT_DIR` at a folder in it.

### 0.2 Conventions
- **Cross-references.** "Tutorial §6.3" means section 6.3 of the deep learning tutorial you studied. "Chapter 4" means a chapter of *this* notebook.
- **Exercises** (🧪) are followed by an empty cell for your attempt, then a **collapsed solution cell**. In Colab the solution shows as a titled cell with its code hidden; double-click it, or use the "Show code" option in its ⋮ menu. In JupyterLab, click the collapsed-input bar. Solution cells are runnable, and nothing later in the notebook depends on them.
- **Check yourself** boxes end each chapter. Answers are collapsed. Commit to an answer before you expand it.
- **⚠️ Pitfall** marks a mistake that real engineers make in real code. Several of them produce *no error at all*, which is exactly why they are dangerous.
- **🏭 Industry note** marks practices that matter more at work than in a course.

- **Self-checks** (🔎) follow some exercises. They test the function you wrote, by name, and print ✅ or ❌ without showing the solution.
- **Numbers in the text are estimates.** Accuracies and speed-ups quoted in the prose are typical values, not guarantees. The notebook records **your** measured values in a `RESULTS` dictionary and prints them as a table in §17.6. When the text and your run disagree, your run is the ground truth (Tutorial §0's rule).

### 0.3 Notebook hygiene, restarts and resume points
A notebook has hidden state: whatever cells you ran, in whatever order. Three rules keep you out of trouble:

1. **Run cells top to bottom.** Several functions (`train_one_epoch`, `evaluate`, `fit`) are deliberately *re-defined* as they gain features. The later definition replaces the earlier one, and running an old cell after a new one silently brings back the old version. When in doubt, use *Runtime → Restart session and run all*.
2. **Names like `m`, `o`, `x` and `h` are scratch variables** that cells reuse. Never rely on one surviving from a distant cell.
3. **Colab recycles idle runtimes.** You lose all variables and the local files, including checkpoints. To continue without re-running everything, run the **setup cell** below, then the **⏯️ Resume point** cell at the start of the chapter you want. Resume cells exist before Chapters 12, 13, 14 and 15. They re-create every function and class defined earlier, and the data that chapter needs. Running one when nothing was lost is harmless. Set `USE_DRIVE = True` to keep checkpoints in Google Drive, so long training runs survive a reset.

### 0.4 How to study this
Type the code instead of only running it. Then break it on purpose (change a shape, delete `zero_grad()`, swap `eval()` for `train()`) and **predict what will happen before you run it**. A predicted error teaches you more than a working script.

### 0.5 The `FAST_DEV_RUN` switch
The setup cell defines `FAST_DEV_RUN = False`. Setting it to `True` makes every training cell use a tiny slice of data and a single epoch, so the whole notebook runs in a few minutes. This is a real industry habit (PyTorch Lightning even has a flag with this exact name): **smoke-test the full pipeline end to end before you spend GPU hours on it.** Accuracy numbers are meaningless in this mode. Leave it `False` for real study.

In [ ]:
# ---- Setup: run this cell first -------------------------------------------
import os, sys, math, time, random, copy, json, platform
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

FAST_DEV_RUN = False          # True = smoke-test mode (tiny data, 1 epoch)
USE_DRIVE = False             # True = keep checkpoints/exports in Google Drive, so they survive runtime resets
DATA_DIR = Path("./data")             # datasets are downloaded here (re-downloading is cheap)
ARTIFACT_DIR = Path("./artifacts")    # checkpoints / exported models go here
if USE_DRIVE:
    from google.colab import drive    # only available on Colab
    drive.mount("/content/drive")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/pytorch_course/artifacts")
DATA_DIR.mkdir(parents=True, exist_ok=True); ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
import subprocess

# Pick the best available device: NVIDIA GPU, then Apple GPU, then CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def set_seed(seed: int = 0) -> None:
    # Seed every RNG that affects a typical PyTorch run (Chapter 9 explains this).
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)       # also seeds all CUDA devices

def n_epochs(full: int) -> int:
    # Number of epochs to train: `full` normally, 1 in smoke-test mode.
    return 1 if FAST_DEV_RUN else full

def maybe_subset(ds, n: int = 512):
    # Return the whole dataset normally, its first n items in smoke-test mode.
    if FAST_DEV_RUN:
        return torch.utils.data.Subset(ds, range(min(n, len(ds))))
    return ds

NUM_WORKERS = min(2, os.cpu_count() or 1)  # Colab has 2 CPU cores

# ---- Bookkeeping: your measured results (printed in §17.6) ---------------------
RESULTS = {}
def record(key, value):
    # Store a measured number (accuracy, seconds, ...) so you can compare it with the text's estimates.
    RESULTS[key] = value
    return value

# ---- Self-checks for exercises ---------------------------------------------------
def self_check(name, test):
    # Run `test(fn)` on the function you defined as `name`; report pass/fail without revealing the solution.
    fn = globals().get(name)
    if fn is None:
        print(f"ℹ️  Define `{name}` in the cell above, then run this cell again.")
        return
    try:
        test(fn)
        print(f"✅ `{name}` passed all checks.")
    except AssertionError as e:
        print(f"❌ `{name}` failed a check: {e}")
    except Exception as e:
        print(f"❌ `{name}` raised {type(e).__name__}: {e}")

# ---- A tiny box-and-arrow diagram helper (used for the figures in this notebook) ---
def flow_diagram(nodes, edges, title="", size=(10, 3)):
    # nodes: {key: (x, y, text, colour)}; edges: [(from, to, label, colour)] with x, y in [0, 1]
    fig, ax = plt.subplots(figsize=size); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    for k, (x, y, text, col) in nodes.items():
        ax.text(x, y, text, ha="center", va="center", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.4", fc=col, ec="0.3"))
    for a, b, label, col in edges:
        (x0, y0), (x1, y1) = nodes[a][:2], nodes[b][:2]
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle="->", color=col, lw=1.4, shrinkA=18, shrinkB=18))
        if label:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2 + 0.06, label, ha="center", fontsize=8, color=col)
    ax.set_title(title, fontsize=10); plt.tight_layout(); plt.show()

print(f"python      {platform.python_version()}")
print(f"torch       {torch.__version__}")
print(f"torchvision {torchvision.__version__}")
print(f"device:     {device}")
if device.type == "cuda":
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
set_seed(0)

> **Version note.** PyTorch currently ships a new minor release roughly every two months. Everything in this notebook uses stable, documented APIs that have been in place since at least PyTorch 2.6, so small version differences on Colab will not matter. Where a behaviour changed recently, the notebook says so. When a release note and this notebook disagree, trust the release note and your own run.

---
# Part I — The foundations

# Chapter 1. Tensors

A **tensor** is PyTorch's n-dimensional array. If you know NumPy, you already know about 80% of the tensor API, because the names, slicing and broadcasting rules are almost identical. A tensor adds three things a NumPy array does not have:

1. **A device.** It can live in GPU memory, and operations on it then run on the GPU.
2. **Autograd.** It can record the operations applied to it, so gradients can be computed automatically (Chapter 2).
3. **Deep-learning dtypes.** Low-precision types such as `float16` and `bfloat16`, which are central to fast training (Chapter 11).

This chapter is about writing tensor code that is correct on the first try. In practice that mostly means getting **shapes** right. Recall the advice from Tutorial §0: roughly half of all real deep learning bugs are shape bugs.

🎯 **In this chapter you will learn to:** create tensors and control their dtype and device; reshape, index and broadcast without shape bugs; time GPU code correctly.

## 1.1 Creating tensors

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])      # from a (nested) Python list
print(a)
print("shape:", a.shape, "| dtype:", a.dtype, "| device:", a.device, "| requires_grad:", a.requires_grad)

# Factory functions -- the same ideas as NumPy
print(torch.zeros(2, 3))
print(torch.ones(2, 3, dtype=torch.int64))
print(torch.full((2, 2), 7.0))
print(torch.arange(0, 10, 2))                # like range(): end is exclusive
print(torch.linspace(0, 1, 5))               # 5 evenly spaced points, end inclusive
print(torch.eye(3))
print(torch.rand(2, 3))                      # uniform on [0, 1)
print(torch.randn(2, 3))                     # standard normal N(0, 1)
print(torch.randint(0, 10, (3,)))            # integers in [0, 10)
print(torch.zeros_like(a), torch.randn_like(a), sep="\n")  # same shape/dtype/device as a

### dtypes: the first place PyTorch differs from NumPy

| You write | NumPy gives | PyTorch gives |
| --- | --- | --- |
| `[1.0, 2.0]` | `float64` | **`float32`** |
| `[1, 2]` | `int64` | `int64` |
| `[True, False]` | `bool` | `bool` |

PyTorch defaults to `float32` because that is what GPUs are fast at, and deep learning does not need 64-bit precision. Model parameters are `float32` by default too. The trouble starts when **NumPy data** (usually `float64`) meets a **`float32` model**:

In [ ]:
x_np = np.random.rand(4, 3)               # float64, as NumPy always does
layer = nn.Linear(3, 2)                   # parameters are float32

try:
    layer(torch.from_numpy(x_np))
except RuntimeError as e:
    print("RuntimeError:", e)

# The fix: convert once, at the boundary between NumPy and PyTorch.
x = torch.from_numpy(x_np).float()        # or .to(torch.float32)
print(layer(x).shape)

### Copy or share? `torch.tensor` vs `torch.from_numpy`

- `torch.tensor(data)` **always copies**.
- `torch.from_numpy(arr)` and `tensor.numpy()` **share memory** on the CPU. Changing one changes the other.

Sharing is fast, which is good, but it can surprise you:

In [ ]:
arr = np.zeros(3)
t_shared = torch.from_numpy(arr)
t_copy = torch.tensor(arr)
arr[0] = 42.0
print("shared:", t_shared)   # sees the change
print("copy:  ", t_copy)     # does not

## 1.2 Shapes: `reshape`, `view`, `permute`, `squeeze`, `unsqueeze`

| Operation | What it does | Typical use |
| --- | --- | --- |
| `x.reshape(a, b)` / `x.view(a, b)` | Same data, new shape (element count unchanged). `-1` means "infer this one" | Flatten images for an MLP |
| `x.flatten(start_dim=1)` | Flatten every dimension from `start_dim` onwards | `(N, C, H, W) → (N, C·H·W)` |
| `x.permute(…)` / `x.transpose(i, j)` | Reorder dimensions | `(H, W, C) → (C, H, W)` for images |
| `x.unsqueeze(d)` / `x[:, None]` | Insert a size-1 dimension | Make a single example into a batch of 1 |
| `x.squeeze(d)` | Remove a size-1 dimension | `(N, 1) → (N,)` |

In [ ]:
x = torch.arange(24)
print(x.reshape(2, 3, 4).shape, x.reshape(4, -1).shape)

imgs = torch.randn(8, 1, 28, 28)              # (N, C, H, W): PyTorch's image layout
print("flattened:", imgs.flatten(start_dim=1).shape)   # (8, 784) -- one row per image

hwc = torch.randn(32, 32, 3)                  # how PIL/NumPy store an image
chw = hwc.permute(2, 0, 1)                    # PyTorch wants channels first
print("HWC -> CHW:", hwc.shape, "->", chw.shape)

single = torch.randn(3, 32, 32)
print("batch of one:", single.unsqueeze(0).shape)     # models always expect a batch dim
print("squeeze:", torch.randn(5, 1).squeeze(1).shape)

**`view` vs `reshape`.** A tensor is a block of memory plus **strides**, which say how many elements to skip to move one step along each dimension. `transpose` and `permute` do not move any data; they only change the strides. The result is "non-contiguous": walking it in row-major order no longer walks memory in order. `view` refuses to work on such tensors. `reshape` works on anything, copying the data only if it has to.

In [ ]:
m = torch.arange(6).reshape(2, 3)
mt = m.T                                  # transpose: a new view with swapped strides
print("strides:", m.stride(), "->", mt.stride(), "| contiguous?", mt.is_contiguous())
try:
    mt.view(6)
except RuntimeError as e:
    print("view failed:", str(e)[:90], "...")
print("reshape works:", mt.reshape(6))
print("or: ", mt.contiguous().view(6))

## 1.3 Indexing
Indexing follows NumPy exactly: slices, integer arrays, boolean masks, and `None` to insert dimensions.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print(x)
print("row 1:          ", x[1])
print("column 2:       ", x[:, 2])
print("last two cols:\n", x[:, -2:])
print("rows 0 and 2:\n", x[[0, 2]])
print("elements > 5:   ", x[x > 5])            # boolean mask -> 1-D result
x2 = x.clone()                                 # clone() = an independent copy
x2[x2 % 2 == 0] = -1                           # masked assignment
print(x2)

# A pattern you will use constantly: pick one entry per row,
# e.g. the predicted probability of the *correct* class.
probs = torch.tensor([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1]])
labels = torch.tensor([0, 1])
print("p(correct class):", probs[torch.arange(2), labels])      # tensor([0.7, 0.8])
print("same with gather:", probs.gather(1, labels[:, None]).squeeze(1))

## 1.4 Broadcasting, and the most common silent bug in regression

Broadcasting follows the NumPy rules. Line up the shapes **from the right**. Two dimensions are compatible if they are equal or one of them is 1, and a missing dimension counts as 1. This is how `b` is added to every row in `Z = X Wᵀ + b` (Tutorial §4.2).

In [ ]:
X = torch.randn(5, 3)        # 5 examples, 3 features
b = torch.randn(3)           # one bias per feature
print((X + b).shape)         # (5, 3) + (3,) -> (5, 3): b is added to every row

# Standardise each feature (column) -- reductions with keepdim=True keep shapes broadcastable
Xs = (X - X.mean(dim=0, keepdim=True)) / X.std(dim=0, keepdim=True)
print(Xs.mean(dim=0), Xs.std(dim=0))   # ~0 and ~1 per column

> ⚠️ **Pitfall: the `(N,)` vs `(N, 1)` bug.** A regression model outputs shape `(N, 1)`. If your targets have shape `(N,)`, then `pred - y` broadcasts to **`(N, N)`**. Every prediction is compared with every target. The loss is a perfectly finite number, training "runs", and the model learns to predict the mean. No error is raised when you write the loss by hand.

In [ ]:
N = 4
pred = torch.randn(N, 1)       # model output
y = torch.randn(N)             # targets, as they often come out of a CSV
print("shape of pred - y:", (pred - y).shape)     # (4, 4), not (4,) or (4, 1)!

wrong = ((pred - y) ** 2).mean()                  # silently wrong
right = ((pred.squeeze(1) - y) ** 2).mean()       # make the shapes match explicitly
print(f"wrong: {wrong:.3f}   right: {right:.3f}")

# PyTorch's own loss at least warns you (look for the UserWarning below):
_ = F.mse_loss(pred, y)

The defence is a habit: **assert shapes at boundaries.** For example, write `assert pred.shape == y.shape` before computing a loss by hand. It costs nothing and catches this whole class of bugs.

## 1.5 Reductions and the meaning of `dim`
A good rule of thumb: **the dimension you reduce over disappears** (unless you pass `keepdim=True`).

In [ ]:
x = torch.tensor([[1.0, 5.0, 3.0],
                  [4.0, 2.0, 6.0]])           # shape (2, 3)
print("sum over everything:", x.sum())
print("sum over dim 0 (down the rows -> one per column):", x.sum(dim=0))   # shape (3,)
print("mean over dim 1 (across columns -> one per row): ", x.mean(dim=1))  # shape (2,)
vals, idx = x.max(dim=1)                        # max along a dim returns values AND indices
print("max per row:", vals, "at", idx)
print("argmax per row:", x.argmax(dim=1))       # exactly how we turn logits into predictions
print("keepdim:", x.sum(dim=1, keepdim=True).shape)

## 1.6 Matrix multiplication and `einsum`

- `A @ B` is matrix multiplication. With 3-D or higher inputs it does a **batched** matmul over the leading dimensions, which broadcast.
- `*` is **elementwise** (the Hadamard product ⊙ of Tutorial §6).
- `torch.einsum` spells out the index pattern explicitly. It is excellent for readability when shapes get complicated.

This also confirms what Tutorial §4.2 told you about `nn.Linear`: it stores `W` with shape `(out, in)` and computes `x @ W.T + b`.

In [ ]:
lin = nn.Linear(3, 2)
x = torch.randn(5, 3)
manual = x @ lin.weight.T + lin.bias
print("W shape:", lin.weight.shape, "| matches nn.Linear:", torch.allclose(lin(x), manual))

A = torch.randn(10, 4, 5)    # a batch of 10 matrices, each 4x5
B = torch.randn(10, 5, 6)
print("batched matmul:", (A @ B).shape)
print("einsum agrees: ", torch.allclose(A @ B, torch.einsum("bij,bjk->bik", A, B), atol=1e-6))

## 1.7 Low-precision dtypes, in brief

| dtype | Bits | Range (max) | Precision | Used for |
| --- | --- | --- | --- | --- |
| `float32` | 32 | ~3.4e38 | ~7 decimal digits | Default for parameters and training |
| `float16` | 16 | **65,504** | ~3 digits | Mixed precision on older GPUs such as the T4 (Chapter 11) |
| `bfloat16` | 16 | ~3.4e38 | ~2 digits | Mixed precision on Ampere-or-newer GPUs and TPUs |
| `int64` | 64 | — | exact | Class labels and indices |

`float16` has a tiny range, so large values overflow to `inf` and very small gradients underflow to 0. That is why fp16 training needs a *gradient scaler* (Chapter 11). `bfloat16` keeps the full float32 range at lower precision, so it does not need one.

In [ ]:
print(torch.finfo(torch.float16))
print(torch.tensor(70000.0).half())                     # overflows to inf
print(torch.tensor(70000.0).to(torch.bfloat16))         # fine, just less precise
print(torch.tensor(1e-8).half())                        # underflows to 0
print("int / int:", torch.tensor(7) / torch.tensor(2), "| int // int:", torch.tensor(7) // torch.tensor(2))

## 1.8 Devices: CPU and GPU

Every tensor lives on one device. **All tensors in an operation must be on the same device**, or you get the error you will see most often in your first month of PyTorch: *"Expected all tensors to be on the same device"*.

Moving data between CPU and GPU is slow compared with compute, so the rule is to **move data to the GPU once per batch and keep it there**. Bring results back with `.item()` (for a scalar) or `.cpu()` (for a tensor) only when you actually need them on the host.

In [ ]:
x_cpu = torch.randn(3, 3)
x_dev = x_cpu.to(device)                 # returns a copy on `device` (a no-op if already there)
print(x_dev.device)

if device.type != "cpu":
    try:
        x_cpu + x_dev
    except RuntimeError as e:
        print("RuntimeError:", str(e)[:100], "...")

loss = (x_dev ** 2).mean()
print("python float:", loss.item())               # .item(): 1-element tensor -> Python number
print("numpy array:\n", x_dev.cpu().numpy())      # GPU tensors must come to the CPU before .numpy()

### Timing GPU code correctly

GPU kernels run **asynchronously**. Python queues the work and moves on immediately. If you time GPU code without `torch.cuda.synchronize()`, you measure how long it took to *queue* the work, not to *do* it. Also, the first CUDA call in a process pays one-off initialisation costs, so always do a warm-up run before timing.

In [ ]:
def time_matmul(dev, n=2048, reps=10):
    a = torch.randn(n, n, device=dev)
    b = torch.randn(n, n, device=dev)
    a @ b                                        # warm-up
    if dev.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps):
        a @ b
    if dev.type == "cuda": torch.cuda.synchronize()   # wait for the GPU to actually finish
    return (time.perf_counter() - t0) / reps

n = 512 if FAST_DEV_RUN else 2048
t_cpu = time_matmul(torch.device("cpu"), n)
print(f"CPU: {t_cpu*1e3:7.2f} ms per {n}x{n} matmul")
if device.type == "cuda":
    t_gpu = time_matmul(device, n)
    print(f"GPU: {t_gpu*1e3:7.2f} ms  (~{t_cpu/t_gpu:.0f}x faster)")

The GPU wins massively on large, parallel work. On **tiny** tensors it can lose, because every kernel launch costs a few microseconds of overhead. That is one reason batching matters (Tutorial §7.6).

## 1.9 In-place operations
Methods ending in an underscore (`add_`, `mul_`, `zero_`, `clamp_`) modify a tensor **in place**, and so do `x += 1` and `x[idx] = v`. They save memory, but they can corrupt values that autograd saved for the backward pass (§2.6 shows the error). Treat in-place ops as an optimisation for code you understand well. Do not use them by default.

In [ ]:
x = torch.ones(3)
y = x.add(1)       # new tensor; x unchanged
x.add_(1)          # x itself is modified
print(x, y)

## 🧪 Exercises

**Exercise 1.1: pairwise distances without loops.** `★★☆ · ~20 min` Given `A` with shape `(n, d)` and `B` with shape `(m, d)`, compute the `(n, m)` matrix `D` with `D[i, j] = ‖A[i] − B[j]‖²`. Use broadcasting or the identity `‖a−b‖² = ‖a‖² + ‖b‖² − 2a·b`, with no Python loops. Check your answer against `torch.cdist(A, B) ** 2`. This is the core of k-nearest-neighbours, k-means and attention-style computations. Write it as a function `pairwise_sq_dists(A, B)`. The 🔎 self-check after your attempt tests it.

In [ ]:
# ✍️ Your attempt here.
# Hint: A[:, None, :] has shape (n, 1, d) and B[None, :, :] has shape (1, m, d).

In [ ]:
# 🔎 Self-check for Exercise 1.1: tests YOUR `pairwise_sq_dists` (run after your attempt)
def _test(fn):
    A, B = torch.randn(7, 4), torch.randn(5, 4)
    D = fn(A, B)
    assert D.shape == (7, 5), f'shape {tuple(D.shape)}, expected (7, 5)'
    assert torch.allclose(D, torch.cdist(A, B) ** 2, atol=1e-4), 'values differ from torch.cdist(A, B)**2'

self_check("pairwise_sq_dists", _test)

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def pairwise_sq_dists(A, B):
    # Method 1 (broadcasting): (n,1,d) - (1,m,d) -> (n,m,d), then sum over d.
    # Clear, but uses O(n*m*d) memory.
    return ((A[:, None, :] - B[None, :, :]) ** 2).sum(dim=-1)

def pairwise_sq_dists_fast(A, B):
    # Method 2 (expansion): O(n*m) memory, and uses a fast matmul.
    a2 = (A ** 2).sum(dim=1, keepdim=True)        # (n, 1)
    b2 = (B ** 2).sum(dim=1)[None, :]             # (1, m)
    return (a2 + b2 - 2 * A @ B.T).clamp_min(0)   # clamp: rounding can give tiny negatives

A, B = torch.randn(5, 3), torch.randn(4, 3)
ref = torch.cdist(A, B) ** 2
print(torch.allclose(pairwise_sq_dists(A, B), ref, atol=1e-5),
      torch.allclose(pairwise_sq_dists_fast(A, B), ref, atol=1e-5))

**Exercise 1.2: a numerically stable softmax.** `★☆☆ · ~10 min` Implement `softmax(z, dim)` using `exp` and `sum`. First write the naive version, then the stable version that subtracts the maximum along `dim` (Tutorial §5.6). Show that the naive version returns `nan` for `z = torch.tensor([[1000., 1001., 1002.]])` and the stable one matches `torch.softmax`. Name the stable version `softmax_stable(z, dim=-1)`. The 🔎 self-check after your attempt tests it.

In [ ]:
# ✍️ Your attempt here.
# Hint: softmax(z) = softmax(z - c) for any constant c. Pick c = max(z) along dim, with keepdim=True.

In [ ]:
# 🔎 Self-check for Exercise 1.2: tests YOUR `softmax_stable` (run after your attempt)
def _test(fn):
    z = torch.tensor([[1000., 1001., 1002.], [-5., 0., 5.]])
    out = fn(z, dim=-1)
    assert torch.isfinite(out).all(), 'non-finite values: did you subtract the max?'
    assert torch.allclose(out, torch.softmax(z, -1), atol=1e-6), 'differs from torch.softmax'
    r = torch.randn(4, 6)
    assert torch.allclose(fn(r, dim=0), torch.softmax(r, 0), atol=1e-6), 'wrong result for dim=0'

self_check("softmax_stable", _test)

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def softmax_naive(z, dim=-1):
    e = torch.exp(z)
    return e / e.sum(dim=dim, keepdim=True)

def softmax_stable(z, dim=-1):
    z = z - z.amax(dim=dim, keepdim=True)   # largest entry becomes 0, so exp() cannot overflow
    e = torch.exp(z)
    return e / e.sum(dim=dim, keepdim=True)

z = torch.tensor([[1000., 1001., 1002.]])
print("naive: ", softmax_naive(z))           # exp(1000) = inf -> inf/inf = nan
print("stable:", softmax_stable(z))
print("torch: ", torch.softmax(z, dim=-1))

**Exercise 1.3: preparing an image batch.** `★☆☆ · ~10 min` You receive `raw` with shape `(N, H, W, 3)` and dtype `uint8` (values 0–255), which is how images come out of most image libraries. Produce a `float32` tensor with shape `(N, 3, H, W)`, scaled to `[0, 1]` and then normalised per channel with `mean = (0.5, 0.4, 0.3)` and `std = (0.2, 0.2, 0.2)`. Do not use any loops. Verify the shape, dtype, and that channel 0 of the result equals `(raw[..., 0] / 255 - 0.5) / 0.2`. Write it as a function `prepare_batch(raw, mean, std)` where `mean` and `std` are sequences of 3 floats. The 🔎 self-check after your attempt tests it.

In [ ]:
# ✍️ Your attempt here.
# Hint: permute first, then .float() / 255, then reshape mean and std to (1, 3, 1, 1) so they broadcast.

In [ ]:
# 🔎 Self-check for Exercise 1.3: tests YOUR `prepare_batch` (run after your attempt)
def _test(fn):
    raw = torch.randint(0, 256, (2, 6, 6, 3), dtype=torch.uint8)
    out = fn(raw, (0.5, 0.4, 0.3), (0.2, 0.2, 0.2))
    assert out.shape == (2, 3, 6, 6), f'shape {tuple(out.shape)}'
    assert out.dtype == torch.float32, f'dtype {out.dtype}'
    exp = (raw.permute(0, 3, 1, 2).float() / 255 - torch.tensor([0.5, 0.4, 0.3]).view(1, 3, 1, 1)) / 0.2
    assert torch.allclose(out, exp, atol=1e-5), 'values differ'

self_check("prepare_batch", _test)

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def prepare_batch(raw, mean, std):
    mean = torch.as_tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)   # broadcast over N, H, W
    std = torch.as_tensor(std, dtype=torch.float32).view(1, 3, 1, 1)
    x = raw.permute(0, 3, 1, 2).float() / 255.0     # (N, 3, H, W) in [0, 1]
    return (x - mean) / std

raw = torch.randint(0, 256, (4, 8, 8, 3), dtype=torch.uint8)
x = prepare_batch(raw, (0.5, 0.4, 0.3), (0.2, 0.2, 0.2))
print(x.shape, x.dtype)
print(torch.allclose(x[:, 0], (raw[..., 0].float() / 255 - 0.5) / 0.2))

### 📌 Key takeaways

- Default float dtype is `float32`; convert NumPy's `float64` at the boundary.
- Broadcasting aligns shapes from the right: `(N,1)` vs `(N,)` silently becomes `(N,N)`.
- GPU work is asynchronous: synchronise before timing.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** `torch.tensor([1, 2, 3]) / 2` — what dtype is the result? What about `torch.tensor([1, 2, 3]) // 2`?

<details>
<summary>Answer</summary>

`/` is true division and returns `float32`: `tensor([0.5, 1.0, 1.5])`. `//` is floor division and keeps `int64`: `tensor([0, 1, 1])`.

</details>

**2.** You run `pred - y` with `pred.shape == (64, 1)` and `y.shape == (64,)`. What shape is the result, and why is that dangerous?

<details>
<summary>Answer</summary>

`(64, 64)`. Broadcasting aligns from the right, so `(64, 1)` and `(1, 64)` expand to `(64, 64)`. Every prediction is compared with every target. A hand-written loss silently computes the wrong thing and the model learns to predict roughly the mean.

</details>

**3.** Why does `x.T.view(-1)` fail while `x.T.reshape(-1)` works?

<details>
<summary>Answer</summary>

Transposing changes only the strides, so `x.T` is non-contiguous. `view` must reinterpret existing memory without copying, which is impossible here. `reshape` falls back to copying the data into a new contiguous tensor.

</details>

**4.** You time `a @ b` on the GPU without `torch.cuda.synchronize()` and get 0.02 ms for a huge matrix. What did you actually measure?

<details>
<summary>Answer</summary>

Only the time to *launch* (queue) the kernel. CUDA runs asynchronously, so the multiply was still running when the timer stopped. Synchronise before reading the clock, and do a warm-up run first.

</details>

---
# Chapter 2. Autograd: the backpropagation you derived, done for you

In Tutorial §6 you derived backpropagation by hand: a forward pass that stores intermediate values, then a backward pass that multiplies local derivatives along the graph. **Autograd is exactly that algorithm, implemented once, for every operation PyTorch has.** When you compute with tensors that have `requires_grad=True`, PyTorch records a graph of the operations. Calling `.backward()` on a scalar walks that graph in reverse and accumulates `∂L/∂(leaf)` into each leaf tensor's `.grad`.

## 2.1 Your hand calculation from Tutorial §6.3, checked by autograd

Recall the tiny network: one input, one sigmoid hidden unit, one sigmoid output, and squared error. The parameters were `w₁ = 0.5, b₁ = 0.1, w₂ = 0.8, b₂ = −0.3`, with `x = 2.0` and `y = 1.0`. By hand you obtained

`∂L/∂w₂ = −0.0780, ∂L/∂b₂ = −0.1040, ∂L/∂w₁ = −0.0312, ∂L/∂b₁ = −0.0156`.

🎯 **In this chapter you will learn to:** explain what autograd records and how `backward()` uses it; avoid gradient accumulation, double-backward and in-place pitfalls; verify gradients and write a custom `autograd.Function`.

In [ ]:
# Leaf tensors we want gradients for: requires_grad=True
w1 = torch.tensor(0.5, requires_grad=True)
b1 = torch.tensor(0.1, requires_grad=True)
w2 = torch.tensor(0.8, requires_grad=True)
b2 = torch.tensor(-0.3, requires_grad=True)
x, y = torch.tensor(2.0), torch.tensor(1.0)      # data: no gradients needed

# Forward pass -- exactly the lines of Tutorial §6.3
z1 = w1 * x + b1
a1 = torch.sigmoid(z1)
z2 = w2 * a1 + b2
a2 = torch.sigmoid(z2)
L = 0.5 * (a2 - y) ** 2
print(f"z1={z1.item():.4f}  a1={a1.item():.4f}  z2={z2.item():.4f}  a2={a2.item():.4f}  L={L.item():.4f}")

L.backward()          # backward pass: fills .grad of every leaf with requires_grad=True

for name, p in [("w2", w2), ("b2", b2), ("w1", w1), ("b1", b1)]:
    print(f"dL/d{name} = {p.grad.item():+.4f}")

Here is the graph autograd recorded for that computation. Forward arrows (black) are the operations. During `backward()`, each node multiplies the incoming gradient by its **local derivative** (red, as in Tutorial §6.3) and passes the result on.

In [ ]:
flow_diagram(
    {"x": (0.04, 0.75, "x = 2", "#eeeeee"), "w1": (0.04, 0.45, "w₁ (leaf)", "#cfe8ff"), "b1": (0.04, 0.15, "b₁ (leaf)", "#cfe8ff"),
     "z1": (0.24, 0.45, "z₁ = w₁x + b₁", "#fff2cc"), "a1": (0.42, 0.45, "a₁ = σ(z₁)", "#fff2cc"),
     "w2": (0.42, 0.85, "w₂ (leaf)", "#cfe8ff"), "b2": (0.42, 0.08, "b₂ (leaf)", "#cfe8ff"),
     "z2": (0.60, 0.45, "z₂ = w₂a₁ + b₂", "#fff2cc"), "a2": (0.78, 0.45, "a₂ = σ(z₂)", "#fff2cc"),
     "L": (0.95, 0.45, "L = ½(a₂ − y)²", "#f8cecc")},
    [("x", "z1", "", "k"), ("w1", "z1", "", "k"), ("b1", "z1", "", "k"), ("z1", "a1", "", "k"),
     ("a1", "z2", "", "k"), ("w2", "z2", "", "k"), ("b2", "z2", "", "k"), ("z2", "a2", "", "k"), ("a2", "L", "", "k"),
     ("L", "a2", "a₂ − y", "tab:red"), ("a2", "z2", "a₂(1 − a₂)", "tab:red"), ("z2", "a1", "w₂", "tab:red"),
     ("a1", "z1", "a₁(1 − a₁)", "tab:red")],
    title="The recorded graph (black: forward) and the local derivatives used by backward() (red)", size=(12, 3.6))

The numbers agree with your hand calculation to every printed digit. From now on you will rarely compute a gradient by hand, but you will **always need to know what autograd is computing**, because that is how you debug it.

## 2.2 The graph, leaves and `grad_fn`

- A **leaf** is a tensor you created directly, such as parameters and inputs. Gradients are stored only for leaves with `requires_grad=True`.
- Every non-leaf tensor produced by an operation has a `grad_fn`, a node that knows how to compute that operation's local backward. These nodes link to one another, and together they form the graph.
- PyTorch's graph is **dynamic** ("define-by-run"). It is rebuilt on every forward pass, so ordinary Python `if` statements and loops work inside models. The graph is simply whatever actually ran.

In [ ]:
print("L.grad_fn:      ", L.grad_fn)                     # last op: multiplication by 0.5
print("its inputs:     ", L.grad_fn.next_functions)       # links back to PowBackward, ...
print("is w1 a leaf?   ", w1.is_leaf, "| is a1 a leaf?", a1.is_leaf)
print("a1.grad:        ", a1.grad)   # None: non-leaf gradients are freed (call a1.retain_grad() before backward to keep it)

## 2.3 Gradients accumulate, which is why `zero_grad()` exists

`.backward()` **adds** into `.grad`; it never overwrites it. This is deliberate. It makes gradient accumulation over several micro-batches possible (Chapter 11). But it means you must clear gradients before each new step. Forgetting to do so is the first of the "three classic errors" in Tutorial §7.4.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
for step in range(3):
    loss = w ** 2            # dloss/dw = 2w = 6
    loss.backward()
    print(f"after backward #{step+1}: w.grad = {w.grad.item()}")   # 6, 12, 18 -- accumulating!

w.grad = None                # what optimizer.zero_grad() does by default (set_to_none=True)
(w ** 2).backward()
print("after clearing:", w.grad.item())

Two related rules:

- **A graph can be backpropagated through only once.** After `backward()`, the intermediate values saved for the backward pass are freed. To backpropagate twice through the *same* graph you would need `backward(retain_graph=True)`. In ordinary training you never need this; you run a fresh forward pass each step.
- **`backward()` needs a scalar.** For a non-scalar output `v`, PyTorch computes a vector–Jacobian product, so you must pass the vector: `v.backward(gradient=u)`. In training, the loss is always a scalar, so this rarely comes up.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
loss = w ** 2
loss.backward()
try:
    loss.backward()
except RuntimeError as e:
    print("RuntimeError:", str(e)[:110], "...")

v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v ** 2                       # non-scalar
try:
    out.backward()
except RuntimeError as e:
    print("RuntimeError:", e)
out.sum().backward()               # reduce to a scalar first
print("d(sum v^2)/dv =", v.grad)   # 2v

## 2.4 Turning autograd off: `no_grad`, `inference_mode`, `detach`

Recording a graph costs memory and time, so turn it off whenever you do not need gradients.

| Tool | Use it for |
| --- | --- |
| `with torch.no_grad():` | Evaluation loops, and manual parameter updates |
| `with torch.inference_mode():` | Pure inference. Slightly faster than `no_grad`, but the tensors it creates can never be used in autograd later |
| `t.detach()` | A tensor that shares data with `t` but is cut out of the graph, for example for logging or a stop-gradient |
| `p.requires_grad_(False)` | **Freezing** parameters (transfer learning, Chapter 14) |

In [ ]:
w = torch.randn(3, requires_grad=True)
with torch.no_grad():
    y = w * 2
print("inside no_grad -> requires_grad:", y.requires_grad)
print("detach ->", (w * 2).detach().requires_grad)

# ⚠️ Pitfall: summing losses for logging without .item() keeps every graph alive -> memory grows each step
running = 0.0
for _ in range(3):
    loss = (w ** 2).sum()
    running += loss.item()        # .item() gives a plain float: no graph is kept
print("running loss:", running)

## 2.5 Gradient checking, the PyTorch way (Tutorial §6.8)

`torch.autograd.gradcheck` compares autograd's analytic gradient against centred finite differences. As Tutorial §6.8 warned, use **float64**: float32 rounding swamps the finite-difference signal.

In [ ]:
def tiny_net(x, W1, b1, W2, b2):
    return (torch.tanh(x @ W1 + b1) @ W2 + b2).pow(2).sum()

args = [torch.randn(4, 3, dtype=torch.float64, requires_grad=True),
        torch.randn(3, 5, dtype=torch.float64, requires_grad=True),
        torch.randn(5,    dtype=torch.float64, requires_grad=True),
        torch.randn(5, 2, dtype=torch.float64, requires_grad=True),
        torch.randn(2,    dtype=torch.float64, requires_grad=True)]
print("gradcheck passed:", torch.autograd.gradcheck(tiny_net, args))

## 2.6 In-place operations and autograd

Many backward formulas need values saved during the forward pass. For example, the sigmoid's backward uses its own output, `σ'(z) = σ(z)(1 − σ(z))` (Tutorial §3.2). If you modify a saved tensor in place, autograd detects the corruption and refuses to continue:

In [ ]:
z = torch.randn(3, requires_grad=True)
s = torch.sigmoid(z)        # autograd saves s for the backward pass
s.mul_(2)                   # in-place change to a saved tensor
try:
    s.sum().backward()
except RuntimeError as e:
    print("RuntimeError:", str(e)[:150], "...")

s = torch.sigmoid(z) * 2    # out-of-place version: fine
s.sum().backward()
print("ok:", z.grad)

## 2.7 Writing your own backward: `torch.autograd.Function`

Occasionally you need an operation whose backward you define yourself. Typical reasons are a numerically better gradient, a custom CUDA kernel, or a deliberate "straight-through" trick. You subclass `torch.autograd.Function` and implement two static methods. This is exactly Tutorial §6's local-derivative view: `backward` receives `∂L/∂output` and must return `∂L/∂input`.

In [ ]:
class MySigmoid(torch.autograd.Function):
    @staticmethod
    def forward(ctx, z):
        s = 1.0 / (1.0 + torch.exp(-z))
        ctx.save_for_backward(s)           # stash what backward needs
        return s

    @staticmethod
    def backward(ctx, grad_out):           # grad_out = dL/ds
        (s,) = ctx.saved_tensors
        return grad_out * s * (1 - s)      # dL/dz = dL/ds * s(1-s)

z = torch.randn(5, dtype=torch.float64, requires_grad=True)
print("matches torch.sigmoid:", torch.allclose(MySigmoid.apply(z), torch.sigmoid(z)))
print("gradcheck:", torch.autograd.gradcheck(MySigmoid.apply, (z,)))

## 2.8 `torch.autograd.grad` and higher derivatives
`torch.autograd.grad(outputs, inputs)` returns gradients directly instead of accumulating them into `.grad`. With `create_graph=True`, the gradient computation itself is recorded, so you can differentiate again. This is used in gradient penalties, meta-learning, and physics-informed networks.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
f = x ** 3
(df,) = torch.autograd.grad(f, x, create_graph=True)   # 3x^2 = 12
(d2f,) = torch.autograd.grad(df, x)                     # 6x   = 12
print(df.item(), d2f.item())

## 🧪 Exercises

**Exercise 2.1: the softmax + cross-entropy gradient (Tutorial §6.6).** `★★☆ · ~15 min` You proved that for one example, `∂L/∂z = softmax(z) − onehot(y)`. For a batch of `N` examples with **mean** reduction, the gradient picks up a factor of `1/N`. Verify this with autograd: create random `logits` of shape `(8, 5)` with `requires_grad=True` and random labels, compute `F.cross_entropy(logits, labels)`, backpropagate, and compare `logits.grad` with your formula.

In [ ]:
# ✍️ Your attempt here.
# Hint: F.one_hot(labels, num_classes=5).float() gives the one-hot matrix.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
logits = torch.randn(8, 5, requires_grad=True)
labels = torch.randint(0, 5, (8,))
F.cross_entropy(logits, labels).backward()        # reduction='mean' by default

expected = (torch.softmax(logits.detach(), dim=1) - F.one_hot(labels, 5).float()) / 8
print("autograd matches (p - y)/N:", torch.allclose(logits.grad, expected, atol=1e-6))

**Exercise 2.2: gradient descent with bare autograd.** `★☆☆ · ~15 min` Minimise `f(w) = (w₀ − 3)² + 10·(w₁ + 1)²` starting from `w = (0, 0)` using only autograd and a hand-written update `w ← w − η∇f`, with `η = 0.05` and 100 steps. You should end near `(3, −1)`. Two details matter: (a) the update must not be recorded in the graph, and (b) the gradient must be cleared every step. What happens with `η = 0.11`, and why? (Tutorial §7.3 and §10.1 explain it.)

In [ ]:
# ✍️ Your attempt here.
# Hint: Use `with torch.no_grad(): w -= lr * w.grad` and then `w.grad = None`.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def minimise(lr, steps=100):
    w = torch.zeros(2, requires_grad=True)
    for _ in range(steps):
        f = (w[0] - 3) ** 2 + 10 * (w[1] + 1) ** 2
        f.backward()
        with torch.no_grad():          # (a) parameter updates must not be part of the graph
            w -= lr * w.grad
        w.grad = None                  # (b) otherwise gradients accumulate across steps
    return w.detach(), f.item()

print("lr=0.05:", minimise(0.05))
print("lr=0.11:", minimise(0.11))
# With lr=0.11 the steep w1 direction (curvature 2*10 = 20) diverges: gradient descent on a
# quadratic is stable only if lr < 2/curvature = 0.1. This is the ill-conditioning problem
# of Tutorial §10.1 -- the steepest direction caps the learning rate for all directions.

### 📌 Key takeaways

- `backward()` accumulates into `.grad` of leaves; clear it every step.
- Disable recording with `no_grad`/`inference_mode` whenever you do not need gradients.
- In-place edits of saved tensors break the backward pass.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** You call `loss.backward()` twice in a training step without zeroing gradients in between. What is in `p.grad`?

<details>
<summary>Answer</summary>

The **sum** of both gradients. `.backward()` accumulates into `.grad`. (If both calls used the same graph, the second would also raise an error, because the saved tensors were freed after the first.)

</details>

**2.** Why must a manual parameter update be wrapped in `torch.no_grad()`?

<details>
<summary>Answer</summary>

Otherwise the update `w - lr * w.grad` becomes part of the autograd graph. For a leaf tensor that requires grad, an in-place update outside `no_grad` is an error. Out of place, it would silently turn `w` into a non-leaf with a growing history.

</details>

**3.** What is the difference between `torch.no_grad()` and `tensor.detach()`?

<details>
<summary>Answer</summary>

`no_grad` is a context: nothing computed inside it is recorded. `detach()` acts on one tensor: it returns a view that shares the data but is cut out of the graph, while the rest of the computation can still record.

</details>

**4.** When is a tensor's `.grad` populated after `backward()`?

<details>
<summary>Answer</summary>

Only for **leaf** tensors with `requires_grad=True`, such as parameters. Gradients of intermediate (non-leaf) tensors are freed unless you call `.retain_grad()` on them before the backward pass.

</details>

---
# Chapter 3. Linear regression four ways: the ladder of abstraction

PyTorch code looks magical until you see exactly what each abstraction replaces. We fit the same linear regression four times, climbing one rung each time. By the end you will be able to read any PyTorch training loop as "the manual algorithm, with these pieces automated".

The data is `y = 3·x₁ − 2·x₂ + 1 + noise`. The model is `ŷ = X wᵀ + b`, and the loss is mean squared error (Tutorial §5.2).

🎯 **In this chapter you will learn to:** map every PyTorch abstraction to the manual step it replaces; read any training loop as 'forward, loss, zero, backward, step'.

In [ ]:
set_seed(0)
N = 200
X = torch.randn(N, 2)
true_w, true_b = torch.tensor([3.0, -2.0]), 1.0
y = (X @ true_w + true_b + 0.1 * torch.randn(N)).unsqueeze(1)   # shape (N, 1) -- see §1.4!
print(X.shape, y.shape)
LR, STEPS = 0.1, 200

### Rung 1: gradients by hand
For `J = (1/N) Σ (ŷ − y)²` with `ŷ = Xw + b` (where `w` has shape `(2, 1)`), the derivatives are `∂J/∂w = (2/N) Xᵀ(ŷ − y)` and `∂J/∂b = (2/N) Σ(ŷ − y)`. Everything is manual.

In [ ]:
w = torch.zeros(2, 1)
b = torch.zeros(1)
for step in range(STEPS):
    y_hat = X @ w + b
    err = y_hat - y
    grad_w = 2 / N * X.T @ err        # derived on paper
    grad_b = 2 / N * err.sum(0)
    w -= LR * grad_w
    b -= LR * grad_b
print("rung 1:", w.squeeze().tolist(), b.item())

### Rung 2: autograd replaces the derivation

In [ ]:
w = torch.zeros(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
for step in range(STEPS):
    loss = ((X @ w + b - y) ** 2).mean()
    loss.backward()                   # <- replaces the two gradient lines
    with torch.no_grad():
        w -= LR * w.grad
        b -= LR * b.grad
    w.grad = None; b.grad = None
print("rung 2:", w.detach().squeeze().tolist(), b.item())

### Rung 3: an optimizer replaces the update and the zeroing

In [ ]:
w = torch.zeros(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
opt = torch.optim.SGD([w, b], lr=LR)
for step in range(STEPS):
    loss = ((X @ w + b - y) ** 2).mean()
    opt.zero_grad()                   # <- replaces "w.grad = None"
    loss.backward()
    opt.step()                        # <- replaces the no_grad update
print("rung 3:", w.detach().squeeze().tolist(), b.item())

### Rung 4: `nn.Module` and a loss object replace the parameters and the formula

In [ ]:
model = nn.Linear(2, 1)              # owns w (shape (1, 2)) and b, with requires_grad=True
nn.init.zeros_(model.weight); nn.init.zeros_(model.bias)     # match the other rungs
loss_fn = nn.MSELoss()
opt = torch.optim.SGD(model.parameters(), lr=LR)
for step in range(STEPS):
    loss = loss_fn(model(X), y)      # the five lines of Tutorial §7.4, in order...
    opt.zero_grad()
    loss.backward()
    opt.step()
print("rung 4:", model.weight.detach().squeeze().tolist(), model.bias.item())

All four rungs recover `w ≈ (3, −2)`, `b ≈ 1`. Here is what each abstraction took over:

| You stopped writing… | …because this does it |
| --- | --- |
| The derivative formulas | `loss.backward()` (autograd) |
| The update rule and gradient zeroing | `torch.optim.SGD` with `step()` / `zero_grad()` |
| Parameter creation and shapes | `nn.Linear` (an `nn.Module`) |
| The loss formula | `nn.MSELoss` |

Swapping `SGD` for `AdamW`, or `nn.Linear` for a 50-layer network, changes **one line** each. That is the whole point of the design.

**On the order of the five lines.** Rung 4 computes the loss first and then calls `zero_grad()`. Tutorial §7.4 calls `zero_grad()` first. Both are correct. The only requirement is that `zero_grad()` happens **before** `backward()` within each step.

## 🧪 Exercise 3.1 `★☆☆ · ~10 min`
Repeat rung 4 with `y_flat = y.squeeze(1)` (shape `(N,)`) as the target. Note the warning PyTorch prints, and the final weights. Then explain, using §1.4, why the model fails to learn `w`.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
y_flat = y.squeeze(1)
model_bad = nn.Linear(2, 1)
opt_bad = torch.optim.SGD(model_bad.parameters(), lr=LR)
for step in range(STEPS):
    loss = F.mse_loss(model_bad(X), y_flat)     # (N,1) vs (N,) -> broadcast to (N,N): PyTorch warns
    opt_bad.zero_grad(); loss.backward(); opt_bad.step()
print("weights:", model_bad.weight.detach().squeeze().tolist(), "bias:", model_bad.bias.item())
# Each prediction is compared with *every* target, so the loss is minimised by predicting
# the mean of y for every input: w -> ~0 and b -> mean(y) ~ 1. The model learns nothing about X.

### 📌 Key takeaways

- autograd replaces derivatives, `optim` replaces updates, `nn.Module` replaces parameter bookkeeping, loss objects replace formulas.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** In rung 4, which object owns the parameters, and how does the optimizer find them?

<details>
<summary>Answer</summary>

The `nn.Linear` module owns `weight` and `bias`, which are registered `nn.Parameter`s. `model.parameters()` yields them, and the optimizer stores references to those exact tensors when it is constructed.

</details>

**2.** What does `opt.step()` do for plain SGD, in one line of math?

<details>
<summary>Answer</summary>

`p ← p − η·p.grad` for every parameter `p` it manages, performed without recording autograd history.

</details>

---
# Part II — Building and training models

# Chapter 4. `nn.Module` in depth

Every model, every layer, and even every loss in PyTorch is an `nn.Module`. A module does four jobs:

1. It **registers** its parameters, buffers and submodules, so that `.parameters()`, `.to(device)`, `.state_dict()` and `.train()/.eval()` all reach them, recursively.
2. It defines the computation in **`forward`**.
3. It carries a **mode flag** (`training`), which layers such as Dropout and BatchNorm read.
4. It supports **hooks**, which let you inspect or modify activations and gradients without editing the model's code.

## 4.1 Anatomy of a module

🎯 **In this chapter you will learn to:** write, inspect and count the parameters of `nn.Module`s; know exactly what `train()` / `eval()` change; initialise models and instrument them with hooks.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=(256, 128), out_dim=10, p_drop=0.2):
        super().__init__()                    # ALWAYS first: sets up the registration machinery
        self.fc1 = nn.Linear(in_dim, hidden[0])   # assigning a Module to an attribute registers it
        self.fc2 = nn.Linear(hidden[0], hidden[1])
        self.head = nn.Linear(hidden[1], out_dim)
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):                     # x: (N, 1, 28, 28) or (N, 784)
        x = x.flatten(start_dim=1)            # -> (N, 784)
        x = self.drop(F.relu(self.fc1(x)))    # -> (N, 256)
        x = self.drop(F.relu(self.fc2(x)))    # -> (N, 128)
        return self.head(x)                   # -> (N, 10) logits: no softmax (Chapter 5)

model = MLP()
print(model)
logits = model(torch.randn(4, 1, 28, 28))     # call the module, NOT model.forward(x)
print(logits.shape)

> ⚠️ **Always call `model(x)`, never `model.forward(x)`.** `__call__` runs the registered hooks and some framework machinery, and then calls `forward`. Calling `forward` directly silently skips the hooks.

## 4.2 Inspecting parameters, and counting them (Tutorial §4.4)

A fully connected layer has `n_in · n_out + n_out` parameters. For 784 → 256 → 128 → 10, that is

`(784·256 + 256) + (256·128 + 128) + (128·10 + 10) = 200,960 + 32,896 + 1,290 = 235,146`.

In [ ]:
for name, p in model.named_parameters():
    print(f"{name:12s} {str(tuple(p.shape)):12s} {p.numel():>8,d}  requires_grad={p.requires_grad}")

def count_params(m: nn.Module, trainable_only: bool = True) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

print(f"total: {count_params(model):,}")
assert count_params(model) == 235_146

## 4.3 Containers: `Sequential`, `ModuleList`, `ModuleDict`

- **`nn.Sequential(...)`**: the output of each module feeds the next. It is concise and fine for simple stacks. Pass an `OrderedDict` if you want readable names.
- **`nn.ModuleList([...])`**: a list whose modules are registered. Use it when `forward` needs custom logic, such as skip connections or a loop over layers.
- **`nn.ModuleDict({...})`**: the dictionary version, for example one head per task.

> ⚠️ **Pitfall: a plain Python list of layers is invisible to PyTorch.** The layers still run in `forward`, but their parameters are not registered. The optimizer never updates them, `.to(device)` does not move them, and `state_dict()` does not save them. There is no error until something downstream breaks in a confusing way.

In [ ]:
class Broken(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(10, 10) for _ in range(3)]       # plain list: NOT registered

class Fixed(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(3)])
    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return x

print("Broken params:", count_params(Broken()), "| Fixed params:", count_params(Fixed()))

from collections import OrderedDict
seq = nn.Sequential(OrderedDict([
    ("flatten", nn.Flatten()),
    ("fc1", nn.Linear(784, 128)), ("act1", nn.ReLU()),
    ("out", nn.Linear(128, 10)),
]))
print(seq.fc1)            # named access
print(seq[1] is seq.fc1)  # index access too

## 4.4 Module or functional? `nn.ReLU()` vs `F.relu`

Tutorial §3.5 showed both styles. The rule:

- **Stateless operations** (`relu`, `gelu`, `softmax`, `flatten`, pooling) can be written either way. `F.*` inside `forward` is common and clean.
- **Anything with parameters or buffers** (`Linear`, `Conv2d`, `BatchNorm`) must be a module created in `__init__`, otherwise its weights are re-created on every call and never learned.
- **Anything that depends on train/eval mode** (`Dropout`, `BatchNorm`) should be a module, so that `model.eval()` switches it automatically.

> ⚠️ **Pitfall:** `F.dropout(x, p=0.5)` defaults to `training=True`, so it stays **on during evaluation** unless you write `F.dropout(x, 0.5, training=self.training)`. Using `nn.Dropout` avoids the trap entirely.

## 4.5 `train()` and `eval()`: what actually changes (Tutorial §11.2, §12.4)

`model.train()` and `model.eval()` set the `training` flag on the module **and every submodule**. Only a few layers read it:

| Layer | `train()` | `eval()` |
| --- | --- | --- |
| `Dropout(p)` | Zeroes each unit with probability `p`, and scales survivors by `1/(1−p)` ("inverted dropout") | Identity |
| `BatchNorm` | Normalises with the **batch's** statistics and updates the running averages | Normalises with the stored **running** statistics and updates nothing |
| Everything else | — | — |

Note that `eval()` does **not** turn off autograd. You still need `torch.no_grad()` or `torch.inference_mode()` for that. These are two separate switches, and evaluation code needs both.

In [ ]:
drop = nn.Dropout(p=0.5)
x = torch.ones(1, 10)
drop.train(); print("train:", drop(x))      # about half zeros, survivors scaled to 2.0
drop.eval();  print("eval: ", drop(x))      # identity

bn = nn.BatchNorm1d(3)
print("buffers:", [n for n, _ in bn.named_buffers()])   # running_mean, running_var, num_batches_tracked
bn.train(); _ = bn(torch.randn(64, 3) * 5 + 10)          # one training batch updates the running stats
print("running_mean after one train batch:", bn.running_mean)  # moved 10% (momentum=0.1) toward ~10
bn.eval()
x_test = torch.randn(4, 3) * 5 + 10
print("eval uses running stats -> output not centred yet:", bn(x_test).mean(0))

## 4.6 Buffers: state that is saved but not trained

`self.register_buffer("name", tensor)` stores a tensor that **moves with `.to(device)`** and **is saved in `state_dict()`**, but is **not** a parameter, so the optimizer never touches it. BatchNorm's running statistics are buffers.

🏭 **Industry note.** Put input-normalisation constants inside the model as buffers. The saved model then carries its own preprocessing, and the classic deployment bug ("we normalised differently in production") disappears. Chapter 15 builds on this.

In [ ]:
class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer("mean", torch.as_tensor(mean, dtype=torch.float32).view(1, -1, 1, 1))
        self.register_buffer("std", torch.as_tensor(std, dtype=torch.float32).view(1, -1, 1, 1))
    def forward(self, x):
        return (x - self.mean) / self.std

norm = Normalize([0.5, 0.4, 0.3], [0.2, 0.2, 0.2])
print("parameters:", list(norm.parameters()))          # [] -- nothing to train
print("state_dict:", list(norm.state_dict().keys()))   # ['mean', 'std'] -- but it is saved

## 4.7 Initialisation, and seeing Tutorial §9.5 happen again

`nn.Linear` and `nn.Conv2d` default to Kaiming-uniform with `a = √5`, which gives weight variance `1/(3·n_in)` (Tutorial §9.6). To override it, write a function and apply it **recursively** with `model.apply(fn)`.

Below we build a 20-layer ReLU MLP and record each layer's activation standard deviation with **forward hooks**, without touching the model's code. This reproduces the experiment of Tutorial §9.5 with real `nn.Module`s.

In [ ]:
def deep_relu_mlp(depth=20, width=256):
    layers = []
    for _ in range(depth):
        layers += [nn.Linear(width, width, bias=False), nn.ReLU()]
    return nn.Sequential(*layers)

def activation_stds(model, x):
    stds, handles = [], []
    for m in model.modules():
        if isinstance(m, nn.ReLU):
            # forward hook signature: hook(module, inputs, output)
            handles.append(m.register_forward_hook(lambda mod, inp, out: stds.append(out.std().item())))
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()          # always remove hooks you no longer need
    return stds

inits = {
    "PyTorch default": None,
    "std=0.01":        lambda w: nn.init.normal_(w, std=0.01),
    "Xavier":          nn.init.xavier_normal_,
    "He (Kaiming)":    lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"),
}
set_seed(0)
x = torch.randn(1000, 256)
plt.figure(figsize=(7, 4))
for label, fn in inits.items():
    net = deep_relu_mlp()
    if fn is not None:
        net.apply(lambda m: fn(m.weight) if isinstance(m, nn.Linear) else None)
    plt.semilogy(range(1, 21), activation_stds(net, x), marker="o", label=label)
plt.xlabel("layer"); plt.ylabel("std of activations (log scale)"); plt.legend(); plt.title("Signal propagation in a 20-layer ReLU MLP")
plt.show()

Only He initialisation keeps the signal at a constant scale. Notice also that the **PyTorch default decays**, because `1/(3·n_in)` is smaller than He's `2/n_in`. For the shallow networks we train early on this rarely matters: over two or three layers the signal shrinks by a modest factor, not by orders of magnitude. BatchNorm or residual connections make it irrelevant in deep ones (Chapter 13). But it is the reason to set initialisation explicitly for deep, un-normalised networks.

## 4.8 Hooks for gradients, and checking gradient flow

Forward hooks see activations. To see gradients, read `p.grad` after `backward()`, or register a hook on a tensor with `t.register_hook(fn)`. A per-layer gradient-norm printout is one of the most useful debugging tools you have (Chapter 10).

In [ ]:
def grad_norms(model):
    return {n: p.grad.norm().item() for n, p in model.named_parameters() if p.grad is not None}

set_seed(0)
net = deep_relu_mlp(depth=10)
net.apply(lambda m: nn.init.normal_(m.weight, std=0.01) if isinstance(m, nn.Linear) else None)
net(torch.randn(64, 256)).pow(2).mean().backward()
for name, g in list(grad_norms(net).items())[::3]:
    print(f"{name:10s} grad norm = {g:.2e}")    # vanishing: tiny everywhere with std=0.01 init

## 4.9 `state_dict`: what gets saved
`model.state_dict()` is an ordered dictionary mapping names to tensors, covering **parameters and persistent buffers**. It does *not* contain the code or the architecture. You rebuild the architecture in code, then load the weights into it. Chapter 9 covers checkpoints in full.

In [ ]:
sd = MLP().state_dict()
print(list(sd.keys()))
fresh = MLP()
print(fresh.load_state_dict(sd))           # <All keys matched successfully>
# strict=True (the default) raises if keys are missing or unexpected -- keep it on unless you mean otherwise.

## 🧪 Exercises

**Exercise 4.1: a configurable MLP factory.** `★★☆ · ~20 min` Write `make_mlp(in_dim, hidden_dims, out_dim, act=nn.ReLU, p_drop=0.0, norm=False)` that returns an `nn.Sequential`. It should start with `nn.Flatten()`. Each hidden block should be `Linear → [BatchNorm1d] → activation → [Dropout]`, and the network should end with a plain `Linear` that produces logits. Then **predict by hand** the parameter count of `make_mlp(784, [512, 256], 10, norm=True)` and check it with `count_params`. (Remember that BatchNorm1d has `2·n` trainable parameters: γ and β.)

In [ ]:
# ✍️ Your attempt here.
# Hint: Build a list of layers and return nn.Sequential(*layers).

In [ ]:
# 🔎 Self-check for Exercise 4.1: tests YOUR `make_mlp` (run after your attempt)
def _test(fn):
    m = fn(784, [512, 256], 10, norm=True)
    assert isinstance(m, nn.Sequential), 'return an nn.Sequential'
    n = sum(p.numel() for p in m.parameters())
    assert n == 537_354, f'{n:,} parameters, expected 537,354'
    m.eval()
    assert fn(784, [64], 10)(torch.randn(4, 1, 28, 28)).shape == (4, 10), 'output shape should be (4, 10)'

self_check("make_mlp", _test)

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def make_mlp(in_dim, hidden_dims, out_dim, act=nn.ReLU, p_drop=0.0, norm=False):
    layers = [nn.Flatten()]
    d = in_dim
    for h in hidden_dims:
        layers.append(nn.Linear(d, h))
        if norm:
            layers.append(nn.BatchNorm1d(h))
        layers.append(act())
        if p_drop > 0:
            layers.append(nn.Dropout(p_drop))
        d = h
    layers.append(nn.Linear(d, out_dim))
    return nn.Sequential(*layers)

m = make_mlp(784, [512, 256], 10, norm=True)
by_hand = (784*512 + 512) + 2*512 + (512*256 + 256) + 2*256 + (256*10 + 10)
print(f"by hand: {by_hand:,}   count_params: {count_params(m):,}")   # 537,354 both
assert by_hand == count_params(m)

**Exercise 4.2: a residual MLP block.** `★★☆ · ~20 min` Implement `ResidualMLPBlock(dim)` that computes `x + Linear(ReLU(Linear(x)))`, using `nn.LayerNorm(dim)` on the input of the branch (so `out = x + f(LN(x))`, the "pre-norm" layout used in Transformers). Stack 20 of them using `nn.Sequential`, and use your forward-hook function from §4.7 (adapted to hook the blocks) to show that the activation scale stays healthy without any special initialisation. This previews why residual connections (Tutorial §9.7, item 3) dominate modern architectures.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
class ResidualMLPBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
    def forward(self, x):
        return x + self.fc2(F.relu(self.fc1(self.norm(x))))

set_seed(0)
res_net = nn.Sequential(*[ResidualMLPBlock(256) for _ in range(20)])
stds, hs = [], []
for blk in res_net:
    hs.append(blk.register_forward_hook(lambda m, i, o: stds.append(o.std().item())))
with torch.no_grad():
    res_net(torch.randn(1000, 256))
for h in hs: h.remove()
print([round(s, 2) for s in stds])
# The std grows slowly and smoothly (each block adds a bounded update to x) instead of
# collapsing or exploding: the identity path keeps the signal alive at any depth.

### 📌 Key takeaways

- Register submodules as attributes or in `ModuleList`/`ModuleDict`, never in plain lists.
- `eval()` changes Dropout/BatchNorm behaviour; it does not stop autograd.
- Buffers are saved and moved but not trained: ideal for preprocessing constants.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Your model's optimizer reports far fewer parameters than you expect, and some layers never change. What is the most likely cause?

<details>
<summary>Answer</summary>

The layers are stored in a plain Python `list` or `dict` instead of `nn.ModuleList` or `nn.ModuleDict`, so they are never registered. (A second possibility is that they were created inside `forward` instead of `__init__`.)

</details>

**2.** Does `model.eval()` stop gradients from being computed?

<details>
<summary>Answer</summary>

No. It only flips the `training` flag that Dropout and BatchNorm read. Use `torch.no_grad()` or `torch.inference_mode()` to stop autograd from recording.

</details>

**3.** What is the difference between a parameter and a buffer?

<details>
<summary>Answer</summary>

Both move with `.to()` and are saved in `state_dict()`. Parameters are returned by `.parameters()` and updated by the optimizer. Buffers are not; they are updated, if at all, by the module's own code (for example BatchNorm's running mean).

</details>

---
# Chapter 5. Loss functions and the logits rule

Tutorial §5 explained *why* each loss is the right one. This chapter is about the **contract** of each loss in PyTorch: what it expects as input and target, in which shapes and dtypes. Most loss bugs are contract violations, and several of them are silent.

## 5.1 The table you will use daily

| Task | Last layer outputs | Loss | Target dtype and shape | Turning outputs into predictions |
| --- | --- | --- | --- | --- |
| Regression | `(N, 1)` raw values | `nn.MSELoss`, `nn.L1Loss`, `nn.HuberLoss` | `float32`, `(N, 1)` | Use the output as is |
| Binary classification | `(N, 1)` **logits** | `nn.BCEWithLogitsLoss` | `float32` 0./1., `(N, 1)` | `logit > 0` (equivalent to `sigmoid > 0.5`) |
| Multi-class, one label | `(N, C)` **logits** | `nn.CrossEntropyLoss` | **`int64` class indices, `(N,)`** | `argmax(dim=1)` |
| Multi-label | `(N, C)` logits | `nn.BCEWithLogitsLoss` | `float32` multi-hot, `(N, C)` | `logits > 0`, per class |

**The logits rule:** the model outputs raw scores (logits), and the loss applies the sigmoid or softmax **internally**, in a numerically stable fused form (Tutorial §5.6). You apply `sigmoid` or `softmax` yourself **only** when you need probabilities, at inference time.

🎯 **In this chapter you will learn to:** choose the right loss, output shape and target dtype for each task; spot the silent double-softmax and sigmoid+BCE bugs.

In [ ]:
# The example of Tutorial §5.7
logits = torch.tensor([[2.0, 1.0, 0.1],
                       [0.5, 2.5, 0.3],
                       [1.2, 0.2, 3.1],
                       [0.1, 0.1, 0.1]])
targets = torch.tensor([0, 1, 2, 0])            # int64 class indices

loss = nn.CrossEntropyLoss()(logits, targets)
print(f"CrossEntropyLoss = {loss.item():.4f}")

# CrossEntropyLoss == log_softmax + negative log-likelihood, averaged over the batch:
log_p = torch.log_softmax(logits, dim=1)
print(f"by hand          = {-log_p[torch.arange(4), targets].mean().item():.4f}")
print(f"last example     = {-log_p[3, 0].item():.4f}  (= ln 3, all logits equal)")

> **A number that does not reproduce.** Tutorial §5.7 prints `0.6161` for this example, but the correct value is **`0.4805`**. You can check it by hand: the four per-example losses are 0.417, 0.220, 0.186 and 1.099, and their mean is 0.4805. The per-example value of 1.0986 for the last row is correct. This is exactly the situation Tutorial §0 prepared you for: *trust your own run*.

## 5.2 Silent bug #1: a softmax before `CrossEntropyLoss`
If the model ends in `softmax` and you then call `CrossEntropyLoss`, the loss applies softmax a second time. Nothing errors, but the "logits" are now confined to `[0, 1]`. The loss can never approach 0, and the gradients are badly squashed:

In [ ]:
confident = torch.tensor([[10.0, -10.0, -10.0]])    # a very confident, correct prediction
t = torch.tensor([0])
print("correct (logits in):       ", F.cross_entropy(confident, t).item())
print("bug (softmax, then CE):    ", F.cross_entropy(torch.softmax(confident, 1), t).item())
print("bug's floor, as softmax -> one-hot:", F.cross_entropy(torch.tensor([[1.0, 0.0, 0.0]]), t).item())

A perfect prediction still costs 0.55 under the double softmax, so the network keeps receiving gradient, but in a distorted form. Accuracy usually ends up worse than it should be, and nothing tells you why.

## 5.3 Target formats for `CrossEntropyLoss`
Tutorial §5.7 says one-hot targets raise a dtype error. That is only half true in current PyTorch:
- **`int64` class indices of shape `(N,)`** are the standard format.
- **`float` class probabilities of shape `(N, C)`** are also accepted, and are used for label smoothing, mixup and distillation. A float one-hot matrix therefore gives *the same loss* as the indices.
- An **`int64` one-hot** matrix of shape `(N, C)` raises an error.

In [ ]:
onehot = F.one_hot(targets, num_classes=3)
print("indices:        ", F.cross_entropy(logits, targets).item())
print("float one-hot:  ", F.cross_entropy(logits, onehot.float()).item())
try:
    F.cross_entropy(logits, onehot)          # int64 one-hot
except RuntimeError as e:
    print("int64 one-hot:  ", "RuntimeError:", e)
try:
    F.cross_entropy(logits, torch.tensor([0, 1, 3, 0]))   # label 3 with only 3 classes
except IndexError as e:
    print("label out of range:", "IndexError:", e)

> ⚠️ **On the GPU, an out-of-range label does not give a tidy `IndexError`.** It shows up as `CUDA error: device-side assert triggered`, often reported at a *later* line, because CUDA errors are asynchronous. After that, the CUDA context is broken and you must restart the runtime. When you see that error, the first thing to check is `labels.min()` and `labels.max()` against the number of classes. Running once on the CPU, or with the environment variable `CUDA_LAUNCH_BLOCKING=1`, will show you the true line.

## 5.4 Binary losses: why `BCEWithLogitsLoss` and not `sigmoid` + `BCELoss`

In [ ]:
for z0 in (-30.0, -200.0):          # a confident "negative" -- but the true label is positive
    target = torch.tensor([[1.0]])
    z = torch.tensor([[z0]], requires_grad=True)
    unfused = F.binary_cross_entropy(torch.sigmoid(z), target); unfused.backward(); g_unfused = z.grad.item()
    z = torch.tensor([[z0]], requires_grad=True)
    fused = F.binary_cross_entropy_with_logits(z, target); fused.backward(); g_fused = z.grad.item()
    print(f"z={z0:6.0f}: sigmoid={torch.sigmoid(torch.tensor(z0)).item():.2e} | "
          f"BCELoss={unfused.item():6.2f} (grad {g_unfused:+.2f}) | "
          f"BCEWithLogits={fused.item():6.2f} (grad {g_fused:+.2f})")

The exact loss is `−log σ(z) = log(1 + e^{−z})`, which is 30 for `z = −30` and 200 for `z = −200`, and the exact gradient is `σ(z) − 1 ≈ −1`, a strong push in the right direction. At `z = −30` the two *losses* still agree, but the unfused *gradient* is already more than ten times too small. `BCELoss` clamps an internal denominator at 1e-12, which caps how hard it can push. At `z = −200`, `σ(z)` underflows to exactly 0 in float32. `BCELoss` then has to take `log(0)`; it clamps the log at −100 to avoid `inf`, so it reports the wrong loss (100). Worse, the gradient flowing back through the saturated sigmoid is **zero**, so the network gets no signal to fix its most confident mistake. The fused `BCEWithLogitsLoss` computes the same quantity analytically and stays exact. In `float16`, the sigmoid underflows far sooner. **Always use the `WithLogits` version.**

## 5.5 The options you will actually use

In [ ]:
# Class imbalance (Tutorial §5.7): weight rare classes more heavily
ce_weighted = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.0, 5.0]))
# Label smoothing (Tutorial §5.8, §12.7)
ce_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)
# Ignore some targets, e.g. padding tokens in sequence models (default ignore_index = -100)
ce_ignore = nn.CrossEntropyLoss(ignore_index=-100)
print("ignore_index:", ce_ignore(logits, torch.tensor([0, 1, -100, 0])).item())   # mean over the 3 kept examples
# Binary imbalance: pos_weight = (#negatives / #positives) up-weights the positive class
bce_imb = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([9.0]))
# Per-example losses for analysis (e.g. find the hardest examples)
per_example = F.cross_entropy(logits, targets, reduction="none")
print("per-example:", per_example)

## 🧪 Exercises

**Exercise 5.1: cross-entropy from scratch, with label smoothing.** `★★★ · ~30 min` Implement `my_cross_entropy(logits, targets, label_smoothing=0.0)` using `torch.logsumexp` (which avoids computing the softmax at all). PyTorch defines label smoothing as training against the target distribution `(1−ε)·onehot + ε/C`. Show that this gives `loss = (1−ε)·NLL + ε·mean_c(−log p_c)`, implement it, and check against `F.cross_entropy(..., label_smoothing=ε)` for `ε ∈ {0, 0.1}`.

In [ ]:
# ✍️ Your attempt here.
# Hint: log p = logits - logsumexp(logits, dim=1, keepdim=True).

In [ ]:
# 🔎 Self-check for Exercise 5.1: tests YOUR `my_cross_entropy` (run after your attempt)
def _test(fn):
    z, t = torch.randn(16, 7), torch.randint(0, 7, (16,))
    for eps in (0.0, 0.1):
        assert torch.allclose(fn(z, t, eps), F.cross_entropy(z, t, label_smoothing=eps), atol=1e-6), f'mismatch for label_smoothing={eps}'
    big = torch.tensor([[1000., 0., -1000.]])
    assert torch.isfinite(fn(big, torch.tensor([2]))), 'not numerically stable'

self_check("my_cross_entropy", _test)

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def my_cross_entropy(logits, targets, label_smoothing=0.0):
    log_p = logits - torch.logsumexp(logits, dim=1, keepdim=True)        # stable log-softmax
    nll = -log_p.gather(1, targets[:, None]).squeeze(1)                    # -log p_correct
    smooth = -log_p.mean(dim=1)                                            # mean over classes of -log p_c
    eps = label_smoothing
    return ((1 - eps) * nll + eps * smooth).mean()

z = torch.randn(16, 7)
t = torch.randint(0, 7, (16,))
for eps in (0.0, 0.1):
    print(eps, torch.allclose(my_cross_entropy(z, t, eps), F.cross_entropy(z, t, label_smoothing=eps)))
# Derivation: CE against q = (1-eps)*onehot + eps/C is  -sum_c q_c log p_c
#   = (1-eps)(-log p_y) + (eps/C) sum_c(-log p_c) = (1-eps) NLL + eps * mean_c(-log p_c).

**Exercise 5.2: choose the loss.** `★☆☆ · ~10 min` For each task below, write down the output shape of the last layer, the loss, and the target dtype and shape. Then expand the answer.

1. Predict a house's price from 12 features.
2. Detect whether an email is spam.
3. Classify a photo as one of 37 dog and cat breeds.
4. Tag a photo with any subset of {beach, sunset, people, dog}.
5. Predict the next character from a vocabulary of 96, at every position in a batch of sequences shaped `(N, T)`.

<details>
<summary><b>Show answer</b></summary>

1. `(N, 1)` output, `MSELoss` (or `HuberLoss` if there are outliers), `float32` targets of shape `(N, 1)`. Standardising the targets usually helps.
2. `(N, 1)` logits, `BCEWithLogitsLoss` (add `pos_weight` if spam is rare), `float32` 0/1 targets of shape `(N, 1)`.
3. `(N, 37)` logits, `CrossEntropyLoss`, `int64` targets of shape `(N,)`.
4. `(N, 4)` logits, `BCEWithLogitsLoss`, `float32` multi-hot targets of shape `(N, 4)`. These are four independent yes/no questions, **not** a softmax.
5. Logits of shape `(N, T, 96)`. `CrossEntropyLoss` wants the class dimension second, so either reshape to `(N·T, 96)` with targets of shape `(N·T,)`, or permute the logits to `(N, 96, T)` with `int64` targets of shape `(N, T)`. Set padded positions to `ignore_index`.

</details>

### 📌 Key takeaways

- Models output logits; the loss applies softmax/sigmoid internally.
- `CrossEntropyLoss` wants `int64` indices `(N,)` (or float probabilities `(N, C)`).
- Prefer `BCEWithLogitsLoss` to `sigmoid` + `BCELoss`, always.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Your classifier ends with `nn.Softmax(dim=1)` and trains with `nn.CrossEntropyLoss`. Will you get an error?

<details>
<summary>Answer</summary>

No, and that is the danger. Softmax is applied twice. The loss has a non-zero floor, gradients are distorted, and accuracy usually suffers. Remove the final softmax and output logits.

</details>

**2.** Which target dtype and shape does `CrossEntropyLoss` expect for 10-class classification with a batch of 32?

<details>
<summary>Answer</summary>

`int64` class indices of shape `(32,)`. (`float32` class probabilities of shape `(32, 10)` are also accepted.)

</details>

**3.** Your GPU training crashes with `device-side assert triggered`. What do you check first?

<details>
<summary>Answer</summary>

That the labels are in the range `[0, C−1]`, for example a 1-indexed dataset or a wrong `num_classes`. Reproduce on the CPU, or with `CUDA_LAUNCH_BLOCKING=1`, to get the real error and line.

</details>

---
# Chapter 6. Data: `Dataset`, `DataLoader` and transforms

PyTorch splits input handling into two clean responsibilities:

- A **`Dataset`** answers two questions: *how many examples are there?* (`__len__`) and *what is example `i`?* (`__getitem__`). It knows nothing about batches.
- A **`DataLoader`** wraps a dataset and handles everything else: shuffling, batching, collating examples into tensors, and loading in parallel worker processes.

In industry, getting this layer right (correct splits, no leakage, fast enough to keep the GPU busy) is often more work than the model itself.

🎯 **In this chapter you will learn to:** write `Dataset`s and configure `DataLoader`s correctly and fast; split data without leakage; handle images, sequences, tables with categorical features, and streams.

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset, random_split

## 6.1 A custom `Dataset`

The pattern is always the same. Load or index your data in `__init__`, and return **one** example from `__getitem__`. Below, we wrap NumPy arrays, the most common situation for tabular data.

In [ ]:
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=0)   # float64, int64

class ArrayDataset(Dataset):
    # Wraps feature/label arrays. Optionally standardises features with GIVEN statistics.
    def __init__(self, X, y, mean=None, std=None):
        self.X = torch.as_tensor(X, dtype=torch.float32)         # convert once, here
        self.y = torch.as_tensor(y, dtype=torch.int64)
        if mean is not None:
            self.X = (self.X - mean) / std

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]                           # one (features, label) pair

ds = ArrayDataset(X_np, y_np)
print(len(ds), ds[0])

## 6.2 Splits, and how to avoid leakage

**Data leakage** means information from the validation or test set influencing training. The most common form is computing normalisation statistics on the *full* dataset before splitting. The rule is: **split first, then fit any preprocessing (mean, std, vocabulary, imputation values) on the training split only, and apply it unchanged everywhere else.**

In [ ]:
# 1) Split indices reproducibly. A fixed generator means the same split on every run.
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(X_np), generator=g).numpy()
train_idx, val_idx, test_idx = perm[:700], perm[700:850], perm[850:]

# 2) Fit preprocessing on the TRAINING split only
mean = torch.tensor(X_np[train_idx].mean(axis=0), dtype=torch.float32)
std = torch.tensor(X_np[train_idx].std(axis=0), dtype=torch.float32)

# 3) Apply the same statistics everywhere
train_ds = ArrayDataset(X_np[train_idx], y_np[train_idx], mean, std)
val_ds = ArrayDataset(X_np[val_idx], y_np[val_idx], mean, std)
test_ds = ArrayDataset(X_np[test_idx], y_np[test_idx], mean, std)
print(len(train_ds), len(val_ds), len(test_ds))

# For classification with imbalanced classes, prefer a *stratified* split:
from sklearn.model_selection import train_test_split
tr, te = train_test_split(np.arange(len(y_np)), test_size=0.15, stratify=y_np, random_state=0)
print("class balance train/test:", y_np[tr].mean().round(3), y_np[te].mean().round(3))

`torch.utils.data.random_split(ds, [0.8, 0.2], generator=g)` is a convenient one-liner when no preprocessing has to be fitted on the training part. It returns `Subset` objects that index into the original dataset.

## 6.3 The `DataLoader`

In [ ]:
loader = DataLoader(train_ds, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))
print("one batch:", xb.shape, xb.dtype, yb.shape, yb.dtype)
print("batches per epoch:", len(loader), "| last batch size:", len(train_ds) % 64)

loader_drop = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)
print("with drop_last:", len(loader_drop), "batches")

| Argument | Typical value | Why |
| --- | --- | --- |
| `batch_size` | 32–512 | Trade-offs in Tutorial §7.6 |
| `shuffle` | `True` for training, **`False`** for evaluation | Shuffling breaks harmful orderings during training (Chapter 10 shows what happens without it). Evaluation must be deterministic |
| `drop_last` | `True` for training when a tiny last batch would upset BatchNorm | In train mode BatchNorm fails if a channel has only **one** value to normalise (batch size 1 with `BatchNorm1d`, or 1×1 feature maps). Otherwise it works, but statistics from a handful of examples are very noisy |
| `num_workers` | `2` on Colab, up to the number of CPU cores elsewhere | Loads batches in parallel worker processes |
| `pin_memory` | `True` when training on CUDA | Page-locked host memory makes CPU→GPU copies faster and lets `.to(device, non_blocking=True)` overlap with compute |
| `persistent_workers` | `True` when `num_workers > 0` | Keeps workers alive between epochs instead of restarting them |
| `collate_fn` | default | Custom batching, for example padding variable-length sequences (§6.4) |
| `generator` | `torch.Generator().manual_seed(s)` | Makes the shuffle order reproducible (Chapter 9) |

> ⚠️ **Workers and notebooks.** On Linux (including Colab), workers are created by `fork` and can use classes defined in the notebook. On macOS and Windows they are created by `spawn` and must re-import your code. A `Dataset` class defined in a notebook cell can then fail with a pickling or attribute error. If that happens, move the class into a `.py` file and import it, or use `num_workers=0`.

## 6.4 Custom collation: variable-length sequences

The default `collate_fn` stacks examples, so every example must have the same shape. Text, audio and time series do not. The standard solution is to **pad** each batch to its longest sequence and return the lengths (or a mask) so that the model can ignore the padding.

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class RandomSequences(Dataset):
    def __init__(self, n=100):
        g = torch.Generator().manual_seed(0)
        self.lengths = torch.randint(3, 10, (n,), generator=g)
        self.data = [torch.randn(int(L), 4, generator=g) for L in self.lengths]   # each (L_i, 4)
        self.labels = torch.randint(0, 2, (n,), generator=g)
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i], self.labels[i]

def pad_collate(batch):
    seqs, labels = zip(*batch)                          # batch is a list of (seq, label) pairs
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True)       # (B, L_max, 4), zero-padded
    mask = torch.arange(padded.size(1))[None, :] < lengths[:, None]   # (B, L_max): True = real data
    return padded, lengths, mask, torch.stack(labels)

seq_loader = DataLoader(RandomSequences(), batch_size=4, collate_fn=pad_collate)
padded, lengths, mask, labels = next(iter(seq_loader))
print(padded.shape, lengths.tolist())
print(mask.int())
# e.g. a mean over the *real* time steps only:
mean_feat = (padded * mask[..., None]).sum(1) / lengths[:, None]
print(mean_feat.shape)

## 6.5 Image data with torchvision

`torchvision.datasets` provides standard datasets with a common interface, and `torchvision.transforms.v2` provides the preprocessing and augmentation pipeline. (The older `torchvision.transforms` API still works, but `v2` is the recommended one: it is faster, and it handles bounding boxes and masks as well as images.)

We will use **Fashion-MNIST**: 70,000 grayscale 28×28 images of clothing in 10 classes, split into 60,000 for training and 10,000 for testing. It is a drop-in replacement for MNIST that is hard enough to separate good models from bad ones.

First we **split**, and only then compute the normalisation statistics, using the training split alone (§6.2):

In [ ]:
from torchvision import datasets
from torchvision.transforms import v2

fmnist_raw = datasets.FashionMNIST(DATA_DIR, train=True, download=True)   # no transform: PIL images
print(fmnist_raw)
FMNIST_CLASSES = fmnist_raw.classes
print(FMNIST_CLASSES)

# 1) Split the 60k training images into 55k train / 5k validation, reproducibly
perm = torch.randperm(len(fmnist_raw), generator=torch.Generator().manual_seed(0))
fm_train_idx, fm_val_idx = perm[:55_000].tolist(), perm[55_000:].tolist()

# 2) Statistics from the TRAINING split only
pixels = fmnist_raw.data[fm_train_idx].float() / 255.0     # (55000, 28, 28) uint8 -> float in [0, 1]
FMNIST_MEAN, FMNIST_STD = pixels.mean().item(), pixels.std().item()
print(f"mean={FMNIST_MEAN:.4f}  std={FMNIST_STD:.4f}  (training split only)")
del pixels

A transform pipeline turns each PIL image into a normalised `float32` tensor of shape `(C, H, W)`:

In [ ]:
fmnist_tf = v2.Compose([
    v2.ToImage(),                                   # PIL -> uint8 tensor (1, 28, 28)
    v2.ToDtype(torch.float32, scale=True),          # -> float32 in [0, 1]  (scale=True divides by 255)
    v2.Normalize(mean=[FMNIST_MEAN], std=[FMNIST_STD]),
])

full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=fmnist_tf)
fmnist_test = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=fmnist_tf)

# 3) The same split indices, now on the transformed dataset
fmnist_train, fmnist_val = Subset(full_train, fm_train_idx), Subset(full_train, fm_val_idx)
fmnist_train, fmnist_val, fmnist_test = maybe_subset(fmnist_train), maybe_subset(fmnist_val), maybe_subset(fmnist_test)

x0, y0 = full_train[0]
print("one example:", x0.shape, x0.dtype, "label:", y0, FMNIST_CLASSES[y0])
print("sizes:", len(fmnist_train), len(fmnist_val), len(fmnist_test))

In [ ]:
def show_images(images, titles=None, ncols=8, cmap="gray", figsize_per=1.4):
    # images: (N, C, H, W) tensor, already in a displayable range
    n = len(images); nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per, nrows * figsize_per * 1.15))
    for i, ax in enumerate(np.array(axes).reshape(-1)):
        ax.axis("off")
        if i < n:
            img = images[i]
            img = img[0] if img.shape[0] == 1 else img.permute(1, 2, 0)   # CHW -> HW or HWC
            ax.imshow(img.clamp(0, 1).numpy(), cmap=cmap if img.ndim == 2 else None)
            if titles is not None:
                ax.set_title(titles[i], fontsize=7)
    plt.tight_layout(); plt.show()

# ALWAYS look at your data. Undo the normalisation for display.
xs = fmnist_raw.data[:16, None].float() / 255          # raw uint8 images -> (16, 1, 28, 28) in [0, 1]
show_images(xs, [FMNIST_CLASSES[int(t)] for t in fmnist_raw.targets[:16]])

## 6.6 Is your data pipeline fast enough?

Before training anything, measure how fast the loader alone can deliver batches. If it is slower than the model can consume them, the GPU sits idle, and no model optimisation will help.

In [ ]:
def loader_throughput(ds, num_workers, batch_size=256, max_batches=50):
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                    pin_memory=(device.type == "cuda"), persistent_workers=num_workers > 0)
    it = iter(dl); next(it)                         # start the workers before timing
    t0 = time.perf_counter(); n = 0
    for i, (xb, yb) in enumerate(it):
        n += len(xb)
        if i + 1 >= max_batches: break
    return n / (time.perf_counter() - t0)

for nw in sorted({0, NUM_WORKERS}):
    print(f"num_workers={nw}: {loader_throughput(fmnist_train, nw):8.0f} images/s")

Expect roughly a 1.5–2× improvement from 2 workers on Colab's 2 CPU cores. For small images the per-image Python overhead (PIL conversion and the transforms) dominates. Chapter 11 shows a far faster option for datasets that fit in GPU memory.

## 6.7 Tabular data: numeric and categorical features

Most industry data is a **table**: some numeric columns (age, income) and some **categorical** ones (city, device type, product ID). Categorical IDs must not be fed to a network as numbers, because city 37 is not "more" than city 12. The standard deep learning tool is **`nn.Embedding(num_categories, dim)`**: a learnable lookup table that maps each ID to a dense vector. It is mathematically identical to a one-hot vector times a weight matrix, but it never builds the one-hot vector.

We generate data with a **known** ground truth, so we can see exactly what the model can and cannot learn. Spend depends on income and age, plus a hidden per-city effect and a per-device effect, plus noise with standard deviation 5. That noise is the floor: no model can beat an RMSE of about 5. This section is also our first proper **regression** project, so note the target standardisation and the `(N, 1)` target shape (§1.4).

In [ ]:
g = torch.Generator().manual_seed(0)
N_TAB, N_CITIES, N_DEVICES = 20_000, 50, 3
city = torch.randint(0, N_CITIES, (N_TAB,), generator=g)
dev_type = torch.randint(0, N_DEVICES, (N_TAB,), generator=g)
age = torch.rand(N_TAB, generator=g) * 50 + 18
income = torch.randn(N_TAB, generator=g) * 15 + 60
city_effect = torch.randn(N_CITIES, generator=g) * 20           # hidden: the model must discover these
device_effect = torch.tensor([0.0, 15.0, -10.0])
signal = 50 + 0.8 * income + 0.02 * (age - 40) ** 2 + city_effect[city] + device_effect[dev_type]
spend = signal + 5 * torch.randn(N_TAB, generator=g)          # noise std 5 = the irreducible error

X_num = torch.stack([age, income], dim=1)                      # (N, 2) float
X_cat = torch.stack([city, dev_type], dim=1)                   # (N, 2) int64 IDs
y_tab = spend.unsqueeze(1)                                     # (N, 1) -- same shape as the model output

idx = torch.randperm(N_TAB, generator=g)
tr, va, te = idx[:14_000], idx[14_000:17_000], idx[17_000:]
num_mean, num_std = X_num[tr].mean(0), X_num[tr].std(0)       # fit preprocessing on TRAIN only
y_mean, y_std = y_tab[tr].mean(), y_tab[tr].std()
def tab_split(i):
    return TensorDataset((X_num[i] - num_mean) / num_std, X_cat[i], (y_tab[i] - y_mean) / y_std)
tab_train, tab_val, tab_test = tab_split(tr), tab_split(va), tab_split(te)
print("one example:", [t.shape for t in tab_train[0]], [t.dtype for t in tab_train[0]])

In [ ]:
class TabularNet(nn.Module):
    # Numeric features + one embedding per categorical column -> MLP -> one regression output.
    def __init__(self, n_num, cardinalities=(), emb_dims=(), hidden=64):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(c, d) for c, d in zip(cardinalities, emb_dims)])
        self.mlp = nn.Sequential(nn.Linear(n_num + sum(emb_dims), hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x_num, x_cat):
        parts = [x_num] + [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]   # each (N, d_i)
        return self.mlp(torch.cat(parts, dim=1))

def train_regressor(model, epochs=n_epochs(15), lr=3e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loader = DataLoader(tab_train, batch_size=256, shuffle=True)
    for _ in range(epochs):
        model.train()
        for xn, xc, yb in loader:
            loss = F.mse_loss(model(xn.to(device), xc.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def rmse(model, ds):
    model.eval()
    xn, xc, yb = (t.to(device) for t in ds.tensors)
    mse_standardised = F.mse_loss(model(xn, xc), yb)
    return (mse_standardised.sqrt() * y_std).item()              # back in original units

set_seed(0)
with_cats = train_regressor(TabularNet(2, cardinalities=(N_CITIES, N_DEVICES), emb_dims=(8, 2)))
set_seed(0)
numeric_only = train_regressor(TabularNet(2))                   # ignores the categorical columns
# Theoretical floors, computed exactly on the TEST split from the true generating process:
noise_te = (spend - signal)[te]
cat_eff = city_effect[city] + device_effect[dev_type]
floor_all = noise_te.pow(2).mean().sqrt().item()                                    # knows everything except the noise
floor_numeric = (cat_eff[te] - cat_eff[tr].mean() + noise_te).pow(2).mean().sqrt().item()   # cannot see city/device
print(f"predict-the-mean baseline : RMSE {(y_tab[te] - y_tab[tr].mean()).pow(2).mean().sqrt().item():6.2f}")
print(f"numeric features only     : RMSE {rmse(numeric_only, tab_test):6.2f}  (theoretical best ≈ {floor_numeric:.1f})")
print(f"with embeddings           : RMSE {rmse(with_cats, tab_test):6.2f}  (theoretical best ≈ {floor_all:.1f}, the noise)")

With enough epochs, the embedding model approaches the noise floor. The numeric-only model cannot get below its own, much higher floor (a model can land a hair below a 'theoretical best' only by chance, through small accidental correlations in a finite test set), because the city and device effects are invisible to it. After training, each city's embedding vector encodes what the model learned about that city. Similar cities end up with similar vectors, which is why embeddings are also used for recommendation and search.

🏭 **Industry note.** On tabular data, gradient-boosted trees (XGBoost, LightGBM, CatBoost) are very strong baselines, and they often beat neural networks. Always compare against one before choosing a network. Networks shine when tables are combined with text, images, or very high-cardinality IDs.

## 6.8 Data that does not fit in memory: `IterableDataset`

A map-style `Dataset` needs random access to item `i`. Logs, event streams and datasets stored as many large files on cloud storage are naturally read **sequentially**. For those, an **`IterableDataset`** implements `__iter__` and yields examples one at a time, usually by reading one file ("shard") after another.

There is one trap. With `num_workers > 0`, **every worker gets a copy of the dataset and iterates all of it**, so each example is produced once per worker. You must split the shards between workers yourself, using `get_worker_info()`:

In [ ]:
from torch.utils.data import IterableDataset, get_worker_info

shard_dir = DATA_DIR / "shards"; shard_dir.mkdir(exist_ok=True)
for s in range(4):                                             # 4 shards x 250 examples
    torch.save((torch.randn(250, 3), torch.arange(s * 250, (s + 1) * 250)), shard_dir / f"shard{s}.pt")
shard_files = sorted(shard_dir.glob("shard*.pt"))

class ShardStream(IterableDataset):
    def __init__(self, files, split_by_worker=True):
        self.files, self.split_by_worker = list(files), split_by_worker
    def __iter__(self):
        files = self.files
        info = get_worker_info()                               # None in the main process
        if self.split_by_worker and info is not None:
            files = files[info.id::info.num_workers]           # worker k reads shards k, k+W, k+2W, ...
        for f in files:
            xs, ys = torch.load(f, weights_only=True)
            yield from zip(xs, ys)

for split in (False, True):
    ids = torch.cat([yb for _, yb in DataLoader(ShardStream(shard_files, split), batch_size=100, num_workers=2)])
    print(f"split_by_worker={split}: {len(ids)} examples seen, {len(ids.unique())} unique")

Without splitting, 1,000 examples become 2,000, and every example is seen twice per epoch. With splitting, each one is seen exactly once. Two more facts about iterable datasets: `shuffle=True` is not allowed, so you shuffle the *shard order* each epoch and keep a small in-memory shuffle buffer; and `len()` may be unknown. Libraries such as WebDataset and Hugging Face `datasets` (streaming mode) implement these patterns at scale.

## 🧪 Exercises

**Exercise 6.1: streaming statistics.** `★★☆ · ~20 min` Real datasets rarely fit in memory, so compute the per-channel mean and standard deviation of a dataset **in one pass over a DataLoader**, accumulating `Σx` and `Σx²` per channel. Test it on Fashion-MNIST with a `ToImage + ToDtype(scale=True)` transform (no normalisation), and compare with `FMNIST_MEAN` and `FMNIST_STD`. Use `float64` accumulators: summing millions of `float32` values loses precision. Write it as a function `channel_stats(ds)` returning `(mean, std)` tensors of shape `(C,)`. The 🔎 self-check after your attempt tests it.

In [ ]:
# ✍️ Your attempt here.
# Hint: For a batch (B, C, H, W): x.sum(dim=(0, 2, 3)) gives per-channel sums. var = E[x²] − E[x]².

In [ ]:
# 🔎 Self-check for Exercise 6.1: tests YOUR `channel_stats` (run after your attempt)
def _test(fn):
    x = torch.rand(300, 3, 4, 4, generator=torch.Generator().manual_seed(0)) * torch.tensor([1., 2., 3.]).view(1, 3, 1, 1)
    m, s = fn(TensorDataset(x, torch.zeros(300)))
    assert torch.allclose(torch.as_tensor(m).float(), x.mean((0, 2, 3)), atol=1e-4), 'mean differs'
    assert torch.allclose(torch.as_tensor(s).float(), x.std((0, 2, 3), unbiased=False), atol=1e-4), 'std differs'

self_check("channel_stats", _test)

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def channel_stats(ds, batch_size=1000):
    s = s2 = None; count = 0
    for xb, _ in DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS):
        xb = xb.double()
        s = xb.sum(dim=(0, 2, 3)) if s is None else s + xb.sum(dim=(0, 2, 3))
        s2 = (xb ** 2).sum(dim=(0, 2, 3)) if s2 is None else s2 + (xb ** 2).sum(dim=(0, 2, 3))
        count += xb.numel() // xb.shape[1]           # pixels per channel
    mean = s / count
    std = (s2 / count - mean ** 2).sqrt()            # population std (fine for normalisation)
    return mean, std

raw_tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
m, s = channel_stats(maybe_subset(datasets.FashionMNIST(DATA_DIR, train=True, transform=raw_tf), 2000))
print("streamed:", m.item(), s.item(), "| direct:", FMNIST_MEAN, FMNIST_STD)

**Exercise 6.2: a transform wrapper.** `★★☆ · ~15 min` `random_split` returns `Subset`s that share the *same* underlying dataset, and therefore the same transform. That is a problem when the training split needs augmentation and the validation split must not have it. Write `TransformDataset(base, transform)`: a `Dataset` that returns `(transform(x), y)` for each `(x, y)` in `base`, where `base` returns *untransformed* items. Use it to build a training split with `v2.RandomHorizontalFlip(p=1.0)` (so you can see it working) and a plain validation split from the same `random_split` of the raw Fashion-MNIST dataset. Show one image from each.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
class TransformDataset(Dataset):
    def __init__(self, base, transform):
        self.base, self.transform = base, transform
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        return self.transform(x), y

tr_raw, va_raw = random_split(fmnist_raw, [55_000, 5_000], generator=torch.Generator().manual_seed(0))
to_tensor = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
tr_aug = TransformDataset(tr_raw, v2.Compose([v2.RandomHorizontalFlip(p=1.0), to_tensor]))
tr_plain = TransformDataset(tr_raw, to_tensor)
va_plain = TransformDataset(va_raw, to_tensor)
show_images(torch.stack([tr_plain[0][0], tr_aug[0][0], va_plain[0][0]]),
            ["train (no aug)", "train (flipped)", "val"], ncols=3, figsize_per=2)

### 📌 Key takeaways

- Split first, then fit preprocessing on the training split only.
- Validation and test pipelines must be deterministic: augmentation is for training only.
- Categorical IDs go through `nn.Embedding`; iterable datasets must split shards across workers.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** You standardise features using the mean and std of the whole dataset, then split into train and test. What is wrong, and does it matter?

<details>
<summary>Answer</summary>

The test statistics leak into training preprocessing. For large i.i.d. data the effect is tiny. With small datasets, time series, or any learned preprocessing, it gives optimistic test results. Always fit preprocessing on the training split only.

</details>

**2.** Why `shuffle=False` for validation and test loaders?

<details>
<summary>Answer</summary>

The metrics do not depend on the order, and a fixed order makes evaluation deterministic and easy to compare (for example, inspecting prediction `i` across runs). Shuffling there only adds noise and cost.

</details>

**3.** What does `pin_memory=True` buy you, and when?

<details>
<summary>Answer</summary>

When training on CUDA, batches are placed in page-locked host memory. Copies to the GPU are then faster and can run asynchronously with `.to(device, non_blocking=True)`, overlapping the transfer with computation.

</details>

---
# Chapter 7. The training loop, and your first complete project

Tutorial §7.4 gave you the skeleton. Here we turn it into **reusable, correct functions**, which we will extend in later chapters, and use them on Fashion-MNIST. We start from baselines, train an MLP, and evaluate honestly.

## 7.1 Reusable `train_one_epoch` and `evaluate`

🎯 **In this chapter you will learn to:** write reusable, correct training and evaluation functions; compare against baselines and evaluate honestly; use precision/recall/AUC and thresholds for imbalanced problems.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device, scheduler=None, max_grad_norm=None):
    # One pass over `loader`. `scheduler` (optional) is stepped once per batch.
    # Returns (mean loss, accuracy) over the epoch.
    model.train()                                            # Dropout/BN in training mode
    total_loss, total_correct, total_n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)                                    # 1. forward
        loss = loss_fn(logits, y)                            # 2. loss
        optimizer.zero_grad(set_to_none=True)                # 3. clear old gradients
        loss.backward()                                      # 4. backward
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()                                     # 5. update
        if scheduler is not None:
            scheduler.step()
        n = y.size(0)                                        # weight by batch size: the last batch may be smaller
        total_loss += loss.item() * n
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_n += n
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()                                             # no graph during evaluation
def evaluate(model, loader, loss_fn, device):
    model.eval()                                             # Dropout off, BN uses running stats
    total_loss, total_correct, total_n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_n += y.size(0)
    return total_loss / total_n, total_correct / total_n


def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, device,
        scheduler=None, max_grad_norm=None, verbose=True):
    # Train for `epochs`, evaluating on `val_loader` after each one. Returns a history dict.
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device, scheduler, max_grad_norm)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn, device)
        for k, v in zip(hist, (tr_loss, tr_acc, va_loss, va_acc)):
            hist[k].append(v)
        if verbose:
            print(f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} | {time.perf_counter() - t0:.1f}s")
    return hist


def plot_history(hist, title=""):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
    ep = range(1, len(hist["train_loss"]) + 1)
    a1.plot(ep, hist["train_loss"], label="train"); a1.plot(ep, hist["val_loss"], label="val")
    a2.plot(ep, hist["train_acc"], label="train"); a2.plot(ep, hist["val_acc"], label="val")
    a1.set_title("loss"); a2.set_title("accuracy")
    for a in (a1, a2):
        a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
    fig.suptitle(title); plt.tight_layout(); plt.show()

Three details in this code are easy to get wrong:

1. **Weighting by batch size.** Averaging per-batch mean losses over-weights a small last batch. Summing `loss × n` and dividing by the total count gives the true mean over all examples.
2. **`.item()` when accumulating.** It converts to a Python float. Without it you would keep every batch's autograd graph alive (§2.4).
3. **`model.train()` at the start of every epoch.** `evaluate` switches the model to eval mode, so training must switch it back. Forgetting this is a classic silent bug: you would train the rest of the run with dropout off and frozen BatchNorm statistics.

## 7.2 Fashion-MNIST: data loaders

In [ ]:
BATCH = 128
pin = device.type == "cuda"
train_loader = DataLoader(fmnist_train, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=pin, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(fmnist_val, batch_size=512, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(fmnist_test, batch_size=512, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)

## 7.3 Baselines first

A number such as "88% accuracy" means nothing without a reference point. Always establish:

1. **The trivial baseline.** Always predicting the most frequent class gives 10% on this balanced dataset.
2. **A simple model.** Here that is softmax regression, a single `nn.Linear(784, 10)`, which is one neuron per class (Tutorial §2.4).

And before training anything, check the **initial loss**. With 10 balanced classes and an untrained network whose logits are close to equal, the loss should be close to `ln 10 ≈ 2.303` (Tutorial §5.4). A very different value means something is wrong with the data, the labels, or the loss.

In [ ]:
set_seed(0)
softmax_reg = nn.Sequential(nn.Flatten(), nn.Linear(784, 10)).to(device)
loss_fn = nn.CrossEntropyLoss()
init_loss, init_acc = evaluate(softmax_reg, val_loader, loss_fn, device)
print(f"initial loss {init_loss:.3f} (expected ~{math.log(10):.3f}) | initial acc {init_acc:.3f} (expected ~0.1)")

opt = torch.optim.AdamW(softmax_reg.parameters(), lr=1e-3)
hist_lr = fit(softmax_reg, train_loader, val_loader, loss_fn, opt, epochs=n_epochs(5), device=device)
_ = record("Fashion-MNIST softmax regression: best val acc", max(hist_lr["val_acc"]))

Softmax regression reaches roughly 84–85% validation accuracy. Any network we build has to beat that clearly to justify its complexity.

## 7.4 The MLP

In [ ]:
set_seed(0)
mlp = MLP(in_dim=784, hidden=(256, 128), out_dim=10, p_drop=0.2).to(device)    # from Chapter 4
opt = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-2)
hist_mlp = fit(mlp, train_loader, val_loader, loss_fn, opt, epochs=n_epochs(15), device=device)
plot_history(hist_mlp, "MLP on Fashion-MNIST")

**Read the curves the way Tutorial §12.8 taught you.** You should see the training loss keep falling while the validation loss flattens and starts to creep up after roughly ten epochs. Validation accuracy plateaus somewhere around 88–89%. That is the start of overfitting. The gap is modest because dropout and weight decay are already on. Chapter 8 adds early stopping so that we keep the best model rather than the last one.

## 7.5 Honest evaluation on the test set

The test set is used **once**, at the very end, for the model you have chosen. If you look at test accuracy while making design decisions, it stops being an unbiased estimate and becomes just another validation set.

Accuracy alone also hides *where* the model fails. A confusion matrix shows it:

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    # All predictions, true labels and probabilities for a loader (in loader order).
    model.eval()
    preds, labels, probs = [], [], []
    for x, y in loader:
        p = torch.softmax(model(x.to(device)), dim=1).cpu()
        probs.append(p); preds.append(p.argmax(1)); labels.append(y)
    return torch.cat(preds), torch.cat(labels), torch.cat(probs)

def confusion_matrix(preds, labels, num_classes):
    # cm[i, j] = number of examples of true class i predicted as class j
    return torch.bincount(labels * num_classes + preds, minlength=num_classes ** 2).reshape(num_classes, num_classes)

def plot_confusion(cm, class_names, title="Confusion matrix (row-normalised)"):
    cm_n = cm.float() / cm.sum(1, keepdim=True).clamp_min(1)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(cm_n, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)), class_names, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(class_names)), class_names, fontsize=8)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, f"{cm_n[i, j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if cm_n[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

test_preds, test_labels, test_probs = predict(mlp, test_loader, device)
test_acc = record("Fashion-MNIST MLP: test acc", (test_preds == test_labels).float().mean().item())
print(f"MLP test accuracy: {test_acc:.4f}")
cm = confusion_matrix(test_preds, test_labels, 10)
plot_confusion(cm, FMNIST_CLASSES)
per_class = cm.diag() / cm.sum(1).clamp_min(1)
for name, acc in sorted(zip(FMNIST_CLASSES, per_class.tolist()), key=lambda t: t[1])[:4]:
    print(f"  weakest: {name:12s} {acc:.3f}")

The weakest class is almost always **Shirt**, which gets confused with T-shirt/top, Pullover and Coat. That is a genuinely hard distinction at 28×28 resolution. It is also an important lesson: *the aggregate number hides the failure mode*. Looking at the actual mistakes is the next step:

In [ ]:
wrong = (test_preds != test_labels).nonzero().squeeze(1)[:16]
imgs = torch.stack([fmnist_test[i][0] for i in wrong.tolist()]) * FMNIST_STD + FMNIST_MEAN   # undo normalisation
titles = [f"T:{FMNIST_CLASSES[test_labels[i]][:7]}\nP:{FMNIST_CLASSES[test_preds[i]][:7]} {test_probs[i].max():.2f}"
          for i in wrong.tolist()]
show_images(imgs, titles)

**How precise is the test number?** With `n = 10,000` test images and accuracy `p ≈ 0.88`, the standard error of the accuracy is `√(p(1−p)/n) ≈ 0.0033`, so a 95% interval is about **±0.6 percentage points**. Two models that differ by 0.3 points on this test set are statistically indistinguishable. Keep this in mind before you claim one model beats another (Chapter 16 returns to it).

## 7.6 Beyond accuracy: a rare positive class

Real classification problems are often **imbalanced**: fraud, defects, disease. When 2% of examples are positive, a model that always answers "negative" is 98% accurate and completely useless. We build such a task from Fashion-MNIST: *"is it a bag?"*, keeping only enough bags that they make up about 2% of every split.

The metrics that matter, for a decision threshold `t` on the predicted probability:

- **Precision** = TP / (TP + FP): of the items we flagged, how many were really positive?
- **Recall** = TP / (TP + FN): of the real positives, how many did we catch?
- **F1** = the harmonic mean of precision and recall.
- **ROC-AUC** and **average precision (PR-AUC)** summarise performance across *all* thresholds. With rare positives, PR-AUC is the more informative of the two, because ROC-AUC can look excellent while precision is poor.

In [ ]:
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import roc_auc_score, average_precision_score

def base_indices(ds):
    # Map a (possibly nested) Subset back to indices of its underlying dataset.
    idx = list(range(len(ds)))
    while isinstance(ds, Subset):
        idx = [ds.indices[i] for i in idx]; ds = ds.dataset
    return idx, ds

def rare_class_split(ds, positive=8, prevalence=0.02, seed=0):
    idx, base = base_indices(ds)
    labels = torch.as_tensor(base.targets)[idx]
    pos = [i for i, l in zip(idx, labels.tolist()) if l == positive]
    neg = [i for i, l in zip(idx, labels.tolist()) if l != positive]
    n_pos = max(1, round(prevalence * len(neg) / (1 - prevalence)))
    pos = [pos[i] for i in torch.randperm(len(pos), generator=torch.Generator().manual_seed(seed))[:n_pos].tolist()]
    return Subset(base, neg + pos), [0] * len(neg) + [1] * len(pos)

class BinaryTarget(Dataset):
    # Turn integer class labels into float {0., 1.} targets of shape (1,) for BCEWithLogitsLoss.
    def __init__(self, base, positive): self.base, self.positive = base, positive
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        return x, torch.tensor([float(y == self.positive)])

(bag_tr, ytr_bin), (bag_va, yva_bin), (bag_te, yte_bin) = (rare_class_split(d) for d in (fmnist_train, fmnist_val, fmnist_test))
bag_tr, bag_va, bag_te = BinaryTarget(bag_tr, 8), BinaryTarget(bag_va, 8), BinaryTarget(bag_te, 8)
n_pos, n_neg = sum(ytr_bin), len(ytr_bin) - sum(ytr_bin)
print(f"train: {n_pos} positives / {n_neg} negatives ({n_pos / len(ytr_bin):.1%}) | val {sum(yva_bin)} pos | test {sum(yte_bin)} pos")

In [ ]:
@torch.no_grad()
def predict_proba(model, ds):
    model.eval(); ps, ys = [], []
    for x, y in DataLoader(ds, batch_size=512):
        ps.append(torch.sigmoid(model(x.to(device))).squeeze(1).cpu()); ys.append(y.squeeze(1))
    return torch.cat(ps), torch.cat(ys).long()

def binary_metrics(p, y, t=0.5):
    pred = (p >= t).long()
    tp = ((pred == 1) & (y == 1)).sum().item(); fp = ((pred == 1) & (y == 0)).sum().item()
    fn = ((pred == 0) & (y == 1)).sum().item()
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"acc": (pred == y).float().mean().item(), "precision": prec, "recall": rec, "F1": f1}

def train_binary(model, loader, loss_fn, epochs=n_epochs(3)):
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            loss = loss_fn(model(x.to(device)), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

# Three ways to handle the imbalance
sample_w = torch.tensor([1.0 / n_neg if l == 0 else 1.0 / n_pos for l in ytr_bin])   # equal total weight per class
variants = {
    "plain BCE":            (DataLoader(bag_tr, batch_size=128, shuffle=True), nn.BCEWithLogitsLoss()),
    "BCE + pos_weight":     (DataLoader(bag_tr, batch_size=128, shuffle=True),
                             nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg / n_pos], device=device))),
    "balanced sampler":     (DataLoader(bag_tr, batch_size=128,
                             sampler=WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)),
                             nn.BCEWithLogitsLoss()),
}
bin_models = {}
print(f"{'variant':18s} {'acc':>6s} {'prec':>6s} {'recall':>6s} {'F1':>6s} {'ROC-AUC':>8s} {'PR-AUC':>7s}   (validation, t = 0.5)")
for name, (ld, lf) in variants.items():
    set_seed(0)
    m = train_binary(MLP(hidden=(128, 64), out_dim=1).to(device), ld, lf)
    bin_models[name] = m
    p, y = predict_proba(m, bag_va)
    r = binary_metrics(p, y)
    print(f"{name:18s} {r['acc']:6.3f} {r['precision']:6.3f} {r['recall']:6.3f} {r['F1']:6.3f} "
          f"{roc_auc_score(y, p):8.3f} {average_precision_score(y, p):7.3f}")
print(f"{'always negative':18s} {1 - sum(yva_bin) / len(yva_bin):6.3f}  0.000  0.000  0.000    0.500       --")

Read the table with these points in mind:

- **Accuracy hardly separates the models**, and the useless "always negative" rule scores about 98%.
- Reweighting (`pos_weight`) or resampling mostly **moves the operating point**. It raises recall at 0.5 and costs precision. The ranking quality, measured by AUC, often changes much less. Both are legitimate tools, and both mean that "probability ≥ 0.5" no longer corresponds to the true 2% base rate.
- The deeper lesson: **the threshold is a product decision, made on validation data.** For example: "we must catch 90% of bags; maximise precision subject to that."

In [ ]:
def threshold_for_recall(p, y, target_recall=0.9):
    # Highest threshold (hence best precision) whose recall on (p, y) is still >= target_recall.
    for t in torch.sort(p.unique(), descending=True).values.tolist():
        if binary_metrics(p, y, t)["recall"] >= target_recall:
            return t
    return 0.0

m = bin_models["plain BCE"]
p_va, y_va = predict_proba(m, bag_va)
t_star = threshold_for_recall(p_va, y_va, 0.9)                 # chosen on VALIDATION...
p_te, y_te = predict_proba(m, bag_te)
r = binary_metrics(p_te, y_te, t_star)                         # ...reported once on TEST
print(f"threshold chosen on validation: {t_star:.4f}")
print(f"test at that threshold: precision {r['precision']:.3f}, recall {r['recall']:.3f}, F1 {r['F1']:.3f}")
_ = record("Rare 'bag' detector: test PR-AUC (plain BCE)", average_precision_score(y_te, p_te))

The test recall lands *near* the 90% target rather than exactly on it: with only about 20 positives in the validation set, the threshold itself is a noisy estimate. In a real project you would want many more positives in validation, or you would report a confidence interval.

## 🧪 Exercises

**Exercise 7.1: a bigger MLP, and the bias–variance picture.** `★☆☆ · ~15 min + training` Train `MLP(hidden=(1024, 512), p_drop=0.0)` with `weight_decay=0` for 15 epochs, and plot its history next to `hist_mlp`. Then try `p_drop=0.3`. Describe, in terms of Tutorial §12.1 and §12.8, what changes in the train/validation gap and why.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
results = {}
for p in (0.0, 0.3):
    set_seed(0)
    big = MLP(hidden=(1024, 512), p_drop=p).to(device)
    o = torch.optim.AdamW(big.parameters(), lr=1e-3, weight_decay=0.0)
    results[p] = fit(big, train_loader, val_loader, loss_fn, o, epochs=n_epochs(15), device=device, verbose=False)
    print(f"p_drop={p}: final train acc {results[p]['train_acc'][-1]:.4f} | best val acc {max(results[p]['val_acc']):.4f} "
          f"| final val loss {results[p]['val_loss'][-1]:.4f}")
for p, h in results.items():
    plot_history(h, f"MLP 1024-512, dropout={p}")
# Without regularisation, the larger network drives training accuracy towards ~95%+ while the
# validation loss rises clearly after a few epochs: high variance (overfitting). Dropout 0.3 narrows
# the gap and usually gives an equal or better best-validation accuracy: we traded a little bias
# for a lot less variance.

**Exercise 7.2: binary classification end to end.** `★★☆ · ~30 min` Turn Fashion-MNIST into the binary task "is it footwear?" (Sandal = 5, Sneaker = 7, Ankle boot = 9). Wrap the datasets so that labels become `float32` tensors of shape `(1,)`, train a small MLP with **one** output and `BCEWithLogitsLoss` for 3 epochs, and report accuracy. You will need your own train and evaluate loop, because ours assumes `argmax` over classes. What accuracy does always predicting "not footwear" achieve?

In [ ]:
# ✍️ Your attempt here.
# Hint: The prediction is (logit > 0). About 30% of the images are footwear.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
class Footwear(Dataset):
    def __init__(self, base): self.base = base
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        return x, torch.tensor([float(y in (5, 7, 9))])      # float32, shape (1,)

bin_train = DataLoader(Footwear(fmnist_train), batch_size=128, shuffle=True, num_workers=NUM_WORKERS)
bin_val = DataLoader(Footwear(fmnist_val), batch_size=512, num_workers=NUM_WORKERS)
set_seed(0)
bin_model = MLP(hidden=(128, 64), out_dim=1).to(device)
bce = nn.BCEWithLogitsLoss()
o = torch.optim.AdamW(bin_model.parameters(), lr=1e-3)
for epoch in range(n_epochs(3)):
    bin_model.train()
    for x, y in bin_train:
        x, y = x.to(device), y.to(device)
        loss = bce(bin_model(x), y)
        o.zero_grad(); loss.backward(); o.step()
    bin_model.eval(); correct = total = pos = 0
    with torch.no_grad():
        for x, y in bin_val:
            x, y = x.to(device), y.to(device)
            correct += ((bin_model(x) > 0).float() == y).sum().item()
            total += y.numel(); pos += y.sum().item()
    print(f"epoch {epoch+1}: val acc {correct/total:.4f} | 'always no' baseline {1 - pos/total:.4f}")

### 📌 Key takeaways

- Weight batch statistics by batch size and accumulate with `.item()`.
- Baselines and confidence intervals give a number its meaning.
- With rare positives, accuracy is useless: report precision/recall/PR-AUC and choose thresholds on validation.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Your first-batch loss on a 100-class problem is 46.0. What does that suggest?

<details>
<summary>Answer</summary>

An untrained network should give about `ln 100 ≈ 4.6`. A loss of 46 suggests very large initial logits, for example from bad initialisation or unnormalised inputs (raw 0–255 pixels), or a wrong loss or label format. Investigate before training.

</details>

**2.** Why does `evaluate` call `model.eval()`, and why must `train_one_epoch` call `model.train()`?

<details>
<summary>Answer</summary>

Dropout and BatchNorm behave differently in the two modes (Tutorial §11.2). Evaluation must use the deterministic eval behaviour. Training must switch back each epoch, because the eval call left the model in eval mode.

</details>

**3.** Two models score 88.4% and 88.7% on a 10,000-image test set. Is the second one better?

<details>
<summary>Answer</summary>

Not demonstrably. The 95% interval for either number is about ±0.6 points. You would need repeated runs over several seeds, a larger test set, or a paired test on the same examples to support the claim.

</details>

---
# Chapter 8. Optimizers, schedules and regularisation in code

Tutorial §7, §10 and §12 gave you the theory of learning rates, momentum, Adam, AdamW, schedules and regularisers. This chapter is about using them correctly in PyTorch. That means knowing what the optimizer stores, how parameter groups work, where `scheduler.step()` goes, and how to find a learning rate in the first place.

## 8.1 What an optimizer object holds

An optimizer has two pieces of state:

- **`param_groups`**: a list of dicts, each holding some parameters and the hyper-parameters for them (`lr`, `weight_decay`, `betas`, and so on). Schedulers work by editing `lr` here.
- **`state`**: the per-parameter buffers. For SGD with momentum this is the velocity. For Adam it is the first and second moment estimates `m` and `v` of Tutorial §10.5.

🎯 **In this chapter you will learn to:** use optimizers, parameter groups and schedulers correctly; find a learning rate with a range test; add regularisation and early stopping.

In [ ]:
p = nn.Parameter(torch.tensor([1.0, -2.0]))
opt = torch.optim.Adam([p], lr=0.1, betas=(0.9, 0.999))
print("param_groups[0] keys:", sorted(opt.param_groups[0].keys()))

p.grad = torch.tensor([0.5, 0.5])
opt.step()
st = opt.state[p]
print("state keys:", list(st.keys()))
print("m after one step  = (1-β1)·g  :", st["exp_avg"])         # 0.1 * 0.5 = 0.05
print("v after one step  = (1-β2)·g² :", st["exp_avg_sq"])      # 0.001 * 0.25 = 0.00025
print("p moved by ≈ lr·sign(g) on step 1 (bias correction):", p.data)

The last line is Tutorial §10.5's bias correction in action. On step 1, `m̂ = g` and `v̂ = g²`, so the update is `lr · g/|g|`, which is `lr` in the direction of the gradient's sign. Each parameter moves by about `0.1` regardless of the gradient's scale.

## 8.2 The three optimizers you need

```python
torch.optim.SGD(params, lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)  # CNNs from scratch
torch.optim.Adam(params, lr=1e-3)                                                 # fast default, no decay
torch.optim.AdamW(params, lr=1e-3, weight_decay=0.01)                             # the modern default
```

**`Adam(weight_decay=λ)` is not the same as `AdamW(weight_decay=λ)`.** Adam adds `λ·w` to the gradient (L2 regularisation), and that term is then divided by `√v̂` like everything else. AdamW decays the weights directly, `w ← w − η·λ·w`, outside the adaptive scaling. This is the decoupling of Tutorial §10.6, and Exercise 8.3 makes the difference concrete. If you want weight decay with an adaptive optimizer, use AdamW. (Recent PyTorch versions also accept `Adam(..., decoupled_weight_decay=True)`, which behaves like AdamW. You saw the flag in the `param_groups` keys above.)

## 8.3 Parameter groups
Different parameters can have different hyper-parameters. Two uses are universal:

1. **No weight decay on biases and normalisation parameters.** Decaying a BatchNorm or LayerNorm gain `γ` towards 0 fights the normalisation, and decaying biases has no regularising benefit. A simple, standard rule is: decay tensors with `ndim ≥ 2` (weight matrices and conv kernels), and do not decay 1-D tensors.
2. **Different learning rates for different parts of the model.** For example, a smaller learning rate for a pretrained backbone than for a new head (Chapter 14).

In [ ]:
def param_groups_weight_decay(model, weight_decay):
    # Two groups: weight matrices/kernels get decay; biases and norm parameters do not.
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (decay if p.ndim >= 2 else no_decay).append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]

bn_mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 10))
groups = param_groups_weight_decay(bn_mlp, 0.05)
print("tensors per group (decay, no-decay):", end=" ")
print([(len(g["params"]), g["weight_decay"]) for g in groups])
opt = torch.optim.AdamW(groups, lr=1e-3)

# Hyper-parameters can be changed by hand at any time -- this is all a scheduler does:
for g in opt.param_groups:
    g["lr"] = 5e-4
print([g["lr"] for g in opt.param_groups])

## 8.4 Learning-rate schedules (Tutorial §10.8)

A scheduler wraps an optimizer and rewrites `param_groups[i]["lr"]` each time you call `scheduler.step()`. Three rules prevent nearly every scheduler bug:

1. Call `scheduler.step()` **after** `optimizer.step()`. PyTorch warns if you get the order wrong, and the first value of the schedule is then skipped.
2. Know whether your schedule counts **batches or epochs**. `OneCycleLR`, and warmup measured in steps, count batches. `StepLR` usually counts epochs. Either works as long as `T_max` or `total_steps` is in the same unit as your `step()` calls.
3. `ReduceLROnPlateau` is different: you call `scheduler.step(val_metric)` once per epoch.

Let's plot the common schedules by stepping a dummy optimizer:

In [ ]:
from torch.optim import lr_scheduler as S

def lr_trace(make_sched, steps=100, base_lr=0.1):
    p = nn.Parameter(torch.zeros(1))
    opt = torch.optim.SGD([p], lr=base_lr)
    sched = make_sched(opt)
    lrs = []
    for _ in range(steps):
        lrs.append(opt.param_groups[0]["lr"])
        opt.step()               # (no gradient here -- we only care about the lr)
        sched.step()
    return lrs

schedules = {
    "StepLR (÷10 every 30)":      lambda o: S.StepLR(o, step_size=30, gamma=0.1),
    "CosineAnnealingLR":          lambda o: S.CosineAnnealingLR(o, T_max=100),
    "Linear warmup → cosine":     lambda o: S.SequentialLR(o, [S.LinearLR(o, start_factor=0.01, total_iters=10),
                                                               S.CosineAnnealingLR(o, T_max=90)], milestones=[10]),
    "OneCycleLR":                 lambda o: S.OneCycleLR(o, max_lr=0.1, total_steps=100),
}
plt.figure(figsize=(8, 3.5))
for name, mk in schedules.items():
    plt.plot(lr_trace(mk), label=name)
plt.xlabel("step"); plt.ylabel("learning rate"); plt.legend(); plt.grid(alpha=0.3); plt.show()

`OneCycleLR` ignores the optimizer's initial `lr`. It starts at `max_lr / div_factor` (25 by default), warms up to `max_lr` over the first 30% of the steps, and then anneals with a cosine to `max_lr / (div_factor · final_div_factor)`. It also cycles momentum (or Adam's β₁) in the opposite direction.

`ReduceLROnPlateau` reacts to a metric instead of following a fixed curve:

In [ ]:
p = nn.Parameter(torch.zeros(1)); o = torch.optim.SGD([p], lr=0.1)
plateau = S.ReduceLROnPlateau(o, mode="min", factor=0.5, patience=2)
for epoch, val_loss in enumerate([1.0, 0.8, 0.7, 0.71, 0.70, 0.72, 0.69, 0.70, 0.71, 0.72]):
    plateau.step(val_loss)                       # pass the monitored metric
    print(f"epoch {epoch}: val_loss={val_loss:.2f} -> lr={o.param_groups[0]['lr']:.4f}")

## 8.5 Finding a learning rate: the LR range test (Tutorial §7.3)

Tutorial §7.3 described the procedure. Train for a few hundred mini-batches while increasing the learning rate **exponentially** from tiny to huge, record the (smoothed) loss, and pick a learning rate roughly one order of magnitude **below** the point where the loss is lowest. That point sits on the steep downward part of the curve. Here it is as a reusable function. It restores the model's weights afterwards, so the test does not count as training.

In [ ]:
def lr_range_test(model, loader, loss_fn, device, opt_fn=lambda params, lr: torch.optim.AdamW(params, lr=lr),
                  lr_min=1e-6, lr_max=1.0, num_steps=200, beta=0.98):
    saved = copy.deepcopy(model.state_dict())
    opt = opt_fn(model.parameters(), lr_min)
    gamma = (lr_max / lr_min) ** (1 / (num_steps - 1))        # multiply the lr by this every step
    lrs, losses, avg, best = [], [], 0.0, float("inf")
    model.train(); it = iter(loader)
    for step in range(num_steps):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader); x, y = next(it)
        lr = lr_min * gamma ** step
        for g in opt.param_groups:
            g["lr"] = lr
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        avg = beta * avg + (1 - beta) * loss.item()           # exponential moving average...
        smoothed = avg / (1 - beta ** (step + 1))             # ...with Adam-style bias correction
        lrs.append(lr); losses.append(smoothed)
        best = min(best, smoothed)
        if not math.isfinite(smoothed) or smoothed > 4 * best:    # diverged: stop early
            break
    model.load_state_dict(saved)                              # undo the test's training
    return lrs, losses

set_seed(0)
probe = MLP().to(device)
lrs, losses = lr_range_test(probe, train_loader, nn.CrossEntropyLoss(), device,
                            num_steps=50 if FAST_DEV_RUN else 300)
lr_at_min = lrs[int(np.argmin(losses))]
suggested_lr = lr_at_min / 10
plt.figure(figsize=(6, 3.5)); plt.semilogx(lrs, losses)
plt.axvline(lr_at_min, ls="--", c="gray", label=f"min at {lr_at_min:.1e}")
plt.axvline(suggested_lr, ls="--", c="green", label=f"suggested {suggested_lr:.1e}")
plt.xlabel("learning rate (log)"); plt.ylabel("smoothed training loss"); plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f"suggested AdamW lr ≈ {suggested_lr:.1e}")

For this MLP with AdamW you should see a suggestion in the low `1e-3` range, which agrees nicely with the "3e-4 to 1e-3 for Adam" folklore. The range test is a *starting point*, not an oracle. With `OneCycleLR`, a common choice is to use something close to the suggestion, or slightly above it, as `max_lr`.

## 8.6 Regularisation in code (Tutorial §12)

| Technique | In PyTorch |
| --- | --- |
| Weight decay | `AdamW(..., weight_decay=λ)`, with the parameter groups of §8.3 |
| Dropout | `nn.Dropout(p)` modules, switched off by `model.eval()` |
| Label smoothing | `nn.CrossEntropyLoss(label_smoothing=0.1)` |
| Data augmentation | `torchvision.transforms.v2` on the **training** set only (Chapter 13) |
| Gradient clipping | `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)` between `backward()` and `step()`. It returns the pre-clipping norm, which is worth logging |
| Early stopping | A small class that keeps the best weights, below |

> ⚠️ **Pitfall: label smoothing changes the loss scale.** With `label_smoothing=0.1`, even a perfect model has a loss well above 0, so the smoothed training loss cannot be compared with an unsmoothed validation loss. Monitor validation **accuracy**, or compute the validation loss **without** smoothing.

In [ ]:
class EarlyStopping:
    # Stops when the monitored metric has not improved for `patience` epochs,
    # and keeps a CPU copy of the best weights so they can be restored.
    def __init__(self, patience=3, min_delta=0.0, mode="max"):
        self.patience, self.min_delta, self.mode = patience, min_delta, mode
        self.best, self.best_epoch, self.bad_epochs, self.best_state = None, -1, 0, None

    def _improved(self, value):
        if self.best is None:
            return True
        return value > self.best + self.min_delta if self.mode == "max" else value < self.best - self.min_delta

    def step(self, value, model, epoch):
        # Call once per epoch. Returns True when training should stop.
        if self._improved(value):
            self.best, self.best_epoch, self.bad_epochs = value, epoch, 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore_best(self, model):
        model.load_state_dict(self.best_state)

Now we upgrade `fit` so that it supports per-batch, per-epoch and plateau schedulers, a separate evaluation loss, and early stopping. It stays backward compatible with Chapter 7.

In [ ]:
def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, device, scheduler=None,
        scheduler_interval="batch", max_grad_norm=None, early_stopping=None, eval_loss_fn=None, verbose=True):
    # scheduler_interval: "batch" (e.g. OneCycleLR), "epoch" (e.g. StepLR) or "plateau" (ReduceLROnPlateau).
    eval_loss_fn = eval_loss_fn or loss_fn
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        batch_sched = scheduler if scheduler_interval == "batch" else None
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device, batch_sched, max_grad_norm)
        va_loss, va_acc = evaluate(model, val_loader, eval_loss_fn, device)
        if scheduler is not None and scheduler_interval == "epoch":
            scheduler.step()
        elif scheduler is not None and scheduler_interval == "plateau":
            scheduler.step(va_loss)
        for k, v in zip(("train_loss", "train_acc", "val_loss", "val_acc"), (tr_loss, tr_acc, va_loss, va_acc)):
            hist[k].append(v)
        hist["lr"].append(optimizer.param_groups[0]["lr"])
        if verbose:
            print(f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} | lr {hist['lr'][-1]:.2e} | {time.perf_counter() - t0:.1f}s")
        if early_stopping is not None and early_stopping.step(va_acc, model, epoch):
            if verbose:
                print(f"early stopping: no improvement for {early_stopping.patience} epochs "
                      f"(best val acc {early_stopping.best:.4f} at epoch {early_stopping.best_epoch})")
            break
    if early_stopping is not None and early_stopping.best_state is not None:
        early_stopping.restore_best(model)
    return hist

## 8.7 Putting it together: the modern setup of Tutorial §10.9

AdamW with decoupled decay that excludes 1-D parameters, OneCycle stepped per batch with its peak set to the range test's suggestion (OneCycle overwrites the optimizer's `lr` from its first step, so we do not pass one), label smoothing, gradient clipping, and early stopping on validation accuracy:

In [ ]:
set_seed(0)
mlp2 = MLP(hidden=(512, 256), p_drop=0.2).to(device)
EPOCHS = n_epochs(20)
opt = torch.optim.AdamW(param_groups_weight_decay(mlp2, 0.05))    # no lr here: OneCycle sets it
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=suggested_lr, epochs=EPOCHS,   # the LR finder's value
                                            steps_per_epoch=len(train_loader))
hist_modern = fit(mlp2, train_loader, val_loader,
                  loss_fn=nn.CrossEntropyLoss(label_smoothing=0.1),
                  eval_loss_fn=nn.CrossEntropyLoss(),           # unsmoothed, so val loss stays interpretable
                  optimizer=opt, epochs=EPOCHS, device=device,
                  scheduler=sched, scheduler_interval="batch", max_grad_norm=1.0,
                  early_stopping=EarlyStopping(patience=5, mode="max"))
plot_history(hist_modern, "MLP with the modern setup")
_ = record("Fashion-MNIST MLP, modern setup: best val acc", max(hist_modern["val_acc"]))

You should see validation accuracy edge up to around 89–90%. That is a small but typical gain over Chapter 7, and the curves are smoother: the cosine tail of OneCycle anneals into a good minimum. An MLP on 784 raw pixels is close to its ceiling here. The next big jump needs an architecture that understands images (Chapter 12).

> ⚠️ **One subtlety.** When early stopping fires before the last epoch, a OneCycle schedule never reaches its annealing phase. With OneCycle, early stopping is mostly a safety net, and the "best epoch" is usually near the end. With a plateau scheduler, early stopping is the natural partner.

## 🧪 Exercises

**Exercise 8.1: warmup + cosine with `LambdaLR`.** `★★☆ · ~20 min` `LambdaLR(opt, lr_lambda)` sets `lr = base_lr × lr_lambda(step)`. Write the `lr_lambda` that reproduces the "Linear warmup → cosine" curve of §8.4 (start factor 0.01, 10 warmup steps, then a cosine over 90 steps), and check that it matches `lr_trace` of the `SequentialLR` version to within `1e-6`. Many codebases use exactly this kind of lambda, so be able to write it. Name your function `warmup_cosine(step)`. The 🔎 self-check after your attempt tests it.

In [ ]:
# ✍️ Your attempt here.
# Hint: For step k < 10: factor = 0.01 + 0.99·k/10. Afterwards: 0.5·(1 + cos(π·(k−10)/90)).

In [ ]:
# 🔎 Self-check for Exercise 8.1: tests YOUR `warmup_cosine` (run after your attempt)
def _test(fn):
    a = lr_trace(lambda o: S.LambdaLR(o, fn))
    b = lr_trace(schedules['Linear warmup → cosine'])
    d = max(abs(x - y) for x, y in zip(a, b))
    assert d < 1e-6, f'max difference {d:.2e}'

self_check("warmup_cosine", _test)

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def warmup_cosine(step, warmup=10, total=100, start=0.01):
    if step < warmup:
        return start + (1 - start) * step / warmup
    t = (step - warmup) / (total - warmup)
    return 0.5 * (1 + math.cos(math.pi * t))

a = lr_trace(lambda o: S.LambdaLR(o, warmup_cosine))
b = lr_trace(schedules["Linear warmup → cosine"])
print("max difference:", max(abs(x - y) for x, y in zip(a, b)))

**Exercise 8.2: optimizer shoot-out.** `★☆☆ · ~10 min + training` With the same seed and 5 epochs each, train `MLP()` on Fashion-MNIST with (a) plain SGD at `lr=0.1`, (b) SGD with `momentum=0.9, nesterov=True` at `lr=0.05`, (c) Adam at `1e-3`, and (d) AdamW at `1e-3` with `weight_decay=0.05`. Report the best validation accuracy of each. Then connect what you see to Tutorial §10.2–10.7.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
configs = {
    "SGD":            lambda ps: torch.optim.SGD(ps, lr=0.1),
    "SGD+Nesterov":   lambda ps: torch.optim.SGD(ps, lr=0.05, momentum=0.9, nesterov=True),
    "Adam":           lambda ps: torch.optim.Adam(ps, lr=1e-3),
    "AdamW":          lambda ps: torch.optim.AdamW(ps, lr=1e-3, weight_decay=0.05),
}
for name, make_opt in configs.items():
    set_seed(0)
    m = MLP().to(device)
    h = fit(m, train_loader, val_loader, nn.CrossEntropyLoss(), make_opt(m.parameters()),
            epochs=n_epochs(5), device=device, verbose=False)
    print(f"{name:14s} best val acc {max(h['val_acc']):.4f}")
# Typical outcome: all four land within about a percentage point of each other after 5 epochs.
# Momentum speeds up plain SGD; Adam-family optimizers get there fastest with no tuning.
# Differences this small are within seed noise -- rerun with 2-3 seeds before concluding anything.

**Exercise 8.3: Adam's L2 vs AdamW's decoupled decay.** `★★☆ · ~15 min` Take a single parameter `w = 1.0` whose loss gradient is exactly zero, so the *only* force on it is weight decay. Run one step with `lr=0.1, weight_decay=0.01` using (a) `torch.optim.Adam` and (b) `torch.optim.AdamW`, and print `w` after each. Explain the enormous difference using Tutorial §10.6.

In [ ]:
# ✍️ Your attempt here.
# Hint: Set p.grad = torch.zeros(1) before each step.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
for Opt in (torch.optim.Adam, torch.optim.AdamW):
    w = nn.Parameter(torch.tensor([1.0]))
    o = Opt([w], lr=0.1, weight_decay=0.01)
    w.grad = torch.zeros(1)
    o.step()
    print(f"{Opt.__name__:5s}: w = {w.item():.4f}")
# Adam : the decay term λw = 0.01 is ADDED to the gradient and then normalised by sqrt(v̂) = |0.01|,
#        so the step is lr * 0.01/0.01 = 0.1 -> w = 0.9. The "decay" acts like a sign-step: its strength
#        depends on the gradient's scale, not on λ.
# AdamW: w <- w - lr*λ*w = 1 - 0.1*0.01 = 0.999, and the Adam step itself is zero.
#        The decay is exactly the λ you asked for, independent of the adaptive scaling.

### 📌 Key takeaways

- Schedulers edit `param_groups[i]['lr']`; step them after the optimizer, in the unit they were defined in.
- No weight decay on biases and normalisation parameters.
- The LR range test gives a starting point, not an oracle.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** You use `OneCycleLR(..., epochs=20, steps_per_epoch=len(train_loader))` but call `scheduler.step()` once per epoch. What happens?

<details>
<summary>Answer</summary>

The schedule is defined over 20 × len(loader) steps but advances only 20 times. The learning rate barely leaves its tiny starting value (`max_lr/25`) for the whole run, so training is far too slow. OneCycle must be stepped every batch.

</details>

**2.** Why exclude biases and normalisation gains from weight decay?

<details>
<summary>Answer</summary>

Decay pulls parameters toward zero. For a BatchNorm or LayerNorm gain `γ`, that fights the normalisation's purpose of learning the right scale. For biases there is no regularising benefit. Only weight matrices and conv kernels benefit from decay.

</details>

**3.** Your LR range test shows the loss falling until 3e-2 and exploding after 1e-1. Which learning rate do you start with?

<details>
<summary>Answer</summary>

About `3e-3`, one order of magnitude below the minimum, on the steep downward part of the curve (Tutorial §7.3). With OneCycle you might use a value near that as `max_lr`.

</details>

**4.** What must happen between `loss.backward()` and `optimizer.step()` if you clip gradients, and why there?

<details>
<summary>Answer</summary>

`clip_grad_norm_` must run after the gradients exist (after `backward`) and before they are used (before `step`). With mixed precision you must also `unscale_` first (Chapter 11).

</details>

---
# Part III — Engineering discipline

# Chapter 9. Checkpointing and reproducibility

Training runs crash, Colab sessions disconnect, and preemptible cloud GPUs are taken away mid-epoch. Professional training code therefore **saves enough state to resume exactly where it stopped**, and runs can be **reproduced** when someone asks "how did you get that number?".

## 9.1 Saving and loading weights

The recommended format is the **`state_dict`**: a dictionary of tensors. You rebuild the architecture in code and load the weights into it.

🎯 **In this chapter you will learn to:** save, load and resume training exactly; load checkpoints safely; know what reproducibility can and cannot guarantee.

In [ ]:
path = ARTIFACT_DIR / "mlp_fmnist.pt"
torch.save(mlp.state_dict(), path)                       # weights only

restored = MLP().to(device)
restored.load_state_dict(torch.load(path, map_location=device, weights_only=True))
print("restored val acc:", evaluate(restored, val_loader, nn.CrossEntropyLoss(), device)[1])

Three details:

- **`map_location`** puts the tensors on the device you name, whichever device they were saved from. Without it, a checkpoint saved on a GPU cannot be loaded on a CPU-only machine.
- **`weights_only=True`** has been the default since PyTorch 2.6. `torch.save` uses Python's `pickle`, and unpickling an untrusted file **can execute arbitrary code**. With `weights_only=True`, the loader accepts only tensors, primitive types, and containers of them. Keep it on, and load files with `weights_only=False` only when you completely trust their source.
- **Why not `torch.save(model)`?** That pickles the model *object*, which stores references to your class and module path. Rename a file or refactor a class, and the checkpoint no longer loads. A state dict depends only on parameter names.

🏭 **Industry note: `safetensors`.** For sharing weights publicly (it is the default format on the Hugging Face Hub), many teams use the `safetensors` format instead of `torch.save`. It stores *only* tensors, with no pickle, so loading cannot execute code, and files can be memory-mapped for fast loading. It is a separate package (`pip install safetensors`):

```python
from safetensors.torch import save_file, load_file
save_file(model.state_dict(), "model.safetensors")
model.load_state_dict(load_file("model.safetensors"))
```

`save_file` refuses tensors that share memory, such as tied embedding and output weights. Use `safetensors.torch.save_model(model, path)` for those. For *training* checkpoints, with their optimizer state and Python metadata, `torch.save` plus `weights_only=True` loading remains the norm.

## 9.2 A full training checkpoint

To *resume* training, rather than only run inference, you also need the optimizer state (for example Adam's `m` and `v`), the scheduler state, the epoch counter, the best metric so far, and the RNG states.

In [ ]:
def rng_state():
    return {"torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "python": random.getstate()}

def set_rng_state(state):
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state["cuda"]:
        torch.cuda.set_rng_state_all(state["cuda"])
    random.setstate(state["python"])

def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None, config=None, **extra):
    ckpt = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch, "best_metric": best_metric,
        "config": config or {}, "rng": rng_state(),
        "torch_version": str(torch.__version__), **extra,   # str(): see the note below
    }
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(ckpt, tmp)
    os.replace(tmp, path)       # atomic rename: a crash mid-save never leaves a corrupt checkpoint

def load_checkpoint(path, model, optimizer=None, scheduler=None, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location, weights_only=True)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and ckpt["scheduler"] is not None:
        scheduler.load_state_dict(ckpt["scheduler"])
    set_rng_state(ckpt["rng"])
    return ckpt

> ⚠️ **What does not survive `weights_only=True`.** Tensors, numbers, strings, lists, tuples and dicts are fine. NumPy arrays and arbitrary Python objects are not. That is why we store the torch and Python RNG states, but not `np.random.get_state()` (which contains a NumPy array). Convert anything else to tensors or plain Python types before saving. Even `torch.__version__` is a trap: it is a `TorchVersion` object (a `str` subclass), not a plain string, and storing it unconverted makes the checkpoint unloadable with `weights_only=True`. That is why `save_checkpoint` wraps it in `str()`. The failure only shows up when you *load*, which is why you should test loading right after you write a checkpoint function.

## 9.3 Proof that resuming is exact

A good test of a checkpointing system is this: **training for 6 epochs straight must give bit-for-bit the same weights as training 3 epochs, saving, rebuilding everything from scratch, loading, and training 3 more.** For that to hold, we must also restore the state of the DataLoader's shuffling generator and the global RNG, which is used by dropout. We run the test on the CPU, where the arithmetic is deterministic.

In [ ]:
g0 = torch.Generator().manual_seed(42)
Xr, yr = torch.randn(512, 20, generator=g0), torch.randint(0, 3, (512,), generator=g0)   # fixed toy data

def make_run(seed):
    # Build every training object from scratch, as a fresh process would.
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 3))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=6)
    gen = torch.Generator().manual_seed(seed)                       # controls the shuffle order
    loader = DataLoader(TensorDataset(Xr, yr), batch_size=32, shuffle=True, generator=gen)
    return model, opt, sched, loader, gen

def run_epochs(model, opt, sched, loader, n):
    for _ in range(n):
        model.train()
        for xb, yb in loader:
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()

# A: six epochs, uninterrupted
model_a, opt_a, sched_a, loader_a, _ = make_run(seed=0)
run_epochs(model_a, opt_a, sched_a, loader_a, 6)

# B: three epochs, save the checkpoint (including the shuffle generator's state), then "crash"
model_b, opt_b, sched_b, loader_b, gen_b = make_run(seed=0)
run_epochs(model_b, opt_b, sched_b, loader_b, 3)
save_checkpoint(ARTIFACT_DIR / "resume_test.pt", model_b, opt_b, sched_b, epoch=3, loader_gen=gen_b.get_state())
del model_b, opt_b, sched_b, loader_b, gen_b

# C: a fresh "process" -- deliberately built with a DIFFERENT seed -- loads the checkpoint and continues
model_c, opt_c, sched_c, loader_c, gen_c = make_run(seed=123)
ckpt = load_checkpoint(ARTIFACT_DIR / "resume_test.pt", model_c, opt_c, sched_c)
gen_c.set_state(ckpt["loader_gen"])
run_epochs(model_c, opt_c, sched_c, loader_c, 3)

same = all(torch.equal(p, q) for p, q in zip(model_a.state_dict().values(), model_c.state_dict().values()))
print("resumed run identical to uninterrupted run:", same)

If you remove any one restored piece (the optimizer state, the scheduler, the RNG state, or the loader generator), the final weights differ. Try it. In particular, **restarting Adam with an empty state** produces a visible jump in the loss right after resuming, because its moment estimates restart from zero.

Run C was deliberately built with the wrong seed, and it still matches exactly. Everything that matters was restored from the checkpoint, not from the seed.

## 9.4 Reproducibility: what you can and cannot guarantee

| Level | How | Cost |
| --- | --- | --- |
| Same results run to run, same machine, CPU | Seed everything (`set_seed`) and seed the DataLoader `generator` | Free |
| Same results on GPU | Also `torch.use_deterministic_algorithms(True)`, `torch.backends.cudnn.benchmark = False`, and the environment variable `CUBLAS_WORKSPACE_CONFIG=:4096:8` | Often 10–30% slower, and some operations raise an error because they have no deterministic implementation |
| Same results across GPU models, driver or PyTorch versions | **Not guaranteed** | — |

DataLoader workers are handled for you. Each worker seeds its `torch`, `random` and NumPy RNGs from a base seed, which is drawn from the loader's `generator`. Seed the generator, and multi-worker loading is reproducible too.

In [ ]:
def make_deterministic(seed=0):
    # Maximum GPU determinism for debugging or auditing. Not needed (and slower) for normal training.
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # must be set before the first cuBLAS call to take effect
    set_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)   # warn_only: warn instead of raising

# (Defined, not called: we want speed for the rest of the notebook.)

🏭 **Industry note: variance is part of the result.** Bit-exact reproducibility is a debugging tool. Scientific and engineering honesty needs something else: knowing how much your result **varies across seeds**. Report `mean ± std` over 3–5 seeds whenever you compare methods. A 0.3-point "improvement" that disappears under a different seed was never real.

## 🧪 Exercises

**Exercise 9.1: break resume on purpose.** `★★☆ · ~15 min` Repeat the experiment of §9.3, but do *not* load the optimizer state (pass `optimizer=None` to `load_checkpoint`). Print the largest absolute difference between the weights of the two runs. Then restore the optimizer but skip `gen_c.set_state(...)`. Which omission changes the weights more after three epochs, and why?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def resumed_weights(load_opt=True, restore_gen=True):
    m, o, s, ld, g = make_run(seed=123)
    ck = load_checkpoint(ARTIFACT_DIR / "resume_test.pt", m, o if load_opt else None, s)
    if restore_gen:
        g.set_state(ck["loader_gen"])
    run_epochs(m, o, s, ld, 3)
    return m.state_dict()

ref = model_a.state_dict()
for kw in ({"load_opt": False}, {"restore_gen": False}):
    sd = resumed_weights(**kw)
    diff = max((sd[k] - ref[k]).abs().max().item() for k in ref)
    print(kw, f"max |Δw| = {diff:.4f}")
# Both break exactness. Skipping the optimizer state typically changes the weights more: Adam
# restarts with m = v = 0, so its first steps after resuming have size ~lr per parameter (bias
# correction), a jolt to every weight. A different shuffle order only changes which mini-batches
# are seen when -- a smaller, noise-like perturbation.

**Exercise 9.2: seed variance.** `★☆☆ · ~10 min + training` Train `MLP()` on Fashion-MNIST for 3 epochs with seeds 0, 1 and 2 (same hyper-parameters as Chapter 7), and report the best validation accuracy as `mean ± std`. How does the spread compare with the ±0.6-point confidence interval of §7.5?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
accs = []
for seed in (0, 1, 2):
    set_seed(seed)
    m = MLP().to(device)
    h = fit(m, train_loader, val_loader, nn.CrossEntropyLoss(),
            torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-2),
            epochs=n_epochs(3), device=device, verbose=False)
    accs.append(max(h["val_acc"]))
accs = torch.tensor(accs)
print(f"best val acc over seeds: {accs.mean():.4f} ± {accs.std():.4f}   ({accs.tolist()})")
# Typically a few tenths of a point -- the same order as the test-set sampling error. Both sources
# of noise must be smaller than the effect you claim. (Note: the validation set here has only 5,000
# images, so its own sampling error is ~±0.9 points at 95%.)

### 📌 Key takeaways

- Exact resume needs model, optimizer, scheduler, RNG and loader-generator state.
- Keep `weights_only=True`; store only tensors and plain Python types.
- Report variance across seeds, not a single run.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Name everything a training checkpoint needs in order to resume *exactly*.

<details>
<summary>Answer</summary>

Model state dict, optimizer state dict, scheduler state dict, epoch or step counter, best metric (for early stopping or best-model saving), RNG states (torch, CUDA, Python), and the DataLoader generator state. The config is also needed to rebuild the objects.

</details>

**2.** Why is loading checkpoints from the internet with `weights_only=False` dangerous?

<details>
<summary>Answer</summary>

Unpickling can execute arbitrary code embedded in the file. `weights_only=True` restricts loading to tensors and plain data types.

</details>

**3.** You set every seed, yet two GPU runs give slightly different losses. Is something broken?

<details>
<summary>Answer</summary>

Not necessarily. Many CUDA kernels (atomic additions in some backward passes, and cuDNN's algorithm choice when `benchmark=True`) are non-deterministic. Enable deterministic algorithms if you need exactness, and otherwise treat the variation as noise.

</details>

---
# Chapter 10. Debugging training

The hardest neural-network bugs **do not raise errors**. The code runs, the loss goes down a little, and the accuracy is merely disappointing. You then cannot tell whether the idea is bad or the implementation is broken. The cure is a systematic protocol that rules out implementation bugs *before* you start tuning.

## 10.1 The protocol

Run these checks, in order, on every new model and pipeline:

1. **Look at the data after the pipeline.** Plot a batch exactly as the model receives it, with its labels. Check the value ranges, the dtypes and the label distribution.
2. **Check the initial loss.** It should be about `ln C` for C balanced classes (§7.3).
3. **Overfit a single batch.** A correct model and loop can drive the loss on one fixed batch to roughly zero. If it cannot, there is a bug, full stop.
4. **Check gradient flow.** Every parameter should receive a non-`None`, finite, non-zero gradient, and the per-layer norms should be within a few orders of magnitude of each other.
5. **Run a learning-rate range test** (§8.5).
6. **Compare with a baseline.** Is the model beating something simple?
7. **Only now tune** the architecture and the regularisation.

## 10.2 Overfitting one batch

🎯 **In this chapter you will learn to:** apply a systematic debugging protocol; diagnose symptoms from loss curves and gradients; find silent bugs in real code.

In [ ]:
def overfit_one_batch(model, batch, loss_fn, steps=200, lr=1e-3):
    # A correct model and loop should drive the training loss on ONE batch to ~0.
    x, y = (t.to(device) for t in batch)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for step in range(steps):
        logits = model(x)
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 50 == 0 or step == steps - 1:
            acc = (logits.argmax(1) == y).float().mean().item()
            print(f"step {step:3d}: loss {loss.item():.4f}  batch acc {acc:.2f}")
    return loss.item()

one_batch = next(iter(DataLoader(fmnist_train, batch_size=64, shuffle=True)))
set_seed(0)
final = overfit_one_batch(MLP(p_drop=0.0).to(device), one_batch, nn.CrossEntropyLoss())

The loss falls to nearly 0 and the batch accuracy reaches 1.00. Note that we turned **dropout off**. Regularisation is supposed to prevent memorisation, so disable it for this test.

## 10.3 Checking gradient flow

In [ ]:
def gradient_report(model):
    # Call after loss.backward(). Flags parameters with missing, non-finite or zero gradients.
    problems = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.grad is None:
            problems.append(f"{name}: grad is None (unused in forward, or graph cut by .detach()/.item()?)")
        elif not torch.isfinite(p.grad).all():
            problems.append(f"{name}: non-finite gradient")
        elif p.grad.abs().max() == 0:
            problems.append(f"{name}: gradient exactly zero (dead ReLUs? frozen upstream?)")
        else:
            print(f"{name:14s} |grad| = {p.grad.norm():.2e}   |w| = {p.norm():.2e}   ratio = {p.grad.norm() / p.norm():.1e}")
    for msg in problems:
        print("⚠️", msg)

class HasUnusedLayer(MLP):
    def __init__(self):
        super().__init__()
        self.extra = nn.Linear(10, 10)          # defined but never used in forward

m = HasUnusedLayer().to(device)
x, y = (t.to(device) for t in one_batch)
F.cross_entropy(m(x), y).backward()
gradient_report(m)

The ratio `‖∇w‖ / ‖w‖` is useful. Multiplied by the learning rate, it approximates the relative size of each update. Updates of about `1e-3` of the weight per step are healthy. Values of `1e-1` or more suggest instability, and `1e-7` suggests learning has stalled.

## 10.4 Hunting NaNs
A `nan` loss is usually caused by one of five things: a learning rate that is too high, `log(0)` or `sqrt(0)` in a hand-written loss, division by zero in normalisation code, fp16 overflow (Chapter 11), or bad input data (NaNs in the dataset). `torch.autograd.detect_anomaly()` makes the backward pass report **which operation** first produced a non-finite gradient. It is slow, so use it only while debugging.

In [ ]:
x = torch.tensor([0.0, 1.0], requires_grad=True)
y = torch.sqrt(x).sum()                 # d sqrt(x)/dx = 1/(2 sqrt(x)) -> inf at 0
y.backward()
print("grad without anomaly mode:", x.grad)     # inf: will turn into NaNs downstream

x.grad = None
try:
    with torch.autograd.detect_anomaly():
        z = (torch.sqrt(x) * 0).sum()   # inf * 0 = nan in the backward pass
        z.backward()
except RuntimeError as e:
    print("detect_anomaly:", str(e).splitlines()[0])

In production training loops, guard cheaply with `if not torch.isfinite(loss): ...` (skip the batch, log it, or stop) rather than letting one bad batch poison the weights.

## 10.5 A field guide to symptoms

| Symptom | Most likely causes |
| --- | --- |
| Loss is `nan` after a few steps | LR too high; `log(0)` in a custom loss; fp16 overflow without a scaler; NaNs in the data |
| Loss stuck at `ln C`, accuracy at chance | Labels shuffled independently of inputs; LR far too low or far too high; dead network (bad init); the optimizer holds the wrong parameters |
| Training loss decreases very slowly and erratically | `zero_grad()` missing; LR too low; softmax applied before `CrossEntropyLoss` |
| Loss oscillates by class, or the model predicts one class for a while and then another | Training data **not shuffled** (sorted by label) |
| Validation much worse than training from epoch 1 | Different preprocessing for train and validation; augmentation applied to validation; `model.eval()` missing |
| Validation accuracy noisy from run to run on the same model | Dropout active during evaluation (`eval()` missing) |
| Memory grows every step until OOM | Accumulating `loss` instead of `loss.item()`; storing tensors with history in lists |
| Fine on CPU, `device-side assert` on GPU | Label out of range; indexing out of bounds |
| Great validation accuracy, poor in production | Leakage between splits; train/serve preprocessing mismatch; distribution shift |

## 🧪 Exercise 10.1: the buggy training script `★★★ · ~40 min`

The function below trains an MLP on a **label-sorted** subset of Fashion-MNIST. It runs without any error, but it contains **at least six bugs**: five that damage learning or evaluation, and one that wastes memory and compute. Find them all, write a fixed version, and compare the validation accuracy of the two after 2 epochs. Read the code carefully *before* running it, and predict its accuracy.

In [ ]:
# A label-sorted 20k subset of the training split (sorting by label makes one of the bugs bite)
_sub_idx = torch.arange(min(20_000, len(fmnist_train)))
_sub_labels = torch.tensor([fmnist_train[i][1] for i in _sub_idx.tolist()]) if FAST_DEV_RUN else \
              torch.as_tensor(full_train.targets)[torch.as_tensor(fmnist_train.indices)][:20_000]
sorted_train = Subset(fmnist_train, _sub_idx[torch.argsort(_sub_labels, stable=True)].tolist())

def buggy_training(epochs=2):
    set_seed(0)
    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
                          nn.Linear(256, 10), nn.Softmax(dim=1)).to(device)
    loader = DataLoader(sorted_train, batch_size=128, shuffle=False)
    vloader = DataLoader(fmnist_val, batch_size=512)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        running = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            running += loss
        model.eval()
        correct = 0
        for x, y in vloader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
        history.append(correct / len(fmnist_val))
        print(f"epoch {epoch+1}: train loss {running.item() / len(loader):.3f} | val acc {history[-1]:.4f}")
    return history

_ = buggy_training(n_epochs(2))

In [ ]:
# ✍️ Your attempt here.
# Hint: Compare every line with train_one_epoch/evaluate of §7.1 and with the field guide in §10.5.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
# The bugs:
# 1. Softmax at the end of the model + CrossEntropyLoss -> double softmax (§5.2).
# 2. shuffle=False on label-sorted data -> each stretch of batches contains a single class, so the
#    model chases one class after another (catastrophic interference).
# 3. optimizer.zero_grad() missing -> gradients accumulate across all steps (§2.3).
# 4. model.eval() is called after epoch 1 but model.train() is never called again -> dropout is off
#    for the remaining epochs (a silent change of the training procedure).
# 5. `running += loss` accumulates a TENSOR -> keeps every step's autograd graph alive (memory leak).
# 6. Evaluation without torch.no_grad() -> builds graphs for nothing (wasted memory/compute).
# (Also worth fixing: the loss is averaged over batches, not examples -- minor.)

def fixed_training(epochs=2):
    set_seed(0)
    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
                          nn.Linear(256, 10)).to(device)                       # (1) logits out
    loader = DataLoader(sorted_train, batch_size=128, shuffle=True)            # (2) shuffle
    vloader = DataLoader(fmnist_val, batch_size=512)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        model.train()                                                          # (4)
        running, n = 0.0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)                                    # (3)
            loss.backward()
            opt.step()
            running += loss.item() * y.size(0); n += y.size(0)                 # (5)
        model.eval()
        correct = 0
        with torch.no_grad():                                                  # (6)
            for x, y in vloader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
        history.append(correct / len(fmnist_val))
        print(f"epoch {epoch+1}: train loss {running / n:.3f} | val acc {history[-1]:.4f}")
    return history

_ = fixed_training(n_epochs(2))
# Expect the buggy version far below the fixed one (often by tens of points); the fixed one
# reaches ~85% after 2 epochs on 20k images.

### 📌 Key takeaways

- Look at data, check the initial loss, overfit one batch, check gradients, then tune.
- Most damaging bugs raise no error.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Your model cannot overfit a single batch of 32 examples. List three possible causes.

<details>
<summary>Answer</summary>

A bug in the loop (missing `zero_grad`, the optimizer holding the wrong parameters, no `step()`); a broken loss or label contract (double softmax, wrong target format); a model that is too constrained (heavy dropout or weight decay left on, a bottleneck layer, frozen parameters); or a learning rate far too low or high.

</details>

**2.** A parameter's `.grad` is `None` after `backward()`. What does that tell you?

<details>
<summary>Answer</summary>

The parameter did not take part in computing the loss. It is unused in `forward`, or the graph was cut on its path by `.detach()`, `.item()`, `torch.no_grad()`, or a conversion to NumPy.

</details>

**3.** Validation accuracy varies by ±1% when you evaluate the *same* checkpoint twice. Why?

<details>
<summary>Answer</summary>

Something stochastic is active during evaluation. Most often `model.eval()` is missing, so dropout is on (and BatchNorm uses batch statistics). Random augmentation applied to the validation set is another cause.

</details>

---
# Chapter 11. Performance: making training fast

GPU time is money, and fast iteration is how you find good models. This chapter covers the tools that matter most in practice, in the order you should reach for them:

1. **Measure** (§11.1), because intuition about bottlenecks is usually wrong.
2. **Feed the GPU** (§11.2), because a starved GPU makes every other optimisation pointless.
3. **Mixed precision** (§11.3), typically a 2–3× speed-up for one-time setup.
4. **`torch.compile`** (§11.5), often 1.2–2× more, at the cost of compile time.
5. **Profile** (§11.6) to see where the time actually goes.
6. **Manage memory** (§11.7) when the model or batch does not fit.

## 11.1 Measuring correctly

🎯 **In this chapter you will learn to:** measure before optimising; use mixed precision, `torch.compile` and the profiler; estimate memory and train when it runs out.

In [ ]:
def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

def benchmark(fn, warmup=5, iters=20):
    # Average wall-clock seconds per call of fn(), with warm-up and GPU synchronisation (§1.8).
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters

## 11.2 Feeding the GPU: keep small datasets on the device

Fashion-MNIST is 55,000 × 784 floats, about 170 MB as `float32`. That fits comfortably in a T4's 15 GB. Running each image through PIL and the transform pipeline on every epoch is pure overhead. For datasets that fit in GPU memory and need no per-sample random augmentation, **materialise the tensors once, move them to the GPU, and slice batches directly**:

In [ ]:
def materialize(ds, batch_size=2048):
    # Run a dataset's transforms once and stack everything into two tensors.
    xs, ys = [], []
    for x, y in DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS):
        xs.append(x); ys.append(y)
    return torch.cat(xs), torch.cat(ys)

class TensorLoader:
    # A minimal DataLoader replacement for tensors that already live on the target device.
    def __init__(self, X, y, batch_size, shuffle=False, drop_last=False, seed=0):
        self.X, self.y, self.bs, self.shuffle, self.drop_last = X, y, batch_size, shuffle, drop_last
        self.gen = torch.Generator(device=X.device).manual_seed(seed)
    def __len__(self):
        n = len(self.X)
        return n // self.bs if self.drop_last else math.ceil(n / self.bs)
    def __iter__(self):
        n = len(self.X)
        idx = torch.randperm(n, device=self.X.device, generator=self.gen) if self.shuffle else torch.arange(n, device=self.X.device)
        for i in range(len(self)):
            j = idx[i * self.bs:(i + 1) * self.bs]
            yield self.X[j], self.y[j]

t0 = time.perf_counter()
Xtr, ytr = (t.to(device) for t in materialize(fmnist_train))
Xva, yva = (t.to(device) for t in materialize(fmnist_val))
print(f"materialised in {time.perf_counter() - t0:.1f}s:", tuple(Xtr.shape), Xtr.device)
fast_train = TensorLoader(Xtr, ytr, batch_size=128, shuffle=True)
fast_val = TensorLoader(Xva, yva, batch_size=512)

for name, ld in [("torch DataLoader", train_loader), ("GPU TensorLoader", fast_train)]:
    set_seed(0)
    m = MLP().to(device)
    o = torch.optim.AdamW(m.parameters(), lr=1e-3)
    t0 = time.perf_counter()
    train_one_epoch(m, ld, nn.CrossEntropyLoss(), o, device)
    sync()
    print(f"{name:18s}: {record(f'Fashion-MNIST MLP epoch time, {name} (s)', time.perf_counter() - t0):5.2f} s per epoch")

On a T4 the GPU-resident loader is usually several times faster for this small MLP (the printout above is your measurement), because the standard pipeline was the bottleneck all along. The same lesson applies at any scale: profile the input pipeline first. For large image datasets with augmentation, the corresponding tools are more workers, faster decoding, and doing the augmentation on the GPU in batches.

## 11.3 Mixed precision (AMP)

Modern GPUs have **tensor cores** that multiply matrices in 16-bit precision many times faster than in 32-bit. On a T4, the peak fp16 tensor-core throughput is about 8× the fp32 throughput. **Automatic mixed precision** uses them safely:

- `torch.autocast(device_type, dtype)` runs each operation in the precision that suits it. Matmuls and convolutions run in fp16 or bf16. Precision-sensitive operations (softmax, losses, normalisation statistics, large reductions) stay in fp32.
- The **parameters stay in fp32**. Gradients are computed through the fp16 operations and accumulated into fp32 weights, so small updates are not lost to rounding.
- **fp16 only:** a `GradScaler` multiplies the loss by a large factor `S` before `backward()`. Gradients that would underflow to 0 in fp16 (§1.7) are therefore representable. It divides the gradients by `S` again before the optimizer step, skips any step whose gradients contain `inf` or `nan`, and adapts `S` automatically.
- **bf16** (Ampere GPUs such as the A100, and newer) has fp32's exponent range, so it needs **no scaler**. The T4 is a Turing GPU without bf16 tensor cores, so on Colab's T4 we use fp16 with a scaler.

In [ ]:
# What autocast does to dtypes:
lin = nn.Linear(64, 64).to(device)
xx = torch.randn(8, 64, device=device)
amp_demo_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
with torch.autocast(device_type=device.type, dtype=amp_demo_dtype):
    out = lin(xx)
    loss_demo = F.cross_entropy(out, torch.zeros(8, dtype=torch.long, device=device))
print("param dtype:", lin.weight.dtype, "| linear output:", out.dtype, "| loss:", loss_demo.dtype)

In [ ]:
flow_diagram(
    {"w": (0.07, 0.75, "fp32 master\nweights", "#cfe8ff"), "fwd": (0.30, 0.75, "forward under autocast\nmatmul/conv: fp16\nsoftmax/loss: fp32", "#fff2cc"),
     "loss": (0.53, 0.75, "loss (fp32)\n× scale S", "#f8cecc"), "bwd": (0.53, 0.22, "backward\n(fp16 grads, scaled)", "#fff2cc"),
     "uns": (0.30, 0.22, "unscale ÷ S\nskip step if inf/nan", "#e1d5e7"), "step": (0.07, 0.22, "optimizer step\non fp32 weights", "#d5e8d4"),
     "upd": (0.80, 0.22, "scaler.update():\nadapt S", "#e1d5e7")},
    [("w", "fwd", "", "k"), ("fwd", "loss", "", "k"), ("loss", "bwd", "", "k"), ("bwd", "uns", "", "k"),
     ("uns", "step", "", "k"), ("step", "w", "", "k"), ("bwd", "upd", "", "tab:purple")],
    title="One fp16 mixed-precision training step (bf16: the same, without the scale S)", size=(11, 3.4))

The canonical fp16 training step, with **every line in the right order**:

```python
scaler = torch.amp.GradScaler("cuda")
with torch.autocast(device_type="cuda", dtype=torch.float16):
    loss = loss_fn(model(x), y)              # forward + loss under autocast
optimizer.zero_grad(set_to_none=True)
scaler.scale(loss).backward()                # backward on the scaled loss (outside autocast)
scaler.unscale_(optimizer)                   # only needed if you clip or inspect gradients
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)                       # skips the step if gradients are inf/nan
scaler.update()                              # adjust the scale for the next iteration
```

> ⚠️ **Clipping must come after `unscale_`.** Otherwise you clip the *scaled* gradients, which are thousands of times too large, so the clipping does nothing sensible.

Let's measure the speed-up on a wide MLP. Its large matmuls are exactly where tensor cores shine:

In [ ]:
WIDTH = 1024 if FAST_DEV_RUN else 4096
BS_BENCH = 256 if FAST_DEV_RUN else 1024
def wide_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, WIDTH), nn.ReLU(),
                         nn.Linear(WIDTH, WIDTH), nn.ReLU(), nn.Linear(WIDTH, 10)).to(device)

xb = torch.randn(BS_BENCH, 1, 28, 28, device=device)
yb = torch.randint(0, 10, (BS_BENCH,), device=device)
USE_AMP = device.type == "cuda"       # on CPU, autocast exists (bf16) but rarely speeds things up

def make_step(model, use_amp):
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    def step():
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    return step

t_fp32 = benchmark(make_step(wide_mlp(), use_amp=False))
print(f"fp32 step: {t_fp32*1e3:7.1f} ms")
if USE_AMP:
    t_amp = benchmark(make_step(wide_mlp(), use_amp=True))
    print(f"AMP  step: {t_amp*1e3:7.1f} ms   -> {record('AMP speed-up, wide MLP step (x)', t_fp32/t_amp):.1f}x faster")

**On newer GPUs (Ampere, Hopper, Blackwell).** Most industry GPUs are newer than the T4, and two settings matter there:

- **bf16 autocast**: `torch.autocast("cuda", dtype=torch.bfloat16)`, with no `GradScaler`. Our `pick_amp_dtype` (§11.4) chooses it automatically when the hardware supports it.
- **TF32 for fp32 code**: `torch.set_float32_matmul_precision("high")` lets *float32* matrix multiplications use TF32 tensor cores (a reduced-precision internal format). This often gives a large speed-up with no code changes and negligible accuracy impact for deep learning. PyTorch's default, `"highest"`, keeps true fp32 for matmuls. cuDNN convolutions already use TF32 by default (`torch.backends.cudnn.allow_tf32`).

Your GPU:

In [ ]:
if device.type == "cuda":
    major, minor = torch.cuda.get_device_capability()
    print(f"{torch.cuda.get_device_name(0)}: compute capability {major}.{minor}")
    print("bf16 tensor cores + TF32:", "yes (Ampere or newer)" if major >= 8 else "no (e.g. the T4 is Turing, 7.5): use fp16 + GradScaler")
else:
    print("No CUDA GPU: AMP and TF32 do not apply.")

On a T4 the speed-up for this matmul-heavy model is typically around 2–4× (compare with your printout). AMP also halves activation memory, so you can often double the batch size.

## 11.4 The final versions of our training functions

We now fold AMP into `train_one_epoch`, `evaluate` and `fit`. They stay compatible with every earlier call. One more optimisation is included: losses and correct counts are **accumulated on the GPU** and converted with `.item()` once per epoch. Calling `.item()` forces the CPU to wait for the GPU, and doing it every batch stalls the pipeline a little on every step.

In [ ]:
def pick_amp_dtype(device):
    # bf16 where the hardware supports it natively (Ampere+), fp16 on older GPUs (T4), None on CPU/MPS.
    if device.type != "cuda":
        return None
    return torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

def train_one_epoch(model, loader, loss_fn, optimizer, device, scheduler=None, max_grad_norm=None,
                    scaler=None, amp_dtype=None, channels_last=False):
    model.train()
    use_amp = amp_dtype is not None
    if scaler is None:
        scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
    total_loss = torch.zeros((), device=device)
    total_correct = torch.zeros((), device=device, dtype=torch.long)
    total_n = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if channels_last:
            x = x.contiguous(memory_format=torch.channels_last)   # see Chapter 13
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            logits = model(x)
            loss = loss_fn(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if max_grad_norm is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        n = y.size(0)
        total_loss += loss.detach().float() * n
        total_correct += (logits.argmax(dim=1) == y).sum()
        total_n += n
    return total_loss.item() / total_n, total_correct.item() / total_n


@torch.no_grad()
def evaluate(model, loader, loss_fn, device, amp_dtype=None, channels_last=False):
    model.eval()
    total_loss = torch.zeros((), device=device)
    total_correct = torch.zeros((), device=device, dtype=torch.long)
    total_n = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if channels_last:
            x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = model(x)
            loss = loss_fn(logits, y)
        total_loss += loss.float() * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum()
        total_n += y.size(0)
    return total_loss.item() / total_n, total_correct.item() / total_n


def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, device, scheduler=None,
        scheduler_interval="batch", max_grad_norm=None, early_stopping=None, eval_loss_fn=None,
        amp=False, channels_last=False, checkpoint_path=None, verbose=True):
    # The full-featured training driver used for the rest of the notebook.
    eval_loss_fn = eval_loss_fn or loss_fn
    amp_dtype = pick_amp_dtype(device) if amp else None
    scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": [], "epoch_time": []}
    best_acc = -1.0
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device,
                                          scheduler if scheduler_interval == "batch" else None,
                                          max_grad_norm, scaler, amp_dtype, channels_last)
        va_loss, va_acc = evaluate(model, val_loader, eval_loss_fn, device, amp_dtype, channels_last)
        if scheduler is not None and scheduler_interval == "epoch":
            scheduler.step()
        elif scheduler is not None and scheduler_interval == "plateau":
            scheduler.step(va_loss)
        dt = time.perf_counter() - t0
        for k, v in zip(("train_loss", "train_acc", "val_loss", "val_acc", "lr", "epoch_time"),
                        (tr_loss, tr_acc, va_loss, va_acc, optimizer.param_groups[0]["lr"], dt)):
            hist[k].append(v)
        if checkpoint_path is not None and va_acc > best_acc:       # keep the best model on disk
            best_acc = va_acc
            save_checkpoint(checkpoint_path, model, optimizer, scheduler, epoch=epoch, best_metric=va_acc)
        if verbose:
            print(f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} | lr {hist['lr'][-1]:.2e} | {dt:.1f}s")
        if early_stopping is not None and early_stopping.step(va_acc, model, epoch):
            if verbose:
                print(f"early stopping (best val acc {early_stopping.best:.4f} at epoch {early_stopping.best_epoch})")
            break
    if early_stopping is not None and early_stopping.best_state is not None:
        early_stopping.restore_best(model)
    return hist

## 11.5 `torch.compile`

`torch.compile` captures your model's Python code as a graph (via TorchDynamo) and generates fused GPU kernels for it (via TorchInductor, which writes Triton code on GPUs and C++ on CPUs). Fusion matters because a chain of elementwise operations, such as `bias + ReLU + dropout`, otherwise reads and writes memory once per operation.

Use the in-place method **`model.compile()`**. It keeps the `state_dict()` keys unchanged. `compiled = torch.compile(model)` also works, but it returns a wrapper whose state-dict keys gain an `_orig_mod.` prefix, which breaks loading those weights into an uncompiled model.

In [ ]:
m_eager = wide_mlp()
m_comp = wide_mlp()
m_comp.compile()                                  # mode="default"; also "reduce-overhead", "max-autotune"

step_eager = make_step(m_eager, USE_AMP)
step_comp = make_step(m_comp, USE_AMP)
t0 = time.perf_counter(); step_comp(); sync()
print(f"first compiled step (includes compilation): {time.perf_counter() - t0:.1f} s")
t_e, t_c = benchmark(step_eager), benchmark(step_comp)
print(f"eager {t_e*1e3:.1f} ms | compiled {t_c*1e3:.1f} ms | speed-up {record('torch.compile speed-up, wide MLP step (x)', t_e/t_c):.2f}x")
print("state_dict keys unchanged:", list(m_comp.state_dict())[:2])

What to know about `torch.compile`:

- **The first call is slow** (seconds to minutes), and so is every recompilation. It pays off for long runs, not for a quick experiment.
- **Recompilation is triggered by new input shapes.** A smaller last batch triggers a second compilation, after which PyTorch marks that dimension as dynamic. Using `drop_last=True` for training avoids it.
- **Graph breaks.** Code that Dynamo cannot trace (printing tensors, calling `.item()`, data-dependent Python control flow, some third-party libraries) splits the graph into pieces, which reduces the benefit. Run with the environment variable `TORCH_LOGS="graph_breaks"` to see where the breaks happen.
- **Speed-ups depend on the model.** They are largest for many small, fusable operations, and smallest for models already dominated by big matmuls. Our wide MLP is the latter case, so expect a modest gain. **Always measure.**
- `mode="max-autotune"` benchmarks many kernel variants. Compilation takes much longer, and the result is sometimes faster.

## 11.6 Profiling: where does the time go?

`torch.profiler` records every operator and GPU kernel, with timing. Sort the table by GPU time (or by CPU time on a CPU), and look at the top rows.

In [ ]:
from torch.profiler import profile, ProfilerActivity, record_function

activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device.type == "cuda" else [])
prof_step = make_step(wide_mlp(), USE_AMP)
prof_step(); sync()                                       # warm up outside the profiler
with profile(activities=activities) as prof:
    for _ in range(5):
        with record_function("train_step"):              # a named range in the trace
            prof_step()
    sync()
sort_key = "self_cuda_time_total" if device.type == "cuda" else "self_cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=12))
prof.export_chrome_trace(str(ARTIFACT_DIR / "trace.json"))   # open at https://ui.perfetto.dev

For this model, the top rows are the matmul kernels (GEMMs) of the forward and backward passes, which is what you want. In a badly fed pipeline, the trace would instead show **gaps** in the GPU timeline, where the GPU waits for data or for the Python process. The exported trace shows this timeline visually.

## 11.7 Memory

GPU memory during training holds four things:

| Item | Size (fp32 parameters, AdamW) |
| --- | --- |
| Parameters | `4P` bytes |
| Gradients | `4P` bytes |
| Optimizer state (`m`, `v`) | `8P` bytes |
| **Activations** saved for backward | Proportional to **batch size × depth × width**. Often the largest item |

So a model with `P` parameters needs about `16P` bytes **before** activations. A 1-billion-parameter model needs 16 GB just for that, which is why large models need sharding (Chapter 16).

In [ ]:
if device.type == "cuda":
    m = wide_mlp()
    P = count_params(m)
    step_fn = make_step(m, USE_AMP)
    step_fn(); sync()                                    # first step allocates optimizer state
    torch.cuda.reset_peak_memory_stats()
    step_fn(); sync()
    peak = torch.cuda.max_memory_allocated() / 2**20
    print(f"P = {P/1e6:.1f}M parameters -> 16P = {16*P/2**20:.0f} MiB for weights+grads+AdamW state")
    print(f"measured peak during a step: {peak:.0f} MiB  (the rest is activations and workspace)")
    del m, step_fn
    torch.cuda.empty_cache()
else:
    print("CUDA memory statistics are only available on a GPU.")

**When you hit `CUDA out of memory`**, try these in order:

1. Reduce the batch size. If you need the larger effective batch, use **gradient accumulation** (below).
2. Turn on AMP, which roughly halves activation memory.
3. Make sure evaluation runs under `torch.no_grad()` or `torch.inference_mode()`, and that you are not holding tensors with history in Python lists.
4. Use **activation checkpointing**. It stores fewer activations and recomputes them during the backward pass, trading about 30% more compute for much less memory: `torch.utils.checkpoint.checkpoint(block, x, use_reentrant=False)`.
5. Use a smaller model, or shard across GPUs.

**Gradient accumulation** reproduces a large batch with several small ones. Gradients add up across `backward()` calls (§2.3), so you divide each micro-batch loss by the number of micro-batches and step once per group:

```python
ACCUM = 4                                   # effective batch = ACCUM × micro-batch size
optimizer.zero_grad(set_to_none=True)
for i, (x, y) in enumerate(loader):
    loss = loss_fn(model(x), y) / ACCUM     # scale, so the summed gradient is the mean
    loss.backward()
    if (i + 1) % ACCUM == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        # scheduler.step() goes here: once per *optimizer* step
```

There are two caveats. BatchNorm still sees only the micro-batch statistics, so accumulation is not exactly equivalent for BN models. And with AMP you use `scaler.scale(loss).backward()` for every micro-batch, with `scaler.step` and `scaler.update` only on the stepping iterations.

## 🧪 Exercises

**Exercise 11.1: prove gradient accumulation is exact.** `★★☆ · ~20 min` For an MLP **without** dropout or BatchNorm, compute the gradient of the mean loss on one batch of 256 examples. Then compute it again with 4 micro-batches of 64, each loss divided by 4, accumulating with `backward()`. Show the two gradients match (`torch.allclose`, `atol=1e-6`). Then add `nn.Dropout(0.5)` and explain why exact equality is no longer expected, even though the gradient is still unbiased.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def grads_of(model):
    return [p.grad.detach().clone() for p in model.parameters()]

set_seed(0)
net = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
xa = torch.randn(256, 1, 28, 28, device=device)
ya = torch.randint(0, 10, (256,), device=device)

net.zero_grad(); F.cross_entropy(net(xa), ya).backward()
g_full = grads_of(net)

net.zero_grad()
for xm, ym in zip(xa.chunk(4), ya.chunk(4)):
    (F.cross_entropy(net(xm), ym) / 4).backward()
g_acc = grads_of(net)
print("accumulated == full batch:", all(torch.allclose(a, b, atol=1e-6) for a, b in zip(g_full, g_acc)))
# With dropout, each forward pass samples new random masks, so the micro-batch run sees different
# masks from the full-batch run: the two gradients are different samples of the same random variable.
# (With BatchNorm, statistics over 64 examples differ from those over 256 -- a systematic difference.)

**Exercise 11.2: the full speed stack.** `★★☆ · ~20 min` Train `MLP()` on Fashion-MNIST for 5 epochs using `fast_train`/`fast_val` (§11.2), `fit(..., amp=True)`, and `model.compile()`, and compare the total time with 5 epochs of the Chapter 7 setup (`train_loader`, no AMP, no compile). Report the speed-up, and check that the validation accuracies are comparable. Which of the three changes contributed most here, and why?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def timed_fit(model, tl, vl, amp, epochs):
    o = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    t0 = time.perf_counter()
    h = fit(model, tl, vl, nn.CrossEntropyLoss(), o, epochs, device, amp=amp, verbose=False)
    sync()
    return time.perf_counter() - t0, max(h["val_acc"])

E = n_epochs(5)
set_seed(0); t_base, a_base = timed_fit(MLP().to(device), train_loader, val_loader, False, E)
set_seed(0); fast_model = MLP().to(device); fast_model.compile()
t_fast, a_fast = timed_fit(fast_model, fast_train, fast_val, True, E)
print(f"baseline: {t_base:6.1f}s, best val acc {a_base:.4f}")
print(f"fast:     {t_fast:6.1f}s, best val acc {a_fast:.4f}  -> {t_base/t_fast:.1f}x faster (incl. compile time)")
# The GPU-resident data dominates: this small MLP was input-bound, and AMP/compile can only speed up
# work the GPU is actually doing. For compute-bound models (big CNNs, Transformers), AMP matters most.

### 📌 Key takeaways

- Feed the GPU first; then AMP; then `torch.compile`; always measure.
- fp16 needs a GradScaler (and `unscale_` before clipping); bf16 does not.
- Memory ≈ 16 bytes/parameter for AdamW fp32 training, plus activations.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Why does fp16 training need a `GradScaler` but bf16 training does not?

<details>
<summary>Answer</summary>

fp16 has a tiny exponent range (max ≈ 65,504, smallest normal ≈ 6e-5), so small gradients underflow to zero. Scaling the loss shifts them into range. bf16 has the same exponent range as fp32, so underflow is not a problem.

</details>

**2.** You enable AMP and clip gradients to norm 1.0, but training behaves as if there were no clipping. What did you forget?

<details>
<summary>Answer</summary>

`scaler.unscale_(optimizer)` before `clip_grad_norm_`. Without it you clip the scaled gradients, which are larger by the loss-scale factor (often 65,536).

</details>

**3.** `torch.compile` made your 2-minute experiment slower. Why might that be expected?

<details>
<summary>Answer</summary>

Compilation itself can take tens of seconds to minutes, and every new input shape can trigger a recompile. The speed-up only pays off over longer runs with stable shapes.

</details>

**4.** Estimate the GPU memory for weights, gradients and AdamW state of a 350M-parameter model in fp32.

<details>
<summary>Answer</summary>

About 16 bytes × 350M ≈ 5.6 GB, before activations.

</details>

⏯️ **Resume point.** Continuing in a *fresh* runtime? Run the setup cell (Chapter 0), then the hidden cell below. It re-creates every function and class from the earlier chapters and the data Chapter 12 needs. In a normal top-to-bottom run it detects that nothing is missing and does nothing.

In [ ]:
# @title ⏯️ Resume point for Chapter 12: run only in a fresh runtime, after the setup cell { display-mode: "form" }
if "set_seed" not in globals():
    raise RuntimeError("Run the setup cell of Chapter 0 first, then this cell.")
if "fast_train" in globals() and "fit" in globals() and "TensorLoader" in globals():
    print("Chapter 12: the state it needs is already present -- nothing to restore.")
else:
    from collections import OrderedDict
    from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset, random_split
    from sklearn.datasets import make_moons
    from sklearn.model_selection import train_test_split
    from torch.nn.utils.rnn import pad_sequence
    from torchvision import datasets
    from torchvision.transforms import v2
    from torch.utils.data import IterableDataset, get_worker_info
    from torch.utils.data import WeightedRandomSampler
    from sklearn.metrics import roc_auc_score, average_precision_score
    from torch.optim import lr_scheduler as S
    from torch.profiler import profile, ProfilerActivity, record_function

    def time_matmul(dev, n=2048, reps=10):
        a = torch.randn(n, n, device=dev)
        b = torch.randn(n, n, device=dev)
        a @ b                                        # warm-up
        if dev.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(reps):
            a @ b
        if dev.type == "cuda": torch.cuda.synchronize()   # wait for the GPU to actually finish
        return (time.perf_counter() - t0) / reps

    def tiny_net(x, W1, b1, W2, b2):
        return (torch.tanh(x @ W1 + b1) @ W2 + b2).pow(2).sum()

    class MySigmoid(torch.autograd.Function):
        @staticmethod
        def forward(ctx, z):
            s = 1.0 / (1.0 + torch.exp(-z))
            ctx.save_for_backward(s)           # stash what backward needs
            return s

        @staticmethod
        def backward(ctx, grad_out):           # grad_out = dL/ds
            (s,) = ctx.saved_tensors
            return grad_out * s * (1 - s)      # dL/dz = dL/ds * s(1-s)

    class MLP(nn.Module):
        def __init__(self, in_dim=784, hidden=(256, 128), out_dim=10, p_drop=0.2):
            super().__init__()                    # ALWAYS first: sets up the registration machinery
            self.fc1 = nn.Linear(in_dim, hidden[0])   # assigning a Module to an attribute registers it
            self.fc2 = nn.Linear(hidden[0], hidden[1])
            self.head = nn.Linear(hidden[1], out_dim)
            self.drop = nn.Dropout(p_drop)

        def forward(self, x):                     # x: (N, 1, 28, 28) or (N, 784)
            x = x.flatten(start_dim=1)            # -> (N, 784)
            x = self.drop(F.relu(self.fc1(x)))    # -> (N, 256)
            x = self.drop(F.relu(self.fc2(x)))    # -> (N, 128)
            return self.head(x)                   # -> (N, 10) logits: no softmax (Chapter 5)

    def count_params(m: nn.Module, trainable_only: bool = True) -> int:
        return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

    class Broken(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = [nn.Linear(10, 10) for _ in range(3)]       # plain list: NOT registered

    class Fixed(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(3)])
        def forward(self, x):
            for layer in self.layers:
                x = F.relu(layer(x))
            return x

    class Normalize(nn.Module):
        def __init__(self, mean, std):
            super().__init__()
            self.register_buffer("mean", torch.as_tensor(mean, dtype=torch.float32).view(1, -1, 1, 1))
            self.register_buffer("std", torch.as_tensor(std, dtype=torch.float32).view(1, -1, 1, 1))
        def forward(self, x):
            return (x - self.mean) / self.std

    def deep_relu_mlp(depth=20, width=256):
        layers = []
        for _ in range(depth):
            layers += [nn.Linear(width, width, bias=False), nn.ReLU()]
        return nn.Sequential(*layers)

    def activation_stds(model, x):
        stds, handles = [], []
        for m in model.modules():
            if isinstance(m, nn.ReLU):
                # forward hook signature: hook(module, inputs, output)
                handles.append(m.register_forward_hook(lambda mod, inp, out: stds.append(out.std().item())))
        with torch.no_grad():
            model(x)
        for h in handles:
            h.remove()          # always remove hooks you no longer need
        return stds

    def grad_norms(model):
        return {n: p.grad.norm().item() for n, p in model.named_parameters() if p.grad is not None}

    class ArrayDataset(Dataset):
        # Wraps feature/label arrays. Optionally standardises features with GIVEN statistics.
        def __init__(self, X, y, mean=None, std=None):
            self.X = torch.as_tensor(X, dtype=torch.float32)         # convert once, here
            self.y = torch.as_tensor(y, dtype=torch.int64)
            if mean is not None:
                self.X = (self.X - mean) / std

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]                           # one (features, label) pair

    class RandomSequences(Dataset):
        def __init__(self, n=100):
            g = torch.Generator().manual_seed(0)
            self.lengths = torch.randint(3, 10, (n,), generator=g)
            self.data = [torch.randn(int(L), 4, generator=g) for L in self.lengths]   # each (L_i, 4)
            self.labels = torch.randint(0, 2, (n,), generator=g)
        def __len__(self): return len(self.data)
        def __getitem__(self, i): return self.data[i], self.labels[i]

    def pad_collate(batch):
        seqs, labels = zip(*batch)                          # batch is a list of (seq, label) pairs
        lengths = torch.tensor([len(s) for s in seqs])
        padded = pad_sequence(seqs, batch_first=True)       # (B, L_max, 4), zero-padded
        mask = torch.arange(padded.size(1))[None, :] < lengths[:, None]   # (B, L_max): True = real data
        return padded, lengths, mask, torch.stack(labels)

    def show_images(images, titles=None, ncols=8, cmap="gray", figsize_per=1.4):
        # images: (N, C, H, W) tensor, already in a displayable range
        n = len(images); nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per, nrows * figsize_per * 1.15))
        for i, ax in enumerate(np.array(axes).reshape(-1)):
            ax.axis("off")
            if i < n:
                img = images[i]
                img = img[0] if img.shape[0] == 1 else img.permute(1, 2, 0)   # CHW -> HW or HWC
                ax.imshow(img.clamp(0, 1).numpy(), cmap=cmap if img.ndim == 2 else None)
                if titles is not None:
                    ax.set_title(titles[i], fontsize=7)
        plt.tight_layout(); plt.show()

    def loader_throughput(ds, num_workers, batch_size=256, max_batches=50):
        dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                        pin_memory=(device.type == "cuda"), persistent_workers=num_workers > 0)
        it = iter(dl); next(it)                         # start the workers before timing
        t0 = time.perf_counter(); n = 0
        for i, (xb, yb) in enumerate(it):
            n += len(xb)
            if i + 1 >= max_batches: break
        return n / (time.perf_counter() - t0)

    def tab_split(i):
        return TensorDataset((X_num[i] - num_mean) / num_std, X_cat[i], (y_tab[i] - y_mean) / y_std)

    class TabularNet(nn.Module):
        # Numeric features + one embedding per categorical column -> MLP -> one regression output.
        def __init__(self, n_num, cardinalities=(), emb_dims=(), hidden=64):
            super().__init__()
            self.embs = nn.ModuleList([nn.Embedding(c, d) for c, d in zip(cardinalities, emb_dims)])
            self.mlp = nn.Sequential(nn.Linear(n_num + sum(emb_dims), hidden), nn.ReLU(),
                                     nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

        def forward(self, x_num, x_cat):
            parts = [x_num] + [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]   # each (N, d_i)
            return self.mlp(torch.cat(parts, dim=1))

    def train_regressor(model, epochs=n_epochs(15), lr=3e-3):
        model = model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        loader = DataLoader(tab_train, batch_size=256, shuffle=True)
        for _ in range(epochs):
            model.train()
            for xn, xc, yb in loader:
                loss = F.mse_loss(model(xn.to(device), xc.to(device)), yb.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
        return model

    @torch.no_grad()
    def rmse(model, ds):
        model.eval()
        xn, xc, yb = (t.to(device) for t in ds.tensors)
        mse_standardised = F.mse_loss(model(xn, xc), yb)
        return (mse_standardised.sqrt() * y_std).item()              # back in original units

    class ShardStream(IterableDataset):
        def __init__(self, files, split_by_worker=True):
            self.files, self.split_by_worker = list(files), split_by_worker
        def __iter__(self):
            files = self.files
            info = get_worker_info()                               # None in the main process
            if self.split_by_worker and info is not None:
                files = files[info.id::info.num_workers]           # worker k reads shards k, k+W, k+2W, ...
            for f in files:
                xs, ys = torch.load(f, weights_only=True)
                yield from zip(xs, ys)

    def plot_history(hist, title=""):
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
        ep = range(1, len(hist["train_loss"]) + 1)
        a1.plot(ep, hist["train_loss"], label="train"); a1.plot(ep, hist["val_loss"], label="val")
        a2.plot(ep, hist["train_acc"], label="train"); a2.plot(ep, hist["val_acc"], label="val")
        a1.set_title("loss"); a2.set_title("accuracy")
        for a in (a1, a2):
            a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
        fig.suptitle(title); plt.tight_layout(); plt.show()

    @torch.no_grad()
    def predict(model, loader, device):
        # All predictions, true labels and probabilities for a loader (in loader order).
        model.eval()
        preds, labels, probs = [], [], []
        for x, y in loader:
            p = torch.softmax(model(x.to(device)), dim=1).cpu()
            probs.append(p); preds.append(p.argmax(1)); labels.append(y)
        return torch.cat(preds), torch.cat(labels), torch.cat(probs)

    def confusion_matrix(preds, labels, num_classes):
        # cm[i, j] = number of examples of true class i predicted as class j
        return torch.bincount(labels * num_classes + preds, minlength=num_classes ** 2).reshape(num_classes, num_classes)

    def plot_confusion(cm, class_names, title="Confusion matrix (row-normalised)"):
        cm_n = cm.float() / cm.sum(1, keepdim=True).clamp_min(1)
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
        im = ax.imshow(cm_n, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(class_names)), class_names, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(len(class_names)), class_names, fontsize=8)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                ax.text(j, i, f"{cm_n[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cm_n[i, j] > 0.5 else "black")
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
        plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

    def base_indices(ds):
        # Map a (possibly nested) Subset back to indices of its underlying dataset.
        idx = list(range(len(ds)))
        while isinstance(ds, Subset):
            idx = [ds.indices[i] for i in idx]; ds = ds.dataset
        return idx, ds

    def rare_class_split(ds, positive=8, prevalence=0.02, seed=0):
        idx, base = base_indices(ds)
        labels = torch.as_tensor(base.targets)[idx]
        pos = [i for i, l in zip(idx, labels.tolist()) if l == positive]
        neg = [i for i, l in zip(idx, labels.tolist()) if l != positive]
        n_pos = max(1, round(prevalence * len(neg) / (1 - prevalence)))
        pos = [pos[i] for i in torch.randperm(len(pos), generator=torch.Generator().manual_seed(seed))[:n_pos].tolist()]
        return Subset(base, neg + pos), [0] * len(neg) + [1] * len(pos)

    class BinaryTarget(Dataset):
        # Turn integer class labels into float {0., 1.} targets of shape (1,) for BCEWithLogitsLoss.
        def __init__(self, base, positive): self.base, self.positive = base, positive
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            x, y = self.base[i]
            return x, torch.tensor([float(y == self.positive)])

    @torch.no_grad()
    def predict_proba(model, ds):
        model.eval(); ps, ys = [], []
        for x, y in DataLoader(ds, batch_size=512):
            ps.append(torch.sigmoid(model(x.to(device))).squeeze(1).cpu()); ys.append(y.squeeze(1))
        return torch.cat(ps), torch.cat(ys).long()

    def binary_metrics(p, y, t=0.5):
        pred = (p >= t).long()
        tp = ((pred == 1) & (y == 1)).sum().item(); fp = ((pred == 1) & (y == 0)).sum().item()
        fn = ((pred == 0) & (y == 1)).sum().item()
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        return {"acc": (pred == y).float().mean().item(), "precision": prec, "recall": rec, "F1": f1}

    def train_binary(model, loader, loss_fn, epochs=n_epochs(3)):
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        for _ in range(epochs):
            model.train()
            for x, y in loader:
                loss = loss_fn(model(x.to(device)), y.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
        return model

    def threshold_for_recall(p, y, target_recall=0.9):
        # Highest threshold (hence best precision) whose recall on (p, y) is still >= target_recall.
        for t in torch.sort(p.unique(), descending=True).values.tolist():
            if binary_metrics(p, y, t)["recall"] >= target_recall:
                return t
        return 0.0

    def param_groups_weight_decay(model, weight_decay):
        # Two groups: weight matrices/kernels get decay; biases and norm parameters do not.
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            (decay if p.ndim >= 2 else no_decay).append(p)
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    def lr_trace(make_sched, steps=100, base_lr=0.1):
        p = nn.Parameter(torch.zeros(1))
        opt = torch.optim.SGD([p], lr=base_lr)
        sched = make_sched(opt)
        lrs = []
        for _ in range(steps):
            lrs.append(opt.param_groups[0]["lr"])
            opt.step()               # (no gradient here -- we only care about the lr)
            sched.step()
        return lrs

    def lr_range_test(model, loader, loss_fn, device, opt_fn=lambda params, lr: torch.optim.AdamW(params, lr=lr),
                      lr_min=1e-6, lr_max=1.0, num_steps=200, beta=0.98):
        saved = copy.deepcopy(model.state_dict())
        opt = opt_fn(model.parameters(), lr_min)
        gamma = (lr_max / lr_min) ** (1 / (num_steps - 1))        # multiply the lr by this every step
        lrs, losses, avg, best = [], [], 0.0, float("inf")
        model.train(); it = iter(loader)
        for step in range(num_steps):
            try:
                x, y = next(it)
            except StopIteration:
                it = iter(loader); x, y = next(it)
            lr = lr_min * gamma ** step
            for g in opt.param_groups:
                g["lr"] = lr
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            avg = beta * avg + (1 - beta) * loss.item()           # exponential moving average...
            smoothed = avg / (1 - beta ** (step + 1))             # ...with Adam-style bias correction
            lrs.append(lr); losses.append(smoothed)
            best = min(best, smoothed)
            if not math.isfinite(smoothed) or smoothed > 4 * best:    # diverged: stop early
                break
        model.load_state_dict(saved)                              # undo the test's training
        return lrs, losses

    class EarlyStopping:
        # Stops when the monitored metric has not improved for `patience` epochs,
        # and keeps a CPU copy of the best weights so they can be restored.
        def __init__(self, patience=3, min_delta=0.0, mode="max"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best, self.best_epoch, self.bad_epochs, self.best_state = None, -1, 0, None

        def _improved(self, value):
            if self.best is None:
                return True
            return value > self.best + self.min_delta if self.mode == "max" else value < self.best - self.min_delta

        def step(self, value, model, epoch):
            # Call once per epoch. Returns True when training should stop.
            if self._improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, epoch, 0
                self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore_best(self, model):
            model.load_state_dict(self.best_state)

    def rng_state():
        return {"torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate()}

    def set_rng_state(state):
        torch.set_rng_state(state["torch"])
        if torch.cuda.is_available() and state["cuda"]:
            torch.cuda.set_rng_state_all(state["cuda"])
        random.setstate(state["python"])

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None, config=None, **extra):
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch, "best_metric": best_metric,
            "config": config or {}, "rng": rng_state(),
            "torch_version": str(torch.__version__), **extra,   # str(): see the note below
        }
        path = Path(path)
        tmp = path.with_name(path.name + ".tmp")
        torch.save(ckpt, tmp)
        os.replace(tmp, path)       # atomic rename: a crash mid-save never leaves a corrupt checkpoint

    def load_checkpoint(path, model, optimizer=None, scheduler=None, map_location="cpu"):
        ckpt = torch.load(path, map_location=map_location, weights_only=True)
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        set_rng_state(ckpt["rng"])
        return ckpt

    def make_run(seed):
        # Build every training object from scratch, as a fresh process would.
        torch.manual_seed(seed)
        model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 3))
        opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=6)
        gen = torch.Generator().manual_seed(seed)                       # controls the shuffle order
        loader = DataLoader(TensorDataset(Xr, yr), batch_size=32, shuffle=True, generator=gen)
        return model, opt, sched, loader, gen

    def run_epochs(model, opt, sched, loader, n):
        for _ in range(n):
            model.train()
            for xb, yb in loader:
                loss = F.cross_entropy(model(xb), yb)
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()

    def make_deterministic(seed=0):
        # Maximum GPU determinism for debugging or auditing. Not needed (and slower) for normal training.
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # must be set before the first cuBLAS call to take effect
        set_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)   # warn_only: warn instead of raising

    def overfit_one_batch(model, batch, loss_fn, steps=200, lr=1e-3):
        # A correct model and loop should drive the training loss on ONE batch to ~0.
        x, y = (t.to(device) for t in batch)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        model.train()
        for step in range(steps):
            logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            if step % 50 == 0 or step == steps - 1:
                acc = (logits.argmax(1) == y).float().mean().item()
                print(f"step {step:3d}: loss {loss.item():.4f}  batch acc {acc:.2f}")
        return loss.item()

    def gradient_report(model):
        # Call after loss.backward(). Flags parameters with missing, non-finite or zero gradients.
        problems = []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if p.grad is None:
                problems.append(f"{name}: grad is None (unused in forward, or graph cut by .detach()/.item()?)")
            elif not torch.isfinite(p.grad).all():
                problems.append(f"{name}: non-finite gradient")
            elif p.grad.abs().max() == 0:
                problems.append(f"{name}: gradient exactly zero (dead ReLUs? frozen upstream?)")
            else:
                print(f"{name:14s} |grad| = {p.grad.norm():.2e}   |w| = {p.norm():.2e}   ratio = {p.grad.norm() / p.norm():.1e}")
        for msg in problems:
            print("⚠️", msg)

    class HasUnusedLayer(MLP):
        def __init__(self):
            super().__init__()
            self.extra = nn.Linear(10, 10)          # defined but never used in forward

    def buggy_training(epochs=2):
        set_seed(0)
        model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
                              nn.Linear(256, 10), nn.Softmax(dim=1)).to(device)
        loader = DataLoader(sorted_train, batch_size=128, shuffle=False)
        vloader = DataLoader(fmnist_val, batch_size=512)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()
        history = []
        for epoch in range(epochs):
            running = 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                loss = loss_fn(model(x), y)
                loss.backward()
                opt.step()
                running += loss
            model.eval()
            correct = 0
            for x, y in vloader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
            history.append(correct / len(fmnist_val))
            print(f"epoch {epoch+1}: train loss {running.item() / len(loader):.3f} | val acc {history[-1]:.4f}")
        return history

    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    def benchmark(fn, warmup=5, iters=20):
        # Average wall-clock seconds per call of fn(), with warm-up and GPU synchronisation (§1.8).
        for _ in range(warmup):
            fn()
        sync()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        sync()
        return (time.perf_counter() - t0) / iters

    def materialize(ds, batch_size=2048):
        # Run a dataset's transforms once and stack everything into two tensors.
        xs, ys = [], []
        for x, y in DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS):
            xs.append(x); ys.append(y)
        return torch.cat(xs), torch.cat(ys)

    class TensorLoader:
        # A minimal DataLoader replacement for tensors that already live on the target device.
        def __init__(self, X, y, batch_size, shuffle=False, drop_last=False, seed=0):
            self.X, self.y, self.bs, self.shuffle, self.drop_last = X, y, batch_size, shuffle, drop_last
            self.gen = torch.Generator(device=X.device).manual_seed(seed)
        def __len__(self):
            n = len(self.X)
            return n // self.bs if self.drop_last else math.ceil(n / self.bs)
        def __iter__(self):
            n = len(self.X)
            idx = torch.randperm(n, device=self.X.device, generator=self.gen) if self.shuffle else torch.arange(n, device=self.X.device)
            for i in range(len(self)):
                j = idx[i * self.bs:(i + 1) * self.bs]
                yield self.X[j], self.y[j]

    def wide_mlp():
        return nn.Sequential(nn.Flatten(), nn.Linear(784, WIDTH), nn.ReLU(),
                             nn.Linear(WIDTH, WIDTH), nn.ReLU(), nn.Linear(WIDTH, 10)).to(device)

    def make_step(model, use_amp):
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
        def step():
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        return step

    def pick_amp_dtype(device):
        # bf16 where the hardware supports it natively (Ampere+), fp16 on older GPUs (T4), None on CPU/MPS.
        if device.type != "cuda":
            return None
        return torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

    def train_one_epoch(model, loader, loss_fn, optimizer, device, scheduler=None, max_grad_norm=None,
                        scaler=None, amp_dtype=None, channels_last=False):
        model.train()
        use_amp = amp_dtype is not None
        if scaler is None:
            scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
        total_loss = torch.zeros((), device=device)
        total_correct = torch.zeros((), device=device, dtype=torch.long)
        total_n = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if channels_last:
                x = x.contiguous(memory_format=torch.channels_last)   # see Chapter 13
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()
            n = y.size(0)
            total_loss += loss.detach().float() * n
            total_correct += (logits.argmax(dim=1) == y).sum()
            total_n += n
        return total_loss.item() / total_n, total_correct.item() / total_n

    @torch.no_grad()
    def evaluate(model, loader, loss_fn, device, amp_dtype=None, channels_last=False):
        model.eval()
        total_loss = torch.zeros((), device=device)
        total_correct = torch.zeros((), device=device, dtype=torch.long)
        total_n = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if channels_last:
                x = x.contiguous(memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                logits = model(x)
                loss = loss_fn(logits, y)
            total_loss += loss.float() * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum()
            total_n += y.size(0)
        return total_loss.item() / total_n, total_correct.item() / total_n

    def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, device, scheduler=None,
            scheduler_interval="batch", max_grad_norm=None, early_stopping=None, eval_loss_fn=None,
            amp=False, channels_last=False, checkpoint_path=None, verbose=True):
        # The full-featured training driver used for the rest of the notebook.
        eval_loss_fn = eval_loss_fn or loss_fn
        amp_dtype = pick_amp_dtype(device) if amp else None
        scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
        hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": [], "epoch_time": []}
        best_acc = -1.0
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device,
                                              scheduler if scheduler_interval == "batch" else None,
                                              max_grad_norm, scaler, amp_dtype, channels_last)
            va_loss, va_acc = evaluate(model, val_loader, eval_loss_fn, device, amp_dtype, channels_last)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            dt = time.perf_counter() - t0
            for k, v in zip(("train_loss", "train_acc", "val_loss", "val_acc", "lr", "epoch_time"),
                            (tr_loss, tr_acc, va_loss, va_acc, optimizer.param_groups[0]["lr"], dt)):
                hist[k].append(v)
            if checkpoint_path is not None and va_acc > best_acc:       # keep the best model on disk
                best_acc = va_acc
                save_checkpoint(checkpoint_path, model, optimizer, scheduler, epoch=epoch, best_metric=va_acc)
            if verbose:
                print(f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                      f"val loss {va_loss:.4f} acc {va_acc:.4f} | lr {hist['lr'][-1]:.2e} | {dt:.1f}s")
            if early_stopping is not None and early_stopping.step(va_acc, model, epoch):
                if verbose:
                    print(f"early stopping (best val acc {early_stopping.best:.4f} at epoch {early_stopping.best_epoch})")
                break
        if early_stopping is not None and early_stopping.best_state is not None:
            early_stopping.restore_best(model)
        return hist

    fmnist_raw = datasets.FashionMNIST(DATA_DIR, train=True, download=True)
    FMNIST_CLASSES = fmnist_raw.classes
    perm = torch.randperm(len(fmnist_raw), generator=torch.Generator().manual_seed(0))
    fm_train_idx, fm_val_idx = perm[:55_000].tolist(), perm[55_000:].tolist()
    pixels = fmnist_raw.data[fm_train_idx].float() / 255.0
    FMNIST_MEAN, FMNIST_STD = pixels.mean().item(), pixels.std().item(); del pixels
    fmnist_tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=[FMNIST_MEAN], std=[FMNIST_STD])])
    full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=fmnist_tf)
    fmnist_test = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=fmnist_tf)
    fmnist_train, fmnist_val = Subset(full_train, fm_train_idx), Subset(full_train, fm_val_idx)
    fmnist_train, fmnist_val, fmnist_test = maybe_subset(fmnist_train), maybe_subset(fmnist_val), maybe_subset(fmnist_test)
    BATCH, pin, USE_AMP = 128, device.type == "cuda", device.type == "cuda"
    train_loader = DataLoader(fmnist_train, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=NUM_WORKERS > 0)
    val_loader = DataLoader(fmnist_val, batch_size=512, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
    test_loader = DataLoader(fmnist_test, batch_size=512, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
    Xtr, ytr = (t.to(device) for t in materialize(fmnist_train))
    Xva, yva = (t.to(device) for t in materialize(fmnist_val))
    fast_train = TensorLoader(Xtr, ytr, batch_size=128, shuffle=True)
    fast_val = TensorLoader(Xva, yva, batch_size=512)
    print("Chapter 12: functions, classes and data from earlier chapters restored.")


---
# Part IV — Convolutional networks

# Chapter 12. Convolution in PyTorch

Your theory course stopped at MLPs, so this chapter teaches convolutional networks from the ground up, with the code alongside.

## 12.1 Why MLPs are the wrong tool for images

An MLP treats an image as a flat vector, and that throws away three facts about images:

1. **Locality.** A pixel is strongly related to its neighbours and weakly related to distant pixels. An MLP connects every input to every hidden unit and has to *learn* that locality from data.
2. **Translation.** A sneaker is a sneaker wherever it appears in the frame. An MLP must learn the same detector separately for every position.
3. **Scale.** A modest 224×224 RGB image has 150,528 inputs. A single hidden layer of 1,000 units would need **150 million** weights in its first layer alone.

A convolution layer builds the first two facts in, and that solves the third. Each output looks only at a small **local** window (a *kernel*, typically 3×3), and the **same** kernel weights are slid across every position (*weight sharing*).

## 12.2 The operation

For an input `x` with `C_in` channels, a conv layer with `C_out` output channels and `k×k` kernels computes, for every output channel `o` and position `(i, j)`:

```
y[o, i, j] = b[o] + Σ_c Σ_u Σ_v  W[o, c, u, v] · x[c, i·s + u, j·s + v]
             (c = 1..C_in,  u, v = 0..k−1)
```

with stride `s`. (Deep learning libraries compute this *cross-correlation* and call it convolution. A mathematician's convolution flips the kernel first. Since the kernel is learned, the distinction does not matter.) Each output value is a **dot product between the kernel and one patch of the input**, exactly the `wᵀx + b` of a neuron (Tutorial §2.2), applied at every location with shared `w`.

The picture: the kernel slides over the input. Each output value is one dot product.

🎯 **In this chapter you will learn to:** explain convolution, padding, stride, pooling and receptive fields; compute output shapes and parameter counts by hand; build and train a CNN.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
def grid(ax, n, title, hi=None, col="#fff2cc"):
    for i in range(n):
        for j in range(n):
            on = hi is not None and hi[0] <= i < hi[0] + hi[2] and hi[1] <= j < hi[1] + hi[2]
            ax.add_patch(plt.Rectangle((j, n - 1 - i), 1, 1, fc=col if on else "white", ec="0.4"))
    ax.set_xlim(-0.2, n + 0.2); ax.set_ylim(-0.2, n + 0.2); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title, fontsize=9)
grid(axes[0], 5, "input 5×5 (one channel)\nhighlighted: the 3×3 patch", hi=(1, 2, 3))
grid(axes[1], 3, "kernel 3×3 (shared weights)", hi=(0, 0, 3), col="#cfe8ff")
grid(axes[2], 3, "output 3×3 (stride 1, no padding)\nhighlighted: Σ patch ⊙ kernel + b", hi=(1, 2, 1), col="#f8cecc")
plt.tight_layout(); plt.show()

A tiny example, by hand and with `F.conv2d`:

In [ ]:
x = torch.tensor([[1., 2., 0., 1.],
                  [0., 1., 3., 1.],
                  [2., 1., 0., 0.],
                  [1., 0., 1., 2.]]).view(1, 1, 4, 4)       # (N=1, C=1, H=4, W=4)
k = torch.tensor([[1., 0.],
                  [0., -1.]]).view(1, 1, 2, 2)              # (C_out=1, C_in=1, kH=2, kW=2)

# By hand: top-left output = 1*1 + 2*0 + 0*0 + 1*(-1) = 0
manual = torch.zeros(3, 3)
for i in range(3):
    for j in range(3):
        manual[i, j] = (x[0, 0, i:i+2, j:j+2] * k[0, 0]).sum()
print(manual)
print(F.conv2d(x, k)[0, 0])
print("match:", torch.allclose(manual, F.conv2d(x, k)[0, 0]))

## 12.3 `nn.Conv2d`: shapes and parameter counts

`nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True)` takes input `(N, C_in, H, W)` and produces output `(N, C_out, H_out, W_out)`, where

```
H_out = floor( (H + 2p − d·(k − 1) − 1) / s ) + 1        (the same formula for W_out)
```

Two consequences are worth memorising:

- **"Same" padding.** With `k = 3, p = 1, s = 1` (or generally `p = (k−1)/2` for odd `k`), the spatial size is unchanged. You can also write `padding="same"`.
- **Halving.** With `k = 3, p = 1, s = 2`, the spatial size halves (rounding up).

The weight has shape `(C_out, C_in, k, k)`, so the layer has **`C_out · C_in · k · k + C_out`** parameters. That number is independent of the image size.

In [ ]:
def conv_out(h, k, s=1, p=0, d=1):
    return (h + 2 * p - d * (k - 1) - 1) // s + 1

conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
print("weight:", tuple(conv.weight.shape), "| bias:", tuple(conv.bias.shape),
      "| params:", count_params(conv), "=", 16 * 3 * 3 * 3 + 16)

x = torch.randn(8, 3, 32, 32)
for kw in [dict(kernel_size=3, padding=1), dict(kernel_size=3, padding=1, stride=2),
           dict(kernel_size=5), dict(kernel_size=3, padding=2, dilation=2), dict(kernel_size=1)]:
    y = nn.Conv2d(3, 16, **kw)(x)
    pred = conv_out(32, kw["kernel_size"], kw.get("stride", 1), kw.get("padding", 0), kw.get("dilation", 1))
    print(f"{str(kw):55s} -> {tuple(y.shape)}   formula: {pred}")

# Compare with a dense layer mapping the same input to an output of the same size:
print(f"\nconv 3->16 (3x3): {count_params(conv):,} params | dense 3*32*32 -> 16*32*32: {(3*32*32)*(16*32*32) + 16*32*32:,} params")

**1×1 convolutions** (the last row above) mix channels at each pixel without looking at neighbours. They are a per-pixel `nn.Linear` over the channel dimension, and are used to change the number of channels cheaply.

## 12.4 What a filter detects

A hand-designed kernel shows what learned kernels do. Below, the Sobel kernels respond to horizontal and vertical intensity edges. The first layer of a trained CNN learns edge and colour detectors very much like these. Deeper layers combine them into textures, parts and objects: the **representation learning** of Tutorial §8.2.

In [ ]:
sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
sobel = torch.stack([sobel_x, sobel_x.T]).unsqueeze(1)          # (2, 1, 3, 3): two output channels
img = fmnist_raw.data[0].float().div(255).view(1, 1, 28, 28)    # one Fashion-MNIST image
edges = F.conv2d(img, sobel, padding=1)                         # (1, 2, 28, 28)
show_images(torch.cat([img, edges.abs()[0:1, 0:1] / edges.abs().max(), edges.abs()[0:1, 1:2] / edges.abs().max()]),
            ["input", "vertical edges (Sobel x)", "horizontal edges (Sobel y)"], ncols=3, figsize_per=2.2)

## 12.5 Translation equivariance

Because the same kernel is applied everywhere, **shifting the input shifts the output by the same amount**. This property is called *equivariance*. It means a pattern learned at one position is automatically recognised at every position.

In [ ]:
conv = nn.Conv2d(1, 4, 3, padding=1, bias=False)
x = torch.zeros(1, 1, 16, 16); x[..., 4:7, 4:7] = 1.0             # a small square
x_shift = torch.roll(x, shifts=(5, 3), dims=(2, 3))               # move it down 5, right 3
with torch.no_grad():
    y, y_shift = conv(x), conv(x_shift)
print("conv(shift(x)) == shift(conv(x)):",
      torch.allclose(y_shift, torch.roll(y, shifts=(5, 3), dims=(2, 3)), atol=1e-6))

(The equality is exact here because the square stays away from the borders. Zero padding breaks equivariance slightly near the edges.)

## 12.6 Pooling and global average pooling

- **`nn.MaxPool2d(2)`** takes the maximum over each 2×2 window with stride 2. It halves the height and width, and adds a little local translation *invariance*.
- **`nn.AvgPool2d(2)`** does the same with the mean.
- **`nn.AdaptiveAvgPool2d(1)`** averages each channel over the *whole* spatial extent, whatever its size, giving `(N, C, 1, 1)`. This "global average pooling" replaced the huge fully connected layers of early CNNs. It also lets one network accept different input sizes.

Pooling has **no parameters**. Modern networks often downsample with stride-2 convolutions instead, letting the network learn how to downsample.

In [ ]:
x = torch.arange(16.).view(1, 1, 4, 4)
print(x[0, 0]); print("max pool:\n", F.max_pool2d(x, 2)[0, 0]); print("avg pool:\n", F.avg_pool2d(x, 2)[0, 0])
gap = nn.AdaptiveAvgPool2d(1)
print("GAP on 7x7 and 13x13 inputs:", gap(torch.randn(2, 64, 7, 7)).shape, gap(torch.randn(2, 64, 13, 13)).shape)

## 12.7 Receptive fields

The **receptive field** of a unit is the region of the *input image* that can influence it. Stacking layers grows it. For layer `l` with kernel `k_l` and stride `s_l`, track the receptive field `r` and the "jump" `j` (the distance in input pixels between adjacent units):

```
r_l = r_(l−1) + (k_l − 1) · j_(l−1)        j_l = j_(l−1) · s_l        r_0 = j_0 = 1
```

So two stacked 3×3 convolutions see a 5×5 region, and three see 7×7. That is the observation behind VGG: a stack of small kernels has the receptive field of one large kernel, with fewer parameters and more non-linearities. Downsampling multiplies the growth rate of every later layer.

We can also measure a receptive field **empirically with autograd**. Take the gradient of one output unit with respect to the input. The input pixels with non-zero gradient are exactly the unit's receptive field.

In [ ]:
def receptive_field(layers):
    # layers: list of (kernel, stride). Returns the theoretical receptive field size.
    r, j = 1, 1
    for k, s in layers:
        r, j = r + (k - 1) * j, j * s
    return r

stack = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Conv2d(4, 4, 3, padding=1), nn.ReLU(),
                      nn.MaxPool2d(2), nn.Conv2d(4, 4, 3, padding=1))
spec = [(3, 1), (3, 1), (2, 2), (3, 1)]
x = torch.randn(1, 1, 32, 32, requires_grad=True)
out = stack(x)
out[0, 0, 8, 8].backward()                          # one unit in the middle of the output map
nz = (x.grad[0, 0].abs() > 0).nonzero()
measured = (nz.max(0).values - nz.min(0).values + 1).tolist()
print("theory:", receptive_field(spec), "| measured (rows, cols):", measured)

## 12.8 Batch normalisation for convolutions

`nn.BatchNorm2d(C)` normalises **each channel** using statistics over the batch *and* all spatial positions, `(N, H, W)`, and learns one `γ` and one `β` per channel (Tutorial §11.1). Because BN subtracts the mean, a convolution bias placed right before it is redundant. Standard practice is therefore `nn.Conv2d(..., bias=False)` followed by `nn.BatchNorm2d`.

## 12.9 A CNN for Fashion-MNIST

The classic pattern is a set of **stages**. Each stage is a few `conv → BN → ReLU` blocks at one resolution, followed by downsampling, with the channel count doubled as the resolution halves. The network finishes with global average pooling and a linear classifier.

In [ ]:
def conv_bn_relu(c_in, c_out, stride=1):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False),
                         nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

class SmallCNN(nn.Module):
    def __init__(self, in_ch=1, num_classes=10, p_drop=0.3):
        super().__init__()
        self.features = nn.Sequential(
            conv_bn_relu(in_ch, 32), conv_bn_relu(32, 32), nn.MaxPool2d(2),     # 28 -> 14
            conv_bn_relu(32, 64), conv_bn_relu(64, 64), nn.MaxPool2d(2),        # 14 -> 7
            conv_bn_relu(64, 128),                                              # 7x7x128
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p_drop), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.head(self.features(x))

cnn = SmallCNN().to(device)
print(f"SmallCNN parameters: {count_params(cnn):,}  (the MLP of Chapter 7 had {count_params(MLP()):,})")
# Shape trace: write it in your shapes notebook (Tutorial §0)!
h = torch.randn(2, 1, 28, 28, device=device)
for name, layer in cnn.features.named_children():
    h = layer(h); print(f"after features[{name}]: {tuple(h.shape)}")

In [ ]:
set_seed(0)
cnn = SmallCNN().to(device)
EPOCHS_CNN = n_epochs(12)
opt = torch.optim.AdamW(param_groups_weight_decay(cnn, 0.05), lr=2e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, epochs=EPOCHS_CNN, steps_per_epoch=len(fast_train))
hist_cnn = fit(cnn, fast_train, fast_val, nn.CrossEntropyLoss(label_smoothing=0.1), opt, EPOCHS_CNN, device,
               scheduler=sched, eval_loss_fn=nn.CrossEntropyLoss(), amp=True)
plot_history(hist_cnn, "SmallCNN on Fashion-MNIST")
cnn_test_acc = record("Fashion-MNIST SmallCNN: test acc", evaluate(cnn, test_loader, nn.CrossEntropyLoss(), device)[1])
mlp_ref = RESULTS.get("Fashion-MNIST MLP: test acc")
mlp_txt = "n/a (Chapter 7 not run in this runtime)" if mlp_ref is None else f"{mlp_ref:.4f}"
print(f"\nTest accuracy -- MLP (Chapter 7): {mlp_txt} | CNN: {cnn_test_acc:.4f}")

With **fewer parameters** than the MLP, the CNN typically reaches about **92–93%** test accuracy, against the MLP's 88–89%. In error terms, it removes roughly a third of the MLP's mistakes. That is the payoff of building the right assumptions (locality and weight sharing) into the architecture.

## 🧪 Exercises

**Exercise 12.1: predict the shapes.** `★★☆ · ~20 min` For an input of shape `(16, 3, 64, 64)`, write down the output shape and parameter count of each layer below **by hand**, then verify with code: `Conv2d(3, 32, 7, stride=2, padding=3)` → `MaxPool2d(3, stride=2, padding=1)` → `Conv2d(32, 64, 3, padding=1, bias=False)` → `BatchNorm2d(64)` → `Conv2d(64, 64, 1)` → `AdaptiveAvgPool2d(1)`. (The first two layers are the stem of a standard ResNet.)

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
layers = [nn.Conv2d(3, 32, 7, stride=2, padding=3), nn.MaxPool2d(3, stride=2, padding=1),
          nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.Conv2d(64, 64, 1), nn.AdaptiveAvgPool2d(1)]
by_hand = [((16, 32, 32, 32), 3*32*49 + 32),   # (64+6-6-1)//2+1 = 32
           ((16, 32, 16, 16), 0),              # (32+2-2-1)//2+1 = 16
           ((16, 64, 16, 16), 32*64*9),        # no bias
           ((16, 64, 16, 16), 2*64),           # gamma, beta
           ((16, 64, 16, 16), 64*64 + 64),
           ((16, 64, 1, 1), 0)]
h = torch.randn(16, 3, 64, 64)
for layer, (shape, params) in zip(layers, by_hand):
    h = layer(h)
    print(f"{layer.__class__.__name__:18s} {str(tuple(h.shape)):18s} params {count_params(layer):6d} | by hand {shape} {params} "
          f"{'✓' if tuple(h.shape) == shape and count_params(layer) == params else '✗'}")

**Exercise 12.2: convolution as a matrix multiplication (im2col).** `★★★ · ~30 min` GPUs implement convolution largely as one big matmul. `F.unfold(x, kernel_size, padding=…, stride=…)` extracts every k×k patch of every channel as a column, giving a tensor of shape `(N, C_in·k·k, L)`, where `L` is the number of output positions. Use it to implement `conv2d_via_matmul(x, weight, bias, padding)` for stride 1, and check it against `F.conv2d` on random inputs of shape `(2, 3, 10, 10)` with a `(5, 3, 3, 3)` weight.

In [ ]:
# ✍️ Your attempt here.
# Hint: weight.view(C_out, -1) @ patches gives (N, C_out, L); reshape L back to (H_out, W_out).

In [ ]:
# 🔎 Self-check for Exercise 12.2: tests YOUR `conv2d_via_matmul` (run after your attempt)
def _test(fn):
    x, w, b = torch.randn(2, 3, 9, 9), torch.randn(4, 3, 3, 3), torch.randn(4)
    for p in (0, 1):
        assert torch.allclose(fn(x, w, b, padding=p), F.conv2d(x, w, b, padding=p), atol=1e-4), f'mismatch for padding={p}'

self_check("conv2d_via_matmul", _test)

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def conv2d_via_matmul(x, weight, bias=None, padding=0):
    N, C, H, W = x.shape
    C_out, _, k, _ = weight.shape
    patches = F.unfold(x, kernel_size=k, padding=padding)          # (N, C*k*k, L)
    out = weight.view(C_out, -1) @ patches                          # (C_out, C*k*k) @ (N, C*k*k, L) -> (N, C_out, L)
    if bias is not None:
        out = out + bias.view(1, -1, 1)
    H_out, W_out = H + 2 * padding - k + 1, W + 2 * padding - k + 1
    return out.view(N, C_out, H_out, W_out)

x = torch.randn(2, 3, 10, 10); w = torch.randn(5, 3, 3, 3); b = torch.randn(5)
print(torch.allclose(conv2d_via_matmul(x, w, b, padding=1), F.conv2d(x, w, b, padding=1), atol=1e-5))

**Exercise 12.3: the permuted-pixel experiment.** `★★☆ · ~20 min` Pick one fixed random permutation of the 784 pixel positions and apply it to every image (train, validation and test alike). Train the `MLP` and the `SmallCNN` for 3 epochs each on the permuted data, using `Xtr`/`Xva` from §11.2. Predict the results first. Then explain them using §12.1.

In [ ]:
# ✍️ Your attempt here.
# Hint: perm = torch.randperm(784); Xp = X.flatten(1)[:, perm].view_as(X)

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
perm = torch.randperm(784, generator=torch.Generator().manual_seed(0)).to(device)
permute = lambda X: X.flatten(1)[:, perm].view_as(X)
ptrain = TensorLoader(permute(Xtr), ytr, 128, shuffle=True)
pval = TensorLoader(permute(Xva), yva, 512)
for name, make in [("MLP", lambda: MLP()), ("SmallCNN", lambda: SmallCNN())]:
    for data_name, (tl, vl) in {"original": (fast_train, fast_val), "permuted": (ptrain, pval)}.items():
        set_seed(0); m = make().to(device)
        h = fit(m, tl, vl, nn.CrossEntropyLoss(), torch.optim.AdamW(m.parameters(), lr=1e-3),
                n_epochs(3), device, amp=True, verbose=False)
        print(f"{name:9s} {data_name:9s} best val acc {max(h['val_acc']):.4f}")
# The MLP is unaffected: it has no notion of which pixels are neighbours, so a fixed permutation is
# just a relabelling of its inputs. The CNN drops sharply: its locality and weight-sharing assumptions
# are now false, so its inductive bias works against it. Architecture = assumptions about the data.

### 📌 Key takeaways

- A conv layer is a shared-weight local neuron: `C_out·C_in·k·k + C_out` parameters, independent of image size.
- Output size = floor((H + 2p − d(k−1) − 1)/s) + 1.
- Architecture encodes assumptions; break them (permuted pixels) and the advantage disappears.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** How many parameters does `nn.Conv2d(64, 128, kernel_size=3)` have? Does it depend on the image size?

<details>
<summary>Answer</summary>

`128·64·3·3 + 128 = 73,856`. No: the same kernels are applied at every position, so the parameter count is independent of the image size.

</details>

**2.** An input of 32×32 passes through `Conv2d(k=3, s=2, p=1)` three times. What is the output's spatial size?

<details>
<summary>Answer</summary>

32 → 16 → 8 → 4. Each stride-2 'same-padded' convolution halves it (`⌊(H+2−3)/2⌋+1`).

</details>

**3.** Why do conv layers that are followed by BatchNorm use `bias=False`?

<details>
<summary>Answer</summary>

BatchNorm subtracts the per-channel mean, which cancels any constant bias, and then adds its own learned shift `β`. The conv bias would be redundant parameters.

</details>

**4.** What is the receptive field of three stacked 3×3 stride-1 convolutions? How many weights do they need (per input/output channel pair) compared with one 7×7?

<details>
<summary>Answer</summary>

7×7. Three 3×3 kernels use 3·9 = 27 weights against 49 for one 7×7, and add two extra non-linearities.

</details>

⏯️ **Resume point.** Continuing in a *fresh* runtime? Run the setup cell (Chapter 0), then the hidden cell below. It re-creates every function and class from the earlier chapters and the data Chapter 13 needs. In a normal top-to-bottom run it detects that nothing is missing and does nothing.

In [ ]:
# @title ⏯️ Resume point for Chapter 13: run only in a fresh runtime, after the setup cell { display-mode: "form" }
if "set_seed" not in globals():
    raise RuntimeError("Run the setup cell of Chapter 0 first, then this cell.")
if "fit" in globals() and "TensorLoader" in globals() and "param_groups_weight_decay" in globals():
    print("Chapter 13: the state it needs is already present -- nothing to restore.")
else:
    from collections import OrderedDict
    from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset, random_split
    from sklearn.datasets import make_moons
    from sklearn.model_selection import train_test_split
    from torch.nn.utils.rnn import pad_sequence
    from torchvision import datasets
    from torchvision.transforms import v2
    from torch.utils.data import IterableDataset, get_worker_info
    from torch.utils.data import WeightedRandomSampler
    from sklearn.metrics import roc_auc_score, average_precision_score
    from torch.optim import lr_scheduler as S
    from torch.profiler import profile, ProfilerActivity, record_function

    def time_matmul(dev, n=2048, reps=10):
        a = torch.randn(n, n, device=dev)
        b = torch.randn(n, n, device=dev)
        a @ b                                        # warm-up
        if dev.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(reps):
            a @ b
        if dev.type == "cuda": torch.cuda.synchronize()   # wait for the GPU to actually finish
        return (time.perf_counter() - t0) / reps

    def tiny_net(x, W1, b1, W2, b2):
        return (torch.tanh(x @ W1 + b1) @ W2 + b2).pow(2).sum()

    class MySigmoid(torch.autograd.Function):
        @staticmethod
        def forward(ctx, z):
            s = 1.0 / (1.0 + torch.exp(-z))
            ctx.save_for_backward(s)           # stash what backward needs
            return s

        @staticmethod
        def backward(ctx, grad_out):           # grad_out = dL/ds
            (s,) = ctx.saved_tensors
            return grad_out * s * (1 - s)      # dL/dz = dL/ds * s(1-s)

    class MLP(nn.Module):
        def __init__(self, in_dim=784, hidden=(256, 128), out_dim=10, p_drop=0.2):
            super().__init__()                    # ALWAYS first: sets up the registration machinery
            self.fc1 = nn.Linear(in_dim, hidden[0])   # assigning a Module to an attribute registers it
            self.fc2 = nn.Linear(hidden[0], hidden[1])
            self.head = nn.Linear(hidden[1], out_dim)
            self.drop = nn.Dropout(p_drop)

        def forward(self, x):                     # x: (N, 1, 28, 28) or (N, 784)
            x = x.flatten(start_dim=1)            # -> (N, 784)
            x = self.drop(F.relu(self.fc1(x)))    # -> (N, 256)
            x = self.drop(F.relu(self.fc2(x)))    # -> (N, 128)
            return self.head(x)                   # -> (N, 10) logits: no softmax (Chapter 5)

    def count_params(m: nn.Module, trainable_only: bool = True) -> int:
        return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

    class Broken(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = [nn.Linear(10, 10) for _ in range(3)]       # plain list: NOT registered

    class Fixed(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(3)])
        def forward(self, x):
            for layer in self.layers:
                x = F.relu(layer(x))
            return x

    class Normalize(nn.Module):
        def __init__(self, mean, std):
            super().__init__()
            self.register_buffer("mean", torch.as_tensor(mean, dtype=torch.float32).view(1, -1, 1, 1))
            self.register_buffer("std", torch.as_tensor(std, dtype=torch.float32).view(1, -1, 1, 1))
        def forward(self, x):
            return (x - self.mean) / self.std

    def deep_relu_mlp(depth=20, width=256):
        layers = []
        for _ in range(depth):
            layers += [nn.Linear(width, width, bias=False), nn.ReLU()]
        return nn.Sequential(*layers)

    def activation_stds(model, x):
        stds, handles = [], []
        for m in model.modules():
            if isinstance(m, nn.ReLU):
                # forward hook signature: hook(module, inputs, output)
                handles.append(m.register_forward_hook(lambda mod, inp, out: stds.append(out.std().item())))
        with torch.no_grad():
            model(x)
        for h in handles:
            h.remove()          # always remove hooks you no longer need
        return stds

    def grad_norms(model):
        return {n: p.grad.norm().item() for n, p in model.named_parameters() if p.grad is not None}

    class ArrayDataset(Dataset):
        # Wraps feature/label arrays. Optionally standardises features with GIVEN statistics.
        def __init__(self, X, y, mean=None, std=None):
            self.X = torch.as_tensor(X, dtype=torch.float32)         # convert once, here
            self.y = torch.as_tensor(y, dtype=torch.int64)
            if mean is not None:
                self.X = (self.X - mean) / std

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]                           # one (features, label) pair

    class RandomSequences(Dataset):
        def __init__(self, n=100):
            g = torch.Generator().manual_seed(0)
            self.lengths = torch.randint(3, 10, (n,), generator=g)
            self.data = [torch.randn(int(L), 4, generator=g) for L in self.lengths]   # each (L_i, 4)
            self.labels = torch.randint(0, 2, (n,), generator=g)
        def __len__(self): return len(self.data)
        def __getitem__(self, i): return self.data[i], self.labels[i]

    def pad_collate(batch):
        seqs, labels = zip(*batch)                          # batch is a list of (seq, label) pairs
        lengths = torch.tensor([len(s) for s in seqs])
        padded = pad_sequence(seqs, batch_first=True)       # (B, L_max, 4), zero-padded
        mask = torch.arange(padded.size(1))[None, :] < lengths[:, None]   # (B, L_max): True = real data
        return padded, lengths, mask, torch.stack(labels)

    def show_images(images, titles=None, ncols=8, cmap="gray", figsize_per=1.4):
        # images: (N, C, H, W) tensor, already in a displayable range
        n = len(images); nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per, nrows * figsize_per * 1.15))
        for i, ax in enumerate(np.array(axes).reshape(-1)):
            ax.axis("off")
            if i < n:
                img = images[i]
                img = img[0] if img.shape[0] == 1 else img.permute(1, 2, 0)   # CHW -> HW or HWC
                ax.imshow(img.clamp(0, 1).numpy(), cmap=cmap if img.ndim == 2 else None)
                if titles is not None:
                    ax.set_title(titles[i], fontsize=7)
        plt.tight_layout(); plt.show()

    def loader_throughput(ds, num_workers, batch_size=256, max_batches=50):
        dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                        pin_memory=(device.type == "cuda"), persistent_workers=num_workers > 0)
        it = iter(dl); next(it)                         # start the workers before timing
        t0 = time.perf_counter(); n = 0
        for i, (xb, yb) in enumerate(it):
            n += len(xb)
            if i + 1 >= max_batches: break
        return n / (time.perf_counter() - t0)

    def tab_split(i):
        return TensorDataset((X_num[i] - num_mean) / num_std, X_cat[i], (y_tab[i] - y_mean) / y_std)

    class TabularNet(nn.Module):
        # Numeric features + one embedding per categorical column -> MLP -> one regression output.
        def __init__(self, n_num, cardinalities=(), emb_dims=(), hidden=64):
            super().__init__()
            self.embs = nn.ModuleList([nn.Embedding(c, d) for c, d in zip(cardinalities, emb_dims)])
            self.mlp = nn.Sequential(nn.Linear(n_num + sum(emb_dims), hidden), nn.ReLU(),
                                     nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

        def forward(self, x_num, x_cat):
            parts = [x_num] + [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]   # each (N, d_i)
            return self.mlp(torch.cat(parts, dim=1))

    def train_regressor(model, epochs=n_epochs(15), lr=3e-3):
        model = model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        loader = DataLoader(tab_train, batch_size=256, shuffle=True)
        for _ in range(epochs):
            model.train()
            for xn, xc, yb in loader:
                loss = F.mse_loss(model(xn.to(device), xc.to(device)), yb.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
        return model

    @torch.no_grad()
    def rmse(model, ds):
        model.eval()
        xn, xc, yb = (t.to(device) for t in ds.tensors)
        mse_standardised = F.mse_loss(model(xn, xc), yb)
        return (mse_standardised.sqrt() * y_std).item()              # back in original units

    class ShardStream(IterableDataset):
        def __init__(self, files, split_by_worker=True):
            self.files, self.split_by_worker = list(files), split_by_worker
        def __iter__(self):
            files = self.files
            info = get_worker_info()                               # None in the main process
            if self.split_by_worker and info is not None:
                files = files[info.id::info.num_workers]           # worker k reads shards k, k+W, k+2W, ...
            for f in files:
                xs, ys = torch.load(f, weights_only=True)
                yield from zip(xs, ys)

    def plot_history(hist, title=""):
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
        ep = range(1, len(hist["train_loss"]) + 1)
        a1.plot(ep, hist["train_loss"], label="train"); a1.plot(ep, hist["val_loss"], label="val")
        a2.plot(ep, hist["train_acc"], label="train"); a2.plot(ep, hist["val_acc"], label="val")
        a1.set_title("loss"); a2.set_title("accuracy")
        for a in (a1, a2):
            a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
        fig.suptitle(title); plt.tight_layout(); plt.show()

    @torch.no_grad()
    def predict(model, loader, device):
        # All predictions, true labels and probabilities for a loader (in loader order).
        model.eval()
        preds, labels, probs = [], [], []
        for x, y in loader:
            p = torch.softmax(model(x.to(device)), dim=1).cpu()
            probs.append(p); preds.append(p.argmax(1)); labels.append(y)
        return torch.cat(preds), torch.cat(labels), torch.cat(probs)

    def confusion_matrix(preds, labels, num_classes):
        # cm[i, j] = number of examples of true class i predicted as class j
        return torch.bincount(labels * num_classes + preds, minlength=num_classes ** 2).reshape(num_classes, num_classes)

    def plot_confusion(cm, class_names, title="Confusion matrix (row-normalised)"):
        cm_n = cm.float() / cm.sum(1, keepdim=True).clamp_min(1)
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
        im = ax.imshow(cm_n, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(class_names)), class_names, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(len(class_names)), class_names, fontsize=8)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                ax.text(j, i, f"{cm_n[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cm_n[i, j] > 0.5 else "black")
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
        plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

    def base_indices(ds):
        # Map a (possibly nested) Subset back to indices of its underlying dataset.
        idx = list(range(len(ds)))
        while isinstance(ds, Subset):
            idx = [ds.indices[i] for i in idx]; ds = ds.dataset
        return idx, ds

    def rare_class_split(ds, positive=8, prevalence=0.02, seed=0):
        idx, base = base_indices(ds)
        labels = torch.as_tensor(base.targets)[idx]
        pos = [i for i, l in zip(idx, labels.tolist()) if l == positive]
        neg = [i for i, l in zip(idx, labels.tolist()) if l != positive]
        n_pos = max(1, round(prevalence * len(neg) / (1 - prevalence)))
        pos = [pos[i] for i in torch.randperm(len(pos), generator=torch.Generator().manual_seed(seed))[:n_pos].tolist()]
        return Subset(base, neg + pos), [0] * len(neg) + [1] * len(pos)

    class BinaryTarget(Dataset):
        # Turn integer class labels into float {0., 1.} targets of shape (1,) for BCEWithLogitsLoss.
        def __init__(self, base, positive): self.base, self.positive = base, positive
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            x, y = self.base[i]
            return x, torch.tensor([float(y == self.positive)])

    @torch.no_grad()
    def predict_proba(model, ds):
        model.eval(); ps, ys = [], []
        for x, y in DataLoader(ds, batch_size=512):
            ps.append(torch.sigmoid(model(x.to(device))).squeeze(1).cpu()); ys.append(y.squeeze(1))
        return torch.cat(ps), torch.cat(ys).long()

    def binary_metrics(p, y, t=0.5):
        pred = (p >= t).long()
        tp = ((pred == 1) & (y == 1)).sum().item(); fp = ((pred == 1) & (y == 0)).sum().item()
        fn = ((pred == 0) & (y == 1)).sum().item()
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        return {"acc": (pred == y).float().mean().item(), "precision": prec, "recall": rec, "F1": f1}

    def train_binary(model, loader, loss_fn, epochs=n_epochs(3)):
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        for _ in range(epochs):
            model.train()
            for x, y in loader:
                loss = loss_fn(model(x.to(device)), y.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
        return model

    def threshold_for_recall(p, y, target_recall=0.9):
        # Highest threshold (hence best precision) whose recall on (p, y) is still >= target_recall.
        for t in torch.sort(p.unique(), descending=True).values.tolist():
            if binary_metrics(p, y, t)["recall"] >= target_recall:
                return t
        return 0.0

    def param_groups_weight_decay(model, weight_decay):
        # Two groups: weight matrices/kernels get decay; biases and norm parameters do not.
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            (decay if p.ndim >= 2 else no_decay).append(p)
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    def lr_trace(make_sched, steps=100, base_lr=0.1):
        p = nn.Parameter(torch.zeros(1))
        opt = torch.optim.SGD([p], lr=base_lr)
        sched = make_sched(opt)
        lrs = []
        for _ in range(steps):
            lrs.append(opt.param_groups[0]["lr"])
            opt.step()               # (no gradient here -- we only care about the lr)
            sched.step()
        return lrs

    def lr_range_test(model, loader, loss_fn, device, opt_fn=lambda params, lr: torch.optim.AdamW(params, lr=lr),
                      lr_min=1e-6, lr_max=1.0, num_steps=200, beta=0.98):
        saved = copy.deepcopy(model.state_dict())
        opt = opt_fn(model.parameters(), lr_min)
        gamma = (lr_max / lr_min) ** (1 / (num_steps - 1))        # multiply the lr by this every step
        lrs, losses, avg, best = [], [], 0.0, float("inf")
        model.train(); it = iter(loader)
        for step in range(num_steps):
            try:
                x, y = next(it)
            except StopIteration:
                it = iter(loader); x, y = next(it)
            lr = lr_min * gamma ** step
            for g in opt.param_groups:
                g["lr"] = lr
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            avg = beta * avg + (1 - beta) * loss.item()           # exponential moving average...
            smoothed = avg / (1 - beta ** (step + 1))             # ...with Adam-style bias correction
            lrs.append(lr); losses.append(smoothed)
            best = min(best, smoothed)
            if not math.isfinite(smoothed) or smoothed > 4 * best:    # diverged: stop early
                break
        model.load_state_dict(saved)                              # undo the test's training
        return lrs, losses

    class EarlyStopping:
        # Stops when the monitored metric has not improved for `patience` epochs,
        # and keeps a CPU copy of the best weights so they can be restored.
        def __init__(self, patience=3, min_delta=0.0, mode="max"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best, self.best_epoch, self.bad_epochs, self.best_state = None, -1, 0, None

        def _improved(self, value):
            if self.best is None:
                return True
            return value > self.best + self.min_delta if self.mode == "max" else value < self.best - self.min_delta

        def step(self, value, model, epoch):
            # Call once per epoch. Returns True when training should stop.
            if self._improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, epoch, 0
                self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore_best(self, model):
            model.load_state_dict(self.best_state)

    def rng_state():
        return {"torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate()}

    def set_rng_state(state):
        torch.set_rng_state(state["torch"])
        if torch.cuda.is_available() and state["cuda"]:
            torch.cuda.set_rng_state_all(state["cuda"])
        random.setstate(state["python"])

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None, config=None, **extra):
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch, "best_metric": best_metric,
            "config": config or {}, "rng": rng_state(),
            "torch_version": str(torch.__version__), **extra,   # str(): see the note below
        }
        path = Path(path)
        tmp = path.with_name(path.name + ".tmp")
        torch.save(ckpt, tmp)
        os.replace(tmp, path)       # atomic rename: a crash mid-save never leaves a corrupt checkpoint

    def load_checkpoint(path, model, optimizer=None, scheduler=None, map_location="cpu"):
        ckpt = torch.load(path, map_location=map_location, weights_only=True)
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        set_rng_state(ckpt["rng"])
        return ckpt

    def make_run(seed):
        # Build every training object from scratch, as a fresh process would.
        torch.manual_seed(seed)
        model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 3))
        opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=6)
        gen = torch.Generator().manual_seed(seed)                       # controls the shuffle order
        loader = DataLoader(TensorDataset(Xr, yr), batch_size=32, shuffle=True, generator=gen)
        return model, opt, sched, loader, gen

    def run_epochs(model, opt, sched, loader, n):
        for _ in range(n):
            model.train()
            for xb, yb in loader:
                loss = F.cross_entropy(model(xb), yb)
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()

    def make_deterministic(seed=0):
        # Maximum GPU determinism for debugging or auditing. Not needed (and slower) for normal training.
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # must be set before the first cuBLAS call to take effect
        set_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)   # warn_only: warn instead of raising

    def overfit_one_batch(model, batch, loss_fn, steps=200, lr=1e-3):
        # A correct model and loop should drive the training loss on ONE batch to ~0.
        x, y = (t.to(device) for t in batch)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        model.train()
        for step in range(steps):
            logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            if step % 50 == 0 or step == steps - 1:
                acc = (logits.argmax(1) == y).float().mean().item()
                print(f"step {step:3d}: loss {loss.item():.4f}  batch acc {acc:.2f}")
        return loss.item()

    def gradient_report(model):
        # Call after loss.backward(). Flags parameters with missing, non-finite or zero gradients.
        problems = []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if p.grad is None:
                problems.append(f"{name}: grad is None (unused in forward, or graph cut by .detach()/.item()?)")
            elif not torch.isfinite(p.grad).all():
                problems.append(f"{name}: non-finite gradient")
            elif p.grad.abs().max() == 0:
                problems.append(f"{name}: gradient exactly zero (dead ReLUs? frozen upstream?)")
            else:
                print(f"{name:14s} |grad| = {p.grad.norm():.2e}   |w| = {p.norm():.2e}   ratio = {p.grad.norm() / p.norm():.1e}")
        for msg in problems:
            print("⚠️", msg)

    class HasUnusedLayer(MLP):
        def __init__(self):
            super().__init__()
            self.extra = nn.Linear(10, 10)          # defined but never used in forward

    def buggy_training(epochs=2):
        set_seed(0)
        model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
                              nn.Linear(256, 10), nn.Softmax(dim=1)).to(device)
        loader = DataLoader(sorted_train, batch_size=128, shuffle=False)
        vloader = DataLoader(fmnist_val, batch_size=512)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()
        history = []
        for epoch in range(epochs):
            running = 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                loss = loss_fn(model(x), y)
                loss.backward()
                opt.step()
                running += loss
            model.eval()
            correct = 0
            for x, y in vloader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
            history.append(correct / len(fmnist_val))
            print(f"epoch {epoch+1}: train loss {running.item() / len(loader):.3f} | val acc {history[-1]:.4f}")
        return history

    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    def benchmark(fn, warmup=5, iters=20):
        # Average wall-clock seconds per call of fn(), with warm-up and GPU synchronisation (§1.8).
        for _ in range(warmup):
            fn()
        sync()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        sync()
        return (time.perf_counter() - t0) / iters

    def materialize(ds, batch_size=2048):
        # Run a dataset's transforms once and stack everything into two tensors.
        xs, ys = [], []
        for x, y in DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS):
            xs.append(x); ys.append(y)
        return torch.cat(xs), torch.cat(ys)

    class TensorLoader:
        # A minimal DataLoader replacement for tensors that already live on the target device.
        def __init__(self, X, y, batch_size, shuffle=False, drop_last=False, seed=0):
            self.X, self.y, self.bs, self.shuffle, self.drop_last = X, y, batch_size, shuffle, drop_last
            self.gen = torch.Generator(device=X.device).manual_seed(seed)
        def __len__(self):
            n = len(self.X)
            return n // self.bs if self.drop_last else math.ceil(n / self.bs)
        def __iter__(self):
            n = len(self.X)
            idx = torch.randperm(n, device=self.X.device, generator=self.gen) if self.shuffle else torch.arange(n, device=self.X.device)
            for i in range(len(self)):
                j = idx[i * self.bs:(i + 1) * self.bs]
                yield self.X[j], self.y[j]

    def wide_mlp():
        return nn.Sequential(nn.Flatten(), nn.Linear(784, WIDTH), nn.ReLU(),
                             nn.Linear(WIDTH, WIDTH), nn.ReLU(), nn.Linear(WIDTH, 10)).to(device)

    def make_step(model, use_amp):
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
        def step():
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        return step

    def pick_amp_dtype(device):
        # bf16 where the hardware supports it natively (Ampere+), fp16 on older GPUs (T4), None on CPU/MPS.
        if device.type != "cuda":
            return None
        return torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

    def train_one_epoch(model, loader, loss_fn, optimizer, device, scheduler=None, max_grad_norm=None,
                        scaler=None, amp_dtype=None, channels_last=False):
        model.train()
        use_amp = amp_dtype is not None
        if scaler is None:
            scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
        total_loss = torch.zeros((), device=device)
        total_correct = torch.zeros((), device=device, dtype=torch.long)
        total_n = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if channels_last:
                x = x.contiguous(memory_format=torch.channels_last)   # see Chapter 13
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()
            n = y.size(0)
            total_loss += loss.detach().float() * n
            total_correct += (logits.argmax(dim=1) == y).sum()
            total_n += n
        return total_loss.item() / total_n, total_correct.item() / total_n

    @torch.no_grad()
    def evaluate(model, loader, loss_fn, device, amp_dtype=None, channels_last=False):
        model.eval()
        total_loss = torch.zeros((), device=device)
        total_correct = torch.zeros((), device=device, dtype=torch.long)
        total_n = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if channels_last:
                x = x.contiguous(memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                logits = model(x)
                loss = loss_fn(logits, y)
            total_loss += loss.float() * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum()
            total_n += y.size(0)
        return total_loss.item() / total_n, total_correct.item() / total_n

    def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, device, scheduler=None,
            scheduler_interval="batch", max_grad_norm=None, early_stopping=None, eval_loss_fn=None,
            amp=False, channels_last=False, checkpoint_path=None, verbose=True):
        # The full-featured training driver used for the rest of the notebook.
        eval_loss_fn = eval_loss_fn or loss_fn
        amp_dtype = pick_amp_dtype(device) if amp else None
        scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
        hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": [], "epoch_time": []}
        best_acc = -1.0
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device,
                                              scheduler if scheduler_interval == "batch" else None,
                                              max_grad_norm, scaler, amp_dtype, channels_last)
            va_loss, va_acc = evaluate(model, val_loader, eval_loss_fn, device, amp_dtype, channels_last)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            dt = time.perf_counter() - t0
            for k, v in zip(("train_loss", "train_acc", "val_loss", "val_acc", "lr", "epoch_time"),
                            (tr_loss, tr_acc, va_loss, va_acc, optimizer.param_groups[0]["lr"], dt)):
                hist[k].append(v)
            if checkpoint_path is not None and va_acc > best_acc:       # keep the best model on disk
                best_acc = va_acc
                save_checkpoint(checkpoint_path, model, optimizer, scheduler, epoch=epoch, best_metric=va_acc)
            if verbose:
                print(f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                      f"val loss {va_loss:.4f} acc {va_acc:.4f} | lr {hist['lr'][-1]:.2e} | {dt:.1f}s")
            if early_stopping is not None and early_stopping.step(va_acc, model, epoch):
                if verbose:
                    print(f"early stopping (best val acc {early_stopping.best:.4f} at epoch {early_stopping.best_epoch})")
                break
        if early_stopping is not None and early_stopping.best_state is not None:
            early_stopping.restore_best(model)
        return hist

    def grid(ax, n, title, hi=None, col="#fff2cc"):
        for i in range(n):
            for j in range(n):
                on = hi is not None and hi[0] <= i < hi[0] + hi[2] and hi[1] <= j < hi[1] + hi[2]
                ax.add_patch(plt.Rectangle((j, n - 1 - i), 1, 1, fc=col if on else "white", ec="0.4"))
        ax.set_xlim(-0.2, n + 0.2); ax.set_ylim(-0.2, n + 0.2); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title, fontsize=9)

    def conv_out(h, k, s=1, p=0, d=1):
        return (h + 2 * p - d * (k - 1) - 1) // s + 1

    def receptive_field(layers):
        # layers: list of (kernel, stride). Returns the theoretical receptive field size.
        r, j = 1, 1
        for k, s in layers:
            r, j = r + (k - 1) * j, j * s
        return r

    def conv_bn_relu(c_in, c_out, stride=1):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False),
                             nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

    class SmallCNN(nn.Module):
        def __init__(self, in_ch=1, num_classes=10, p_drop=0.3):
            super().__init__()
            self.features = nn.Sequential(
                conv_bn_relu(in_ch, 32), conv_bn_relu(32, 32), nn.MaxPool2d(2),     # 28 -> 14
                conv_bn_relu(32, 64), conv_bn_relu(64, 64), nn.MaxPool2d(2),        # 14 -> 7
                conv_bn_relu(64, 128),                                              # 7x7x128
            )
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p_drop), nn.Linear(128, num_classes))

        def forward(self, x):
            return self.head(self.features(x))

    pin, USE_AMP = device.type == "cuda", device.type == "cuda"
    print("Chapter 13: functions, classes and data from earlier chapters restored.")


---
# Chapter 13. Residual networks and the CIFAR-10 project

## 13.1 Twenty-five years of architectures in one table

| Year | Network | Key idea | What it taught the field |
| --- | --- | --- | --- |
| 1998 | LeNet-5 | conv → pool → conv → pool → dense | Convolutions plus backprop work for vision |
| 2012 | AlexNet | Bigger, ReLU, dropout, GPUs, data augmentation | Scale, plus the right tricks, wins (Tutorial §8.4) |
| 2014 | VGG | Only 3×3 convolutions, stacked deep | Small kernels stacked deep beat large kernels (§12.7) |
| 2015 | **ResNet** | **Residual connections**: `y = x + F(x)` | Depth stops hurting. Networks of 100+ layers train reliably |
| 2019+ | EfficientNet, ConvNeXt, ViT | Systematic scaling; Transformer-era design | Covered by the "next course" (Chapter 17) |

## 13.2 Why plain deep networks fail, and the residual fix

Stack more `conv → BN → ReLU` layers onto a working network and, beyond a certain depth, **training** accuracy gets *worse*. This is not overfitting: the deeper network could in principle copy the shallower one and set its extra layers to the identity. But learning an identity mapping through a stack of non-linear layers turns out to be hard for SGD.

A **residual block** makes the identity the default. It computes `y = ReLU(x + F(x))`, where `F` is a small stack of layers. If the extra layers are not useful, the block only has to learn `F ≈ 0`, which is easy. In the backward pass, `∂y/∂x = I + ∂F/∂x` provides a path with derivative exactly 1. That gradient highway is the strongest defence against vanishing gradients (Tutorial §9.7, item 3).

🎯 **In this chapter you will learn to:** explain and implement residual blocks; train a ResNet on CIFAR-10 with augmentation, AMP and OneCycle; find the input-pipeline bottleneck.

In [ ]:
flow_diagram(
    {"x": (0.05, 0.5, "x", "#cfe8ff"), "c1": (0.24, 0.72, "conv 3×3 → BN → ReLU", "#fff2cc"),
     "c2": (0.50, 0.72, "conv 3×3 → BN\n(γ initialised to 0)", "#fff2cc"), "add": (0.70, 0.5, "+", "#f8cecc"),
     "relu": (0.84, 0.5, "ReLU", "#fff2cc"), "y": (0.96, 0.5, "y", "#cfe8ff"),
     "sc": (0.37, 0.18, "shortcut: identity, or 1×1 conv + BN\nwhen the shape changes", "#e1d5e7")},
    [("x", "c1", "", "k"), ("c1", "c2", "", "k"), ("c2", "add", "F(x)", "k"), ("x", "sc", "", "tab:purple"),
     ("sc", "add", "x", "tab:purple"), ("add", "relu", "", "k"), ("relu", "y", "", "k")],
    title="A residual BasicBlock: y = ReLU(F(x) + shortcut(x)). Gradients flow back through the shortcut unchanged.", size=(11, 3.2))

When a block changes the resolution or the channel count, `x` and `F(x)` have different shapes. The shortcut then uses a **projection**, a 1×1 convolution with the same stride, followed by BN.

In [ ]:
class BasicBlock(nn.Module):
    # The ResNet-18/34 block: two 3x3 convs with a residual connection.
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        nn.init.zeros_(self.bn2.weight)       # zero-init the last BN gamma: each block starts as the identity (Tutorial §9.6)
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:      # shapes differ -> projection shortcut
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class CifarResNet(nn.Module):
    # A ResNet-18-style network for 32x32 inputs: a 3x3 stem (no early downsampling), 4 stages, GAP, linear head.
    def __init__(self, widths=(64, 128, 256, 512), blocks=(2, 2, 2, 2), num_classes=10, in_ch=3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, widths[0], 3, padding=1, bias=False),
                                  nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        stages, c_in = [], widths[0]
        for i, (w, n) in enumerate(zip(widths, blocks)):
            stride = 1 if i == 0 else 2                          # halve the resolution at each new stage
            layers = [BasicBlock(c_in, w, stride)] + [BasicBlock(w, w) for _ in range(n - 1)]
            stages.append(nn.Sequential(*layers)); c_in = w
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c_in, num_classes))
        for m in self.modules():                                 # He init for convs (Tutorial §9.6)
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x):
        return self.head(self.stages(self.stem(x)))

CIFAR_ARCH = dict(widths=(32, 64, 128, 256), blocks=(2, 2, 2, 2))   # half-width ResNet-18: fast on a T4
probe = CifarResNet(**CIFAR_ARCH)
print(f"CifarResNet{CIFAR_ARCH}: {count_params(probe)/1e6:.2f}M parameters "
      f"(full-width ResNet-18 layout: {count_params(CifarResNet())/1e6:.1f}M)")
h = torch.randn(2, 3, 32, 32); h = probe.stem(h); print("stem:", tuple(h.shape))
for i, st in enumerate(probe.stages):
    h = st(h); print(f"stage {i+1}:", tuple(h.shape))

## 13.3 CIFAR-10 data, augmentation, and the transform trap

CIFAR-10 has 60,000 32×32 colour images in 10 classes: 50,000 for training and 10,000 for testing. We hold out 5,000 training images for validation.

**Data augmentation** (Tutorial §12.6) is the most effective regulariser here. The standard CIFAR recipe is a **random crop** with 4 pixels of padding plus a **horizontal flip**. Augmentation is applied **only to the training split**, so we use the `TransformDataset` pattern of Exercise 6.2, which gives each split its own transform:

In [ ]:
class TransformDataset(Dataset):
    # Apply `transform` to the inputs of an (untransformed) base dataset.
    def __init__(self, base, transform):
        self.base, self.transform = base, transform
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        return self.transform(x), y

cifar_raw = datasets.CIFAR10(DATA_DIR, train=True, download=True)            # PIL images, no transform
cifar_test_raw = datasets.CIFAR10(DATA_DIR, train=False, download=True)
CIFAR_CLASSES = cifar_raw.classes

perm = torch.randperm(len(cifar_raw), generator=torch.Generator().manual_seed(0))
tr_idx, va_idx = perm[:45_000].tolist(), perm[45_000:].tolist()

# Normalisation statistics from the TRAINING split only (cifar_raw.data: uint8 array of shape (50000, 32, 32, 3))
px = torch.from_numpy(cifar_raw.data[tr_idx]).float().div(255)
CIFAR_MEAN, CIFAR_STD = px.mean(dim=(0, 1, 2)).tolist(), px.std(dim=(0, 1, 2)).tolist()
del px
print("mean:", [round(v, 4) for v in CIFAR_MEAN], "std:", [round(v, 4) for v in CIFAR_STD])

to_normed = [v2.ToDtype(torch.float32, scale=True), v2.Normalize(CIFAR_MEAN, CIFAR_STD)]
cifar_train_tf = v2.Compose([v2.ToImage(), v2.RandomCrop(32, padding=4, padding_mode="reflect"),
                             v2.RandomHorizontalFlip(), *to_normed])
cifar_eval_tf = v2.Compose([v2.ToImage(), *to_normed])

cifar_train = maybe_subset(TransformDataset(Subset(cifar_raw, tr_idx), cifar_train_tf))
cifar_val = maybe_subset(TransformDataset(Subset(cifar_raw, va_idx), cifar_eval_tf))
cifar_test = maybe_subset(TransformDataset(cifar_test_raw, cifar_eval_tf))
print(len(cifar_train), len(cifar_val), len(cifar_test))

# Look at the augmentation: the same image, 8 random augmentations
base_img = cifar_raw[tr_idx[0]][0]
aug = torch.stack([cifar_train_tf(base_img) for _ in range(8)])
mean_t, std_t = torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1), torch.tensor(CIFAR_STD).view(1, 3, 1, 1)
show_images(aug * std_t + mean_t, [CIFAR_CLASSES[cifar_raw[tr_idx[0]][1]]] * 8)

## 13.4 Training on the GPU, properly configured

The training settings, and the reason for each:

| Setting | Value | Why |
| --- | --- | --- |
| Optimizer | SGD, momentum 0.9, Nesterov | A strong, standard choice for ResNets trained from scratch on CIFAR (AdamW also works well with tuning) |
| Learning rate | OneCycle, peak 0.1 | Warm-up stabilises the early, high-gradient phase. The cosine tail anneals into a good minimum |
| Weight decay | 5e-4, on weights only (§8.3) | The classic CIFAR value |
| Label smoothing | 0.1 | Small, consistent gain (Tutorial §12.7) |
| Batch size | 256, `drop_last=True` | Good GPU use. A constant batch shape avoids recompiles and BN edge cases |
| AMP | fp16 on the T4 | About 2× faster (Chapter 11) |
| `channels_last` | on | Tensor-core convolutions prefer NHWC memory layout. Often 10–30% faster with AMP |
| `cudnn.benchmark` | `True` | Input shapes are fixed, so cuDNN may time several convolution algorithms once and cache the fastest |

In [ ]:
torch.backends.cudnn.benchmark = True

CIFAR_BATCH = 256
cifar_train_loader = DataLoader(cifar_train, batch_size=CIFAR_BATCH, shuffle=True, drop_last=True,
                                num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=NUM_WORKERS > 0)
cifar_val_loader = DataLoader(cifar_val, batch_size=512, num_workers=NUM_WORKERS, pin_memory=pin)
cifar_test_loader = DataLoader(cifar_test, batch_size=512, num_workers=NUM_WORKERS, pin_memory=pin)

set_seed(0)
resnet = CifarResNet(**CIFAR_ARCH).to(device).to(memory_format=torch.channels_last)
EPOCHS_CIFAR = n_epochs(30)
opt = torch.optim.SGD(param_groups_weight_decay(resnet, 5e-4), lr=0.1, momentum=0.9, nesterov=True)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.1, epochs=EPOCHS_CIFAR,
                                            steps_per_epoch=len(cifar_train_loader), pct_start=0.15)
CIFAR_CKPT = ARTIFACT_DIR / "cifar_resnet_best.pt"
hist_resnet = fit(resnet, cifar_train_loader, cifar_val_loader,
                  loss_fn=nn.CrossEntropyLoss(label_smoothing=0.1), eval_loss_fn=nn.CrossEntropyLoss(),
                  optimizer=opt, epochs=EPOCHS_CIFAR, device=device, scheduler=sched,
                  amp=True, channels_last=True, checkpoint_path=CIFAR_CKPT)
print(f"total training time: {record('CIFAR-10 ResNet: training time (min)', sum(hist_resnet['epoch_time'])/60):.1f} min")
plot_history(hist_resnet, "CifarResNet on CIFAR-10")

On a Colab T4, expect roughly 10–15 minutes and a final validation accuracy in the **low 90s**. These are estimates; your measured values are recorded in `RESULTS` (§17.6). Colab's two CPU cores, which run the per-image augmentation, are often the bottleneck. Watch the epoch time, and compare it with Exercise 13.3.

Early in the run, training accuracy is often *below* validation accuracy. That is normal here (Tutorial §12.8): training batches are augmented, and they are measured while the weights are still changing under a high learning rate, whereas validation images are clean. As the learning rate anneals, training accuracy typically overtakes validation accuracy.

## 13.5 Final evaluation

We reload the **best** checkpoint by validation accuracy (saved by `fit`), then touch the test set exactly once.

In [ ]:
best = CifarResNet(**CIFAR_ARCH).to(device).to(memory_format=torch.channels_last)
ckpt = torch.load(CIFAR_CKPT, map_location=device, weights_only=True)
best.load_state_dict(ckpt["model"])
print(f"best checkpoint: epoch {ckpt['epoch']}, val acc {ckpt['best_metric']:.4f}")

c_preds, c_labels, c_probs = predict(best, cifar_test_loader, device)
cifar_test_acc = record("CIFAR-10 ResNet: test acc", (c_preds == c_labels).float().mean().item())
se = math.sqrt(cifar_test_acc * (1 - cifar_test_acc) / len(c_labels))
print(f"CIFAR-10 test accuracy: {cifar_test_acc:.4f} ± {1.96 * se:.4f} (95% CI)")
plot_confusion(confusion_matrix(c_preds, c_labels, 10), CIFAR_CLASSES)

The usual confusion is **cat ↔ dog**, followed by the vehicle pairs (automobile ↔ truck) and the animal pairs (deer ↔ horse, bird ↔ airplane). These are errors a human would understand at 32×32 pixels.

## 13.6 Looking inside: feature maps

Forward hooks (§4.7) let us capture each stage's output for one image. Early stages respond to edges and colours at full resolution. Later stages are coarse and abstract.

In [ ]:
feats = {}
hooks = [best.stages[i].register_forward_hook(lambda m, inp, out, i=i: feats.__setitem__(i, out.detach().float().cpu()))
         for i in range(4)]
img, label = cifar_test[3]
with torch.no_grad():
    best.eval(); best(img.unsqueeze(0).to(device).contiguous(memory_format=torch.channels_last))
for h_ in hooks:
    h_.remove()
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
axes[0].imshow((img * std_t[0] + mean_t[0]).permute(1, 2, 0).clamp(0, 1)); axes[0].set_title(CIFAR_CLASSES[label])
for i in range(4):
    fm = feats[i][0]
    axes[i + 1].imshow(fm[fm.mean((1, 2)).argmax()], cmap="viridis")     # the most active channel
    axes[i + 1].set_title(f"stage {i+1}: {tuple(fm.shape)}", fontsize=8)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## 🧪 Exercises

**Exercise 13.1: test-time augmentation (TTA).** `★☆☆ · ~15 min` A cheap, widely used trick is to average the model's softmax outputs over the original test image and its horizontal flip. Implement `predict_tta(model, loader)` and compare its test accuracy with `cifar_test_acc`. Why does this help, and what does it cost at inference time?

In [ ]:
# ✍️ Your attempt here.
# Hint: torch.flip(x, dims=[3]) flips the width dimension of an (N, C, H, W) batch.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
@torch.no_grad()
def predict_tta(model, loader):
    model.eval(); correct = n = 0
    for x, y in loader:
        x, y = x.to(device).contiguous(memory_format=torch.channels_last), y.to(device)
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            p = torch.softmax(model(x), 1) + torch.softmax(model(torch.flip(x, dims=[3])), 1)
        correct += (p.argmax(1) == y).sum().item(); n += y.numel()
    return correct / n

print(f"plain: {cifar_test_acc:.4f} | with flip TTA: {predict_tta(best, cifar_test_loader):.4f}")
# Typically +0.2 to +0.5 points: the two views make partly independent errors, and averaging is a tiny
# ensemble. It costs 2x the inference compute -- often unacceptable in latency-critical production.

**Exercise 13.2: the value of augmentation.** `★☆☆ · ~10 min + training` Retrain the same network for 15 epochs **without** the random crop and flip (use `cifar_eval_tf` for training too), and plot its history next to the augmented run. Interpret the train–validation gap using Tutorial §12.6.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
noaug_train = maybe_subset(TransformDataset(Subset(cifar_raw, tr_idx), cifar_eval_tf))
noaug_loader = DataLoader(noaug_train, batch_size=CIFAR_BATCH, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=pin)
set_seed(0)
m = CifarResNet(**CIFAR_ARCH).to(device).to(memory_format=torch.channels_last)
E = n_epochs(15)
o = torch.optim.SGD(param_groups_weight_decay(m, 5e-4), lr=0.1, momentum=0.9, nesterov=True)
s = torch.optim.lr_scheduler.OneCycleLR(o, max_lr=0.1, epochs=E, steps_per_epoch=len(noaug_loader), pct_start=0.15)
h_noaug = fit(m, noaug_loader, cifar_val_loader, nn.CrossEntropyLoss(label_smoothing=0.1), o, E, device,
              scheduler=s, eval_loss_fn=nn.CrossEntropyLoss(), amp=True, channels_last=True, verbose=False)
plot_history(h_noaug, "No augmentation")
print(f"no augmentation: final train acc {h_noaug['train_acc'][-1]:.4f}, best val acc {max(h_noaug['val_acc']):.4f}")
# Without augmentation the network memorises the training set (train acc -> ~100%) and validation
# accuracy plateaus several points lower (typically high 80s): a large variance gap. Augmentation
# effectively multiplies the dataset with label-preserving variations, closing most of that gap.

**Exercise 13.3: is the GPU waiting for data?** `★★☆ · ~20 min` Measure (a) the loader-only throughput of `cifar_train_loader` (images/s, using `loader_throughput` from §6.6) and (b) the model-only training throughput on a fixed GPU batch (with `benchmark` from §11.1, AMP and `channels_last` on). Which is the bottleneck on your Colab instance? Propose two fixes.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
data_ips = loader_throughput(cifar_train, NUM_WORKERS, batch_size=CIFAR_BATCH, max_batches=30)
m = CifarResNet(**CIFAR_ARCH).to(device).to(memory_format=torch.channels_last)
o = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
sc = torch.amp.GradScaler(device.type, enabled=USE_AMP)
xb_ = torch.randn(CIFAR_BATCH, 3, 32, 32, device=device).contiguous(memory_format=torch.channels_last)
yb_ = torch.randint(0, 10, (CIFAR_BATCH,), device=device)
def model_step():
    with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
        loss = F.cross_entropy(m(xb_), yb_)
    o.zero_grad(); sc.scale(loss).backward(); sc.step(o); sc.update()
model_ips = CIFAR_BATCH / benchmark(model_step, warmup=3, iters=10)
print(f"data loader: {data_ips:8.0f} img/s | model: {model_ips:8.0f} img/s")
print("bottleneck:", "DATA" if data_ips < model_ips else "MODEL")
# On a 2-core Colab T4 the loader is usually the bottleneck for this small model. Fixes: (1) keep the
# uint8 dataset on the GPU and do crop+flip in batch on the GPU (no per-image Python); (2) more CPU
# cores / workers; (3) a bigger model, which would make the GPU the bottleneck (the data cost is "free").

**Exercise 13.4: augmentation on the GPU.** `★★★ · ~45 min` Implement fix (1) from Exercise 13.3. Keep the training images as one `uint8` tensor on the GPU, about 140 MB for 45,000 images. Then implement `gpu_crop_flip(x_u8, pad=4)`, which applies a *different* random 32×32 crop (from a reflect-padded 40×40 image) and a random horizontal flip to every image of a batch, **without a Python loop over images**. Normalise and train one epoch with it. Compare the epoch time with one epoch of `cifar_train_loader`.

In [ ]:
# ✍️ Your attempt here.
# Hint: Build per-image row and column index tensors: rows = offset_y[:, None] + arange(32), and index the padded batch with broadcasting.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def gpu_crop_flip(x_u8, pad=4):
    N, C, H, W = x_u8.shape
    xp = F.pad(x_u8.float(), (pad, pad, pad, pad), mode="reflect")          # (N, C, H+2p, W+2p)
    oy = torch.randint(0, 2 * pad + 1, (N,), device=x_u8.device)
    ox = torch.randint(0, 2 * pad + 1, (N,), device=x_u8.device)
    rows = oy[:, None] + torch.arange(H, device=x_u8.device)                # (N, H)
    cols = ox[:, None] + torch.arange(W, device=x_u8.device)                # (N, W)
    n_idx = torch.arange(N, device=x_u8.device)[:, None, None, None]
    c_idx = torch.arange(C, device=x_u8.device)[None, :, None, None]
    out = xp[n_idx, c_idx, rows[:, None, :, None], cols[:, None, None, :]]   # (N, C, H, W): one crop per image
    flip = torch.rand(N, device=x_u8.device) < 0.5
    return torch.where(flip[:, None, None, None], out.flip(3), out)          # values still in 0..255

class GPUAugLoader:
    # Yields normalised, augmented batches from a uint8 tensor that lives on the GPU.
    def __init__(self, X_u8, y, batch_size, mean, std):
        self.base = TensorLoader(X_u8, y, batch_size, shuffle=True, drop_last=True)
        self.mean = torch.tensor(mean, device=X_u8.device).view(1, 3, 1, 1) * 255
        self.std = torch.tensor(std, device=X_u8.device).view(1, 3, 1, 1) * 255
    def __len__(self): return len(self.base)
    def __iter__(self):
        for xb, yb in self.base:
            yield (gpu_crop_flip(xb) - self.mean) / self.std, yb

n_use = len(cifar_train)                                                    # 45,000 (512 in smoke-test mode)
Xc = torch.from_numpy(cifar_raw.data[tr_idx[:n_use]]).permute(0, 3, 1, 2).contiguous().to(device)   # uint8 (N,3,32,32)
yc = torch.tensor([cifar_raw.targets[i] for i in tr_idx[:n_use]], device=device)
gpu_loader = GPUAugLoader(Xc, yc, CIFAR_BATCH, CIFAR_MEAN, CIFAR_STD)
show_images(gpu_crop_flip(Xc[:1].repeat(8, 1, 1, 1)).cpu() / 255, ncols=8)   # one image, 8 random augmentations

times = {}
for name, ld in [("CPU DataLoader", cifar_train_loader), ("GPU augmentation", gpu_loader)]:
    set_seed(0)
    m = CifarResNet(**CIFAR_ARCH).to(device).to(memory_format=torch.channels_last)
    o = torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9, nesterov=True)
    t0 = time.perf_counter()
    train_one_epoch(m, ld, nn.CrossEntropyLoss(), o, device, amp_dtype=pick_amp_dtype(device), channels_last=True)
    sync(); times[name] = time.perf_counter() - t0
    print(f"{name:17s}: {times[name]:6.1f} s per epoch")
del Xc, yc, gpu_loader, m, o
# If the CPU pipeline was the bottleneck (Exercise 13.3), the GPU version is several times faster.

### 📌 Key takeaways

- Residual connections make the identity easy and give gradients a direct path.
- Augment training data only; use `TransformDataset`-style wrappers per split.
- Check whether data loading or the model is the bottleneck.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Why does a residual block help gradients flow?

<details>
<summary>Answer</summary>

`y = x + F(x)` gives `∂y/∂x = I + ∂F/∂x`. The identity term passes the gradient back unchanged, however small `∂F/∂x` becomes, so the gradient does not have to pass through a long product of small Jacobians.

</details>

**2.** When does a residual block need a projection shortcut?

<details>
<summary>Answer</summary>

When `F(x)` changes the spatial size (stride > 1) or the number of channels. `x` must then be mapped to the same shape, with a 1×1 convolution of the same stride followed by BN.

</details>

**3.** You apply `RandomCrop` to the CIFAR validation set by accident. What happens to your metrics?

<details>
<summary>Answer</summary>

Validation accuracy becomes noisy and biased downwards, and it changes from run to run. Model selection and early stopping then use a noisy signal. Validation and test inputs must be deterministic.

</details>

**4.** Why is `cudnn.benchmark = True` safe here but potentially slower for variable-size inputs?

<details>
<summary>Answer</summary>

cuDNN times several algorithms for each new input shape and caches the fastest. With fixed shapes that cost is paid once. With constantly changing shapes it is paid repeatedly.

</details>

⏯️ **Resume point.** Continuing in a *fresh* runtime? Run the setup cell (Chapter 0), then the hidden cell below. It re-creates every function and class from the earlier chapters and the data Chapter 14 needs. In a normal top-to-bottom run it detects that nothing is missing and does nothing.

In [ ]:
# @title ⏯️ Resume point for Chapter 14: run only in a fresh runtime, after the setup cell { display-mode: "form" }
if "set_seed" not in globals():
    raise RuntimeError("Run the setup cell of Chapter 0 first, then this cell.")
if "fit" in globals() and "TransformDataset" in globals() and "pick_amp_dtype" in globals():
    print("Chapter 14: the state it needs is already present -- nothing to restore.")
else:
    from collections import OrderedDict
    from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset, random_split
    from sklearn.datasets import make_moons
    from sklearn.model_selection import train_test_split
    from torch.nn.utils.rnn import pad_sequence
    from torchvision import datasets
    from torchvision.transforms import v2
    from torch.utils.data import IterableDataset, get_worker_info
    from torch.utils.data import WeightedRandomSampler
    from sklearn.metrics import roc_auc_score, average_precision_score
    from torch.optim import lr_scheduler as S
    from torch.profiler import profile, ProfilerActivity, record_function

    def time_matmul(dev, n=2048, reps=10):
        a = torch.randn(n, n, device=dev)
        b = torch.randn(n, n, device=dev)
        a @ b                                        # warm-up
        if dev.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(reps):
            a @ b
        if dev.type == "cuda": torch.cuda.synchronize()   # wait for the GPU to actually finish
        return (time.perf_counter() - t0) / reps

    def tiny_net(x, W1, b1, W2, b2):
        return (torch.tanh(x @ W1 + b1) @ W2 + b2).pow(2).sum()

    class MySigmoid(torch.autograd.Function):
        @staticmethod
        def forward(ctx, z):
            s = 1.0 / (1.0 + torch.exp(-z))
            ctx.save_for_backward(s)           # stash what backward needs
            return s

        @staticmethod
        def backward(ctx, grad_out):           # grad_out = dL/ds
            (s,) = ctx.saved_tensors
            return grad_out * s * (1 - s)      # dL/dz = dL/ds * s(1-s)

    class MLP(nn.Module):
        def __init__(self, in_dim=784, hidden=(256, 128), out_dim=10, p_drop=0.2):
            super().__init__()                    # ALWAYS first: sets up the registration machinery
            self.fc1 = nn.Linear(in_dim, hidden[0])   # assigning a Module to an attribute registers it
            self.fc2 = nn.Linear(hidden[0], hidden[1])
            self.head = nn.Linear(hidden[1], out_dim)
            self.drop = nn.Dropout(p_drop)

        def forward(self, x):                     # x: (N, 1, 28, 28) or (N, 784)
            x = x.flatten(start_dim=1)            # -> (N, 784)
            x = self.drop(F.relu(self.fc1(x)))    # -> (N, 256)
            x = self.drop(F.relu(self.fc2(x)))    # -> (N, 128)
            return self.head(x)                   # -> (N, 10) logits: no softmax (Chapter 5)

    def count_params(m: nn.Module, trainable_only: bool = True) -> int:
        return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

    class Broken(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = [nn.Linear(10, 10) for _ in range(3)]       # plain list: NOT registered

    class Fixed(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(3)])
        def forward(self, x):
            for layer in self.layers:
                x = F.relu(layer(x))
            return x

    class Normalize(nn.Module):
        def __init__(self, mean, std):
            super().__init__()
            self.register_buffer("mean", torch.as_tensor(mean, dtype=torch.float32).view(1, -1, 1, 1))
            self.register_buffer("std", torch.as_tensor(std, dtype=torch.float32).view(1, -1, 1, 1))
        def forward(self, x):
            return (x - self.mean) / self.std

    def deep_relu_mlp(depth=20, width=256):
        layers = []
        for _ in range(depth):
            layers += [nn.Linear(width, width, bias=False), nn.ReLU()]
        return nn.Sequential(*layers)

    def activation_stds(model, x):
        stds, handles = [], []
        for m in model.modules():
            if isinstance(m, nn.ReLU):
                # forward hook signature: hook(module, inputs, output)
                handles.append(m.register_forward_hook(lambda mod, inp, out: stds.append(out.std().item())))
        with torch.no_grad():
            model(x)
        for h in handles:
            h.remove()          # always remove hooks you no longer need
        return stds

    def grad_norms(model):
        return {n: p.grad.norm().item() for n, p in model.named_parameters() if p.grad is not None}

    class ArrayDataset(Dataset):
        # Wraps feature/label arrays. Optionally standardises features with GIVEN statistics.
        def __init__(self, X, y, mean=None, std=None):
            self.X = torch.as_tensor(X, dtype=torch.float32)         # convert once, here
            self.y = torch.as_tensor(y, dtype=torch.int64)
            if mean is not None:
                self.X = (self.X - mean) / std

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]                           # one (features, label) pair

    class RandomSequences(Dataset):
        def __init__(self, n=100):
            g = torch.Generator().manual_seed(0)
            self.lengths = torch.randint(3, 10, (n,), generator=g)
            self.data = [torch.randn(int(L), 4, generator=g) for L in self.lengths]   # each (L_i, 4)
            self.labels = torch.randint(0, 2, (n,), generator=g)
        def __len__(self): return len(self.data)
        def __getitem__(self, i): return self.data[i], self.labels[i]

    def pad_collate(batch):
        seqs, labels = zip(*batch)                          # batch is a list of (seq, label) pairs
        lengths = torch.tensor([len(s) for s in seqs])
        padded = pad_sequence(seqs, batch_first=True)       # (B, L_max, 4), zero-padded
        mask = torch.arange(padded.size(1))[None, :] < lengths[:, None]   # (B, L_max): True = real data
        return padded, lengths, mask, torch.stack(labels)

    def show_images(images, titles=None, ncols=8, cmap="gray", figsize_per=1.4):
        # images: (N, C, H, W) tensor, already in a displayable range
        n = len(images); nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per, nrows * figsize_per * 1.15))
        for i, ax in enumerate(np.array(axes).reshape(-1)):
            ax.axis("off")
            if i < n:
                img = images[i]
                img = img[0] if img.shape[0] == 1 else img.permute(1, 2, 0)   # CHW -> HW or HWC
                ax.imshow(img.clamp(0, 1).numpy(), cmap=cmap if img.ndim == 2 else None)
                if titles is not None:
                    ax.set_title(titles[i], fontsize=7)
        plt.tight_layout(); plt.show()

    def loader_throughput(ds, num_workers, batch_size=256, max_batches=50):
        dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                        pin_memory=(device.type == "cuda"), persistent_workers=num_workers > 0)
        it = iter(dl); next(it)                         # start the workers before timing
        t0 = time.perf_counter(); n = 0
        for i, (xb, yb) in enumerate(it):
            n += len(xb)
            if i + 1 >= max_batches: break
        return n / (time.perf_counter() - t0)

    def tab_split(i):
        return TensorDataset((X_num[i] - num_mean) / num_std, X_cat[i], (y_tab[i] - y_mean) / y_std)

    class TabularNet(nn.Module):
        # Numeric features + one embedding per categorical column -> MLP -> one regression output.
        def __init__(self, n_num, cardinalities=(), emb_dims=(), hidden=64):
            super().__init__()
            self.embs = nn.ModuleList([nn.Embedding(c, d) for c, d in zip(cardinalities, emb_dims)])
            self.mlp = nn.Sequential(nn.Linear(n_num + sum(emb_dims), hidden), nn.ReLU(),
                                     nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

        def forward(self, x_num, x_cat):
            parts = [x_num] + [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]   # each (N, d_i)
            return self.mlp(torch.cat(parts, dim=1))

    def train_regressor(model, epochs=n_epochs(15), lr=3e-3):
        model = model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        loader = DataLoader(tab_train, batch_size=256, shuffle=True)
        for _ in range(epochs):
            model.train()
            for xn, xc, yb in loader:
                loss = F.mse_loss(model(xn.to(device), xc.to(device)), yb.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
        return model

    @torch.no_grad()
    def rmse(model, ds):
        model.eval()
        xn, xc, yb = (t.to(device) for t in ds.tensors)
        mse_standardised = F.mse_loss(model(xn, xc), yb)
        return (mse_standardised.sqrt() * y_std).item()              # back in original units

    class ShardStream(IterableDataset):
        def __init__(self, files, split_by_worker=True):
            self.files, self.split_by_worker = list(files), split_by_worker
        def __iter__(self):
            files = self.files
            info = get_worker_info()                               # None in the main process
            if self.split_by_worker and info is not None:
                files = files[info.id::info.num_workers]           # worker k reads shards k, k+W, k+2W, ...
            for f in files:
                xs, ys = torch.load(f, weights_only=True)
                yield from zip(xs, ys)

    def plot_history(hist, title=""):
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
        ep = range(1, len(hist["train_loss"]) + 1)
        a1.plot(ep, hist["train_loss"], label="train"); a1.plot(ep, hist["val_loss"], label="val")
        a2.plot(ep, hist["train_acc"], label="train"); a2.plot(ep, hist["val_acc"], label="val")
        a1.set_title("loss"); a2.set_title("accuracy")
        for a in (a1, a2):
            a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
        fig.suptitle(title); plt.tight_layout(); plt.show()

    @torch.no_grad()
    def predict(model, loader, device):
        # All predictions, true labels and probabilities for a loader (in loader order).
        model.eval()
        preds, labels, probs = [], [], []
        for x, y in loader:
            p = torch.softmax(model(x.to(device)), dim=1).cpu()
            probs.append(p); preds.append(p.argmax(1)); labels.append(y)
        return torch.cat(preds), torch.cat(labels), torch.cat(probs)

    def confusion_matrix(preds, labels, num_classes):
        # cm[i, j] = number of examples of true class i predicted as class j
        return torch.bincount(labels * num_classes + preds, minlength=num_classes ** 2).reshape(num_classes, num_classes)

    def plot_confusion(cm, class_names, title="Confusion matrix (row-normalised)"):
        cm_n = cm.float() / cm.sum(1, keepdim=True).clamp_min(1)
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
        im = ax.imshow(cm_n, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(class_names)), class_names, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(len(class_names)), class_names, fontsize=8)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                ax.text(j, i, f"{cm_n[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cm_n[i, j] > 0.5 else "black")
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
        plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

    def base_indices(ds):
        # Map a (possibly nested) Subset back to indices of its underlying dataset.
        idx = list(range(len(ds)))
        while isinstance(ds, Subset):
            idx = [ds.indices[i] for i in idx]; ds = ds.dataset
        return idx, ds

    def rare_class_split(ds, positive=8, prevalence=0.02, seed=0):
        idx, base = base_indices(ds)
        labels = torch.as_tensor(base.targets)[idx]
        pos = [i for i, l in zip(idx, labels.tolist()) if l == positive]
        neg = [i for i, l in zip(idx, labels.tolist()) if l != positive]
        n_pos = max(1, round(prevalence * len(neg) / (1 - prevalence)))
        pos = [pos[i] for i in torch.randperm(len(pos), generator=torch.Generator().manual_seed(seed))[:n_pos].tolist()]
        return Subset(base, neg + pos), [0] * len(neg) + [1] * len(pos)

    class BinaryTarget(Dataset):
        # Turn integer class labels into float {0., 1.} targets of shape (1,) for BCEWithLogitsLoss.
        def __init__(self, base, positive): self.base, self.positive = base, positive
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            x, y = self.base[i]
            return x, torch.tensor([float(y == self.positive)])

    @torch.no_grad()
    def predict_proba(model, ds):
        model.eval(); ps, ys = [], []
        for x, y in DataLoader(ds, batch_size=512):
            ps.append(torch.sigmoid(model(x.to(device))).squeeze(1).cpu()); ys.append(y.squeeze(1))
        return torch.cat(ps), torch.cat(ys).long()

    def binary_metrics(p, y, t=0.5):
        pred = (p >= t).long()
        tp = ((pred == 1) & (y == 1)).sum().item(); fp = ((pred == 1) & (y == 0)).sum().item()
        fn = ((pred == 0) & (y == 1)).sum().item()
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        return {"acc": (pred == y).float().mean().item(), "precision": prec, "recall": rec, "F1": f1}

    def train_binary(model, loader, loss_fn, epochs=n_epochs(3)):
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        for _ in range(epochs):
            model.train()
            for x, y in loader:
                loss = loss_fn(model(x.to(device)), y.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
        return model

    def threshold_for_recall(p, y, target_recall=0.9):
        # Highest threshold (hence best precision) whose recall on (p, y) is still >= target_recall.
        for t in torch.sort(p.unique(), descending=True).values.tolist():
            if binary_metrics(p, y, t)["recall"] >= target_recall:
                return t
        return 0.0

    def param_groups_weight_decay(model, weight_decay):
        # Two groups: weight matrices/kernels get decay; biases and norm parameters do not.
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            (decay if p.ndim >= 2 else no_decay).append(p)
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    def lr_trace(make_sched, steps=100, base_lr=0.1):
        p = nn.Parameter(torch.zeros(1))
        opt = torch.optim.SGD([p], lr=base_lr)
        sched = make_sched(opt)
        lrs = []
        for _ in range(steps):
            lrs.append(opt.param_groups[0]["lr"])
            opt.step()               # (no gradient here -- we only care about the lr)
            sched.step()
        return lrs

    def lr_range_test(model, loader, loss_fn, device, opt_fn=lambda params, lr: torch.optim.AdamW(params, lr=lr),
                      lr_min=1e-6, lr_max=1.0, num_steps=200, beta=0.98):
        saved = copy.deepcopy(model.state_dict())
        opt = opt_fn(model.parameters(), lr_min)
        gamma = (lr_max / lr_min) ** (1 / (num_steps - 1))        # multiply the lr by this every step
        lrs, losses, avg, best = [], [], 0.0, float("inf")
        model.train(); it = iter(loader)
        for step in range(num_steps):
            try:
                x, y = next(it)
            except StopIteration:
                it = iter(loader); x, y = next(it)
            lr = lr_min * gamma ** step
            for g in opt.param_groups:
                g["lr"] = lr
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            avg = beta * avg + (1 - beta) * loss.item()           # exponential moving average...
            smoothed = avg / (1 - beta ** (step + 1))             # ...with Adam-style bias correction
            lrs.append(lr); losses.append(smoothed)
            best = min(best, smoothed)
            if not math.isfinite(smoothed) or smoothed > 4 * best:    # diverged: stop early
                break
        model.load_state_dict(saved)                              # undo the test's training
        return lrs, losses

    class EarlyStopping:
        # Stops when the monitored metric has not improved for `patience` epochs,
        # and keeps a CPU copy of the best weights so they can be restored.
        def __init__(self, patience=3, min_delta=0.0, mode="max"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best, self.best_epoch, self.bad_epochs, self.best_state = None, -1, 0, None

        def _improved(self, value):
            if self.best is None:
                return True
            return value > self.best + self.min_delta if self.mode == "max" else value < self.best - self.min_delta

        def step(self, value, model, epoch):
            # Call once per epoch. Returns True when training should stop.
            if self._improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, epoch, 0
                self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore_best(self, model):
            model.load_state_dict(self.best_state)

    def rng_state():
        return {"torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate()}

    def set_rng_state(state):
        torch.set_rng_state(state["torch"])
        if torch.cuda.is_available() and state["cuda"]:
            torch.cuda.set_rng_state_all(state["cuda"])
        random.setstate(state["python"])

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None, config=None, **extra):
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch, "best_metric": best_metric,
            "config": config or {}, "rng": rng_state(),
            "torch_version": str(torch.__version__), **extra,   # str(): see the note below
        }
        path = Path(path)
        tmp = path.with_name(path.name + ".tmp")
        torch.save(ckpt, tmp)
        os.replace(tmp, path)       # atomic rename: a crash mid-save never leaves a corrupt checkpoint

    def load_checkpoint(path, model, optimizer=None, scheduler=None, map_location="cpu"):
        ckpt = torch.load(path, map_location=map_location, weights_only=True)
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        set_rng_state(ckpt["rng"])
        return ckpt

    def make_run(seed):
        # Build every training object from scratch, as a fresh process would.
        torch.manual_seed(seed)
        model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 3))
        opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=6)
        gen = torch.Generator().manual_seed(seed)                       # controls the shuffle order
        loader = DataLoader(TensorDataset(Xr, yr), batch_size=32, shuffle=True, generator=gen)
        return model, opt, sched, loader, gen

    def run_epochs(model, opt, sched, loader, n):
        for _ in range(n):
            model.train()
            for xb, yb in loader:
                loss = F.cross_entropy(model(xb), yb)
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()

    def make_deterministic(seed=0):
        # Maximum GPU determinism for debugging or auditing. Not needed (and slower) for normal training.
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # must be set before the first cuBLAS call to take effect
        set_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)   # warn_only: warn instead of raising

    def overfit_one_batch(model, batch, loss_fn, steps=200, lr=1e-3):
        # A correct model and loop should drive the training loss on ONE batch to ~0.
        x, y = (t.to(device) for t in batch)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        model.train()
        for step in range(steps):
            logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            if step % 50 == 0 or step == steps - 1:
                acc = (logits.argmax(1) == y).float().mean().item()
                print(f"step {step:3d}: loss {loss.item():.4f}  batch acc {acc:.2f}")
        return loss.item()

    def gradient_report(model):
        # Call after loss.backward(). Flags parameters with missing, non-finite or zero gradients.
        problems = []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if p.grad is None:
                problems.append(f"{name}: grad is None (unused in forward, or graph cut by .detach()/.item()?)")
            elif not torch.isfinite(p.grad).all():
                problems.append(f"{name}: non-finite gradient")
            elif p.grad.abs().max() == 0:
                problems.append(f"{name}: gradient exactly zero (dead ReLUs? frozen upstream?)")
            else:
                print(f"{name:14s} |grad| = {p.grad.norm():.2e}   |w| = {p.norm():.2e}   ratio = {p.grad.norm() / p.norm():.1e}")
        for msg in problems:
            print("⚠️", msg)

    class HasUnusedLayer(MLP):
        def __init__(self):
            super().__init__()
            self.extra = nn.Linear(10, 10)          # defined but never used in forward

    def buggy_training(epochs=2):
        set_seed(0)
        model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
                              nn.Linear(256, 10), nn.Softmax(dim=1)).to(device)
        loader = DataLoader(sorted_train, batch_size=128, shuffle=False)
        vloader = DataLoader(fmnist_val, batch_size=512)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()
        history = []
        for epoch in range(epochs):
            running = 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                loss = loss_fn(model(x), y)
                loss.backward()
                opt.step()
                running += loss
            model.eval()
            correct = 0
            for x, y in vloader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
            history.append(correct / len(fmnist_val))
            print(f"epoch {epoch+1}: train loss {running.item() / len(loader):.3f} | val acc {history[-1]:.4f}")
        return history

    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    def benchmark(fn, warmup=5, iters=20):
        # Average wall-clock seconds per call of fn(), with warm-up and GPU synchronisation (§1.8).
        for _ in range(warmup):
            fn()
        sync()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        sync()
        return (time.perf_counter() - t0) / iters

    def materialize(ds, batch_size=2048):
        # Run a dataset's transforms once and stack everything into two tensors.
        xs, ys = [], []
        for x, y in DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS):
            xs.append(x); ys.append(y)
        return torch.cat(xs), torch.cat(ys)

    class TensorLoader:
        # A minimal DataLoader replacement for tensors that already live on the target device.
        def __init__(self, X, y, batch_size, shuffle=False, drop_last=False, seed=0):
            self.X, self.y, self.bs, self.shuffle, self.drop_last = X, y, batch_size, shuffle, drop_last
            self.gen = torch.Generator(device=X.device).manual_seed(seed)
        def __len__(self):
            n = len(self.X)
            return n // self.bs if self.drop_last else math.ceil(n / self.bs)
        def __iter__(self):
            n = len(self.X)
            idx = torch.randperm(n, device=self.X.device, generator=self.gen) if self.shuffle else torch.arange(n, device=self.X.device)
            for i in range(len(self)):
                j = idx[i * self.bs:(i + 1) * self.bs]
                yield self.X[j], self.y[j]

    def wide_mlp():
        return nn.Sequential(nn.Flatten(), nn.Linear(784, WIDTH), nn.ReLU(),
                             nn.Linear(WIDTH, WIDTH), nn.ReLU(), nn.Linear(WIDTH, 10)).to(device)

    def make_step(model, use_amp):
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
        def step():
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        return step

    def pick_amp_dtype(device):
        # bf16 where the hardware supports it natively (Ampere+), fp16 on older GPUs (T4), None on CPU/MPS.
        if device.type != "cuda":
            return None
        return torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

    def train_one_epoch(model, loader, loss_fn, optimizer, device, scheduler=None, max_grad_norm=None,
                        scaler=None, amp_dtype=None, channels_last=False):
        model.train()
        use_amp = amp_dtype is not None
        if scaler is None:
            scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
        total_loss = torch.zeros((), device=device)
        total_correct = torch.zeros((), device=device, dtype=torch.long)
        total_n = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if channels_last:
                x = x.contiguous(memory_format=torch.channels_last)   # see Chapter 13
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()
            n = y.size(0)
            total_loss += loss.detach().float() * n
            total_correct += (logits.argmax(dim=1) == y).sum()
            total_n += n
        return total_loss.item() / total_n, total_correct.item() / total_n

    @torch.no_grad()
    def evaluate(model, loader, loss_fn, device, amp_dtype=None, channels_last=False):
        model.eval()
        total_loss = torch.zeros((), device=device)
        total_correct = torch.zeros((), device=device, dtype=torch.long)
        total_n = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if channels_last:
                x = x.contiguous(memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                logits = model(x)
                loss = loss_fn(logits, y)
            total_loss += loss.float() * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum()
            total_n += y.size(0)
        return total_loss.item() / total_n, total_correct.item() / total_n

    def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, device, scheduler=None,
            scheduler_interval="batch", max_grad_norm=None, early_stopping=None, eval_loss_fn=None,
            amp=False, channels_last=False, checkpoint_path=None, verbose=True):
        # The full-featured training driver used for the rest of the notebook.
        eval_loss_fn = eval_loss_fn or loss_fn
        amp_dtype = pick_amp_dtype(device) if amp else None
        scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
        hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": [], "epoch_time": []}
        best_acc = -1.0
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device,
                                              scheduler if scheduler_interval == "batch" else None,
                                              max_grad_norm, scaler, amp_dtype, channels_last)
            va_loss, va_acc = evaluate(model, val_loader, eval_loss_fn, device, amp_dtype, channels_last)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            dt = time.perf_counter() - t0
            for k, v in zip(("train_loss", "train_acc", "val_loss", "val_acc", "lr", "epoch_time"),
                            (tr_loss, tr_acc, va_loss, va_acc, optimizer.param_groups[0]["lr"], dt)):
                hist[k].append(v)
            if checkpoint_path is not None and va_acc > best_acc:       # keep the best model on disk
                best_acc = va_acc
                save_checkpoint(checkpoint_path, model, optimizer, scheduler, epoch=epoch, best_metric=va_acc)
            if verbose:
                print(f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                      f"val loss {va_loss:.4f} acc {va_acc:.4f} | lr {hist['lr'][-1]:.2e} | {dt:.1f}s")
            if early_stopping is not None and early_stopping.step(va_acc, model, epoch):
                if verbose:
                    print(f"early stopping (best val acc {early_stopping.best:.4f} at epoch {early_stopping.best_epoch})")
                break
        if early_stopping is not None and early_stopping.best_state is not None:
            early_stopping.restore_best(model)
        return hist

    def grid(ax, n, title, hi=None, col="#fff2cc"):
        for i in range(n):
            for j in range(n):
                on = hi is not None and hi[0] <= i < hi[0] + hi[2] and hi[1] <= j < hi[1] + hi[2]
                ax.add_patch(plt.Rectangle((j, n - 1 - i), 1, 1, fc=col if on else "white", ec="0.4"))
        ax.set_xlim(-0.2, n + 0.2); ax.set_ylim(-0.2, n + 0.2); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title, fontsize=9)

    def conv_out(h, k, s=1, p=0, d=1):
        return (h + 2 * p - d * (k - 1) - 1) // s + 1

    def receptive_field(layers):
        # layers: list of (kernel, stride). Returns the theoretical receptive field size.
        r, j = 1, 1
        for k, s in layers:
            r, j = r + (k - 1) * j, j * s
        return r

    def conv_bn_relu(c_in, c_out, stride=1):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False),
                             nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

    class SmallCNN(nn.Module):
        def __init__(self, in_ch=1, num_classes=10, p_drop=0.3):
            super().__init__()
            self.features = nn.Sequential(
                conv_bn_relu(in_ch, 32), conv_bn_relu(32, 32), nn.MaxPool2d(2),     # 28 -> 14
                conv_bn_relu(32, 64), conv_bn_relu(64, 64), nn.MaxPool2d(2),        # 14 -> 7
                conv_bn_relu(64, 128),                                              # 7x7x128
            )
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p_drop), nn.Linear(128, num_classes))

        def forward(self, x):
            return self.head(self.features(x))

    class BasicBlock(nn.Module):
        # The ResNet-18/34 block: two 3x3 convs with a residual connection.
        def __init__(self, c_in, c_out, stride=1):
            super().__init__()
            self.conv1 = nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False)
            self.bn1 = nn.BatchNorm2d(c_out)
            self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=False)
            self.bn2 = nn.BatchNorm2d(c_out)
            nn.init.zeros_(self.bn2.weight)       # zero-init the last BN gamma: each block starts as the identity (Tutorial §9.6)
            self.shortcut = nn.Identity()
            if stride != 1 or c_in != c_out:      # shapes differ -> projection shortcut
                self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))

        def forward(self, x):
            out = F.relu(self.bn1(self.conv1(x)))
            out = self.bn2(self.conv2(out))
            return F.relu(out + self.shortcut(x))

    class CifarResNet(nn.Module):
        # A ResNet-18-style network for 32x32 inputs: a 3x3 stem (no early downsampling), 4 stages, GAP, linear head.
        def __init__(self, widths=(64, 128, 256, 512), blocks=(2, 2, 2, 2), num_classes=10, in_ch=3):
            super().__init__()
            self.stem = nn.Sequential(nn.Conv2d(in_ch, widths[0], 3, padding=1, bias=False),
                                      nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
            stages, c_in = [], widths[0]
            for i, (w, n) in enumerate(zip(widths, blocks)):
                stride = 1 if i == 0 else 2                          # halve the resolution at each new stage
                layers = [BasicBlock(c_in, w, stride)] + [BasicBlock(w, w) for _ in range(n - 1)]
                stages.append(nn.Sequential(*layers)); c_in = w
            self.stages = nn.Sequential(*stages)
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c_in, num_classes))
            for m in self.modules():                                 # He init for convs (Tutorial §9.6)
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

        def forward(self, x):
            return self.head(self.stages(self.stem(x)))

    class TransformDataset(Dataset):
        # Apply `transform` to the inputs of an (untransformed) base dataset.
        def __init__(self, base, transform):
            self.base, self.transform = base, transform
        def __len__(self):
            return len(self.base)
        def __getitem__(self, i):
            x, y = self.base[i]
            return self.transform(x), y

    pin, USE_AMP = device.type == "cuda", device.type == "cuda"
    print("Chapter 14: functions, classes and data from earlier chapters restored.")


---
# Chapter 14. Transfer learning

In industry you will rarely train a vision model from scratch. You will start from a model **pretrained** on a large dataset (most often ImageNet: 1.28 million images in 1,000 classes) and adapt it to your task. Tutorial §12.8 ranked this third among remedies for overfitting, right after more data and augmentation. On small datasets it is frequently transformative.

## 14.1 Why it works, and the three strategies

The early and middle layers of an ImageNet network compute **general-purpose visual features**: edges, textures, shapes and object parts (§12.4, §13.6). Those features are useful for almost any natural-image task. Only the last layer is truly specific to ImageNet's 1,000 classes.

| Strategy | What you train | When to use it |
| --- | --- | --- |
| **Feature extraction (linear probe)** | Only a new final layer, on frozen features | Very little data, or very limited compute. Also a fast baseline |
| **Fine-tuning** | Everything, with a small learning rate for the pretrained part | The default when you have hundreds of images or more |
| **From scratch** | Everything, from random initialisation | Huge datasets, or domains very far from natural images |

We will compare all three on **Oxford Flowers-102**: 102 flower species, with only **10 training images per class** (1,020 in total), 1,020 for validation, and 6,149 for testing.

## 14.2 The `torchvision.models` API

Every pretrained model comes with a **weights enum** that bundles the checkpoint URL, its metadata, and, crucially, the **exact preprocessing** it was trained with.

🎯 **In this chapter you will learn to:** use pretrained torchvision models and their preprocessing; compare linear probing, fine-tuning and training from scratch; freeze layers correctly (including BatchNorm).

In [ ]:
from torchvision import models
from torchvision.models import ResNet50_Weights

print(f"{len(models.list_models(module=models))} classification architectures available, e.g.",
      models.list_models(module=models)[:5])
weights = ResNet50_Weights.DEFAULT                      # currently IMAGENET1K_V2
print(weights, "|", f"{weights.meta['num_params']/1e6:.1f}M params |",
      "ImageNet top-1:", weights.meta["_metrics"]["ImageNet-1K"]["acc@1"])
preprocess = weights.transforms()                       # the inference preprocessing these weights expect
print(preprocess)
IMAGENET_MEAN, IMAGENET_STD = preprocess.mean, preprocess.std

First, use the network as it is, to classify a photo into ImageNet's 1,000 classes. (The cat photo ships with scikit-image, so no download is needed.)

In [ ]:
from skimage import data as skdata

cat = torch.from_numpy(skdata.chelsea()).permute(2, 0, 1)          # (3, H, W) uint8
imagenet_model = models.resnet50(weights=weights).to(device).eval()
with torch.inference_mode():
    probs = torch.softmax(imagenet_model(preprocess(cat).unsqueeze(0).to(device)), dim=1)[0]
top = probs.topk(5)
plt.figure(figsize=(3, 2.4)); plt.imshow(cat.permute(1, 2, 0)); plt.axis("off"); plt.show()
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{weights.meta['categories'][i]:22s} {p:.3f}")
del imagenet_model

> ⚠️ **The preprocessing must match the pretraining.** These weights expect RGB images scaled to `[0, 1]` and normalised with the ImageNet mean and std above. Feeding them inputs normalised with your own dataset's statistics, or BGR images from OpenCV, or 0–255 values, silently costs a lot of accuracy. `weights.transforms()` is the source of truth.

## 14.3 The Flowers-102 data

In [ ]:
flowers = {s: datasets.Flowers102(DATA_DIR, split=s, download=True) for s in ("train", "val", "test")}
FLOWER_CLASSES = datasets.Flowers102.classes
print({s: len(d) for s, d in flowers.items()}, "|", len(FLOWER_CLASSES), "classes, e.g.", FLOWER_CLASSES[:3])

IMG = preprocess.crop_size[0]          # 224 for these weights
RESIZE = preprocess.resize_size[0]     # 232 for IMAGENET1K_V2 -- read it from the preset, don't guess
flower_train_tf = v2.Compose([v2.ToImage(), v2.RandomResizedCrop(IMG, scale=(0.35, 1.0), antialias=True),
                              v2.RandomHorizontalFlip(), v2.ToDtype(torch.float32, scale=True),
                              v2.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
flower_eval_tf = v2.Compose([v2.ToImage(), v2.Resize(RESIZE, antialias=True), v2.CenterCrop(IMG),
                             v2.ToDtype(torch.float32, scale=True), v2.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

fl_train = maybe_subset(TransformDataset(flowers["train"], flower_train_tf), 128)
fl_train_plain = maybe_subset(TransformDataset(flowers["train"], flower_eval_tf), 128)   # for feature caching
fl_val = maybe_subset(TransformDataset(flowers["val"], flower_eval_tf), 128)
fl_test = maybe_subset(TransformDataset(flowers["test"], flower_eval_tf), 128)

FLOWER_BATCH = 16 if FAST_DEV_RUN else 64      # 64 fits a T4 comfortably with AMP

def flower_loader(ds, shuffle=False, bs=None):
    bs = bs or FLOWER_BATCH
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=pin,
                      drop_last=shuffle, persistent_workers=NUM_WORKERS > 0)

fl_train_loader, fl_val_loader, fl_test_loader = flower_loader(fl_train, True), flower_loader(fl_val), flower_loader(fl_test)
im_mean, im_std = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
xs_, ys_ = next(iter(DataLoader(fl_train, batch_size=8, shuffle=True)))
show_images(xs_ * im_std + im_mean, [FLOWER_CLASSES[y][:14] for y in ys_.tolist()], figsize_per=1.6)

## 14.4 Strategy A: a linear probe on frozen, cached features

When the backbone is frozen and there is no augmentation, the features never change. So we compute them **once**, cache them, and train a linear classifier on the cached tensors. That takes seconds instead of minutes, and it is a standard industry trick. We replace the final `fc` layer with `nn.Identity()`, so the network outputs its 2,048-dimensional pooled features.

In [ ]:
@torch.no_grad()
def extract_features(backbone, loader):
    backbone.eval(); feats, labels = [], []
    for x, y in loader:
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            f = backbone(x.to(device, non_blocking=True))
        feats.append(f.float()); labels.append(y.to(device))
    return torch.cat(feats), torch.cat(labels)

backbone = models.resnet50(weights=weights)
backbone.fc = nn.Identity()                 # outputs (N, 2048) pooled features
backbone = backbone.to(device)
t0 = time.perf_counter()
Ftr, Ytr = extract_features(backbone, flower_loader(fl_train_plain))
Fva, Yva = extract_features(backbone, fl_val_loader)
Fte, Yte = extract_features(backbone, fl_test_loader)
print(f"features: train {tuple(Ftr.shape)}, val {tuple(Fva.shape)}, test {tuple(Fte.shape)} in {time.perf_counter()-t0:.0f}s")

set_seed(0)
probe = nn.Linear(Ftr.shape[1], 102).to(device)      # 2048 for ResNet-50; read it, don't hard-code it
opt = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
hist_probe = fit(probe, TensorLoader(Ftr, Ytr, 128, shuffle=True), TensorLoader(Fva, Yva, 1024),
                 nn.CrossEntropyLoss(), opt, epochs=n_epochs(60), device=device, verbose=False)
probe_val = max(hist_probe["val_acc"])
probe_test = evaluate(probe, TensorLoader(Fte, Yte, 1024), nn.CrossEntropyLoss(), device)[1]
print(f"linear probe: best val acc {probe_val:.4f} | test acc {probe_test:.4f}")
del backbone

With only 10 images per class, a linear layer on frozen ImageNet features typically already gets high test accuracy on 102 classes, often in the 80s (an estimate; your run is recorded in `RESULTS`). That alone shows how general the pretrained features are. Training on the 1,020 images from scratch, as we will see, does not come close.

## 14.5 A freezing pitfall: `requires_grad_(False)` does not freeze BatchNorm

To freeze part of a network *while still training other parts* (for example, the full training loop with augmentation and a new head), setting `requires_grad_(False)` stops the **parameters** changing. But BatchNorm's running statistics are **buffers**, and they are updated in every forward pass in train mode.

In [ ]:
frozen = models.resnet50(weights=weights).to(device)
frozen.requires_grad_(False)                     # "freeze" every parameter
bn = frozen.bn1
before = bn.running_mean.clone()
frozen.train()                                   # e.g. inside our train_one_epoch
with torch.no_grad():
    frozen(torch.randn(8, 3, IMG, IMG, device=device))
print("running_mean changed despite requires_grad=False:", not torch.equal(before, bn.running_mean))

def freeze_(module):
    # Truly freeze a module: no gradients AND BatchNorm statistics fixed.
    module.requires_grad_(False)
    module.eval()          # NB: a later model.train() switches it back -- re-apply freeze_ after it
    return module
del frozen

Note the comment in `freeze_`: `model.train()` recursively puts every submodule back into train mode. In a training loop with a partly frozen network, you must call `model.train()` and *then* switch the frozen part back to `.eval()`, or override the model's `train()` method so it does this for you.

## 14.6 Strategy B: full fine-tuning with discriminative learning rates

Now we train everything, with augmentation. The pretrained backbone gets a learning rate **ten times smaller** than the newly initialised head. The backbone's weights are already good, and large updates would destroy them early on, while the random head still produces large, noisy gradients ("catastrophic forgetting"). OneCycle's warm-up protects the backbone further during those first steps.

In [ ]:
def finetune_param_groups(model, head_name, lr_backbone, lr_head, weight_decay):
    # Four groups: (backbone | head) x (decay | no decay).
    head = getattr(model, head_name)
    backbone_part = nn.ModuleList([m for n, m in model.named_children() if n != head_name])
    groups = [dict(g, lr=lr_backbone) for g in param_groups_weight_decay(backbone_part, weight_decay)]
    groups += [dict(g, lr=lr_head) for g in param_groups_weight_decay(head, weight_decay)]
    return groups

def train_flowers(model, lrs_head_backbone, epochs, label=""):
    lr_head, lr_backbone = lrs_head_backbone
    groups = finetune_param_groups(model, "fc", lr_backbone, lr_head, weight_decay=0.01)
    opt = torch.optim.AdamW(groups)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], epochs=epochs,
                                                steps_per_epoch=len(fl_train_loader), pct_start=0.2)
    print(f"--- {label} ---")
    hist = fit(model, fl_train_loader, fl_val_loader, nn.CrossEntropyLoss(label_smoothing=0.1), opt, epochs, device,
               scheduler=sched, eval_loss_fn=nn.CrossEntropyLoss(), amp=True, channels_last=True)
    test = evaluate(model, fl_test_loader, nn.CrossEntropyLoss(), device, pick_amp_dtype(device), True)[1]
    return hist, test

EPOCHS_FT = n_epochs(15)
set_seed(0)
ft_model = models.resnet50(weights=weights)
ft_model.fc = nn.Linear(ft_model.fc.in_features, 102)          # new head for 102 classes
ft_model = ft_model.to(device).to(memory_format=torch.channels_last)
hist_ft, ft_test = train_flowers(ft_model, (1e-3, 1e-4), EPOCHS_FT, "fine-tuning (pretrained)")
print(f"fine-tuning: test acc {ft_test:.4f}")

## 14.7 Strategy C: the same network, from scratch

In [ ]:
set_seed(0)
scratch = models.resnet50(weights=None, num_classes=102).to(device).to(memory_format=torch.channels_last)
hist_scratch, scratch_test = train_flowers(scratch, (1e-3, 1e-3), EPOCHS_FT, "from scratch (random init)")

print(f"\n{'strategy':28s} {'test accuracy':>14s}")
for name, acc in [("A: linear probe (frozen)", probe_test), ("B: fine-tuning", ft_test), ("C: from scratch", scratch_test)]:
    print(f"{name:28s} {record(f'Flowers-102 {name}: test acc', acc):14.4f}")

# Free memory before the next experiment: in a long notebook, every model you keep a reference to
# stays in (GPU) memory. Deleting the references and clearing the CUDA cache is a good habit.
import gc
del ft_model, scratch
_ = gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

The typical ranking is dramatic. Fine-tuning comes first, usually reaching around 90% or more. The linear probe is a strong second. Training from scratch is far behind (expect a small fraction of the fine-tuned accuracy), because 25 million parameters cannot be learned from 1,020 images. This one table is the reason transfer learning is the industry default.

**A practical recipe:**
1. Start with a linear probe. It takes minutes, and it is your baseline.
2. Fine-tune everything, with the backbone at about 1/10 of the head's learning rate, plus warm-up and augmentation.
3. If you are overfitting badly, freeze the early stages (they hold the most generic features) and fine-tune only the later ones.
4. Match the pretraining preprocessing exactly. Choose a backbone whose pretraining data resembles your domain.
5. 🏭 Check the **licence** of both the weights and the pretraining data before you ship a product built on them.

## 🧪 Exercises

**Exercise 14.1: partial fine-tuning.** `★★★ · ~40 min` Fine-tune only `layer4` and `fc` of a pretrained ResNet-50 for the same number of epochs, freezing everything else with `freeze_` (and re-freezing after `model.train()`, for example by overriding `train`). Compare its test accuracy and its epoch time with full fine-tuning.

In [ ]:
# ✍️ Your attempt here.
# Hint: Subclass torchvision's ResNet, or wrap train(): def train(self, mode=True): super().train(mode); freeze_(...) 

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
class PartialResNet(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.net = models.resnet50(weights=weights)
        self.net.fc = nn.Linear(self.net.fc.in_features, num_classes)
        self.frozen = nn.ModuleList([self.net.conv1, self.net.bn1, self.net.layer1, self.net.layer2, self.net.layer3])
        freeze_(self.frozen)
    def train(self, mode=True):
        super().train(mode)
        self.frozen.eval()                     # keep frozen BatchNorm statistics fixed
        return self
    def forward(self, x):
        return self.net(x)

set_seed(0)
pm = PartialResNet().to(device).to(memory_format=torch.channels_last)
trainable = [p for p in pm.parameters() if p.requires_grad]
print(f"trainable parameters: {sum(p.numel() for p in trainable)/1e6:.1f}M of {count_params(pm, False)/1e6:.1f}M")
opt = torch.optim.AdamW([{"params": [p for p in pm.net.layer4.parameters()], "lr": 1e-4},
                         {"params": pm.net.fc.parameters(), "lr": 1e-3}], weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[1e-4, 1e-3], epochs=EPOCHS_FT, steps_per_epoch=len(fl_train_loader))
h = fit(pm, fl_train_loader, fl_val_loader, nn.CrossEntropyLoss(label_smoothing=0.1), opt, EPOCHS_FT, device,
        scheduler=sched, eval_loss_fn=nn.CrossEntropyLoss(), amp=True, channels_last=True, verbose=False)
print(f"partial fine-tuning test acc: {evaluate(pm, fl_test_loader, nn.CrossEntropyLoss(), device)[1]:.4f} "
      f"| mean epoch time {np.mean(h['epoch_time']):.1f}s vs full {np.mean(hist_ft['epoch_time']):.1f}s")
del pm, opt, sched; _ = gc.collect()
# Usually close to full fine-tuning (sometimes slightly better on tiny datasets, as there is less to
# overfit), and faster per epoch because no gradients are computed for the frozen early stages.

**Exercise 14.2: swap the backbone.** `★★☆ · ~30 min` Write `replace_head(model, num_classes)` that works for ResNets (`model.fc`), EfficientNets and ConvNeXts (the last `nn.Linear` inside `model.classifier`). Use it to run a linear probe with `models.get_model("efficientnet_b0", weights="DEFAULT")`, caching features as in §14.4. EfficientNet-B0 has only about 5M parameters. How does its probe accuracy compare with ResNet-50's?

In [ ]:
# ✍️ Your attempt here.
# Hint: For the probe, replace the head with nn.Identity() and read in_features from the old Linear first.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def replace_head(model, num_classes):
    # Swap the final nn.Linear for a new one with num_classes outputs; returns (model, in_features).
    if isinstance(getattr(model, "fc", None), nn.Linear):
        in_f = model.fc.in_features
        model.fc = nn.Linear(in_f, num_classes) if num_classes else nn.Identity()
    elif hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):
        idx = max(i for i, m in enumerate(model.classifier) if isinstance(m, nn.Linear))
        in_f = model.classifier[idx].in_features
        model.classifier[idx] = nn.Linear(in_f, num_classes) if num_classes else nn.Identity()
    else:
        raise ValueError("unknown head layout -- print(model) and look at the last layer")
    return model, in_f

eff_w = models.get_model_weights("efficientnet_b0").DEFAULT
eff, in_f = replace_head(models.get_model("efficientnet_b0", weights=eff_w), None)   # None -> Identity: features out
eff = eff.to(device)
# EfficientNet-B0's preset differs slightly (resize 256, bicubic). For a quick comparison we reuse our
# pipeline (same ImageNet mean/std); for a final number, build the transform from eff_w.transforms().
F_tr, Y_tr = extract_features(eff, flower_loader(fl_train_plain))
F_va, Y_va = extract_features(eff, fl_val_loader)
F_te, Y_te = extract_features(eff, fl_test_loader)
set_seed(0)
p2 = nn.Linear(in_f, 102).to(device)
fit(p2, TensorLoader(F_tr, Y_tr, 128, shuffle=True), TensorLoader(F_va, Y_va, 1024), nn.CrossEntropyLoss(),
    torch.optim.AdamW(p2.parameters(), lr=1e-3, weight_decay=1e-4), n_epochs(60), device, verbose=False)
print(f"EfficientNet-B0 probe ({in_f}-d features, {eff_w.meta['num_params']/1e6:.1f}M params): "
      f"test acc {evaluate(p2, TensorLoader(F_te, Y_te, 1024), nn.CrossEntropyLoss(), device)[1]:.4f} "
      f"| ResNet-50 probe: {probe_test:.4f}")

### 📌 Key takeaways

- Use the weights' own preprocessing (`weights.transforms()`).
- Fine-tune with a smaller backbone learning rate than head learning rate.
- `requires_grad_(False)` does not freeze BatchNorm statistics: use `eval()` too.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Why give the pretrained backbone a smaller learning rate than the new head?

<details>
<summary>Answer</summary>

The backbone already holds good features, and big updates, especially early on when the random head sends large noisy gradients, would destroy them (catastrophic forgetting). The head starts from random and needs to move a lot.

</details>

**2.** You freeze a backbone with `requires_grad_(False)` and train only the head, but the backbone's behaviour still drifts. Why?

<details>
<summary>Answer</summary>

BatchNorm running statistics are buffers. They are updated by every forward pass in train mode, regardless of `requires_grad`. Keep frozen BN layers in `eval()` mode.

</details>

**3.** When is a linear probe on cached features the right choice in production?

<details>
<summary>Answer</summary>

When data or compute is very limited, when you need a fast baseline, or when many tasks share one frozen backbone, so features are computed once and many small heads are trained cheaply.

</details>

⏯️ **Resume point.** Continuing in a *fresh* runtime? Run the setup cell (Chapter 0), then the hidden cell below. It re-creates every function and class from the earlier chapters and the data Chapter 15 needs. In a normal top-to-bottom run it detects that nothing is missing and does nothing.

In [ ]:
# @title ⏯️ Resume point for Chapter 15: run only in a fresh runtime, after the setup cell { display-mode: "form" }
if "set_seed" not in globals():
    raise RuntimeError("Run the setup cell of Chapter 0 first, then this cell.")
if "cifar_test_acc" in globals() and "CifarResNet" in globals() and "cifar_val_loader" in globals():
    print("Chapter 15: the state it needs is already present -- nothing to restore.")
else:
    from collections import OrderedDict
    from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset, random_split
    from sklearn.datasets import make_moons
    from sklearn.model_selection import train_test_split
    from torch.nn.utils.rnn import pad_sequence
    from torchvision import datasets
    from torchvision.transforms import v2
    from torch.utils.data import IterableDataset, get_worker_info
    from torch.utils.data import WeightedRandomSampler
    from sklearn.metrics import roc_auc_score, average_precision_score
    from torch.optim import lr_scheduler as S
    from torch.profiler import profile, ProfilerActivity, record_function
    from torchvision import models
    from torchvision.models import ResNet50_Weights
    from skimage import data as skdata
    import gc

    def time_matmul(dev, n=2048, reps=10):
        a = torch.randn(n, n, device=dev)
        b = torch.randn(n, n, device=dev)
        a @ b                                        # warm-up
        if dev.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(reps):
            a @ b
        if dev.type == "cuda": torch.cuda.synchronize()   # wait for the GPU to actually finish
        return (time.perf_counter() - t0) / reps

    def tiny_net(x, W1, b1, W2, b2):
        return (torch.tanh(x @ W1 + b1) @ W2 + b2).pow(2).sum()

    class MySigmoid(torch.autograd.Function):
        @staticmethod
        def forward(ctx, z):
            s = 1.0 / (1.0 + torch.exp(-z))
            ctx.save_for_backward(s)           # stash what backward needs
            return s

        @staticmethod
        def backward(ctx, grad_out):           # grad_out = dL/ds
            (s,) = ctx.saved_tensors
            return grad_out * s * (1 - s)      # dL/dz = dL/ds * s(1-s)

    class MLP(nn.Module):
        def __init__(self, in_dim=784, hidden=(256, 128), out_dim=10, p_drop=0.2):
            super().__init__()                    # ALWAYS first: sets up the registration machinery
            self.fc1 = nn.Linear(in_dim, hidden[0])   # assigning a Module to an attribute registers it
            self.fc2 = nn.Linear(hidden[0], hidden[1])
            self.head = nn.Linear(hidden[1], out_dim)
            self.drop = nn.Dropout(p_drop)

        def forward(self, x):                     # x: (N, 1, 28, 28) or (N, 784)
            x = x.flatten(start_dim=1)            # -> (N, 784)
            x = self.drop(F.relu(self.fc1(x)))    # -> (N, 256)
            x = self.drop(F.relu(self.fc2(x)))    # -> (N, 128)
            return self.head(x)                   # -> (N, 10) logits: no softmax (Chapter 5)

    def count_params(m: nn.Module, trainable_only: bool = True) -> int:
        return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

    class Broken(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = [nn.Linear(10, 10) for _ in range(3)]       # plain list: NOT registered

    class Fixed(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(3)])
        def forward(self, x):
            for layer in self.layers:
                x = F.relu(layer(x))
            return x

    class Normalize(nn.Module):
        def __init__(self, mean, std):
            super().__init__()
            self.register_buffer("mean", torch.as_tensor(mean, dtype=torch.float32).view(1, -1, 1, 1))
            self.register_buffer("std", torch.as_tensor(std, dtype=torch.float32).view(1, -1, 1, 1))
        def forward(self, x):
            return (x - self.mean) / self.std

    def deep_relu_mlp(depth=20, width=256):
        layers = []
        for _ in range(depth):
            layers += [nn.Linear(width, width, bias=False), nn.ReLU()]
        return nn.Sequential(*layers)

    def activation_stds(model, x):
        stds, handles = [], []
        for m in model.modules():
            if isinstance(m, nn.ReLU):
                # forward hook signature: hook(module, inputs, output)
                handles.append(m.register_forward_hook(lambda mod, inp, out: stds.append(out.std().item())))
        with torch.no_grad():
            model(x)
        for h in handles:
            h.remove()          # always remove hooks you no longer need
        return stds

    def grad_norms(model):
        return {n: p.grad.norm().item() for n, p in model.named_parameters() if p.grad is not None}

    class ArrayDataset(Dataset):
        # Wraps feature/label arrays. Optionally standardises features with GIVEN statistics.
        def __init__(self, X, y, mean=None, std=None):
            self.X = torch.as_tensor(X, dtype=torch.float32)         # convert once, here
            self.y = torch.as_tensor(y, dtype=torch.int64)
            if mean is not None:
                self.X = (self.X - mean) / std

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]                           # one (features, label) pair

    class RandomSequences(Dataset):
        def __init__(self, n=100):
            g = torch.Generator().manual_seed(0)
            self.lengths = torch.randint(3, 10, (n,), generator=g)
            self.data = [torch.randn(int(L), 4, generator=g) for L in self.lengths]   # each (L_i, 4)
            self.labels = torch.randint(0, 2, (n,), generator=g)
        def __len__(self): return len(self.data)
        def __getitem__(self, i): return self.data[i], self.labels[i]

    def pad_collate(batch):
        seqs, labels = zip(*batch)                          # batch is a list of (seq, label) pairs
        lengths = torch.tensor([len(s) for s in seqs])
        padded = pad_sequence(seqs, batch_first=True)       # (B, L_max, 4), zero-padded
        mask = torch.arange(padded.size(1))[None, :] < lengths[:, None]   # (B, L_max): True = real data
        return padded, lengths, mask, torch.stack(labels)

    def show_images(images, titles=None, ncols=8, cmap="gray", figsize_per=1.4):
        # images: (N, C, H, W) tensor, already in a displayable range
        n = len(images); nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per, nrows * figsize_per * 1.15))
        for i, ax in enumerate(np.array(axes).reshape(-1)):
            ax.axis("off")
            if i < n:
                img = images[i]
                img = img[0] if img.shape[0] == 1 else img.permute(1, 2, 0)   # CHW -> HW or HWC
                ax.imshow(img.clamp(0, 1).numpy(), cmap=cmap if img.ndim == 2 else None)
                if titles is not None:
                    ax.set_title(titles[i], fontsize=7)
        plt.tight_layout(); plt.show()

    def loader_throughput(ds, num_workers, batch_size=256, max_batches=50):
        dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                        pin_memory=(device.type == "cuda"), persistent_workers=num_workers > 0)
        it = iter(dl); next(it)                         # start the workers before timing
        t0 = time.perf_counter(); n = 0
        for i, (xb, yb) in enumerate(it):
            n += len(xb)
            if i + 1 >= max_batches: break
        return n / (time.perf_counter() - t0)

    def tab_split(i):
        return TensorDataset((X_num[i] - num_mean) / num_std, X_cat[i], (y_tab[i] - y_mean) / y_std)

    class TabularNet(nn.Module):
        # Numeric features + one embedding per categorical column -> MLP -> one regression output.
        def __init__(self, n_num, cardinalities=(), emb_dims=(), hidden=64):
            super().__init__()
            self.embs = nn.ModuleList([nn.Embedding(c, d) for c, d in zip(cardinalities, emb_dims)])
            self.mlp = nn.Sequential(nn.Linear(n_num + sum(emb_dims), hidden), nn.ReLU(),
                                     nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

        def forward(self, x_num, x_cat):
            parts = [x_num] + [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]   # each (N, d_i)
            return self.mlp(torch.cat(parts, dim=1))

    def train_regressor(model, epochs=n_epochs(15), lr=3e-3):
        model = model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        loader = DataLoader(tab_train, batch_size=256, shuffle=True)
        for _ in range(epochs):
            model.train()
            for xn, xc, yb in loader:
                loss = F.mse_loss(model(xn.to(device), xc.to(device)), yb.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
        return model

    @torch.no_grad()
    def rmse(model, ds):
        model.eval()
        xn, xc, yb = (t.to(device) for t in ds.tensors)
        mse_standardised = F.mse_loss(model(xn, xc), yb)
        return (mse_standardised.sqrt() * y_std).item()              # back in original units

    class ShardStream(IterableDataset):
        def __init__(self, files, split_by_worker=True):
            self.files, self.split_by_worker = list(files), split_by_worker
        def __iter__(self):
            files = self.files
            info = get_worker_info()                               # None in the main process
            if self.split_by_worker and info is not None:
                files = files[info.id::info.num_workers]           # worker k reads shards k, k+W, k+2W, ...
            for f in files:
                xs, ys = torch.load(f, weights_only=True)
                yield from zip(xs, ys)

    def plot_history(hist, title=""):
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
        ep = range(1, len(hist["train_loss"]) + 1)
        a1.plot(ep, hist["train_loss"], label="train"); a1.plot(ep, hist["val_loss"], label="val")
        a2.plot(ep, hist["train_acc"], label="train"); a2.plot(ep, hist["val_acc"], label="val")
        a1.set_title("loss"); a2.set_title("accuracy")
        for a in (a1, a2):
            a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
        fig.suptitle(title); plt.tight_layout(); plt.show()

    @torch.no_grad()
    def predict(model, loader, device):
        # All predictions, true labels and probabilities for a loader (in loader order).
        model.eval()
        preds, labels, probs = [], [], []
        for x, y in loader:
            p = torch.softmax(model(x.to(device)), dim=1).cpu()
            probs.append(p); preds.append(p.argmax(1)); labels.append(y)
        return torch.cat(preds), torch.cat(labels), torch.cat(probs)

    def confusion_matrix(preds, labels, num_classes):
        # cm[i, j] = number of examples of true class i predicted as class j
        return torch.bincount(labels * num_classes + preds, minlength=num_classes ** 2).reshape(num_classes, num_classes)

    def plot_confusion(cm, class_names, title="Confusion matrix (row-normalised)"):
        cm_n = cm.float() / cm.sum(1, keepdim=True).clamp_min(1)
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
        im = ax.imshow(cm_n, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(class_names)), class_names, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(len(class_names)), class_names, fontsize=8)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                ax.text(j, i, f"{cm_n[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cm_n[i, j] > 0.5 else "black")
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
        plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

    def base_indices(ds):
        # Map a (possibly nested) Subset back to indices of its underlying dataset.
        idx = list(range(len(ds)))
        while isinstance(ds, Subset):
            idx = [ds.indices[i] for i in idx]; ds = ds.dataset
        return idx, ds

    def rare_class_split(ds, positive=8, prevalence=0.02, seed=0):
        idx, base = base_indices(ds)
        labels = torch.as_tensor(base.targets)[idx]
        pos = [i for i, l in zip(idx, labels.tolist()) if l == positive]
        neg = [i for i, l in zip(idx, labels.tolist()) if l != positive]
        n_pos = max(1, round(prevalence * len(neg) / (1 - prevalence)))
        pos = [pos[i] for i in torch.randperm(len(pos), generator=torch.Generator().manual_seed(seed))[:n_pos].tolist()]
        return Subset(base, neg + pos), [0] * len(neg) + [1] * len(pos)

    class BinaryTarget(Dataset):
        # Turn integer class labels into float {0., 1.} targets of shape (1,) for BCEWithLogitsLoss.
        def __init__(self, base, positive): self.base, self.positive = base, positive
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            x, y = self.base[i]
            return x, torch.tensor([float(y == self.positive)])

    @torch.no_grad()
    def predict_proba(model, ds):
        model.eval(); ps, ys = [], []
        for x, y in DataLoader(ds, batch_size=512):
            ps.append(torch.sigmoid(model(x.to(device))).squeeze(1).cpu()); ys.append(y.squeeze(1))
        return torch.cat(ps), torch.cat(ys).long()

    def binary_metrics(p, y, t=0.5):
        pred = (p >= t).long()
        tp = ((pred == 1) & (y == 1)).sum().item(); fp = ((pred == 1) & (y == 0)).sum().item()
        fn = ((pred == 0) & (y == 1)).sum().item()
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        return {"acc": (pred == y).float().mean().item(), "precision": prec, "recall": rec, "F1": f1}

    def train_binary(model, loader, loss_fn, epochs=n_epochs(3)):
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        for _ in range(epochs):
            model.train()
            for x, y in loader:
                loss = loss_fn(model(x.to(device)), y.to(device))
                opt.zero_grad(); loss.backward(); opt.step()
        return model

    def threshold_for_recall(p, y, target_recall=0.9):
        # Highest threshold (hence best precision) whose recall on (p, y) is still >= target_recall.
        for t in torch.sort(p.unique(), descending=True).values.tolist():
            if binary_metrics(p, y, t)["recall"] >= target_recall:
                return t
        return 0.0

    def param_groups_weight_decay(model, weight_decay):
        # Two groups: weight matrices/kernels get decay; biases and norm parameters do not.
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            (decay if p.ndim >= 2 else no_decay).append(p)
        return [{"params": decay, "weight_decay": weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    def lr_trace(make_sched, steps=100, base_lr=0.1):
        p = nn.Parameter(torch.zeros(1))
        opt = torch.optim.SGD([p], lr=base_lr)
        sched = make_sched(opt)
        lrs = []
        for _ in range(steps):
            lrs.append(opt.param_groups[0]["lr"])
            opt.step()               # (no gradient here -- we only care about the lr)
            sched.step()
        return lrs

    def lr_range_test(model, loader, loss_fn, device, opt_fn=lambda params, lr: torch.optim.AdamW(params, lr=lr),
                      lr_min=1e-6, lr_max=1.0, num_steps=200, beta=0.98):
        saved = copy.deepcopy(model.state_dict())
        opt = opt_fn(model.parameters(), lr_min)
        gamma = (lr_max / lr_min) ** (1 / (num_steps - 1))        # multiply the lr by this every step
        lrs, losses, avg, best = [], [], 0.0, float("inf")
        model.train(); it = iter(loader)
        for step in range(num_steps):
            try:
                x, y = next(it)
            except StopIteration:
                it = iter(loader); x, y = next(it)
            lr = lr_min * gamma ** step
            for g in opt.param_groups:
                g["lr"] = lr
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            avg = beta * avg + (1 - beta) * loss.item()           # exponential moving average...
            smoothed = avg / (1 - beta ** (step + 1))             # ...with Adam-style bias correction
            lrs.append(lr); losses.append(smoothed)
            best = min(best, smoothed)
            if not math.isfinite(smoothed) or smoothed > 4 * best:    # diverged: stop early
                break
        model.load_state_dict(saved)                              # undo the test's training
        return lrs, losses

    class EarlyStopping:
        # Stops when the monitored metric has not improved for `patience` epochs,
        # and keeps a CPU copy of the best weights so they can be restored.
        def __init__(self, patience=3, min_delta=0.0, mode="max"):
            self.patience, self.min_delta, self.mode = patience, min_delta, mode
            self.best, self.best_epoch, self.bad_epochs, self.best_state = None, -1, 0, None

        def _improved(self, value):
            if self.best is None:
                return True
            return value > self.best + self.min_delta if self.mode == "max" else value < self.best - self.min_delta

        def step(self, value, model, epoch):
            # Call once per epoch. Returns True when training should stop.
            if self._improved(value):
                self.best, self.best_epoch, self.bad_epochs = value, epoch, 0
                self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                self.bad_epochs += 1
            return self.bad_epochs >= self.patience

        def restore_best(self, model):
            model.load_state_dict(self.best_state)

    def rng_state():
        return {"torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python": random.getstate()}

    def set_rng_state(state):
        torch.set_rng_state(state["torch"])
        if torch.cuda.is_available() and state["cuda"]:
            torch.cuda.set_rng_state_all(state["cuda"])
        random.setstate(state["python"])

    def save_checkpoint(path, model, optimizer, scheduler=None, epoch=0, best_metric=None, config=None, **extra):
        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch, "best_metric": best_metric,
            "config": config or {}, "rng": rng_state(),
            "torch_version": str(torch.__version__), **extra,   # str(): see the note below
        }
        path = Path(path)
        tmp = path.with_name(path.name + ".tmp")
        torch.save(ckpt, tmp)
        os.replace(tmp, path)       # atomic rename: a crash mid-save never leaves a corrupt checkpoint

    def load_checkpoint(path, model, optimizer=None, scheduler=None, map_location="cpu"):
        ckpt = torch.load(path, map_location=map_location, weights_only=True)
        model.load_state_dict(ckpt["model"])
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and ckpt["scheduler"] is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        set_rng_state(ckpt["rng"])
        return ckpt

    def make_run(seed):
        # Build every training object from scratch, as a fresh process would.
        torch.manual_seed(seed)
        model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 3))
        opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=6)
        gen = torch.Generator().manual_seed(seed)                       # controls the shuffle order
        loader = DataLoader(TensorDataset(Xr, yr), batch_size=32, shuffle=True, generator=gen)
        return model, opt, sched, loader, gen

    def run_epochs(model, opt, sched, loader, n):
        for _ in range(n):
            model.train()
            for xb, yb in loader:
                loss = F.cross_entropy(model(xb), yb)
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()

    def make_deterministic(seed=0):
        # Maximum GPU determinism for debugging or auditing. Not needed (and slower) for normal training.
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # must be set before the first cuBLAS call to take effect
        set_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)   # warn_only: warn instead of raising

    def overfit_one_batch(model, batch, loss_fn, steps=200, lr=1e-3):
        # A correct model and loop should drive the training loss on ONE batch to ~0.
        x, y = (t.to(device) for t in batch)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        model.train()
        for step in range(steps):
            logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            if step % 50 == 0 or step == steps - 1:
                acc = (logits.argmax(1) == y).float().mean().item()
                print(f"step {step:3d}: loss {loss.item():.4f}  batch acc {acc:.2f}")
        return loss.item()

    def gradient_report(model):
        # Call after loss.backward(). Flags parameters with missing, non-finite or zero gradients.
        problems = []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if p.grad is None:
                problems.append(f"{name}: grad is None (unused in forward, or graph cut by .detach()/.item()?)")
            elif not torch.isfinite(p.grad).all():
                problems.append(f"{name}: non-finite gradient")
            elif p.grad.abs().max() == 0:
                problems.append(f"{name}: gradient exactly zero (dead ReLUs? frozen upstream?)")
            else:
                print(f"{name:14s} |grad| = {p.grad.norm():.2e}   |w| = {p.norm():.2e}   ratio = {p.grad.norm() / p.norm():.1e}")
        for msg in problems:
            print("⚠️", msg)

    class HasUnusedLayer(MLP):
        def __init__(self):
            super().__init__()
            self.extra = nn.Linear(10, 10)          # defined but never used in forward

    def buggy_training(epochs=2):
        set_seed(0)
        model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
                              nn.Linear(256, 10), nn.Softmax(dim=1)).to(device)
        loader = DataLoader(sorted_train, batch_size=128, shuffle=False)
        vloader = DataLoader(fmnist_val, batch_size=512)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()
        history = []
        for epoch in range(epochs):
            running = 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                loss = loss_fn(model(x), y)
                loss.backward()
                opt.step()
                running += loss
            model.eval()
            correct = 0
            for x, y in vloader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
            history.append(correct / len(fmnist_val))
            print(f"epoch {epoch+1}: train loss {running.item() / len(loader):.3f} | val acc {history[-1]:.4f}")
        return history

    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    def benchmark(fn, warmup=5, iters=20):
        # Average wall-clock seconds per call of fn(), with warm-up and GPU synchronisation (§1.8).
        for _ in range(warmup):
            fn()
        sync()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        sync()
        return (time.perf_counter() - t0) / iters

    def materialize(ds, batch_size=2048):
        # Run a dataset's transforms once and stack everything into two tensors.
        xs, ys = [], []
        for x, y in DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS):
            xs.append(x); ys.append(y)
        return torch.cat(xs), torch.cat(ys)

    class TensorLoader:
        # A minimal DataLoader replacement for tensors that already live on the target device.
        def __init__(self, X, y, batch_size, shuffle=False, drop_last=False, seed=0):
            self.X, self.y, self.bs, self.shuffle, self.drop_last = X, y, batch_size, shuffle, drop_last
            self.gen = torch.Generator(device=X.device).manual_seed(seed)
        def __len__(self):
            n = len(self.X)
            return n // self.bs if self.drop_last else math.ceil(n / self.bs)
        def __iter__(self):
            n = len(self.X)
            idx = torch.randperm(n, device=self.X.device, generator=self.gen) if self.shuffle else torch.arange(n, device=self.X.device)
            for i in range(len(self)):
                j = idx[i * self.bs:(i + 1) * self.bs]
                yield self.X[j], self.y[j]

    def wide_mlp():
        return nn.Sequential(nn.Flatten(), nn.Linear(784, WIDTH), nn.ReLU(),
                             nn.Linear(WIDTH, WIDTH), nn.ReLU(), nn.Linear(WIDTH, 10)).to(device)

    def make_step(model, use_amp):
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
        def step():
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        return step

    def pick_amp_dtype(device):
        # bf16 where the hardware supports it natively (Ampere+), fp16 on older GPUs (T4), None on CPU/MPS.
        if device.type != "cuda":
            return None
        return torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

    def train_one_epoch(model, loader, loss_fn, optimizer, device, scheduler=None, max_grad_norm=None,
                        scaler=None, amp_dtype=None, channels_last=False):
        model.train()
        use_amp = amp_dtype is not None
        if scaler is None:
            scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
        total_loss = torch.zeros((), device=device)
        total_correct = torch.zeros((), device=device, dtype=torch.long)
        total_n = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if channels_last:
                x = x.contiguous(memory_format=torch.channels_last)   # see Chapter 13
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(x)
                loss = loss_fn(logits, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()
            n = y.size(0)
            total_loss += loss.detach().float() * n
            total_correct += (logits.argmax(dim=1) == y).sum()
            total_n += n
        return total_loss.item() / total_n, total_correct.item() / total_n

    @torch.no_grad()
    def evaluate(model, loader, loss_fn, device, amp_dtype=None, channels_last=False):
        model.eval()
        total_loss = torch.zeros((), device=device)
        total_correct = torch.zeros((), device=device, dtype=torch.long)
        total_n = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if channels_last:
                x = x.contiguous(memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                logits = model(x)
                loss = loss_fn(logits, y)
            total_loss += loss.float() * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum()
            total_n += y.size(0)
        return total_loss.item() / total_n, total_correct.item() / total_n

    def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, device, scheduler=None,
            scheduler_interval="batch", max_grad_norm=None, early_stopping=None, eval_loss_fn=None,
            amp=False, channels_last=False, checkpoint_path=None, verbose=True):
        # The full-featured training driver used for the rest of the notebook.
        eval_loss_fn = eval_loss_fn or loss_fn
        amp_dtype = pick_amp_dtype(device) if amp else None
        scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
        hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": [], "epoch_time": []}
        best_acc = -1.0
        for epoch in range(1, epochs + 1):
            t0 = time.perf_counter()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device,
                                              scheduler if scheduler_interval == "batch" else None,
                                              max_grad_norm, scaler, amp_dtype, channels_last)
            va_loss, va_acc = evaluate(model, val_loader, eval_loss_fn, device, amp_dtype, channels_last)
            if scheduler is not None and scheduler_interval == "epoch":
                scheduler.step()
            elif scheduler is not None and scheduler_interval == "plateau":
                scheduler.step(va_loss)
            dt = time.perf_counter() - t0
            for k, v in zip(("train_loss", "train_acc", "val_loss", "val_acc", "lr", "epoch_time"),
                            (tr_loss, tr_acc, va_loss, va_acc, optimizer.param_groups[0]["lr"], dt)):
                hist[k].append(v)
            if checkpoint_path is not None and va_acc > best_acc:       # keep the best model on disk
                best_acc = va_acc
                save_checkpoint(checkpoint_path, model, optimizer, scheduler, epoch=epoch, best_metric=va_acc)
            if verbose:
                print(f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                      f"val loss {va_loss:.4f} acc {va_acc:.4f} | lr {hist['lr'][-1]:.2e} | {dt:.1f}s")
            if early_stopping is not None and early_stopping.step(va_acc, model, epoch):
                if verbose:
                    print(f"early stopping (best val acc {early_stopping.best:.4f} at epoch {early_stopping.best_epoch})")
                break
        if early_stopping is not None and early_stopping.best_state is not None:
            early_stopping.restore_best(model)
        return hist

    def grid(ax, n, title, hi=None, col="#fff2cc"):
        for i in range(n):
            for j in range(n):
                on = hi is not None and hi[0] <= i < hi[0] + hi[2] and hi[1] <= j < hi[1] + hi[2]
                ax.add_patch(plt.Rectangle((j, n - 1 - i), 1, 1, fc=col if on else "white", ec="0.4"))
        ax.set_xlim(-0.2, n + 0.2); ax.set_ylim(-0.2, n + 0.2); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title, fontsize=9)

    def conv_out(h, k, s=1, p=0, d=1):
        return (h + 2 * p - d * (k - 1) - 1) // s + 1

    def receptive_field(layers):
        # layers: list of (kernel, stride). Returns the theoretical receptive field size.
        r, j = 1, 1
        for k, s in layers:
            r, j = r + (k - 1) * j, j * s
        return r

    def conv_bn_relu(c_in, c_out, stride=1):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False),
                             nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

    class SmallCNN(nn.Module):
        def __init__(self, in_ch=1, num_classes=10, p_drop=0.3):
            super().__init__()
            self.features = nn.Sequential(
                conv_bn_relu(in_ch, 32), conv_bn_relu(32, 32), nn.MaxPool2d(2),     # 28 -> 14
                conv_bn_relu(32, 64), conv_bn_relu(64, 64), nn.MaxPool2d(2),        # 14 -> 7
                conv_bn_relu(64, 128),                                              # 7x7x128
            )
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p_drop), nn.Linear(128, num_classes))

        def forward(self, x):
            return self.head(self.features(x))

    class BasicBlock(nn.Module):
        # The ResNet-18/34 block: two 3x3 convs with a residual connection.
        def __init__(self, c_in, c_out, stride=1):
            super().__init__()
            self.conv1 = nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False)
            self.bn1 = nn.BatchNorm2d(c_out)
            self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=False)
            self.bn2 = nn.BatchNorm2d(c_out)
            nn.init.zeros_(self.bn2.weight)       # zero-init the last BN gamma: each block starts as the identity (Tutorial §9.6)
            self.shortcut = nn.Identity()
            if stride != 1 or c_in != c_out:      # shapes differ -> projection shortcut
                self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))

        def forward(self, x):
            out = F.relu(self.bn1(self.conv1(x)))
            out = self.bn2(self.conv2(out))
            return F.relu(out + self.shortcut(x))

    class CifarResNet(nn.Module):
        # A ResNet-18-style network for 32x32 inputs: a 3x3 stem (no early downsampling), 4 stages, GAP, linear head.
        def __init__(self, widths=(64, 128, 256, 512), blocks=(2, 2, 2, 2), num_classes=10, in_ch=3):
            super().__init__()
            self.stem = nn.Sequential(nn.Conv2d(in_ch, widths[0], 3, padding=1, bias=False),
                                      nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
            stages, c_in = [], widths[0]
            for i, (w, n) in enumerate(zip(widths, blocks)):
                stride = 1 if i == 0 else 2                          # halve the resolution at each new stage
                layers = [BasicBlock(c_in, w, stride)] + [BasicBlock(w, w) for _ in range(n - 1)]
                stages.append(nn.Sequential(*layers)); c_in = w
            self.stages = nn.Sequential(*stages)
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c_in, num_classes))
            for m in self.modules():                                 # He init for convs (Tutorial §9.6)
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

        def forward(self, x):
            return self.head(self.stages(self.stem(x)))

    class TransformDataset(Dataset):
        # Apply `transform` to the inputs of an (untransformed) base dataset.
        def __init__(self, base, transform):
            self.base, self.transform = base, transform
        def __len__(self):
            return len(self.base)
        def __getitem__(self, i):
            x, y = self.base[i]
            return self.transform(x), y

    def flower_loader(ds, shuffle=False, bs=None):
        bs = bs or FLOWER_BATCH
        return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=pin,
                          drop_last=shuffle, persistent_workers=NUM_WORKERS > 0)

    @torch.no_grad()
    def extract_features(backbone, loader):
        backbone.eval(); feats, labels = [], []
        for x, y in loader:
            with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
                f = backbone(x.to(device, non_blocking=True))
            feats.append(f.float()); labels.append(y.to(device))
        return torch.cat(feats), torch.cat(labels)

    def freeze_(module):
        # Truly freeze a module: no gradients AND BatchNorm statistics fixed.
        module.requires_grad_(False)
        module.eval()          # NB: a later model.train() switches it back -- re-apply freeze_ after it
        return module

    def finetune_param_groups(model, head_name, lr_backbone, lr_head, weight_decay):
        # Four groups: (backbone | head) x (decay | no decay).
        head = getattr(model, head_name)
        backbone_part = nn.ModuleList([m for n, m in model.named_children() if n != head_name])
        groups = [dict(g, lr=lr_backbone) for g in param_groups_weight_decay(backbone_part, weight_decay)]
        groups += [dict(g, lr=lr_head) for g in param_groups_weight_decay(head, weight_decay)]
        return groups

    def train_flowers(model, lrs_head_backbone, epochs, label=""):
        lr_head, lr_backbone = lrs_head_backbone
        groups = finetune_param_groups(model, "fc", lr_backbone, lr_head, weight_decay=0.01)
        opt = torch.optim.AdamW(groups)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], epochs=epochs,
                                                    steps_per_epoch=len(fl_train_loader), pct_start=0.2)
        print(f"--- {label} ---")
        hist = fit(model, fl_train_loader, fl_val_loader, nn.CrossEntropyLoss(label_smoothing=0.1), opt, epochs, device,
                   scheduler=sched, eval_loss_fn=nn.CrossEntropyLoss(), amp=True, channels_last=True)
        test = evaluate(model, fl_test_loader, nn.CrossEntropyLoss(), device, pick_amp_dtype(device), True)[1]
        return hist, test

    pin, USE_AMP = device.type == "cuda", device.type == "cuda"
    torch.backends.cudnn.benchmark = True
    cifar_raw = datasets.CIFAR10(DATA_DIR, train=True, download=True)
    cifar_test_raw = datasets.CIFAR10(DATA_DIR, train=False, download=True)
    CIFAR_CLASSES = cifar_raw.classes
    perm = torch.randperm(len(cifar_raw), generator=torch.Generator().manual_seed(0))
    tr_idx, va_idx = perm[:45_000].tolist(), perm[45_000:].tolist()
    px = torch.from_numpy(cifar_raw.data[tr_idx]).float().div(255)
    CIFAR_MEAN, CIFAR_STD = px.mean(dim=(0, 1, 2)).tolist(), px.std(dim=(0, 1, 2)).tolist(); del px
    to_normed = [v2.ToDtype(torch.float32, scale=True), v2.Normalize(CIFAR_MEAN, CIFAR_STD)]
    cifar_train_tf = v2.Compose([v2.ToImage(), v2.RandomCrop(32, padding=4, padding_mode="reflect"), v2.RandomHorizontalFlip(), *to_normed])
    cifar_eval_tf = v2.Compose([v2.ToImage(), *to_normed])
    cifar_train = maybe_subset(TransformDataset(Subset(cifar_raw, tr_idx), cifar_train_tf))
    cifar_val = maybe_subset(TransformDataset(Subset(cifar_raw, va_idx), cifar_eval_tf))
    cifar_test = maybe_subset(TransformDataset(cifar_test_raw, cifar_eval_tf))
    CIFAR_ARCH, CIFAR_BATCH = dict(widths=(32, 64, 128, 256), blocks=(2, 2, 2, 2)), 256
    cifar_val_loader = DataLoader(cifar_val, batch_size=512, num_workers=NUM_WORKERS, pin_memory=pin)
    cifar_test_loader = DataLoader(cifar_test, batch_size=512, num_workers=NUM_WORKERS, pin_memory=pin)
    CIFAR_CKPT = ARTIFACT_DIR / "cifar_resnet_best.pt"
    if not CIFAR_CKPT.exists():
        print("No CIFAR-10 checkpoint found (Chapter 13 not run in this runtime, and USE_DRIVE is off).")
        print("Training a SHORT model (fewer epochs, lower accuracy) so that Chapter 15 can run...")
        _loader = DataLoader(cifar_train, batch_size=CIFAR_BATCH, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=pin)
        set_seed(0)
        _m = CifarResNet(**CIFAR_ARCH).to(device).to(memory_format=torch.channels_last)
        _E = n_epochs(8)
        _opt = torch.optim.SGD(param_groups_weight_decay(_m, 5e-4), lr=0.1, momentum=0.9, nesterov=True)
        _sched = torch.optim.lr_scheduler.OneCycleLR(_opt, max_lr=0.1, epochs=_E, steps_per_epoch=len(_loader), pct_start=0.15)
        fit(_m, _loader, cifar_val_loader, nn.CrossEntropyLoss(label_smoothing=0.1), _opt, _E, device, scheduler=_sched,
            eval_loss_fn=nn.CrossEntropyLoss(), amp=True, channels_last=True, checkpoint_path=CIFAR_CKPT, verbose=False)
        del _m, _opt, _sched, _loader
    ckpt = torch.load(CIFAR_CKPT, map_location=device, weights_only=True)
    best = CifarResNet(**CIFAR_ARCH).to(device).to(memory_format=torch.channels_last)
    best.load_state_dict(ckpt["model"])
    c_preds, c_labels, c_probs = predict(best, cifar_test_loader, device)
    cifar_test_acc = (c_preds == c_labels).float().mean().item()
    print(f"CIFAR-10 model ready: val acc {ckpt['best_metric']:.4f}, test acc {cifar_test_acc:.4f}")
    print("Chapter 15: functions, classes and data from earlier chapters restored.")


---
# Part V — Production

# Chapter 15. Inference, export and deployment

A trained model sitting in a notebook is worth nothing to a company. This chapter turns the CIFAR-10 ResNet into a **self-contained, versioned artifact** that a separate program can load and run, without your training code and, with ONNX, without PyTorch at all.

## 15.1 The inference checklist

Every inference path, whether a notebook cell, a script or a web service, must:

1. Call **`model.eval()`**, so that dropout is off and BN uses its running statistics.
2. Run under **`torch.inference_mode()`**, so that no autograd bookkeeping happens.
3. Apply **exactly the training preprocessing** (resize, scale, normalise, channel order, dtype).
4. **Batch** requests when throughput matters. Use batch size 1 only when latency matters.
5. Return **probabilities plus metadata**: class names and a model version. If downstream decisions use the probability *values* (thresholds, risk scores), check their **calibration** first (§15.2). Softmax outputs are not calibrated by default.

The most common production bug is item 3, a train/serve preprocessing mismatch. The fix is architectural: **put the preprocessing inside the model**, so the artifact accepts raw `uint8` images and nothing can drift apart.

## 15.2 Are the probabilities trustworthy? Calibration

A classifier is **calibrated** if, among all the predictions it makes with confidence 0.8, about 80% are correct. Modern deep networks trained with cross-entropy are frequently **overconfident** (Guo et al., 2017, *On Calibration of Modern Neural Networks*). Label smoothing, which we used, tends to reduce overconfidence, and can even make a model under-confident (Müller et al., 2019). Either way, you should **measure** calibration before anyone uses the probability values.

- **Expected calibration error (ECE).** Bin the predictions by confidence, and take the weighted average gap between confidence and accuracy over the bins. 0 means perfectly calibrated.
- **Temperature scaling.** The simplest effective fix. Divide the logits by a single scalar `T > 0`, fitted to minimise the NLL on the **validation** set. `T > 1` softens over-confident predictions, and `T < 1` sharpens under-confident ones. Dividing by a positive constant never changes the `argmax`, so **accuracy is unchanged**.

🎯 **In this chapter you will learn to:** build an inference artifact that carries its own preprocessing; measure and fix calibration with temperature scaling; export with `torch.export` and ONNX and serve from a CLI.

In [ ]:
net = CifarResNet(**CIFAR_ARCH)
net.load_state_dict(torch.load(CIFAR_CKPT, map_location="cpu", weights_only=True)["model"])
net.eval()

@torch.no_grad()
def collect_logits(model, loader):
    model.eval(); logits, labels = [], []
    for x, y in loader:
        logits.append(model(x.to(device)).float().cpu()); labels.append(y)
    return torch.cat(logits), torch.cat(labels)

def expected_calibration_error(probs, labels, n_bins=15):
    conf, pred = probs.max(dim=1)
    correct = (pred == labels).float()
    edges = torch.linspace(0, 1, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (conf > lo) & (conf <= hi)
        if in_bin.any():
            ece += in_bin.float().mean().item() * abs(correct[in_bin].mean().item() - conf[in_bin].mean().item())
    return ece

def fit_temperature(logits, labels):
    log_t = torch.zeros(1, requires_grad=True)           # optimise log T, so T = exp(log T) stays positive
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), labels)
        loss.backward()
        return loss
    opt.step(closure)
    return log_t.exp().item()

calib_net = copy.deepcopy(net).to(device)
val_logits, val_y = collect_logits(calib_net, cifar_val_loader)
test_logits, test_y = collect_logits(calib_net, cifar_test_loader)
TEMPERATURE = record("CIFAR-10 ResNet: fitted temperature T", fit_temperature(val_logits, val_y))   # VALIDATION only

fig, ax = plt.subplots(figsize=(4.2, 4))
for name, t in [("T = 1 (raw softmax)", 1.0), (f"T = {TEMPERATURE:.2f} (fitted)", TEMPERATURE)]:
    p = torch.softmax(test_logits / t, dim=1)
    ece = expected_calibration_error(p, test_y)
    print(f"{name:22s} test NLL {F.cross_entropy(test_logits / t, test_y).item():.4f} | ECE {ece:.4f} | "
          f"acc {(p.argmax(1) == test_y).float().mean().item():.4f}")
    record(f"CIFAR-10 ResNet: test ECE, {name}", ece)
    conf, pred = p.max(1); edges = torch.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        b = (conf > lo) & (conf <= hi)
        if b.sum() > 20:
            xs.append(conf[b].mean().item()); ys.append((pred[b] == test_y[b]).float().mean().item())
    ax.plot(xs, ys, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title("Reliability diagram (test)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
del calib_net

The temperature is fitted on validation data and *checked* on test data, the same discipline as any other hyper-parameter. The deployment model below applies it, so the probabilities it returns are the calibrated ones.

## 15.3 Packaging the model with its preprocessing

In [ ]:
class CifarInference(nn.Module):
    # Raw uint8 RGB images (N, 3, 32, 32), values 0..255 -> calibrated class probabilities (N, 10).
    def __init__(self, net, mean, std, temperature=1.0):
        super().__init__()
        self.net = net
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1))
        self.register_buffer("temperature", torch.tensor(float(temperature)))

    def forward(self, x):
        x = x.float() / 255.0                  # same as ToDtype(float32, scale=True)
        x = (x - self.mean) / self.std         # same as Normalize(mean, std)
        return torch.softmax(self.net(x) / self.temperature, dim=1)

infer_model = CifarInference(net, CIFAR_MEAN, CIFAR_STD, TEMPERATURE).eval()   # deployment target: CPU

# Train/serve parity test: raw-uint8 path == the training pipeline's eval path, on real test images
raw_imgs = torch.stack([v2.functional.pil_to_tensor(cifar_test_raw[i][0]) for i in range(16)])  # uint8 (16,3,32,32)
with torch.inference_mode():
    p_serving = infer_model(raw_imgs)
    p_training = torch.softmax(net.eval()(torch.stack([cifar_eval_tf(cifar_test_raw[i][0]) for i in range(16)])) / TEMPERATURE, 1)
print("serving path matches training path:", torch.allclose(p_serving, p_training, atol=1e-5))

That last check is a **parity test**, and it is worth writing for every model you ship. Next, save a versioned artifact: the weights plus everything needed to rebuild and interpret the model.

In [ ]:
artifact = {
    "format_version": 1,
    "arch": "CifarResNet", "arch_kwargs": {k: list(v) for k, v in CIFAR_ARCH.items()},
    "state_dict": infer_model.state_dict(),          # includes the mean/std buffers
    "classes": list(CIFAR_CLASSES),
    "input": {"dtype": "uint8", "shape": [3, 32, 32], "layout": "NCHW", "channels": "RGB"},
    "metrics": {"val_acc": float(ckpt["best_metric"]), "test_acc": float(cifar_test_acc)},
    "temperature": float(TEMPERATURE),
    "torch_version": str(torch.__version__),
}
torch.save(artifact, ARTIFACT_DIR / "cifar_resnet_artifact.pt")
loaded = torch.load(ARTIFACT_DIR / "cifar_resnet_artifact.pt", weights_only=True)   # test loading immediately
rebuilt = CifarInference(CifarResNet(**loaded["arch_kwargs"]), [0.0] * 3, [1.0] * 3)
rebuilt.load_state_dict(loaded["state_dict"])         # overwrites the placeholder mean/std too
print("artifact round-trip OK:", torch.allclose(rebuilt.eval()(raw_imgs), p_serving))

The artifact above still needs the Python class `CifarResNet` to rebuild the model. The next two sections remove that dependency.

## 15.4 `torch.export`: a portable, code-free graph

`torch.export.export(model, example_args)` traces the model into an **`ExportedProgram`**: a complete graph of PyTorch operators with the weights attached, and no Python model code. You can save it, load it in a process that has never seen your class definitions, and compile it ahead of time (AOTInductor) or lower it to mobile and edge runtimes (ExecuTorch). It is the modern replacement for TorchScript, which is deprecated.

By default, every input dimension is fixed to the example's size. We declare the batch dimension **dynamic**, so that one export serves any batch size:

In [ ]:
example = torch.randint(0, 256, (2, 3, 32, 32), dtype=torch.uint8)
batch = torch.export.Dim("batch", min=1, max=1024)
exported = torch.export.export(infer_model, (example,), dynamic_shapes={"x": {0: batch}})   # "x" = forward's arg name
print(type(exported).__name__, "| graph nodes:", len(exported.graph.nodes))
print("inputs:", [s.arg.name for s in exported.graph_signature.input_specs if s.kind.name == "USER_INPUT"])

pt2_path = ARTIFACT_DIR / "cifar_resnet.pt2"
torch.export.save(exported, pt2_path)
reloaded = torch.export.load(pt2_path).module()       # an nn.Module-like callable; no CifarResNet class needed
with torch.inference_mode():
    for n in (1, 7, 64):
        xb_ = torch.randint(0, 256, (n, 3, 32, 32), dtype=torch.uint8)
        assert torch.allclose(reloaded(xb_), infer_model(xb_), atol=1e-5), n
print("exported program matches eager for batch sizes 1, 7, 64")

try:
    reloaded(torch.randint(0, 256, (2, 3, 64, 64), dtype=torch.uint8))     # wrong image size
except Exception as e:
    print("wrong input shape is rejected:", type(e).__name__, "-", str(e).splitlines()[0][:80])

What `torch.export` requires of your model:

- **No data-dependent Python control flow.** `if x.sum() > 0:` cannot be captured as a fixed graph. Use `torch.where`, or the structured control-flow operator `torch.cond` (PyTorch 2.14 adds a multi-way `torch.switch`, still marked unstable).
- **Export in eval mode.** BatchNorm and dropout are then captured with their inference behaviour.
- **Declare which dimensions vary.** Anything not declared dynamic is specialised to the example's value, and other values are rejected at runtime, as you just saw. That is a feature: shape errors are caught at the boundary.

## 15.5 ONNX: run without PyTorch

**ONNX** is an open model format supported by many runtimes: ONNX Runtime (CPU, CUDA, TensorRT, mobile, browser), OpenVINO, and others. Exporting to ONNX lets a team that does not use PyTorch, or a device that cannot run it, serve your model. The modern exporter, `dynamo=True` (the default), is built on `torch.export`.

In [ ]:
import importlib.util, subprocess
_missing = [p for p in ("onnx", "onnxscript", "onnxruntime") if importlib.util.find_spec(p) is None]
if _missing:
    print("installing", _missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
import onnxruntime as ort

onnx_path = ARTIFACT_DIR / "cifar_resnet.onnx"
torch.onnx.export(infer_model, (example,), str(onnx_path), dynamo=True,
                  dynamic_shapes={"x": {0: batch}}, external_data=False)   # external_data=False: one self-contained file
print(f"ONNX file: {onnx_path.stat().st_size / 2**20:.1f} MiB")

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
print("ONNX input:", inp.name, inp.shape, inp.type)
for n in (1, 5, 64):
    xb_ = torch.randint(0, 256, (n, 3, 32, 32), dtype=torch.uint8)
    out = sess.run(None, {inp.name: xb_.numpy()})[0]
    with torch.inference_mode():
        np.testing.assert_allclose(out, infer_model(xb_).numpy(), rtol=1e-4, atol=1e-5)
print("ONNX Runtime matches PyTorch for batch sizes 1, 5, 64")

## 15.6 Latency: how fast is a single prediction?

Production requirements are usually stated as **latency** (for example, "p95 below 20 ms for one image") or as **throughput** (images per second). Measure both on the hardware you will actually deploy to. Here that is Colab's CPU:

In [ ]:
one = torch.randint(0, 256, (1, 3, 32, 32), dtype=torch.uint8)
def lat_ms(fn, iters=50):
    for _ in range(5): fn()
    ts = []
    for _ in range(iters):
        t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1e3)
    ts = np.array(ts)
    return np.median(ts), np.percentile(ts, 95)

with torch.inference_mode():
    for name, fn in [("PyTorch eager (CPU)", lambda: infer_model(one)),
                     ("torch.export (CPU)", lambda: reloaded(one)),
                     ("ONNX Runtime (CPU)", lambda: sess.run(None, {inp.name: one.numpy()}))]:
        med, p95 = lat_ms(fn)
        record(f"CIFAR-10 single-image latency, {name}: median (ms)", med)
        print(f"{name:22s} median {med:6.2f} ms | p95 {p95:6.2f} ms")

ONNX Runtime is often noticeably faster at batch size 1 on a CPU, because it fuses operators and has low per-call overhead. Results vary by machine, so measure on yours.

## 15.7 A standalone command-line predictor

Real consumers of your model are programs, not notebooks. The script below depends only on NumPy, Pillow, and either `torch` (for the `.pt2` file) or `onnxruntime` (for the `.onnx` file). It knows nothing about our training code. Class names travel in a small JSON metadata file.

In [ ]:
meta = {"classes": list(CIFAR_CLASSES), "image_size": 32, "model_version": "cifar-resnet-1.0",
        "metrics": artifact["metrics"]}
(ARTIFACT_DIR / "cifar_meta.json").write_text(json.dumps(meta, indent=2))

sample_dir = ARTIFACT_DIR / "samples"; sample_dir.mkdir(exist_ok=True)
for i in range(6):
    img, y = cifar_test_raw[i]
    img.save(sample_dir / f"test{i}_{CIFAR_CLASSES[y]}.png")     # true class in the file name, for checking
print(sorted(p.name for p in sample_dir.iterdir()))

In [ ]:
%%writefile predict.py
"""Classify images with an exported CIFAR-10 model.

Usage:
    python predict.py --model cifar_resnet.pt2  --meta cifar_meta.json img1.png [img2.png ...]
    python predict.py --model cifar_resnet.onnx --meta cifar_meta.json img1.png
"""
import argparse
import json
import sys
from pathlib import Path

import numpy as np
from PIL import Image


def load_images(paths, size):
    batch = []
    for p in paths:
        img = Image.open(p).convert("RGB")                      # handles grayscale / RGBA / palette inputs
        if img.size != (size, size):
            img = img.resize((size, size), Image.BILINEAR)
        batch.append(np.asarray(img, dtype=np.uint8).transpose(2, 0, 1))   # HWC -> CHW
    return np.stack(batch)                                      # (N, 3, size, size) uint8


def run_model(model_path, x):
    if model_path.endswith(".onnx"):
        import onnxruntime as ort
        sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
        return sess.run(None, {sess.get_inputs()[0].name: x})[0]
    import torch
    model = torch.export.load(model_path).module()
    with torch.inference_mode():
        return model(torch.from_numpy(x)).numpy()


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("images", nargs="+", help="image files to classify")
    ap.add_argument("--model", required=True, help="exported model: .pt2 (torch.export) or .onnx")
    ap.add_argument("--meta", required=True, help="JSON file with 'classes' and 'image_size'")
    ap.add_argument("--topk", type=int, default=3)
    args = ap.parse_args(argv)

    meta = json.loads(Path(args.meta).read_text())
    missing = [p for p in args.images if not Path(p).is_file()]
    if missing:
        sys.exit(f"error: file(s) not found: {missing}")
    probs = run_model(args.model, load_images(args.images, meta["image_size"]))
    for path, p in zip(args.images, probs):
        top = np.argsort(-p)[: args.topk]
        print(f"{Path(path).name:28s} " + "  ".join(f"{meta['classes'][i]} {p[i]:.3f}" for i in top))


if __name__ == "__main__":
    main()

In [ ]:
# {ARTIFACT_DIR} is expanded by IPython, so this also works when ARTIFACT_DIR is on Google Drive
!python predict.py --model "{ARTIFACT_DIR}/cifar_resnet.pt2" --meta "{ARTIFACT_DIR}/cifar_meta.json" {ARTIFACT_DIR}/samples/*.png
print("--- same images through ONNX Runtime ---")
!python predict.py --model "{ARTIFACT_DIR}/cifar_resnet.onnx" --meta "{ARTIFACT_DIR}/cifar_meta.json" --topk 1 {ARTIFACT_DIR}/samples/*.png

## 15.8 Where production goes from here

This notebook stops at a working, tested artifact and a CLI. The next layers of a production system, which you will meet at work, are:

- **Serving.** Wrap `run_model` in a web service (for example FastAPI), or use a dedicated inference server such as NVIDIA Triton Inference Server for dynamic batching and multi-model hosting.
- **Optimisation.** Ahead-of-time compilation with AOTInductor (`torch._inductor.aoti_compile_and_package`) packages a compiled model loadable from C++ or Python. Quantisation to int8 or lower precision (the `torchao` library) reduces size and latency. ExecuTorch targets phones and microcontrollers.
- **Versioning and monitoring.** Store artifacts with their metrics, data version and code commit. In production, monitor the input distribution and prediction confidence, because models degrade silently when the data shifts.
- **Legacy formats.** You will meet TorchScript (`torch.jit.script` and `torch.jit.trace`) in older codebases. It is deprecated, and new work should use `torch.export`.

## 🧪 Exercises

**Exercise 15.1: a reusable export test.** `★★☆ · ~20 min` Write `check_export_parity(eager_model, exported_callable, make_input, batch_sizes=(1, 3, 32), atol=1e-5)`. It should compare outputs for each batch size, raise an `AssertionError` that names the batch size and the maximum difference on failure, and return the maximum difference otherwise. Run it on `reloaded` and on a thin wrapper around the ONNX session. This is precisely the kind of function that belongs in a CI test suite (Chapter 16).

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
def check_export_parity(eager_model, exported_callable, make_input, batch_sizes=(1, 3, 32), atol=1e-5):
    worst = 0.0
    with torch.inference_mode():
        for n in batch_sizes:
            x = make_input(n)
            ref = eager_model(x)
            got = torch.as_tensor(exported_callable(x))
            diff = (ref - got).abs().max().item()
            assert diff <= atol, f"batch size {n}: max abs difference {diff:.2e} > {atol:.0e}"
            worst = max(worst, diff)
    return worst

make_u8 = lambda n: torch.randint(0, 256, (n, 3, 32, 32), dtype=torch.uint8)
onnx_callable = lambda x: sess.run(None, {inp.name: x.numpy()})[0]
print("torch.export max diff:", check_export_parity(infer_model, reloaded, make_u8))
print("ONNX max diff:        ", check_export_parity(infer_model, onnx_callable, make_u8, atol=1e-4))

**Exercise 15.2: latency vs throughput.** `★☆☆ · ~15 min` Using ONNX Runtime on the CPU, measure the median time per call and the throughput in images per second for batch sizes 1, 8, 64 and 256. Plot throughput against batch size. What does the curve suggest about how a server should handle many simultaneous requests?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
sizes, tput, lat = [1, 8, 64, 256], [], []
for n in sizes:
    xb_ = np.random.randint(0, 256, (n, 3, 32, 32), dtype=np.uint8)
    med, _ = lat_ms(lambda: sess.run(None, {inp.name: xb_}), iters=20)
    lat.append(med); tput.append(n / (med / 1e3))
    print(f"batch {n:4d}: {med:8.2f} ms/call | {tput[-1]:8.0f} img/s")
plt.figure(figsize=(5, 3)); plt.semilogx(sizes, tput, marker="o", base=2)
plt.xlabel("batch size"); plt.ylabel("images / s"); plt.grid(alpha=0.3); plt.show()
# Throughput usually rises with batch size (strongly on GPUs and multi-core CPUs, only a little on a
# single slow core), because the fixed per-call overhead is amortised over more images. A server should therefore *dynamically batch* concurrent requests (wait a few ms,
# run them together), trading a little latency for much higher throughput.

### 📌 Key takeaways

- Put preprocessing (and temperature) inside the exported model; test train/serve parity.
- Softmax probabilities are not automatically calibrated: measure ECE, fit a temperature on validation.
- Mark dynamic dimensions when exporting; verify exports numerically.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Why put normalisation inside the exported model instead of in the client code?

<details>
<summary>Answer</summary>

The artifact then carries its own preprocessing, so no client can apply different statistics, the wrong scaling, or the wrong dtype. This eliminates train/serve skew, the most common production bug.

</details>

**2.** Your exported model fails when called with batch size 3, although it worked with 2. What did you forget?

<details>
<summary>Answer</summary>

To mark the batch dimension dynamic (`dynamic_shapes={'x': {0: Dim('batch')}}`). Without that, `torch.export` specialises every dimension to the example's size.

</details>

**3.** What does `torch.export` refuse to capture, and what do you use instead?

<details>
<summary>Answer</summary>

Data-dependent Python control flow, such as `if tensor.sum() > 0`. Use tensor operations like `torch.where`, or the structured control-flow operator `torch.cond`.

</details>

---
# Chapter 16. From notebook to codebase

Notebooks are excellent for learning and exploration. They are a poor home for code that must run unattended, be reviewed, be tested, and be rerun six months later by someone else. In industry, the code that produces a model you ship lives in **version-controlled Python modules and scripts**, with **configuration**, **logging**, **checkpoint/resume**, and **tests**.

In this chapter we turn the Fashion-MNIST CNN into a small but complete project, written to disk from the notebook:

```
fmnist_project/
├── fmnist/
│   ├── __init__.py
│   ├── model.py        # architecture only
│   └── engine.py       # data, training and evaluation functions
├── train.py            # entry point: config -> data -> model -> fit -> checkpoints/logs
└── tests/
    └── test_core.py    # fast unit tests, run with pytest
```

The separation matters. `model.py` can be imported by an inference service without pulling in any training code. `engine.py` can be unit-tested. `train.py` is thin glue that a scheduler or a colleague can run with different flags.

🎯 **In this chapter you will learn to:** structure, configure, log and test a training codebase; resume crashed runs and search hyper-parameters honestly; know the path to multi-GPU training.

In [ ]:
Path("fmnist_project/fmnist").mkdir(parents=True, exist_ok=True)
Path("fmnist_project/tests").mkdir(exist_ok=True)
_ = Path("fmnist_project/fmnist/__init__.py").write_text("")

In [ ]:
%%writefile fmnist_project/fmnist/model.py
"""Model definitions. No training code here, so inference services can import this file alone."""
import torch.nn as nn


def conv_bn_relu(c_in: int, c_out: int) -> nn.Sequential:
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
                         nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


class SmallCNN(nn.Module):
    """Input (N, 1, 28, 28) normalised images -> (N, num_classes) logits."""

    def __init__(self, num_classes: int = 10, width: int = 32, p_drop: float = 0.3):
        super().__init__()
        w = width
        self.features = nn.Sequential(
            conv_bn_relu(1, w), conv_bn_relu(w, w), nn.MaxPool2d(2),
            conv_bn_relu(w, 2 * w), conv_bn_relu(2 * w, 2 * w), nn.MaxPool2d(2),
            conv_bn_relu(2 * w, 4 * w),
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p_drop),
                                  nn.Linear(4 * w, num_classes))

    def forward(self, x):
        return self.head(self.features(x))

In [ ]:
%%writefile fmnist_project/fmnist/engine.py
"""Data loading, training and evaluation."""
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets
from torchvision.transforms import v2


def make_loaders(data_dir, batch_size=128, num_workers=2, subset=0, seed=0):
    """Train/val/test loaders. Normalisation statistics come from the training split only."""
    raw = datasets.FashionMNIST(data_dir, train=True, download=True)
    gen = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(raw), generator=gen)
    tr_idx, va_idx = perm[:55_000], perm[55_000:]
    pixels = raw.data[tr_idx].float() / 255
    mean, std = pixels.mean().item(), pixels.std().item()
    tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize([mean], [std])])
    full = datasets.FashionMNIST(data_dir, train=True, transform=tf)
    test = datasets.FashionMNIST(data_dir, train=False, download=True, transform=tf)
    train, val = Subset(full, tr_idx.tolist()), Subset(full, va_idx.tolist())
    if subset:
        train, val, test = (Subset(d, range(min(subset, len(d)))) for d in (train, val, test))
    kw = dict(num_workers=num_workers, pin_memory=torch.cuda.is_available())
    return (DataLoader(train, batch_size, shuffle=True, drop_last=True, generator=gen, **kw),
            DataLoader(val, 512, **kw), DataLoader(test, 512, **kw), {"mean": mean, "std": std})


def train_one_epoch(model, loader, optimizer, scheduler, device, scaler, amp_dtype, label_smoothing=0.0):
    model.train()
    total, correct, n = torch.zeros((), device=device), torch.zeros((), device=device), 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = model(x)
            loss = F.cross_entropy(logits, y, label_smoothing=label_smoothing)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"non-finite loss {loss.item()}")    # fail loudly, never train on NaNs
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.detach().float() * y.size(0)
        correct += (logits.argmax(1) == y).sum()
        n += y.size(0)
    return {"train_loss": total.item() / n, "train_acc": correct.item() / n}


@torch.no_grad()
def evaluate(model, loader, device, amp_dtype=None):
    model.eval()
    total, correct, n = torch.zeros((), device=device), torch.zeros((), device=device), 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = model(x)
        total += F.cross_entropy(logits.float(), y, reduction="sum")
        correct += (logits.argmax(1) == y).sum()
        n += y.size(0)
    return {"loss": total.item() / n, "acc": correct.item() / n}

In [ ]:
%%writefile fmnist_project/train.py
"""Train SmallCNN on Fashion-MNIST.

Examples:
    python train.py --epochs 10 --out_dir runs/baseline
    python train.py --epochs 20 --out_dir runs/baseline --resume     # continue an interrupted run
"""
import argparse
import json
import random
import subprocess
import time
from dataclasses import asdict, dataclass, fields
from pathlib import Path

import torch

from fmnist.engine import evaluate, make_loaders, train_one_epoch
from fmnist.model import SmallCNN


@dataclass
class Config:
    data_dir: str = "../data"
    out_dir: str = "runs/default"
    epochs: int = 10
    batch_size: int = 128
    lr: float = 2e-3
    weight_decay: float = 0.05
    label_smoothing: float = 0.1
    width: int = 32
    seed: int = 0
    subset: int = 0              # >0: use only this many examples per split (smoke tests)
    num_workers: int = 2
    amp: bool = True
    resume: bool = False
    stop_after: int = 0          # >0: exit after this many epochs (simulates a crash, for testing resume)
    eval_test: bool = True       # --no-eval_test during hyper-parameter search: never peek at the test set


def parse_config() -> Config:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    for f in fields(Config):
        if isinstance(f.default, bool):
            p.add_argument(f"--{f.name}", action=argparse.BooleanOptionalAction, default=f.default)
        else:
            p.add_argument(f"--{f.name}", type=type(f.default), default=f.default)
    return Config(**vars(p.parse_args()))


def git_commit() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "unknown"


def save(path, **state):
    tmp = Path(str(path) + ".tmp")
    torch.save(state, tmp)
    tmp.replace(path)                                   # atomic


def main():
    cfg = parse_config()
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    random.seed(cfg.seed); torch.manual_seed(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp_dtype = None
    if cfg.amp and device.type == "cuda":
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

    train_dl, val_dl, test_dl, norm = make_loaders(cfg.data_dir, cfg.batch_size, cfg.num_workers, cfg.subset, cfg.seed)
    model = SmallCNN(width=cfg.width).to(device)
    decay = [p for p in model.parameters() if p.ndim >= 2]
    no_decay = [p for p in model.parameters() if p.ndim < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": cfg.weight_decay},
                             {"params": no_decay, "weight_decay": 0.0}], lr=cfg.lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, epochs=cfg.epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))

    start_epoch, best_acc = 1, -1.0
    if cfg.resume and (out / "last.pt").exists():
        ck = torch.load(out / "last.pt", map_location=device, weights_only=True)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["rng"]); train_dl.generator.set_state(ck["loader_gen"])
        start_epoch, best_acc = ck["epoch"] + 1, ck["best_acc"]
        print(f"resumed from epoch {ck['epoch']} (best val acc {best_acc:.4f})")

    (out / "config.json").write_text(json.dumps({**asdict(cfg), "git_commit": git_commit(),
                                                 "torch": str(torch.__version__), "normalisation": norm}, indent=2))
    log = open(out / "metrics.jsonl", "a")
    for epoch in range(start_epoch, cfg.epochs + 1):
        t0 = time.perf_counter()
        tr = train_one_epoch(model, train_dl, opt, sched, device, scaler, amp_dtype, cfg.label_smoothing)
        va = evaluate(model, val_dl, device, amp_dtype)
        row = {"epoch": epoch, **tr, "val_loss": va["loss"], "val_acc": va["acc"],
               "lr": opt.param_groups[0]["lr"], "time_s": round(time.perf_counter() - t0, 2)}
        log.write(json.dumps(row) + "\n"); log.flush()
        print(json.dumps(row))
        if va["acc"] > best_acc:
            best_acc = va["acc"]
            save(out / "best.pt", model=model.state_dict(), epoch=epoch, val_acc=best_acc, config=asdict(cfg), norm=norm)
        save(out / "last.pt", model=model.state_dict(), optimizer=opt.state_dict(), scheduler=sched.state_dict(),
             scaler=scaler.state_dict(), rng=torch.get_rng_state(), loader_gen=train_dl.generator.get_state(),
             epoch=epoch, best_acc=best_acc)
        if cfg.stop_after and epoch - start_epoch + 1 >= cfg.stop_after:
            print(f"--stop_after {cfg.stop_after}: exiting early, as if the job had been killed")
            log.close()
            return
    log.close()

    best = torch.load(out / "best.pt", map_location=device, weights_only=True)
    results = {"best_epoch": best["epoch"], "val_acc": best["val_acc"]}
    if cfg.eval_test:                                                  # touched once, at the very end
        model.load_state_dict(best["model"])
        results["test_acc"] = evaluate(model, test_dl, device, amp_dtype)["acc"]
    (out / "results.json").write_text(json.dumps(results, indent=2))
    print(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in results.items()))


if __name__ == "__main__":
    main()

Things to notice in `train.py`, all of them habits worth copying:

- **One `Config` dataclass** is the single source of truth. Every field automatically becomes a command-line flag, and the full config is saved next to the results, together with the git commit and the PyTorch version.
- **Two checkpoints.** `last.pt` holds the full training state, for resuming. `best.pt` holds the weights to keep. Both are written atomically.
- **Machine-readable logs.** One JSON line per epoch can be loaded straight into pandas later.
- **It fails loudly.** A non-finite loss raises an error instead of silently corrupting the weights.
- **The test set is evaluated once,** after loading the best checkpoint.

🏭 **Experiment tracking.** JSON-lines logs are the dependency-free baseline. PyTorch also ships a TensorBoard writer, `from torch.utils.tensorboard import SummaryWriter` (it needs the `tensorboard` package, which Colab preinstalls). `writer.add_scalar("val/acc", acc, epoch)` lets you compare many runs interactively. Hosted trackers such as Weights & Biases or MLflow add sharing and artifact storage.

Let's run it. In smoke-test mode we use a small subset.

In [ ]:
TRAIN_ARGS = "--epochs 1 --subset 1000 --num_workers 0" if FAST_DEV_RUN else "--epochs 3"
!cd fmnist_project && python train.py {TRAIN_ARGS} --out_dir runs/demo

Now simulate a crash. We start a fresh run and kill it after one epoch with `--stop_after 1`. Then we relaunch the **same command** with `--resume`, and the script continues from the last saved epoch instead of starting over:

In [ ]:
RUN = "--epochs 2 --subset 1000 --num_workers 0" if FAST_DEV_RUN else "--epochs 4"
!cd fmnist_project && rm -rf runs/crashy && python train.py {RUN} --out_dir runs/crashy --stop_after 1
print("----- relaunch after the 'crash' -----")
!cd fmnist_project && python train.py {RUN} --out_dir runs/crashy --resume
!cat fmnist_project/runs/crashy/results.json

> **Resume with the *same* config.** OneCycle's total length is fixed when the scheduler is created, and `load_state_dict` restores that length. Resuming with a different `--epochs` would restore a schedule of the wrong length, and PyTorch raises an error as soon as the step count overruns it. Resume is for continuing the *same* run after an interruption. Exercise 16.2 makes the script enforce this.

## 16.2 Testing ML code

ML code needs tests as much as any other code. The useful tests are fast (seconds, on the CPU, with synthetic data) and target the bugs of Chapter 10:

| Test | Catches |
| --- | --- |
| Output shape and dtype for a batch | Architecture and shape bugs |
| Loss at initialisation ≈ `ln C` | Broken init, bad output layer, wrong loss |
| The model overfits a tiny batch | Broken training step or loss contract |
| Two runs with the same seed give identical results | Hidden non-determinism |
| Checkpoint save → load → identical outputs | Serialisation bugs (§9.2's `TorchVersion` surprise) |
| Exported model matches eager (Exercise 15.1) | Export problems |

In [ ]:
%%writefile fmnist_project/tests/test_core.py
import io
import math
import sys
from pathlib import Path

import pytest
import torch
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from fmnist.model import SmallCNN  # noqa: E402


@pytest.fixture
def batch():
    g = torch.Generator().manual_seed(0)
    return torch.randn(16, 1, 28, 28, generator=g), torch.randint(0, 10, (16,), generator=g)


def test_output_shape_and_dtype(batch):
    x, _ = batch
    out = SmallCNN()(x)
    assert out.shape == (16, 10) and out.dtype == torch.float32


def test_initial_loss_near_log_num_classes(batch):
    torch.manual_seed(0)
    x, y = batch
    model = SmallCNN().eval()
    with torch.no_grad():
        loss = F.cross_entropy(model(x), y).item()
    assert abs(loss - math.log(10)) < 0.5, loss


def test_can_overfit_tiny_batch(batch):
    torch.manual_seed(0)
    x, y = batch
    model = SmallCNN(p_drop=0.0)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for _ in range(60):
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    assert loss.item() < 0.05, loss.item()


def test_same_seed_same_result(batch):
    x, _ = batch
    outs = []
    for _ in range(2):
        torch.manual_seed(123)
        outs.append(SmallCNN().eval()(x))
    assert torch.equal(outs[0], outs[1])


def test_checkpoint_roundtrip(batch):
    x, _ = batch
    model = SmallCNN().eval()
    buf = io.BytesIO()
    torch.save({"model": model.state_dict(), "torch": str(torch.__version__)}, buf)
    buf.seek(0)
    restored = SmallCNN().eval()
    restored.load_state_dict(torch.load(buf, weights_only=True)["model"])
    with torch.no_grad():
        assert torch.equal(model(x), restored(x))

In [ ]:
!cd fmnist_project && python -m pytest -q tests

🏭 **Industry note.** In a real repository these tests run automatically on every pull request (continuous integration). A change that breaks the training step, the loss contract or the checkpoint format is caught in minutes, not after a two-day training run.

## 16.3 Hyper-parameter search, done honestly

Tuning one knob at a time by hand is slow, and it misses interactions between knobs. The standard baseline is **random search** (Bergstra & Bengio, 2012). With a fixed budget, it usually beats grid search, because typically only a few hyper-parameters matter much, and random sampling tries many more distinct values of each. The rules:

1. Sample scale-type hyper-parameters (learning rate, weight decay) **log-uniformly**.
2. Fix a **budget** (trials × epochs) in advance.
3. Select on **validation** only. Our script's `--no-eval_test` flag guarantees the search never even computes test accuracy.
4. Re-run the best few configurations with **several seeds** before trusting the ranking, then evaluate the single final choice on test, once.

In [ ]:
SEARCH_TRIALS = 2 if FAST_DEV_RUN else 8
SEARCH_ARGS = "--epochs 1 --subset 1000 --num_workers 0" if FAST_DEV_RUN else "--epochs 2"
rng = random.Random(0)
trials = []
for t in range(SEARCH_TRIALS):
    lr = 10 ** rng.uniform(-3.5, -1.5)          # log-uniform between ~3e-4 and ~3e-2
    wd = 10 ** rng.uniform(-4, -1)              # log-uniform between 1e-4 and 1e-1
    out = f"runs/search/trial{t}"
    subprocess.run(f"cd fmnist_project && python train.py --lr {lr:.3e} --weight_decay {wd:.3e} "
                   f"{SEARCH_ARGS} --no-eval_test --out_dir {out}", shell=True, check=True, stdout=subprocess.DEVNULL)
    res = json.loads(Path(f"fmnist_project/{out}/results.json").read_text())
    assert "test_acc" not in res                 # the search never saw the test set
    trials.append({"lr": lr, "weight_decay": wd, "val_acc": res["val_acc"]})
    print(f"trial {t}: lr {lr:.2e}  wd {wd:.2e}  ->  val acc {res['val_acc']:.4f}")

best_trial = max(trials, key=lambda r: r["val_acc"])
print("best (by validation):", best_trial)
plt.figure(figsize=(5, 3.8))
sc = plt.scatter([r["lr"] for r in trials], [r["weight_decay"] for r in trials], c=[r["val_acc"] for r in trials], s=80, cmap="viridis")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("learning rate"); plt.ylabel("weight decay")
plt.colorbar(sc, label="val acc"); plt.title("Random search (log-uniform)"); plt.tight_layout(); plt.show()

With only 2 epochs per trial, this is a *short-horizon* search. The best short-run learning rate is often higher than the best long-run one. That is a known bias, and it is one reason the final step is to retrain the top configurations at full length, with several seeds.

## 16.4 Scaling out: multiple GPUs (a preview)

When one GPU is not enough, the standard tool is **Distributed Data Parallel (DDP)**. One process runs per GPU. Each holds a full copy of the model and processes a different slice of every batch, and the gradients are averaged across processes during `backward()`. The changes to a training script are small:

```python
# launch with:  torchrun --nproc_per_node=4 train_ddp.py
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

dist.init_process_group("nccl")                      # torchrun sets RANK, LOCAL_RANK, WORLD_SIZE
local_rank = int(os.environ["LOCAL_RANK"]); torch.cuda.set_device(local_rank)
model = DDP(SmallCNN().cuda(), device_ids=[local_rank])
sampler = DistributedSampler(train_ds, shuffle=True)  # each process sees a different shard
loader = DataLoader(train_ds, batch_size=per_gpu_batch, sampler=sampler)   # no shuffle=True with a sampler
for epoch in range(epochs):
    sampler.set_epoch(epoch)                          # different shuffle every epoch
    ...                                               # the usual five lines
if dist.get_rank() == 0:                              # only one process logs and saves
    torch.save(model.module.state_dict(), "model.pt") # .module: the unwrapped model
dist.destroy_process_group()
```

The effective batch size becomes `per_gpu_batch × num_GPUs`, so the learning rate usually needs retuning (Tutorial §7.6). For models too large for one GPU's memory (§11.7), **FSDP** (`torch.distributed.fsdp.fully_shard`) shards parameters, gradients and optimizer state across GPUs. That is the subject of a later course.

## 16.5 An honest-evaluation checklist

Before you report a number in a paper, a pull request or a meeting:

- [ ] **Splits** were made before any preprocessing was fitted, and no example (or near-duplicate, such as the same patient, user or product) appears in more than one split.
- [ ] **The test set** was used once, for the final model only. Every decision was made on validation data.
- [ ] **Baselines** are reported: trivial, simple, and the previous best.
- [ ] **Uncertainty** is reported: a confidence interval from the test-set size (§7.5) and/or mean ± std over seeds (§9.4).
- [ ] **Per-class or per-slice metrics** are inspected, not only the aggregate (§7.5).
- [ ] **The metric matches the goal.** With imbalanced classes, accuracy is misleading. Use precision and recall, F1, or AUROC, at the operating point the product will use.
- [ ] **The configuration, code version and data version** are recorded, so that someone else can reproduce the result.

## 🧪 Exercises

**Exercise 16.1: run a small experiment, the professional way.** `★★☆ · ~30 min` Use `train.py` to compare `--width 16`, `32` and `64` (3 epochs each, separate `--out_dir`s), and also `--seed 1` for width 32. Load all the `metrics.jsonl` and `results.json` files with `json` (or pandas), plot the validation-accuracy curves on one figure, and write two sentences about whether width matters, given the seed-to-seed difference.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click / expand to reveal) { display-mode: "form" }
EXTRA = "--subset 1000 --num_workers 0 --epochs 1" if FAST_DEV_RUN else "--epochs 3"
runs = {"w16": "--width 16", "w32": "--width 32", "w32_seed1": "--width 32 --seed 1", "w64": "--width 64"}
for name, flags in runs.items():
    subprocess.run(f"cd fmnist_project && python train.py {flags} {EXTRA} --out_dir runs/{name}",
                   shell=True, check=True, stdout=subprocess.DEVNULL)
plt.figure(figsize=(6, 3.5))
for name in runs:
    rows = [json.loads(l) for l in open(f"fmnist_project/runs/{name}/metrics.jsonl")]
    res = json.loads(open(f"fmnist_project/runs/{name}/results.json").read())
    plt.plot([r["epoch"] for r in rows], [r["val_acc"] for r in rows], marker="o", label=f"{name} (test {res['test_acc']:.3f})")
plt.xlabel("epoch"); plt.ylabel("val acc"); plt.legend(); plt.grid(alpha=0.3); plt.show()
# Compare the w32 vs w32_seed1 gap with the width-to-width gaps: only differences clearly larger than
# the seed-to-seed spread are evidence that width matters.

**Exercise 16.2: refuse an inconsistent resume.** `★★★ · ~40 min` Modify `train.py` so that `--resume` compares the new config with the saved `config.json`, ignoring `resume` and `epochs`. If any other field differs, it should exit with a clear message instead of resuming. Then handle `epochs` properly: if it differs, explain to the user why OneCycle cannot simply be extended. Implement the comparison as a small function and add a unit test for it in `tests/test_core.py`.

<details>
<summary><b>Show answer</b></summary>

A minimal implementation:

```python
IGNORED = {"resume", "epochs"}

def config_mismatch(saved: dict, current: dict) -> dict:
    # Return {field: (saved, current)} for every relevant field that differs.
    return {k: (saved.get(k), v) for k, v in current.items()
            if k not in IGNORED and saved.get(k) != v}

# in main(), before resuming:
saved = json.loads((out / "config.json").read_text())
diff = config_mismatch(saved, asdict(cfg))
if diff:
    raise SystemExit(f"refusing to resume: config differs from the saved run: {diff}")
if saved["epochs"] != cfg.epochs:
    raise SystemExit("refusing to resume with a different --epochs: the OneCycle schedule's length "
                     "is fixed at creation. Start a new run, or use a schedule defined per epoch.")
```

and a test:

```python
def test_config_mismatch():
    from train import config_mismatch
    assert config_mismatch({"lr": 1e-3, "epochs": 5}, {"lr": 1e-3, "epochs": 9, "resume": True}) == {}
    assert config_mismatch({"lr": 1e-3}, {"lr": 2e-3}) == {"lr": (1e-3, 2e-3)}
```

Write the config **before** training starts (as `train.py` already does), so that a crashed run still leaves the config behind for this check. Note that `train.py` currently rewrites `config.json` on every launch. That write has to move *after* the check.

</details>

### 📌 Key takeaways

- One config object, saved with results; `last.pt` to resume, `best.pt` to ship.
- Fast unit tests catch the bugs of Chapter 10 before long runs.
- Random search with log-uniform sampling, selection on validation only, test once.

### ✔️ Check yourself

Commit to an answer *before* expanding it — retrieval practice only works if you actually try.

**1.** Why keep `model.py` free of training code?

<details>
<summary>Answer</summary>

Inference services, export scripts and tests can then import the architecture without pulling in data loading, optimizers or heavy dependencies. Changes to the training code also cannot break serving.

</details>

**2.** Which two checkpoints does `train.py` keep, and why both?

<details>
<summary>Answer</summary>

`last.pt` holds the complete training state, to resume after an interruption. `best.pt` holds the weights with the best validation score, the model to evaluate and ship. They are usually not from the same epoch.

</details>

**3.** Name three fast tests every training codebase should have.

<details>
<summary>Answer</summary>

Any three of: output shapes; initial loss ≈ ln C; overfitting a tiny batch; determinism under a fixed seed; checkpoint round-trip; export parity.

</details>

---
# Part VI — Review

# Chapter 17. Interview questions, error decoder and cheat sheet

## 17.1 The error-message decoder

Keep this table open during your first months of PyTorch. The messages are abbreviated.

| Error message (abbreviated) | What it means | Usual fix |
| --- | --- | --- |
| `mat1 and mat2 shapes cannot be multiplied (64x3136 and 784x128)` | The input to a `Linear` layer has the wrong number of features | Recompute the flattened size (§12.3), or use `nn.LazyLinear` / global average pooling |
| `mat1 and mat2 must have the same dtype ... Double and Float` | float64 data (from NumPy) met float32 weights | `.float()` at the NumPy boundary (§1.1) |
| `Expected all tensors to be on the same device` | Mixed CPU and GPU tensors | `.to(device)` for the model **and** for every batch (§1.8) |
| `Expected floating point type for target with class probabilities, got Long` | `CrossEntropyLoss` received a `(N, C)` integer target | Pass class indices of shape `(N,)` (§5.3) |
| `Target 10 is out of bounds` / `device-side assert triggered` | A label is ≥ the number of classes (or negative) | Check `labels.min()`, `labels.max()` and `num_classes` (§5.3) |
| `element 0 of tensors does not require grad and does not have a grad_fn` | The loss is not connected to any parameter | You computed it under `no_grad`, used `.detach()` or `.item()`, or froze everything |
| `Trying to backward through the graph a second time` | A second `backward()` on a freed graph | Recompute the forward pass. Do not reuse tensors from a previous step (§2.3) |
| `one of the variables needed for gradient computation has been modified by an inplace operation` | An in-place operation overwrote a saved tensor | Use the out-of-place version (§2.6) |
| `view size is not compatible with input tensor's size and stride` | `.view` on a non-contiguous tensor | `.reshape`, or `.contiguous().view` (§1.2) |
| `CUDA out of memory` | Activations, parameters and optimizer state exceed GPU memory | Smaller batch + gradient accumulation, AMP, `no_grad` in evaluation (§11.7) |
| `Weights only load failed ... Unsupported global` | The checkpoint contains a non-tensor object | Store plain types (`str(torch.__version__)`). Use `weights_only=False` only for trusted files (§9.2) |
| `Error(s) in loading state_dict ... Missing key(s) ... Unexpected key(s) "_orig_mod..."` | Keys saved from a `torch.compile` wrapper or a DDP `.module` wrapper | Save from the unwrapped model, or use `model.compile()` (§11.5) |
| `UserWarning: Using a target size ... different to the input size` | `(N, 1)` vs `(N,)` broadcasting in a loss | Make the shapes identical (§1.4) |
| `DataLoader worker ... exited unexpectedly` | Usually out of shared memory, or an exception inside `__getitem__` | `num_workers=0` to see the real traceback, then fix the dataset |

## 17.2 Interview questions with model answers

These are representative of what junior ML-engineer interviews ask about PyTorch. Answer each out loud before expanding it.

🎯 **In this chapter you will learn to:** decode common error messages; answer standard PyTorch interview questions; compare your measured results with the text.

**Q1.** Write the five essential lines of a training step from memory, in order, and say what each one does.

<details>
<summary><b>Model answer</b></summary>

```python
logits = model(x)                    # forward
loss = loss_fn(logits, y)            # scalar loss
optimizer.zero_grad(set_to_none=True)  # clear accumulated grads
loss.backward()                      # autograd fills p.grad
optimizer.step()                     # update parameters
```
Plus `model.train()` before the loop, and `model.eval()` with `torch.no_grad()` for evaluation. `zero_grad` can go anywhere before `backward`.

</details>

**Q2.** What is the difference between `model.eval()` and `torch.no_grad()`? Do you need both?

<details>
<summary><b>Model answer</b></summary>

`eval()` changes layer *behaviour* (dropout off, BatchNorm uses running statistics). `no_grad()` stops autograd *recording*, which saves memory and time. They are independent, and correct evaluation needs both.

</details>

**Q3.** Why does `CrossEntropyLoss` take logits instead of probabilities?

<details>
<summary><b>Model answer</b></summary>

It fuses log-softmax and NLL using the log-sum-exp trick, which is numerically stable and has a clean gradient, `p − y`. Applying softmax first double-applies it: the loss gets a non-zero floor and the gradients are distorted, with no error raised.

</details>

**Q4.** How does autograd compute gradients? What is a leaf tensor?

<details>
<summary><b>Model answer</b></summary>

During the forward pass it records a graph of operations (each output's `grad_fn`). `backward()` traverses that graph in reverse, applying each operation's local vector–Jacobian product: reverse-mode automatic differentiation, which is backpropagation. Leaves are tensors created directly by the user, such as parameters. Their `.grad` is populated and accumulated.

</details>

**Q5.** Your validation accuracy is much worse than training from the very first epoch. What do you check?

<details>
<summary><b>Model answer</b></summary>

Preprocessing parity between the splits (normalisation, resize), augmentation accidentally applied to validation, `model.eval()` missing, label mapping differences between the splits, and data leakage or distribution differences in how the splits were built.

</details>

**Q6.** Explain mixed-precision training. Why is a gradient scaler needed for fp16?

<details>
<summary><b>Model answer</b></summary>

Autocast runs matmuls and convolutions in 16-bit for tensor-core speed, keeps precision-sensitive operations in fp32, and keeps fp32 master weights. fp16's small exponent range makes small gradients underflow to zero, so the loss is multiplied by a scale factor before `backward` and divided out before the step. Steps with `inf` or `nan` gradients are skipped. bf16 has fp32's range and needs no scaler.

</details>

**Q7.** What is the difference between Adam with `weight_decay` and AdamW?

<details>
<summary><b>Model answer</b></summary>

Adam adds `λw` to the gradient, which then gets divided by `√v̂`, so the effective decay depends on the gradient history. AdamW applies `w ← w − ηλw` directly, decoupled from the adaptive scaling. That is the intended regularisation, and it generalises better.

</details>

**Q8.** How would you fine-tune an ImageNet model on 2,000 images in 20 classes?

<details>
<summary><b>Model answer</b></summary>

Load the pretrained weights and use their preprocessing. Replace the head with a 20-way linear layer. Start with a linear probe as a baseline. Then fine-tune everything with AdamW, a backbone learning rate about 10× smaller than the head's, warm-up plus cosine, augmentation (random resized crop, flip), and label smoothing. Monitor validation, keep the best checkpoint, and test once. If overfitting, freeze the early stages, keeping their BN in eval mode.

</details>

**Q9.** What does `torch.compile` do, and when might it not help?

<details>
<summary><b>Model answer</b></summary>

It captures the Python model as a graph (Dynamo) and generates fused kernels (Inductor and Triton). It may not help when the model is dominated by a few large matmuls, when the run is short (compile time dominates), when shapes keep changing (recompiles), or when there are many graph breaks.

</details>

**Q10.** How do you make a training run resumable *exactly*?

<details>
<summary><b>Model answer</b></summary>

Checkpoint the model, optimizer, scheduler and GradScaler state dicts, the epoch or step, the best metric, the RNG states, and the DataLoader generator state. Write atomically, and restore everything before continuing with the same configuration.

</details>

**Q11.** Your loss becomes NaN after 2,000 steps. Walk through your debugging.

<details>
<summary><b>Model answer</b></summary>

Reproduce with a fixed seed and log the step. Check the gradient norms before the NaN (explosion → lower LR, clipping, warm-up). Check the inputs of the offending batch for NaN or inf. Look for `log`, `sqrt` or division in custom code. With fp16, check for overflow (scaler behaviour, or switch to bf16). Run `torch.autograd.detect_anomaly()` around the failing step to find the operation.

</details>

**Q12.** What is the receptive field of a unit after two 3×3 convolutions and a stride-2 pooling, followed by one more 3×3 convolution?

<details>
<summary><b>Model answer</b></summary>

Using `r ← r + (k−1)·j`, `j ← j·s`: 1 → 3 → 5 → 6 (pool k=2, s=2, j=2) → 6 + 2·2 = 10. So 10×10 pixels.

</details>

**Q13.** Why do ResNets train well at depth, when plain deep CNNs do not?

<details>
<summary><b>Model answer</b></summary>

The residual form `y = x + F(x)` makes the identity the default mapping and gives the gradient a path with derivative 1, so signals and gradients do not have to pass through long products of Jacobians. Together with BN and good initialisation (zero-init of the last BN gamma), very deep networks optimise reliably.

</details>

**Q14.** How do you deploy a PyTorch model to a service that cannot run Python PyTorch?

<details>
<summary><b>Model answer</b></summary>

Put the preprocessing inside the model, switch to eval mode, and export with `torch.export` (with dynamic batch dimensions). Then convert to ONNX and serve with ONNX Runtime, or compile with AOTInductor. Verify numerical parity with tests, and ship class names and a version with the artifact.

</details>

**Q15.** What is data leakage? Give two examples.

<details>
<summary><b>Model answer</b></summary>

Information from evaluation data influencing training or model selection. Examples: normalisation statistics or a vocabulary fitted on the full dataset; the same patient or user in both train and test; tuning hyper-parameters on the test set; features that encode the label from the future.

</details>

## 17.3 Cheat sheet: the complete modern training loop

```python
# ---- setup -----------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(0)
model = Net().to(device).to(memory_format=torch.channels_last)       # channels_last for CNNs
decay = [p for p in model.parameters() if p.ndim >= 2]
no_decay = [p for p in model.parameters() if p.ndim < 2]
optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": 0.05},
                               {"params": no_decay, "weight_decay": 0.0}], lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, epochs=E, steps_per_epoch=len(train_dl))
scaler = torch.amp.GradScaler("cuda")                               # fp16 only; omit for bf16
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# ---- loop ------------------------------------------------------------------
for epoch in range(E):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(x), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step()                                             # per batch for OneCycle
    model.eval()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        val_acc = ...                                                # evaluate; save best & last checkpoints
```

**Shapes and dtypes.** Images are `(N, C, H, W)` `float32`. Class labels are `(N,)` `int64`. Regression targets are `(N, 1)` `float32`, the same shape as the predictions. Binary labels are `(N, 1)` `float32` for `BCEWithLogitsLoss`.

**Always:** look at the data → check the initial loss ≈ ln C → overfit one batch → LR range test → compare with a baseline → only then tune.

## 17.4 Capstone projects

Pick one and treat it as a portfolio piece: a repository with `train.py`, tests, a README with results (mean ± std over seeds), and an exported model with a `predict.py`.

1. **Tabular deep learning vs gradient boosting.** On a public tabular dataset, compare an MLP (with embeddings for categorical features and standardised numerics) against a gradient-boosted-trees baseline. Report honestly; trees often win, and knowing when is valuable.
2. **Your own image classifier.** Collect or choose a small domain dataset (plant diseases, defects, food). Run a linear probe, then fine-tune. Analyse the errors per class, and export to ONNX.
3. **CIFAR-10 speed run.** Reach 94% test accuracy as fast as possible on a T4. Use GPU-side augmentation, AMP, `channels_last`, `torch.compile` and schedule tuning, and document what each change bought.
4. **Robustness study.** Train on CIFAR-10, then evaluate under corruptions you implement (noise, blur, contrast, JPEG compression). Test whether augmentation or transfer learning improves robustness.
5. **Production mini-service.** Wrap the CIFAR artifact in a small HTTP service with request batching, input validation, logging of prediction confidence, and a test suite including export parity.

## 17.5 Where to go next

- **Sequence models and Transformers.** Attention, `nn.MultiheadAttention`, `F.scaled_dot_product_attention`, and tokenisation. This is the "next course" of Tutorial §0.
- **The ecosystem** you deliberately skipped here. Now that you know what they automate, you will use them well: PyTorch Lightning (training-loop structure), Hugging Face `transformers` and `datasets`, `timm` (hundreds of pretrained vision models), TensorBoard or Weights & Biases (experiment tracking), and `torchao` (quantisation).
- **Scale.** DDP and FSDP2 for multi-GPU training, and `torch.distributed.checkpoint` for large-model checkpoints.
- **Primary sources.** The official PyTorch documentation and tutorials at docs.pytorch.org, and the release notes of each new version. Reading them every couple of months keeps your knowledge current, because this field changes quickly.

## 17.6 Your measured results

Throughout the notebook, `record(...)` stored the numbers **you** measured. Compare them with the estimates in the text. Where they disagree, your run is the ground truth, and investigating why (hardware, versions, seeds, data loading) is an excellent exercise in itself.

In [ ]:
width = max((len(k) for k in RESULTS), default=10)
print(f"{'measurement':{width}s}   value")
print("-" * (width + 12))
for k, v in RESULTS.items():
    print(f"{k:{width}s}   {v:.4f}" if isinstance(v, float) else f"{k:{width}s}   {v}")
(ARTIFACT_DIR / "my_results.json").write_text(json.dumps(RESULTS, indent=2))
print(f"\nsaved to {ARTIFACT_DIR / 'my_results.json'}  (device: {device}, torch {torch.__version__}, FAST_DEV_RUN={FAST_DEV_RUN})")

---

**Congratulations.** You have gone from `torch.tensor([1., 2.])` to a tested training codebase and an exported, served model. More importantly, you have built the habits that separate engineers who ship from those who get stuck: check shapes, check the initial loss, overfit one batch, measure before optimising, evaluate honestly, and never trust a number you have not reproduced.